# Council Administration - pardon chisiba

## 1. Administrative Information

Administrative information was collected from the official Kabwe Municipal Council website. 
The information includes the council's organisational structure, departments, administrative 
units, leadership structure, functions, and contact information.

# Kabwe Municipal Council Dataset - Christopher Banda

### 1. Project Overview

This notebook documents the extraction, cleaning, preprocessing, validation, and preparation of data collected from the digital footprints of Kabwe Municipal Council, Zambia.

The project focuses on creating a carefully curated multi-source dataset covering important council functions and financial activities.

### Main focus

The notebook currently covers the Constituency Development Fund (CDF), including:

- CDF allocations and budgets
- CDF funding and disbursements
- CDF expenditure and usage
- CDF community projects
- Project sectors such as education, health, water and sanitation
- CDF output indicators

### Data sources

The data was collected primarily from official Kabwe Municipal Council publications, including:

- Council budgets
- CDF project documents
- CDF financial statements
- Output-based budget documents

The original source documents are preserved in the `data/raw/` directory, while extracted text is stored in `data/extracted/` and cleaned datasets are stored in `data/processed/`.

### Dataset format

The final datasets are stored as CSV files using the pipe (`|`) delimiter as required by the assignment.

The dataset naming convention follows:

`db-unza26-csc4792-[DESCRIPTION].csv`

### Reproducibility

The project uses Python scripts for downloading, extracting, and cleaning data. The notebook documents the process and performs additional validation and quality checks.

Git and GitHub are used for version control so that changes to the dataset, scripts, and documentation can be tracked.

## 2. Import Libraries

The following Python libraries are used throughout the data extraction, cleaning, preprocessing, and validation process.

- requests — downloads data from the Kabwe Municipal Council website.
- BeautifulSoup — parses HTML pages when scraping web content.
- pandas — loads, cleans, transforms, validates, and exports tabular data.
- numpy — supports numerical and missing-value operations.
- re — uses regular expressions for text and financial-value cleaning.
- urllib.parse.urljoin — constructs complete URLs from relative links.
- fitz — extracts text from PDF documents using PyMuPDF.

In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import numpy as np
import re
from urllib.parse import urljoin
import fitz
import pdfplumber

## Configure Project Directory Paths

The project directory paths are defined centrally to support the organised group repository structure.

The notebook is located in the `notebooks/CDF/` directory, so the project root is referenced two levels above the notebook. Separate path variables are created for the CDF raw, extracted, processed, source, and documentation directories.

Using these shared path variables avoids repeatedly hard-coding directory paths throughout the notebook and makes the workflow easier to maintain if the project structure changes.


In [ ]:
from pathlib import Path

PROJECT_ROOT = Path("../..")

RAW_DIR = PROJECT_ROOT / "data" / "raw" / "CDF"
EXTRACTED_DIR = PROJECT_ROOT / "data" / "extracted" / "CDF"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed" / "CDF"
SOURCES_DIR = PROJECT_ROOT / "sources"
DOCS_DIR = PROJECT_ROOT / "docs" / "CDF"

## 2. Data Source

## 3. Define the Data Source

The official Kabwe Municipal Council website is used as the primary digital source for the data collection process.

The base URL is stored in a variable so that it can be reused throughout the notebook when accessing council webpages and publications. This improves code readability and makes the extraction process easier to maintain.

In [ ]:
BASE_URL = "https://www.kabwecouncil.gov.zm"

print(BASE_URL)

https://www.kabwecouncil.gov.zm


## 4. Test the Council Website Connection

A `GET` request is used to test whether the official Kabwe Municipal Council website can be accessed programmatically.

The request includes a timeout to prevent the notebook from waiting indefinitely if the website does not respond. SSL certificate verification is disabled because the council website may return an SSL certificate warning during programmatic access.

The HTTP status code is printed to confirm whether the request was successful. A status code of `200` indicates that the webpage was successfully retrieved.

In [ ]:
import requests
import urllib3

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

response = requests.get(
    BASE_URL,
    timeout=30,
    verify=False
)

print("Status code:", response.status_code)

Status code: 200


## 5. Parse the Webpage

The downloaded HTML content is parsed using `BeautifulSoup`. The `html.parser` parser converts the raw HTML response into a structured object that can be searched and navigated.

This allows the notebook to identify elements such as links, headings, tables, and other information available on the council website for further data extraction.

In [ ]:
soup = BeautifulSoup(response.text, "html.parser")

## 6. Extract Website Links

All hyperlinks available on the Kabwe Municipal Council homepage are collected from the parsed HTML document.

For each link, the notebook extracts:

- The visible link text
- The complete URL associated with the link

`urljoin()` is used to convert relative links into complete URLs based on the council's base URL.

The extracted links are stored in a pandas DataFrame, making them easier to inspect and use when identifying relevant council publications and digital sources for further data extraction.

In [ ]:
links = []

for link in soup.find_all("a", href=True):
    text = link.get_text(" ", strip=True)
    url = urljoin(BASE_URL, link["href"])

    links.append({
        "text": text,
        "url": url
    })

links_df = pd.DataFrame(links)

links_df.head(20)

,text,url
0,,https://www.kabwecouncil.gov.zm#
1,Home,https://www.kabwecouncil.gov.zm/
2,About,https://www.kabwecouncil.gov.zm#
3,About Us,https://www.kabwecouncil.gov.zm/?page_id=2601
4,Mandate,https://www.kabwecouncil.gov.zm/?page_id=169
5,Who we are,https://www.kabwecouncil.gov.zm/?page_id=118
6,Departments,https://www.kabwecouncil.gov.zm/?page_id=770
7,office of the the town clerk,https://www.kabwecouncil.gov.zm/?page_id=2634
8,Dept of Human Resource and Administration,https://www.kabwecouncil.gov.zm/?page_id=2637
9,Dept of Health,https://www.kabwecouncil.gov.zm/?page_id=2640


## Filter Relevant Council Sources

The links collected from the official Kabwe Municipal Council website are filtered using keywords related to the required dataset and potential supporting sources.

Keywords such as CDF, budget, financial reports, development, IDP, wards, revenue, and procurement are used to identify potentially relevant pages and documents.

This filtering step narrows the large set of discovered website links to sources that may contain information useful for the CDF dataset and broader council data collection.


In [ ]:
keywords = [
    "cdf",
    "project",
    "budget",
    "financial",
    "report",
    "development",
    "idp",
    "ward",
    "revenue",
    "procurement",
    "minutes"
]

pattern = "|".join(keywords)

relevant_links = links_df[
    links_df["text"].str.contains(
        pattern,
        case=False,
        na=False
    )
]

relevant_links

,text,url
27,CDF,https://www.kabwecouncil.gov.zm/?page_id=2542
28,CDF GUIDLINES,https://www.kabwecouncil.gov.zm/?page_id=2579
29,CDF branding guidelines,https://www.kabwecouncil.gov.zm/?page_id=3674
66,CDF,https://www.kabwecouncil.gov.zm/?page_id=2542
67,CDF GUIDLINES,https://www.kabwecouncil.gov.zm/?page_id=2579
68,CDF branding guidelines,https://www.kabwecouncil.gov.zm/?page_id=3674
91,CDF Skills Bursaries Applicants,https://www.katetecouncil.gov.zm/wp-content/uploads/2023/11/SKILLS-DEVELOPME...
108,CDF PROJECT MONITORING BY KABWE MUNICIPAL COUNCIL MANAGEMENT TEAM,https://www.kabwecouncil.gov.zm/?p=4386
115,Ministry of Local Government and Rural Development,https://www.mlgrd.gov.zm/


## 7. Download the 2024 Bwacha CDF Project Document

The 2024 Bwacha CDF community projects document is downloaded directly from the official Kabwe Municipal Council website.

The response is validated by checking:

- The HTTP status code to confirm the request was successful.
- The `Content-Type` header to confirm that the returned resource is a PDF.
- The file size to confirm that content was received.
- The first bytes of the response to provide an additional check that the downloaded file is a PDF.

This step forms part of the data extraction process before the document is parsed and converted into structured data.

In [ ]:
import requests

pdf_url = (
    "https://www.kabwecouncil.gov.zm/"
    "wp-content/uploads/2024/11/"
    "2024-Bwacha-community-projects-Recieved.pdf"
)

pdf_response = requests.get(
    pdf_url,
    timeout=60,
    verify=False
)

print("Status code:", pdf_response.status_code)
print("Content-Type:", pdf_response.headers.get("Content-Type"))
print("File size:", len(pdf_response.content), "bytes")
print("First 20 bytes:", pdf_response.content[:20])

Status code: 200
Content-Type: application/pdf
File size: 93259 bytes
First 20 bytes: b'%PDF-1.5\r\n%\xb5\xb5\xb5\xb5\r\n1 0'


## 8. Save the Raw Source Document

The downloaded PDF is saved in the `data/raw/CDF` directory using a descriptive filename.

Keeping the original source document unchanged allows the extraction and cleaning process to be reproduced and provides a reference for verifying the accuracy of the structured dataset.

The raw source is preserved separately from the cleaned data so that the original evidence is not modified during preprocessing.

In [ ]:
pdf_path = RAW_DIR / "2024_bwacha_cdf_projects.pdf"

with open(pdf_path, "wb") as file:
    file.write(pdf_response.content)

print("PDF downloaded successfully.")

PDF downloaded successfully.


## 9. Inspect the PDF Structure

The saved PDF is opened using `pdfplumber` to inspect its structure before extracting the project records.

For each page, the notebook checks how many tables can be detected. This helps determine whether the PDF contains machine-readable tabular data and guides the subsequent extraction process.

The page and table counts are also useful for validating that the expected source document has been loaded correctly.

In [ ]:
with pdfplumber.open(pdf_path) as pdf:
    print("Number of pages:", len(pdf.pages))

    for page_number, page in enumerate(pdf.pages):
        tables = page.extract_tables()

        print(
            f"Page {page_number + 1}: "
            f"{len(tables)} table(s) found"
        )

Number of pages: 5
Page 1: 1 table(s) found
Page 2: 1 table(s) found
Page 3: 1 table(s) found
Page 4: 1 table(s) found
Page 5: 1 table(s) found


## 10. Inspect Extracted Table Data

The first detected table on the first page is inspected by printing a sample of its rows.

This step is used to examine how `pdfplumber` interprets the PDF table and to identify issues such as incorrect column boundaries, merged cells, missing values, or inconsistent formatting before the data is transformed into a structured dataset.

Only a small sample is displayed here to avoid unnecessarily printing the entire table during the inspection stage.

In [ ]:
with pdfplumber.open(pdf_path) as pdf:
    page = pdf.pages[0]
    tables = page.extract_tables()

    table = tables[0]

    for row in table[:10]:
        print(row)

['2024 CDF COMMUNITY PROJECTS SUBMISSION - BWACHA CONSTITUENCY\nKABWE MUNICIPAL COUNCIL', None, None, None, None, None, None, None, None, None]
['No.', 'Project Name', 'Project Description', 'Ward', 'Project\nSite/Location', 'Application\nAmount', "Engineers'\nEstimates", 'Approved\nAmount', 'Contract\nAmount', 'Status']
['Education', None, None, None, None, None, None, None, None, None]
['1', 'Construction of 1X3 Classroom\nBlock and Teachers Houses in\nKangomba ward', 'Construction of 1X3\nClassroom Block and\nTeachers Houses in\nKangomba ward - Kalima zone', 'Kangomba', 'Kangomba\nward', '', '', '', '', '']
['2', 'Construction of 1X4 Classroom\nBlock at Kagomba Primary\nSchool', 'Construction of 1X4\nClassroom Block at Kagomba\nPrimary School', 'Kangomba', 'Kangomba\nPrimary\nSchool', '', '', '', '', '']
['3', 'Construction of 1X4 Classroom\nBlock at Kagomba Primary\nSchool', 'Construction of 1X4\nClassroom Block at Kagomba\nPrimary School', 'Kangomba', 'Kangomba\nPrimary\nSchool', 

## 11. Convert the Extracted Table into a DataFrame

The extracted table is converted into a pandas DataFrame so that the CDF project records can be processed as structured tabular data.

The first row of the extracted table is used as the column headers, while the remaining rows are treated as project records.

A preview of the resulting DataFrame is displayed to verify that the columns and records have been correctly structured before further cleaning and preprocessing.

In [ ]:
df = pd.DataFrame(
    table[1:],
    columns=table[0]
)

df.head()

,2024 CDF COMMUNITY PROJECTS SUBMISSION - BWACHA CONSTITUENCY\nKABWE MUNICIPAL COUNCIL,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
0,No.,Project Name,Project Description,Ward,Project\nSite/Location,Application\nAmount,Engineers'\nEstimates,Approved\nAmount,Contract\nAmount,Status
1,Education,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1,Construction of 1X3 Classroom\nBlock and Teach...,Construction of 1X3\nClassroom Block and\nTeac...,Kangomba,Kangomba\nward,,,,,
3,2,Construction of 1X4 Classroom\nBlock at Kagomb...,Construction of 1X4\nClassroom Block at Kagomb...,Kangomba,Kangomba\nPrimary\nSchool,,,,,
4,3,Construction of 1X4 Classroom\nBlock at Kagomb...,Construction of 1X4\nClassroom Block at Kagomb...,Kangomba,Kangomba\nPrimary\nSchool,,,,,


## 12. Inspect Extracted Data Structure

The shape and column names of the DataFrame are inspected to confirm the number of extracted records and fields.

This provides an initial structural check before the data cleaning and preprocessing stage.

In [ ]:
print(df.shape)
print(df.columns.tolist())

(8, 10)
['2024 CDF COMMUNITY PROJECTS SUBMISSION - BWACHA CONSTITUENCY\nKABWE MUNICIPAL COUNCIL', nan, nan, nan, nan, nan, nan, nan, nan, nan]


## 13. Inspect Data Types and Missing Values

The `info()` method is used to inspect the structure of the extracted DataFrame.

It provides information about:

- The number of records and columns
- Column names
- Data types
- The number of non-null values in each column

This initial inspection helps identify fields that may require type conversion, standardisation, or missing-value handling during the data cleaning stage.

In [ ]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 8 entries, 0 to 7
Data columns (total 10 columns):
 #   Column                                                                                Non-Null Count  Dtype
---  ------                                                                                --------------  -----
 0   2024 CDF COMMUNITY PROJECTS SUBMISSION - BWACHA CONSTITUENCY
KABWE MUNICIPAL COUNCIL  8 non-null      str  
 1   nan                                                                                   7 non-null      str  
 2   nan                                                                                   7 non-null      str  
 3   nan                                                                                   7 non-null      str  
 4   nan                                                                                   7 non-null      str  
 5   nan                                                                                   7 non-null      str  
 6   n

## 14. Extract All Project Records

The notebook now processes every page of the PDF and extracts all detected tables.

Each non-empty row is added to a single list, creating one collection containing the project records from the complete document rather than only the first page.

The total number of extracted rows is reported as an initial completeness check before the records are converted into a structured DataFrame.

In [ ]:
all_rows = []

with pdfplumber.open(pdf_path) as pdf:
    for page_number, page in enumerate(pdf.pages, start=1):
        tables = page.extract_tables()

        for table in tables:
            for row in table:
                if row:
                    all_rows.append(row)

print("Total rows extracted:", len(all_rows))

Total rows extracted: 47


## 15. Identify Valid Project Records

The extracted rows are filtered to identify actual CDF project records.

A valid project row is identified by checking whether the first column contains a numeric project number. Rows that do not contain a numeric project number, such as table headers or other non-project text, are excluded.

This step converts the raw PDF extraction into a collection containing only the project records that can be used for further cleaning and preprocessing.

In [ ]:
project_rows = []

for row in all_rows:
    if row[0] is not None and str(row[0]).strip().isdigit():
        project_rows.append(row)

print("Number of project rows:", len(project_rows))

Number of project rows: 36


## 16. Structure the CDF Project Dataset

The validated project rows are converted into a pandas DataFrame and assigned descriptive column names based on the structure of the source document.

The fields capture key information about each CDF project, including its description, location, financial estimates, approved amount, contract amount, and implementation status.

Structuring the extracted records in this way makes the data suitable for subsequent cleaning, analysis, and export as a CSV dataset.

In [ ]:
columns = [
    "project_number",
    "project_name",
    "project_description",
    "ward",
    "project_site",
    "application_amount",
    "engineers_estimate",
    "approved_amount",
    "contract_amount",
    "status"
]

df = pd.DataFrame(project_rows, columns=columns)

df.head()

,project_number,project_name,project_description,ward,project_site,application_amount,engineers_estimate,approved_amount,contract_amount,status
0,1,Construction of 1X3 Classroom\nBlock and Teach...,Construction of 1X3\nClassroom Block and\nTeac...,Kangomba,Kangomba\nward,,,,,
1,2,Construction of 1X4 Classroom\nBlock at Kagomb...,Construction of 1X4\nClassroom Block at Kagomb...,Kangomba,Kangomba\nPrimary\nSchool,,,,,
2,3,Construction of 1X4 Classroom\nBlock at Kagomb...,Construction of 1X4\nClassroom Block at Kagomb...,Kangomba,Kangomba\nPrimary\nSchool,,,,,
3,4,Construction of 1X3 Classroom\nBlock at Mine P...,Construction of 1X3\nClassroom Block at Mine\n...,Kangomba,Mine Primary\nSchool,,,,,
4,5,"Repairing of the Mono-Pump,\nConstruction of T...","Repairing of the Mono-Pump,\nConstruction of T...",Kangomba,Mary Chidgey\nCommunity\nSchool,,,,,


In [ ]:
for column in df.columns:
    df[column] = (
        df[column]
        .fillna("")
        .astype(str)
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
    )

df.head()

,project_number,project_name,project_description,ward,project_site,application_amount,engineers_estimate,approved_amount,contract_amount,status
0,1,Construction of 1X3 Classroom Block and Teache...,Construction of 1X3 Classroom Block and Teache...,Kangomba,Kangomba ward,,,,,
1,2,Construction of 1X4 Classroom Block at Kagomba...,Construction of 1X4 Classroom Block at Kagomba...,Kangomba,Kangomba Primary School,,,,,
2,3,Construction of 1X4 Classroom Block at Kagomba...,Construction of 1X4 Classroom Block at Kagomba...,Kangomba,Kangomba Primary School,,,,,
3,4,Construction of 1X3 Classroom Block at Mine Pr...,Construction of 1X3 Classroom Block at Mine Pr...,Kangomba,Mine Primary School,,,,,
4,5,"Repairing of the Mono-Pump, Construction of To...","Repairing of the Mono-Pump, Construction of To...",Kangomba,Mary Chidgey Community School,,,,,


## 18. Add Dataset Context

The year and constituency fields are added to each project record because they apply to the entire source document.

The source document contains CDF community projects for **Bwacha Constituency for 2024**, so these values are assigned consistently to every extracted project record.

Adding these fields makes the dataset more informative and allows the records to be distinguished when combined with CDF project data from other years or constituencies.

In [ ]:
df.insert(1, "year", 2024)
df.insert(2, "constituency", "Bwacha")

## 19. Add Source Metadata

Source metadata is added to each project record to preserve the origin of the extracted data.

The `source_url` field records the official URL of the document, while `source_document` identifies the specific council publication from which the records were extracted.

Including source metadata makes individual records traceable to their original source and supports verification and reproducibility of the dataset.

In [ ]:
df["source_url"] = pdf_url
df["source_document"] = "2024 Bwacha CDF Community Projects Submission"

## 20. Inspect Data Types and Missing Values

The `info()` method is used to inspect the structure of the dataset after adding the source metadata.

This provides the number of records and columns, column names, data types, and non-null values. The information helps identify fields that may require data type conversion or missing-value handling during the cleaning stage.

In [ ]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 36 entries, 0 to 35
Data columns (total 14 columns):
 #   Column               Non-Null Count  Dtype
---  ------               --------------  -----
 0   project_number       36 non-null     str  
 1   year                 36 non-null     int64
 2   constituency         36 non-null     str  
 3   project_name         36 non-null     str  
 4   project_description  36 non-null     str  
 5   ward                 36 non-null     str  
 6   project_site         36 non-null     str  
 7   application_amount   36 non-null     str  
 8   engineers_estimate   36 non-null     str  
 9   approved_amount      36 non-null     str  
 10  contract_amount      36 non-null     str  
 11  status               36 non-null     str  
 12  source_url           36 non-null     str  
 13  source_document      36 non-null     str  
dtypes: int64(1), str(13)
memory usage: 4.1 KB


## 21. Inspect Project Distribution by Ward

The `value_counts()` method is used to examine the number of CDF project records associated with each ward.

This provides a basic completeness and consistency check for the ward field and helps identify unusual values, missing ward assignments, or potential inconsistencies that may need to be reviewed during data cleaning.

In [ ]:
df["ward"].value_counts()

ward
Kangomba        8
Kawama          8
Chimaniman i    6
Bwacha          4
Muwowo East     2
Munyama         2
Chililalila     2
Chinyama        2
Ngungu          2
Name: count, dtype: int64

## 22. Inspect Project Status Values

The `value_counts()` method is used to examine the distribution of project status values.

The `dropna=False` argument ensures that missing status values are also counted. This is important because missing values in the status field may indicate incomplete information in the source document and should be identified before deciding how they will be handled during data cleaning.

In [ ]:
df["status"].value_counts(dropna=False)

status
    36
Name: count, dtype: int64

## 23. Inspect Financial Field Formats

The financial fields are inspected before cleaning to identify the different formats used in the extracted PDF data.

The inspection checks the unique values in each financial column for formatting issues such as commas, currency symbols, blank values, dashes, or other non-numeric representations.

This helps determine the appropriate cleaning and numeric conversion rules for the financial fields.

In [ ]:
financial_columns = [
    "application_amount",
    "engineers_estimate",
    "approved_amount",
    "contract_amount"
]

for column in financial_columns:
    print(f"\n--- {column} ---")
    print(df[column].unique()[:20])


--- application_amount ---
<StringArray>
['']
Length: 1, dtype: str

--- engineers_estimate ---
<StringArray>
['']
Length: 1, dtype: str

--- approved_amount ---
<StringArray>
['']
Length: 1, dtype: str

--- contract_amount ---
<StringArray>
['']
Length: 1, dtype: str


## 24. Define Financial Value Cleaning Function

A reusable function is defined to standardise financial values extracted from the PDF.

The function:

- Removes leading and trailing whitespace.
- Converts blank values and placeholders such as `N/A` and `-` to missing values.
- Removes currency symbols, commas, and other non-numeric characters.
- Converts valid financial values to numeric `float` values.

Missing financial values are preserved as missing rather than being replaced with zero, because a blank value in the source does not necessarily mean that no amount existed.

Defining the cleaning logic as a reusable function ensures that the same preprocessing rules can be consistently applied to all financial columns.

In [ ]:
def clean_amount(value):
    value = str(value).strip()

    # Treat blank values as missing
    if value == "" or value.lower() in ["nan", "none", "n/a", "na", "-"]:
        return pd.NA

    # Remove currency symbols, commas and other non-numeric characters
    value = re.sub(r"[^0-9.\-]", "", value)

    if value == "":
        return pd.NA

    return float(value)

## 25. Apply Financial Data Cleaning

The `clean_amount()` function is applied to each financial column in the dataset.

This standardises the financial fields by converting valid values to numeric format and representing blank or placeholder values as missing data.

Applying the same cleaning function to all financial columns ensures consistent preprocessing across the dataset.

In [ ]:
df[financial_columns].head(10)

,application_amount,engineers_estimate,approved_amount,contract_amount
0,<NA>,<NA>,<NA>,<NA>
1,<NA>,<NA>,<NA>,<NA>
2,<NA>,<NA>,<NA>,<NA>
3,<NA>,<NA>,<NA>,<NA>
4,<NA>,<NA>,<NA>,<NA>
5,<NA>,<NA>,<NA>,<NA>
6,<NA>,<NA>,<NA>,<NA>
7,<NA>,<NA>,<NA>,<NA>
8,<NA>,<NA>,<NA>,<NA>
9,<NA>,<NA>,<NA>,<NA>


## 26. Verify Financial Data Types

The data types of the cleaned financial columns are checked to confirm that the preprocessing step converted the financial values into appropriate numeric types.

This verification helps ensure that the financial fields can be used reliably for calculations, comparisons, and further analysis.

In [ ]:
df[financial_columns].dtypes

application_amount    object
engineers_estimate    object
approved_amount       object
contract_amount       object
dtype: object

## 27. Check Missing Financial Values

The number of missing values in each financial column is calculated after cleaning.

This check confirms which financial fields contain unavailable values in the source data. Missing values are retained as missing rather than replaced with zero, since an empty or unavailable value in the source does not necessarily indicate that no amount was involved.

The results help verify that the cleaning process handled missing and placeholder values consistently.

In [ ]:
df[financial_columns].isna().sum()

application_amount    36
engineers_estimate    36
approved_amount       36
contract_amount       36
dtype: int64

## 28. Clean and Standardise Text Fields

The main text fields are cleaned to improve consistency across the project records.

The cleaning process:

- Replaces missing text values with empty strings.
- Converts values to string format.
- Replaces multiple whitespace characters with a single space.
- Removes unnecessary leading and trailing whitespace.

This standardisation reduces formatting inconsistencies while preserving the actual textual content from the source document.

In [ ]:
text_columns = [
    "project_name",
    "project_description",
    "ward",
    "project_site",
    "status"
]

for column in text_columns:
    df[column] = (
        df[column]
        .fillna("")
        .astype(str)
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
    )

## 29. Check for Duplicate Project Records

Potential duplicate project records are identified using a combination of project name, ward, and project site.

These fields provide a practical combination for identifying whether the same project has been extracted more than once, even when other fields such as financial values or status differ.

The number of potential duplicates is calculated and reported as part of the data quality checks.

In [ ]:
duplicates = df.duplicated(
    subset=[
        "project_name",
        "ward",
        "project_site"
    ]
)

print("Duplicate rows:", duplicates.sum())

Duplicate rows: 1


## 30. Export the Cleaned 2024 CDF Dataset

The cleaned Bwacha CDF project records are exported as a CSV file using the pipe (`|`) delimiter required by the assignment.

The output filename follows the prescribed dataset naming convention:

`db-unza26-csc4792-kabwe_cdf_projects_2024.csv`

The cleaned dataset is saved in the `data/processed/CDF` directory, keeping it separate from the original raw source documents and intermediate extracted data.

In [ ]:
processed_path = (
    PROCESSED_DIR /
    "db-unza26-csc4792-kabwe_cdf_projects_2024.csv"
)

df.to_csv(
    processed_path,
    sep="|",
    index=False
)

print("Saved:", processed_path)

Saved: ../data/processed/kabwe_2024_bwacha_cdf_projects.csv


## 31. Validate the Exported CSV

The exported CSV file is read back into pandas using the pipe (`|`) delimiter.

The number of rows and columns is checked to confirm that the exported dataset can be successfully loaded and that its structure has been preserved.

A preview of the reloaded dataset provides an additional verification that the processed CSV was written correctly.

In [ ]:
test_df = pd.read_csv(
    processed_path,
    sep="|"
)

print("Rows:", len(test_df))
print("Columns:", len(test_df.columns))

test_df.head()

Rows: 36
Columns: 14


,project_number,year,constituency,project_name,project_description,ward,project_site,application_amount,engineers_estimate,approved_amount,contract_amount,status,source_url,source_document
0,1,2024,Bwacha,Construction of 1X3 Classroom Block and Teachers Houses in Kangomba ward,Construction of 1X3 Classroom Block and Teachers Houses in Kangomba ward - Kalima zone,Kangomba,Kangomba ward,NaN,NaN,NaN,NaN,NaN,https://www.kabwecouncil.gov.zm/wp-content/uploads/2024/11/2024-Bwacha-community-projects-Reciev...,2024 Bwacha CDF Community Projects Submission
1,2,2024,Bwacha,Construction of 1X4 Classroom Block at Kagomba Primary School,Construction of 1X4 Classroom Block at Kagomba Primary School,Kangomba,Kangomba Primary School,NaN,NaN,NaN,NaN,NaN,https://www.kabwecouncil.gov.zm/wp-content/uploads/2024/11/2024-Bwacha-community-projects-Reciev...,2024 Bwacha CDF Community Projects Submission
2,3,2024,Bwacha,Construction of 1X4 Classroom Block at Kagomba Primary School,Construction of 1X4 Classroom Block at Kagomba Primary School,Kangomba,Kangomba Primary School,NaN,NaN,NaN,NaN,NaN,https://www.kabwecouncil.gov.zm/wp-content/uploads/2024/11/2024-Bwacha-community-projects-Reciev...,2024 Bwacha CDF Community Projects Submission
3,4,2024,Bwacha,Construction of 1X3 Classroom Block at Mine Primary School,Construction of 1X3 Classroom Block at Mine Primary School - Mutwewansofu zone,Kangomba,Mine Primary School,NaN,NaN,NaN,NaN,NaN,https://www.kabwecouncil.gov.zm/wp-content/uploads/2024/11/2024-Bwacha-community-projects-Reciev...,2024 Bwacha CDF Community Projects Submission
4,5,2024,Bwacha,"Repairing of the Mono-Pump, Construction of Toilets for Pre- School Pupils, Procurement of Desks...","Repairing of the Mono-Pump, Construction of Toilets for Pre- School Pupils, Procurement of Desks...",Kangomba,Mary Chidgey Community School,NaN,NaN,NaN,NaN,NaN,https://www.kabwecouncil.gov.zm/wp-content/uploads/2024/11/2024-Bwacha-community-projects-Reciev...,2024 Bwacha CDF Community Projects Submission


## 32. Download and Validate the 2024 Kabwe Central CDF Document

The 2024 Kabwe Central CDF community projects document is downloaded directly from the official Kabwe Municipal Council website.

The response is validated by checking the HTTP status code, content type, file size, and initial bytes of the downloaded content. These checks help confirm that the expected PDF document was successfully retrieved before extraction.

In [ ]:
pdf_url_central = (
    "https://www.kabwecouncil.gov.zm/"
    "wp-content/uploads/2024/11/"
    "2024-Community-Projects-Kabwe-Central-received.pdf"
)

response_central = requests.get(
    pdf_url_central,
    timeout=60,
    verify=False
)

print("Status code:", response_central.status_code)
print("Content-Type:", response_central.headers.get("Content-Type"))
print("File size:", len(response_central.content), "bytes")
print("First 20 bytes:", response_central.content[:20])

Status code: 200
Content-Type: application/pdf
File size: 96930 bytes
First 20 bytes: b'%PDF-1.5\r\n%\xb5\xb5\xb5\xb5\r\n1 0'


## 33. Save the Raw Kabwe Central CDF Document

The downloaded Kabwe Central CDF project document is saved unchanged in the `data/raw/CDF` directory.

Preserving the original PDF separately from the processed dataset allows the extracted records to be traced back to the original council publication and supports reproducibility and verification of the data preparation process.

In [ ]:
pdf_path_central = (
    RAW_DIR /
    "2024_kabwe_central_cdf_projects.pdf"
)

with open(pdf_path_central, "wb") as file:
    file.write(response_central.content)

print("Saved:", pdf_path_central)

Saved: ../data/raw/2024_kabwe_central_cdf_projects.pdf


## 34. Inspect the Kabwe Central PDF Structure

The Kabwe Central CDF project document is inspected page by page using `pdfplumber`.

For each page, the notebook identifies the number of tables detected. This helps determine how the PDF is structured and confirms whether the project records can be extracted as machine-readable tables.

The inspection results guide the extraction method used for the Kabwe Central project records.

In [ ]:
with pdfplumber.open(pdf_path_central) as pdf:
    print("Number of pages:", len(pdf.pages))

    for page_number, page in enumerate(pdf.pages, start=1):
        tables = page.extract_tables()

        print(
            f"Page {page_number}: "
            f"{len(tables)} table(s) found"
        )

Number of pages: 2
Page 1: 1 table(s) found
Page 2: 3 table(s) found


## 35. Inspect the Kabwe Central Extracted Table

A sample of the first table from the Kabwe Central CDF document is displayed to inspect how the PDF table has been interpreted by `pdfplumber`.

The sample is used to identify the table headers, column structure, missing values, and any formatting issues that may need to be considered when structuring and cleaning the extracted records.

Only the first few rows are displayed to keep the notebook output concise.

In [ ]:
with pdfplumber.open(pdf_path_central) as pdf:
    table = pdf.pages[0].extract_tables()[0]

    for row in table[:10]:
        print(row)

['No.', 'Project Name', 'Project Description', 'Type of\nProject', 'Ward', 'Project\nSite/Location', 'Application\nAmount', "Engineers'\nEstimates", 'Approved\nAmount', 'Contract\nAmount', 'Status']
['Education', None, None, None, None, None, None, None, None, None, None]
['1', 'Construction of Secondary\nSchool 1X4 Classroom\nBlock', 'Construction of Secondary\nSchool 1X4 Classroom\nBlock', 'Construction', 'Luangwa', 'Kabwe Trust\nSecondary School', '', '', '', '', '']
['2', 'Construction of 1X3\nClassroom Block', 'Construction of 1X3\nClassroom Block', 'Construction', 'Luangwa', 'Kabwe Central\nHospital Special\nSchool Community', '', '', '', '', '']
['3', 'Construction of 1X2\nClassroom Block', 'Construction of 1X2\nClassroom Block', 'Construction', 'Luangwa', 'Kabwe Trust Primary\nSchool', '', '', '', '', '']
['4', 'Construction of 1X3\nClassroom Block', 'Construction of 1X3\nClassroom Block', 'Construction', 'Mpima', 'Mpima Dairy Scheme', '', '', '', '', '']
['5', 'Construction of

## 35. Inspect the Kabwe Central Extracted Table

A sample of the first table from the Kabwe Central CDF document is displayed to inspect how the PDF table has been interpreted by `pdfplumber`.

The sample is used to identify the table headers, column structure, missing values, and any formatting issues that may need to be considered when structuring and cleaning the extracted records.

Only the first few rows are displayed to keep the notebook output concise.

In [ ]:
central_rows = []

with pdfplumber.open(pdf_path_central) as pdf:
    for page in pdf.pages:
        tables = page.extract_tables()

        for table in tables:
            for row in table:
                if row:
                    # Keep only rows whose first column is a project number
                    if (
                        row[0] is not None
                        and str(row[0]).strip().isdigit()
                    ):
                        central_rows.append(row)

print("Kabwe Central project rows:", len(central_rows))

Kabwe Central project rows: 43


## 36. Structure the Kabwe Central CDF Dataset

The validated Kabwe Central project records are converted into a pandas DataFrame using descriptive column names based on the structure of the source document.

The resulting fields capture project identification, description, location, financial information, and implementation status.

A small preview is displayed to verify that the extracted records have been assigned to the expected columns before further cleaning and preprocessing.

In [ ]:
central_columns = [
    "project_number",
    "project_name",
    "project_description",
    "project_type",
    "ward",
    "project_site",
    "application_amount",
    "engineers_estimate",
    "approved_amount",
    "contract_amount",
    "status"
]

central_df = pd.DataFrame(
    central_rows,
    columns=central_columns
)

central_df.head()

,project_number,project_name,project_description,project_type,ward,project_site,application_amount,engineers_estimate,approved_amount,contract_amount,status
0,1,Construction of Secondary\nSchool 1X4 Classroom\nBlock,Construction of Secondary\nSchool 1X4 Classroom\nBlock,Construction,Luangwa,Kabwe Trust\nSecondary School,,,,,
1,2,Construction of 1X3\nClassroom Block,Construction of 1X3\nClassroom Block,Construction,Luangwa,Kabwe Central\nHospital Special\nSchool Community,,,,,
2,3,Construction of 1X2\nClassroom Block,Construction of 1X2\nClassroom Block,Construction,Luangwa,Kabwe Trust Primary\nSchool,,,,,
3,4,Construction of 1X3\nClassroom Block,Construction of 1X3\nClassroom Block,Construction,Mpima,Mpima Dairy Scheme,,,,,
4,5,Construction of 1X3\nClassroom Block,Construction of 1X3\nClassroom Block,Construction,Mpima,Mpima C,,,,,


## 37. Verify the Kabwe Central Dataset Structure

The number of records, number of columns, and column names are checked after structuring the Kabwe Central project data.

This validation confirms that the extracted dataset has the expected structure before the records are cleaned and combined with the other 2024 CDF project data.

In [ ]:
central_text_columns = [
    "project_name",
    "project_description",
    "project_type",
    "ward",
    "project_site",
    "status"
]

for column in central_text_columns:
    central_df[column] = (
        central_df[column]
        .fillna("")
        .astype(str)
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
    )

## 38. Add Dataset Context

The year and constituency fields are added to the Kabwe Central project records.

The source document contains CDF community projects for **Kabwe Central Constituency for 2024**, so these values are assigned consistently to all extracted records.

These fields allow project records from different constituencies and years to be distinguished when the datasets are combined.

In [ ]:
central_df.insert(1, "year", 2024)
central_df.insert(2, "constituency", "Kabwe Central")

## 39. Add Source Metadata

Source metadata is added to each Kabwe Central project record to preserve its connection to the original council publication.

The `source_url` field stores the official document URL, while `source_document` identifies the specific CDF publication used for the extraction.

This allows individual records to be traced back to their source and supports verification and reproducibility when the Kabwe Central and Bwacha datasets are combined.

In [ ]:
central_df["source_url"] = pdf_url_central
central_df["source_document"] = (
    "2024 Kabwe Central CDF Community Projects Submission"
)

## 40. Clean Kabwe Central Financial Fields

The reusable `clean_amount()` function is applied to the financial fields in the Kabwe Central dataset.

Using the same cleaning function as the Bwacha dataset ensures that financial values from both constituency sources are standardised using consistent preprocessing rules before the datasets are combined.

In [ ]:
financial_columns = [
    "application_amount",
    "engineers_estimate",
    "approved_amount",
    "contract_amount"
]

for column in financial_columns:
    central_df[column] = central_df[column].apply(
        clean_amount
    )

## 41. Check Missing Kabwe Central Financial Values

The number of missing values in each financial field is calculated after applying the cleaning function.

This confirms that missing or unavailable financial values from the source document have been preserved as missing rather than incorrectly converted to zero.

In [ ]:
central_df[financial_columns].isna().sum()

application_amount    43
engineers_estimate    43
approved_amount       43
contract_amount       43
dtype: int64

## 42. Standardise the 2024 CDF Dataset Schema

The Bwacha and Kabwe Central project DataFrames are aligned to the same final column structure before they are combined.

Using a consistent schema ensures that corresponding fields from both constituency sources are placed in the same columns and can be safely concatenated into a single 2024 CDF project dataset.

The final schema contains project identification, year and constituency, project details, location, financial information, status, and source metadata.

In [ ]:
final_columns = [
    "project_number",
    "year",
    "constituency",
    "project_name",
    "project_description",
    "ward",
    "project_site",
    "application_amount",
    "engineers_estimate",
    "approved_amount",
    "contract_amount",
    "status",
    "source_url",
    "source_document"
]

df = df[final_columns]
central_df = central_df[final_columns]

## 43. Combine 2024 CDF Project Records

The cleaned project records from Bwacha and Kabwe Central constituencies are combined into a single DataFrame using `pandas.concat()`.

Both datasets have already been standardised to the same column structure, allowing the records to be safely combined.

`ignore_index=True` creates a new sequential index for the combined dataset.

The resulting record count is printed as a completeness check for the integrated 2024 CDF project dataset.

In [ ]:
kabwe_cdf_projects = pd.concat(
    [df, central_df],
    ignore_index=True
)

print(
    "Total Kabwe CDF projects:",
    len(kabwe_cdf_projects)
)

Total Kabwe CDF projects: 79


## 44. Validate Constituency Distribution

The constituency distribution of the combined 2024 CDF project dataset is checked using `value_counts()`.

This verifies that records from both source constituencies were successfully included after the integration step.

The resulting counts also provide a simple completeness check for the constituency field.

In [ ]:
kabwe_cdf_projects["constituency"].value_counts()

constituency
Kabwe Central    43
Bwacha           36
Name: count, dtype: int64

## 45. Export the Integrated 2024 CDF Dataset

The cleaned and integrated 2024 CDF project records are exported as a CSV file using the pipe (`|`) delimiter required by the assignment.

The dataset combines project records from Bwacha and Kabwe Central constituencies and is saved using the prescribed naming convention:

`db-unza26-csc4792-kabwe_cdf_projects_2024.csv`

The processed dataset is stored in the `data/processed/CDF` directory, while the original council documents remain preserved in `data/raw/CDF`.

In [ ]:
output_path = (
    PROCESSED_DIR /
    "db-unza26-csc4792-kabwe_cdf_projects_2024.csv"
)

kabwe_cdf_projects.to_csv(
    output_path,
    sep="|",
    index=False
)

print("Saved:", output_path)

Saved: ../data/processed/kabwe_cdf_projects_2024.csv


## 46. Inspect the Integrated 2024 CDF Dataset

The `info()` method is used to inspect the structure of the combined 2024 CDF project dataset after integrating the Bwacha and Kabwe Central records.

This verifies the final number of records and fields, data types, and non-null values across the integrated dataset before final quality checks and export validation.

In [ ]:
kabwe_cdf_projects.info()

<class 'pandas.DataFrame'>
RangeIndex: 79 entries, 0 to 78
Data columns (total 15 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   project_number       79 non-null     str   
 1   year                 79 non-null     int64 
 2   constituency         79 non-null     str   
 3   project_name         79 non-null     str   
 4   project_description  79 non-null     str   
 5   project_type         43 non-null     object
 6   ward                 79 non-null     str   
 7   project_site         79 non-null     str   
 8   application_amount   0 non-null      object
 9   engineers_estimate   0 non-null      object
 10  approved_amount      0 non-null      object
 11  contract_amount      0 non-null      object
 12  status               79 non-null     str   
 13  source_url           79 non-null     str   
 14  source_document      79 non-null     str   
dtypes: int64(1), object(5), str(9)
memory usage: 9.4+ KB


## 47. Check Missing Values in the Integrated Dataset

Missing values are counted across all columns in the integrated 2024 CDF project dataset.

This provides an overall data-quality check after combining the two constituency sources and confirms which fields contain incomplete information.

Missing values are not automatically replaced because a missing value in the original council source does not necessarily mean that the value was zero or not applicable.

In [ ]:
missing = kabwe_cdf_projects.isna().sum()

print(missing)

project_number          0
year                    0
constituency            0
project_name            0
project_description     0
project_type           36
ward                    0
project_site            0
application_amount     79
engineers_estimate     79
approved_amount        79
contract_amount        79
status                  0
source_url              0
source_document         0
dtype: int64


## 48. Calculate Missing Value Percentages

The percentage of missing values is calculated for each column in the integrated 2024 CDF dataset.

This provides a clearer measure of data completeness by showing the proportion of records with missing values rather than only the raw count.

The results help identify fields with a relatively high level of missing information and support decisions about appropriate missing-value handling.

In [ ]:
missing_percentage = (
    kabwe_cdf_projects.isna().mean() * 100
).round(2)

print(missing_percentage)

project_number           0.00
year                     0.00
constituency             0.00
project_name             0.00
project_description      0.00
project_type            45.57
ward                     0.00
project_site             0.00
application_amount     100.00
engineers_estimate     100.00
approved_amount        100.00
contract_amount        100.00
status                   0.00
source_url               0.00
source_document          0.00
dtype: float64


## 49. Check for Empty String Values

The dataset is checked for empty string values in addition to standard missing values.

This is necessary because empty strings may be present in text fields without being recognised as `NaN` values by pandas.

Leading and trailing whitespace is removed before checking, ensuring that values containing only spaces are also identified as empty.

In [ ]:
empty_values = (
    kabwe_cdf_projects
    .astype(str)
    .apply(lambda column: column.str.strip().eq("").sum())
)

print(empty_values)

project_number          0
year                    0
constituency            0
project_name            0
project_description     0
project_type            0
ward                    0
project_site            1
application_amount      0
engineers_estimate      0
approved_amount         0
contract_amount         0
status                 79
source_url              0
source_document         0
dtype: int64


## 50. Check for Exact Duplicate Records

The integrated 2024 CDF dataset is checked for exact duplicate rows.

This validation ensures that identical project records were not accidentally introduced during the extraction, cleaning, or integration of the Bwacha and Kabwe Central datasets.

The number of exact duplicate rows is reported as part of the final data-quality checks.

In [ ]:
duplicate_count = kabwe_cdf_projects.duplicated().sum()

print("Exact duplicate rows:", duplicate_count)

Exact duplicate rows: 0


## 51. Check for Potential Duplicate Projects

Potential duplicate project records are identified using key project-identification fields: year, constituency, project name, ward, and project site.

The `keep=False` option marks all records belonging to a duplicate group, allowing the total number of potentially duplicated records to be reported.

This check helps identify duplicate projects that may have different financial or descriptive values but otherwise represent the same project.

In [ ]:
project_duplicates = kabwe_cdf_projects.duplicated(
    subset=[
        "year",
        "constituency",
        "project_name",
        "ward",
        "project_site"
    ],
    keep=False
)

print(
    "Potential duplicate project records:",
    project_duplicates.sum()
)

Potential duplicate project records: 2


## 52. Summarise Financial Data

Descriptive statistics are generated for the financial fields in the integrated 2024 CDF project dataset.

The summary provides measures such as count, mean, standard deviation, minimum, maximum, and quartiles.

This helps assess the range and distribution of the cleaned financial values and can also reveal unusual or potentially inconsistent values that may require further investigation.

In [ ]:
kabwe_cdf_projects[
    financial_columns
].describe()

,application_amount,engineers_estimate,approved_amount,contract_amount
count,0,0,0,0
unique,0,0,0,0
top,NaN,NaN,NaN,NaN
freq,NaN,NaN,NaN,NaN


## 53. Check for Negative Financial Values

The financial fields are checked for negative values after cleaning.

Negative amounts are flagged because the project financial fields represent applications, estimates, approved amounts, and contract amounts, which would normally be recorded as non-negative values.

This check helps identify possible extraction or data-cleaning errors before the dataset is finalised.

In [ ]:
for column in financial_columns:
    negative_values = (
        kabwe_cdf_projects[column] < 0
    ).sum()

    print(
        column,
        "negative values:",
        negative_values
    )

application_amount negative values: 0
engineers_estimate negative values: 0
approved_amount negative values: 0
contract_amount negative values: 0


## 54. Generate a Data Quality Report

A consolidated data-quality report is generated for the integrated 2024 CDF dataset.

The report summarises the data type, number of missing values, and number of unique values for every column.

This provides a compact overview of the dataset's structure, completeness, and variability and supports the final quality-assurance process before the dataset is submitted.

In [ ]:
quality_report = pd.DataFrame({
    "column": kabwe_cdf_projects.columns,
    "data_type": [
        str(dtype)
        for dtype in kabwe_cdf_projects.dtypes
    ],
    "missing_values": [
        kabwe_cdf_projects[column].isna().sum()
        for column in kabwe_cdf_projects.columns
    ],
    "unique_values": [
        kabwe_cdf_projects[column].nunique()
        for column in kabwe_cdf_projects.columns
    ]
})

quality_report

,column,data_type,missing_values,unique_values
0,project_number,str,0,26
1,year,int64,0,1
2,constituency,str,0,2
3,project_name,str,0,75
4,project_description,str,0,75
5,project_type,object,36,8
6,ward,str,0,25
7,project_site,str,0,51
8,application_amount,object,79,0
9,engineers_estimate,object,79,0


## 55. Download and Validate the 2025 Proposed CDF Projects

The 2025 proposed CDF projects document is downloaded directly from the official Kabwe Municipal Council website.

The response is validated by checking the HTTP status code, content type, file size, and first bytes of the downloaded content. These checks help confirm that the expected PDF document was successfully retrieved before the extraction process.

The source URL is retained so that the extracted project records can be traced back to the original council publication.

In [ ]:
pdf_url_2025_central = (
    "https://www.kabwecouncil.gov.zm/"
    "wp-content/uploads/2025/08/"
    "Proposed-2025-CDF-projects.pdf"
)

response_2025_central = requests.get(
    pdf_url_2025_central,
    timeout=60,
    verify=False
)

print("Status code:", response_2025_central.status_code)
print("Content-Type:", response_2025_central.headers.get("Content-Type"))
print("File size:", len(response_2025_central.content), "bytes")
print("First 20 bytes:", response_2025_central.content[:20])

Status code: 200
Content-Type: application/pdf
File size: 2017833 bytes
First 20 bytes: b'%PDF-1.4\n%\xe2\xe3\xcf\xd3\n11 0 '


## 56. Save the Raw 2025 CDF Source Document

The downloaded 2025 proposed CDF projects PDF is saved unchanged in the `data/raw/CDF` directory.

Preserving the original source document allows the extracted project records to be verified against the council's publication and supports reproducibility of the data extraction process.

The raw document is kept separate from the processed dataset so that the original source remains unmodified.

In [ ]:
pdf_path_2025_central = (
    RAW_DIR /
    "2025_proposed_kabwe_central_cdf_projects.pdf"
)

with open(pdf_path_2025_central, "wb") as file:
    file.write(response_2025_central.content)

print("Saved:", pdf_path_2025_central)

Saved: ../data/raw/2025_proposed_kabwe_central_cdf_projects.pdf


In [ ]:
import pdfplumber

## 57. Check Text Extraction from the 2025 CDF PDF

The 2025 proposed CDF projects PDF is checked to determine whether its pages contain machine-readable text.

`pdfplumber` attempts to extract text from each page. The result is reported as either `TEXT FOUND` or `NO TEXT`.

This check is important because the source document is scanned rather than fully machine-readable. Where text cannot be extracted directly, an alternative process such as image-based extraction and manual verification is required to preserve the accuracy of the project records.

In [ ]:
with pdfplumber.open(pdf_path_2025_central) as pdf:
    for page_number, page in enumerate(pdf.pages, start=1):
        text = page.extract_text()

        print(
            f"Page {page_number}:",
            "TEXT FOUND" if text else "NO TEXT"
        )

Page 1: NO TEXT
Page 2: NO TEXT
Page 3: NO TEXT
Page 4: NO TEXT
Page 5: NO TEXT


## 58. Verify Text Extraction with PyMuPDF

PyMuPDF is used as a second text-extraction method to determine whether the 2025 proposed CDF PDF contains machine-readable text.

The number of characters extracted from each page is reported. Very low or zero character counts indicate that the document is primarily image-based and that direct text extraction is not sufficient for recovering the project records.

This validation supports the decision to use image-based extraction followed by manual verification for the scanned CDF document.

In [ ]:
import fitz

document = fitz.open(pdf_path_2025_central)

for page_number, page in enumerate(document, start=1):
    text = page.get_text("text")

    print(
        f"Page {page_number}:",
        len(text),
        "characters"
    )

Page 1: 0 characters
Page 2: 0 characters
Page 3: 0 characters
Page 4: 0 characters
Page 5: 0 characters


## 59. Install OCR Dependencies for the Scanned PDF

The 2025 proposed CDF projects document is image-based, so Optical Character Recognition (OCR) is required to recover text from its scanned pages.

The notebook installs Tesseract OCR and the required Python packages:

- `pytesseract` provides a Python interface to Tesseract.
- `pdf2image` converts PDF pages into images that can be processed by OCR.
- `tesseract-ocr` provides the OCR engine used to recognise text from the page images.

These tools are used only for the scanned 2025 CDF document because conventional PDF text extraction did not provide usable text.

In [ ]:
import pytesseract

pytesseract.pytesseract.tesseract_cmd = (
    r"C:\Program Files\Tesseract-OCR\tesseract.exe"
)

print("Tesseract path:", pytesseract.pytesseract.tesseract_cmd)
print("Tesseract version:")
print(pytesseract.get_tesseract_version())

Tesseract path: C:\Program Files\Tesseract-OCR\tesseract.exe
Tesseract version:
5.5.3.20260724


In [ ]:
from pdf2image import convert_from_path

print("pdf2image is ready")

pdf2image is ready


## 60. Convert the Scanned PDF into Images

The scanned 2025 CDF PDF is converted into page images using `pdf2image`.

A resolution of 300 DPI is used to provide sufficient image quality for Optical Character Recognition (OCR). Poppler is specified explicitly because the notebook is being executed on Windows.

Each PDF page is converted into an image that can subsequently be processed by Tesseract OCR.

The number of converted pages is reported as a check that the complete PDF has been processed.

In [ ]:
from pdf2image import convert_from_path

poppler_path = r"C:\poppler-26.07.0\Library\bin"

pages = convert_from_path(
    pdf_path_2025_central,
    dpi=300,
    poppler_path=poppler_path
)

print("Pages converted:", len(pages))

Pages converted: 5


## 61. Extract Text from the Scanned CDF Page Using OCR

Tesseract OCR is applied to the first converted PDF page to recover text from the scanned document.

The `--psm 6` configuration treats the page as a single uniform block of text, which is suitable for testing the scanned page before processing the complete document.

Only the first 5,000 characters are displayed to inspect the OCR output without producing excessive notebook output.

The extracted text is then used to guide the structuring and verification of the 2025 CDF project records.

In [ ]:
page_text = pytesseract.image_to_string(
    pages[0],
    config="--psm 6"
)

print(page_text[:5000])

_
CDF 2025 PROPOSED COMMUNITY PROJECT KABWE CENTRAL CONSTITUENCY ©
©
NO | NAME OF COMMUNITY PROJECT APPLIED FORSHORTLUSTED; WARD SECTOR COMMENT O
1 Proposed Construction of a standard Maternity Annex at Mpima 5
Mpima health center
.
— =
Proposed Construction of a 1x4 Classroom block ( CRB) at High ridge Approved =
Lukanga Secondary School i
o
7 Proposed Construction of a 1x3 Classroom block (CRB) at Chirwa =
Kasanda Malombe Secondary School Oo
O
4 | Proposed Construction and installation of 02 Solar powered | Waya Water and Approved
Water reticulated Systems with lockable kiosks in Waya Sanitation
communities
Proposed Construction and installation of 02 Solar powered | Kalonga Water and Approved
Water reticulated Systems at Kamushanga Market Shelter Sanitation
Proposed Construction and installation of 02 Solar powered | Kaputula Education Approved
Water reticulated Systems at C-gate Priamary School
7 | Proposed Construction and installation of 02 Solar powered | Nijanji Education Appro

## 62. Save the OCR Output

The OCR text extracted from the scanned CDF document is saved as a UTF-8 text file in the `data/raw/CDF` directory.

Preserving the OCR output provides an intermediate record of the extraction process and makes it possible to review the text produced by Tesseract before converting it into structured project records.

At this stage, the saved output represents the OCR test performed on the first page. The complete document will be processed in the subsequent extraction step.

In [ ]:
ocr_path = (
    RAW_DIR /
    "2025_kabwe_central_cdf_ocr.txt"
)

with open(ocr_path, "w", encoding="utf-8") as file:
    file.write(page_text)

print("OCR text saved:", ocr_path)

OCR text saved: ../data/raw/2025_kabwe_central_cdf_ocr.txt


## 63. Perform OCR on All PDF Pages

Tesseract OCR is applied to every page of the scanned 2025 CDF document.

The extracted text from each page is stored in the `all_ocr_text` list, preserving the OCR output for the complete document.

A progress message is printed after each page is processed so that the extraction progress can be monitored during execution.

This step produces the complete OCR text that will be used to identify and structure the 2025 CDF project records.

In [ ]:
all_ocr_text = []

for page_number, page in enumerate(pages, start=1):
    text = pytesseract.image_to_string(
        page,
        config="--psm 6"
    )

    all_ocr_text.append(text)

    print(f"Page {page_number} OCR complete")

Page 1 OCR complete
Page 2 OCR complete
Page 3 OCR complete
Page 4 OCR complete
Page 5 OCR complete


## 64. Combine and Inspect the Complete OCR Output

The OCR results from all pages are combined into a single text string using newline separators.

The first 10,000 characters are displayed to inspect the overall OCR output and identify whether the project information has been captured sufficiently for subsequent structuring and verification.

The complete OCR text remains stored in `full_ocr_text`, while the displayed output is limited to avoid excessive notebook output.

In [ ]:
full_ocr_text = "\n".join(all_ocr_text)

print(full_ocr_text[:10000])

_
CDF 2025 PROPOSED COMMUNITY PROJECT KABWE CENTRAL CONSTITUENCY ©
©
NO | NAME OF COMMUNITY PROJECT APPLIED FORSHORTLUSTED; WARD SECTOR COMMENT O
1 Proposed Construction of a standard Maternity Annex at Mpima 5
Mpima health center
.
— =
Proposed Construction of a 1x4 Classroom block ( CRB) at High ridge Approved =
Lukanga Secondary School i
o
7 Proposed Construction of a 1x3 Classroom block (CRB) at Chirwa =
Kasanda Malombe Secondary School Oo
O
4 | Proposed Construction and installation of 02 Solar powered | Waya Water and Approved
Water reticulated Systems with lockable kiosks in Waya Sanitation
communities
Proposed Construction and installation of 02 Solar powered | Kalonga Water and Approved
Water reticulated Systems at Kamushanga Market Shelter Sanitation
Proposed Construction and installation of 02 Solar powered | Kaputula Education Approved
Water reticulated Systems at C-gate Priamary School
7 | Proposed Construction and installation of 02 Solar powered | Nijanji Education Appro

## 65. Save the Complete OCR Output

The complete OCR text generated from all pages of the scanned 2025 CDF document is saved as a UTF-8 text file in the `data/raw/CDF` directory.

Saving the complete OCR output preserves an intermediate extraction artifact that can be reviewed and used to reproduce the subsequent data structuring and verification process.

The OCR output is kept alongside the original PDF while the final structured project dataset is stored separately in `data/processed/CDF`.

In [ ]:
ocr_path = (
    RAW_DIR /
    "2025_kabwe_central_cdf_ocr.txt"
)

with open(ocr_path, "w", encoding="utf-8") as file:
    file.write(full_ocr_text)

print("Complete OCR saved:", ocr_path)

## 66. Identify Project Records from OCR Text

The OCR output is processed line by line to identify lines that appear to represent CDF project records.

A regular expression is used to detect lines beginning with a numeric project number. This filters out headings, labels, and other non-project text from the OCR output.

The identified project lines are stored in `project_lines` for further structuring and verification.

In [ ]:
project_lines = []

for line in lines:
    line = line.strip()

    if re.match(r"^\d+\s", line):
        project_lines.append(line)

for line in project_lines:
    print(line)

1 Proposed Construction of a standard Maternity Annex at Mpima 5
7 Proposed Construction of a 1x3 Classroom block (CRB) at Chirwa =
4 | Proposed Construction and installation of 02 Solar powered | Waya Water and Approved
7 | Proposed Construction and installation of 02 Solar powered | Nijanji Education Approved
10 | Proposed Construction of an Ablution block at BOCCs Katondo Approved c
11 | Procurement of a Hydraulic Tipper Truck All Transport Approved OD
12 | Proposed extension and rehabilitation of Waya market Waya Commerce and | Approved 140]
13 | Proposed Completion of Nakoli Market shelter Nakoli Commerce and | Approved
14 | Completion of Kabwe General Hospital’s Relative waiting Luangwa Health Approved
15 | Construction of an Ablution Block at Mpima Prison Primary | MPIMA Approved
17 | Additional Roads Funding for roads ALL Transport Approved =
19 | Additional funding for Kasanda Market Justin Commerce and | Approved =
20 | Proposed Construction of a 1x4 Classroom block ( CRB) at

## 67. Save the Extracted OCR Project Lines

The project lines identified from the OCR output are saved as a separate UTF-8 text file in the `data/raw/CDF` directory.

This intermediate file preserves the lines identified as potential CDF project records before they are transformed into the final structured dataset.

Saving this intermediate extraction artifact supports reproducibility and provides an additional reference for verifying the manually structured project records against the OCR output.

In [ ]:
ocr_rows_path = (
    RAW_DIR /
    "2025_kabwe_central_cdf_project_rows_ocr.txt"
)

with open(ocr_rows_path, "w", encoding="utf-8") as file:
    for i, line in enumerate(project_lines, start=1):
        file.write(f"{i}: {line}\n")

print("Saved:", ocr_rows_path)

Saved: ../data/raw/2025_kabwe_central_cdf_project_rows_ocr.txt


## 68. Extract OCR Text with Positional Information

Tesseract OCR is used to extract text from the first page together with its positional information.

Unlike `image_to_string()`, `image_to_data()` provides the location of each recognised text element on the page, including its horizontal and vertical position and width.

The extracted OCR data is cleaned by removing missing and empty text values and sorted according to its position on the page.

The positional information can be used to help reconstruct the structure of the scanned project table during the data preparation process.

In [ ]:
from pytesseract import Output

ocr_data = pytesseract.image_to_data(
    pages[0],
    config="--psm 6",
    output_type=Output.DATAFRAME
)

ocr_data = ocr_data.dropna(subset=["text"])

ocr_data["text"] = (
    ocr_data["text"]
    .astype(str)
    .str.strip()
)

ocr_data = ocr_data[ocr_data["text"] != ""]

ocr_data = ocr_data.sort_values(
    ["top", "left"]
)



NameError: name 'pages' is not defined

## 69. Extract Project Numbers from OCR Records

The project numbers are extracted from the OCR-filtered project lines using a regular expression.

The pattern identifies numeric values at the beginning of each project line and converts them to integers.

The extracted project numbers provide a structured identifier for the project records and can be used to validate that the expected project sequence was captured from the scanned document.

In [ ]:
project_numbers = []

for line in project_lines:
    match = re.match(r"^(\d+)", line)

    if match:
        project_numbers.append(int(match.group(1)))

print(project_numbers)

[1, 7, 4, 7, 10, 11, 12, 13, 14, 15, 17, 19, 20, 21, 22, 23, 24, 26, 27, 28, 29, 30, 31, 32, 33]


## 70. Identify Project Numbers Using OCR Coordinates

OCR positional information is used to identify project numbers in the `NO` column of the scanned CDF table.

The OCR results are filtered to retain numeric text located within the horizontal region corresponding to the project-number column.

This approach uses both the recognised text and its position on the page to help reconstruct structured project records from the scanned document.

In [ ]:
# Recreate OCR data for page 1
page1_ocr = pytesseract.image_to_data(
    pages[0],
    config="--psm 6",
    output_type=Output.DATAFRAME
)

page1_ocr = page1_ocr.dropna(subset=["text"])

page1_ocr["text"] = (
    page1_ocr["text"]
    .astype(str)
    .str.strip()
)

page1_ocr = page1_ocr[
    page1_ocr["text"] != ""
]

# Find numbers in the NO column
page1_numbers = page1_ocr[
    (page1_ocr["left"] < 210) &
    (page1_ocr["text"].str.match(r"^\d+$", na=False))
]

 top  left text
 587   111    1
1026    76    7
1296   100    4
2177    93    7


## 71. Create a Working DataFrame from OCR Records

The OCR-filtered project lines are converted into a pandas DataFrame for further processing.

Each extracted line is assigned a sequential `ocr_row` identifier, while the original OCR text is preserved in the `ocr_text` column.

This working structure provides an intermediate representation of the scanned project records before the OCR text is manually verified and transformed into the final structured CDF dataset.

In [ ]:
working_df = pd.DataFrame({
    "ocr_row": range(1, len(project_lines) + 1),
    "ocr_text": project_lines
})

working_df

,ocr_row,ocr_text
0,1,1 Proposed Construction of a standard Maternit...
1,2,7 Proposed Construction of a 1x3 Classroom blo...
2,3,4 | Proposed Construction and installation of ...
3,4,7 | Proposed Construction and installation of ...
4,5,10 | Proposed Construction of an Ablution bloc...
5,6,11 | Procurement of a Hydraulic Tipper Truck A...
6,7,12 | Proposed extension and rehabilitation of ...
7,8,13 | Proposed Completion of Nakoli Market shel...
8,9,14 | Completion of Kabwe General Hospital’s Re...
9,10,15 | Construction of an Ablution Block at Mpim...


## 72. Extract Project Numbers from OCR Text

The project number is extracted from the beginning of each OCR project line using a regular expression.

The extracted number is stored in a separate `no_ocr` column, while the original OCR text is preserved in `ocr_text`.

Separating the project number from the OCR text creates a structured identifier that can be used to organise, verify, and match the extracted project records during subsequent data preparation.

In [ ]:
working_df["no_ocr"] = working_df["ocr_text"].str.extract(
    r"^(\d+)"
)[0]

print(working_df.columns.tolist())

['ocr_row', 'ocr_text', 'no_ocr']


## 73. Separate Project Numbers from OCR Text

The project number is removed from the beginning of each OCR record using a regular expression.

The remaining text is stored in the `project_text` column, while the extracted project number remains in `no_ocr` and the original OCR text is preserved in `ocr_text`.

This creates a cleaner intermediate representation of each project record and prepares the OCR text for further parsing and verification.

In [ ]:
working_df["project_text"] = (
    working_df["ocr_text"]
    .str.replace(r"^\d+\s*", "", regex=True)
    .str.strip()
)

working_df[
    ["ocr_row", "no_ocr", "project_text"]
]

,ocr_row,no_ocr,project_text
0,1,1,Proposed Construction of a standard Maternity ...
1,2,7,Proposed Construction of a 1x3 Classroom block...
2,3,4,| Proposed Construction and installation of 02...
3,4,7,| Proposed Construction and installation of 02...
4,5,10,| Proposed Construction of an Ablution block a...
5,6,11,| Procurement of a Hydraulic Tipper Truck All ...
6,7,12,| Proposed extension and rehabilitation of Way...
7,8,13,| Proposed Completion of Nakoli Market shelter...
8,9,14,| Completion of Kabwe General Hospital’s Relat...
9,10,15,| Construction of an Ablution Block at Mpima P...


## 74. Clean OCR Formatting Artifacts

The extracted project text is cleaned to remove trailing characters such as `=` and em dashes that may have been introduced during OCR.

After removing these artifacts, surrounding whitespace is stripped from the text.

This step improves the consistency of the OCR-derived project text while preserving the substantive information extracted from the source document.

In [ ]:
working_df["project_text"] = (
    working_df["project_text"]
    .str.replace(r"\s*[=—]+\s*$", "", regex=True)
    .str.strip()
)

working_df[
    ["ocr_row", "no_ocr", "project_text"]
]

,ocr_row,no_ocr,project_text
0,1,1,Proposed Construction of a standard Maternity ...
1,2,7,Proposed Construction of a 1x3 Classroom block...
2,3,4,| Proposed Construction and installation of 02...
3,4,7,| Proposed Construction and installation of 02...
4,5,10,| Proposed Construction of an Ablution block a...
5,6,11,| Procurement of a Hydraulic Tipper Truck All ...
6,7,12,| Proposed extension and rehabilitation of Way...
7,8,13,| Proposed Completion of Nakoli Market shelter...
8,9,14,| Completion of Kabwe General Hospital’s Relat...
9,10,15,| Construction of an Ablution Block at Mpima P...


## 75. Reconstruct Project Rows Using OCR Coordinates

A reusable function is defined to reconstruct text belonging to a specific project row from the positional OCR data.

The function filters OCR elements using their vertical (`top`) and horizontal (`left`) coordinates, sorts the elements according to their position on the page, and combines the recognised text into a single string.

The function is tested on the first project row to verify that the OCR elements can be grouped into a coherent project record.

This approach helps transform the scanned table's positional OCR output into structured information for subsequent verification.

In [ ]:
def get_row_text(data, start_top, end_top, min_left=0, max_left=3000):
    row = data[
        (data["top"] >= start_top) &
        (data["top"] < end_top) &
        (data["left"] >= min_left) &
        (data["left"] < max_left)
    ].copy()

    row = row.sort_values(["top", "left"])

    return " ".join(row["text"].tolist())


# Test with the first project
row1_text = get_row_text(
    page1_ocr,
    500,
    780,
    50,
    3000
)

print(row1_text)

standard of Maternity Mpima Annex Proposed Construction at 1 a health Mpima center


In [ ]:
# Page 5
page5 = pages[4]

# Row ranges for Page 5
row_ranges_p5 = [
    (30, 150, 440),
    (31, 440, 690),
    (32, 690, 1150),
    (33, 1150, 2400),
]

all_rows_p5 = []

for row_number, y1, y2 in row_ranges_p5:
    row_data = {"no": row_number}

    for column_name, (x1, x2) in columns.items():

        if column_name == "NO":
            continue

        crop = page5.crop((x1, y1, x2, y2))

        text = pytesseract.image_to_string(
            crop,
            config="--psm 6"
        )

        text = text.replace("|", " ")
        text = re.sub(r"\s+", " ", text)
        text = text.strip()

        row_data[column_name.lower()] = text

    all_rows_p5.append(row_data)

page5_df = pd.DataFrame(all_rows_p5)

print(page5_df.to_string(index=False))

 no                                                                                   project             ward       sector                                                                                                                                                 comment
 30                 ) Procurement of equipment for Magandanyama Maternit Annex and laboratory    y oavd Ramust    Health 10                                                                                              Not approved due to Insuffi funds to cater for all project
 31                                        Procurement of equipment for Mpima Maternity Annex            Mpima       Health                                                                            Not approved as another pr under the health sector was approved in this ward
 32                                      Procurement of equipment for Kamakuti Maternity Anne           x Waya       Health Not approved due to Insuffi funds to cater for a

In [ ]:
# Combine all 33 extracted rows from Pages 1–5

proposed_2025_df = pd.concat(
    [
        page1_df,
        page2_df,
        page3_df,
        page4_df,
        page5_df
    ],
    ignore_index=True
)

# Sort by project number
proposed_2025_df = proposed_2025_df.sort_values(
    by="no"
).reset_index(drop=True)

print("Total rows:", len(proposed_2025_df))
print("Total columns:", len(proposed_2025_df.columns))

display(proposed_2025_df)

Total rows: 27
Total columns: 5


,no,project,ward,sector,comment
0,1,Proposed Construction of a standard Maternity ...,Mpima,Health,Approved
1,2,Proposed Construction of a 1x4 Classroom block...,High ridge,Education,Approved
2,3,Proposed Construction and installation of 02 S...,Waya,Water and Sanitation,Approved
3,4,Proposed Construction and installation of 02 S...,Kalonga,Water and Sanitation,Approved
4,10,Proposed Construction of an Ablution block at ...,Katondo,Education,Approved
5,11,Procurement of a Hydraulic Tipper Truck,,Transport,Approved
6,12,sinning Proposed extension and rehabilitation ...,Waya,Commerce ai Trade,nd Approved
7,13,Proposed Completion of Nakoli Market shelter,Nakoli,Commerce al Trade,id Approved
8,14,Completion of Kabwe General Hospital’s Relativ...,Luangwa,Health,Approved
9,15,Construction of an Ablution Block at Mpima Pri...,y MPIMA,Education,Approved


In [ ]:
verified_1_33 = [
    [1, "Proposed Construction of a standard Maternity Annex at Mpima health center",
     "Mpima", "Health", "Approved"],

    [2, "Proposed Construction of a 1x4 Classroom block (CRB) at Lukanga Secondary School",
     "High ridge", "Education", "Approved"],

    [3, "Proposed Construction of a 1x3 Classroom block (CRB) at Kasanda Malombe Secondary School",
     "Chirwa", "Education", "Approved"],

    [4, "Proposed Construction and installation of 02 Solar powered Water reticulated Systems with lockable kiosks in Waya communities",
     "Waya", "Water and Sanitation", "Approved"],

    [5, "Proposed Construction and installation of 02 Solar powered Water reticulated Systems at Kamushanga Market Shelter",
     "Kalonga", "Water and Sanitation", "Approved"],

    [6, "Proposed Construction and installation of 02 Solar powered Water reticulated Systems at C-gate Primary School",
     "Kaputula", "Education", "Approved"],

    [7, "Proposed Construction and installation of 02 Solar powered Water reticulated Systems at Njanji Market facilities",
     "Njanji", "Education", "Approved"],

    [8, "Proposed Construction and installation of 02 Solar powered Water reticulated Systems at Kamakuti Maternity annex",
     "Waya", "Health", "Approved"],

    [9, "Proposed Construction of an Ablution block at C-gate Primary School",
     "Kaputula", "Education", "Approved"],

    [10, "Proposed Construction of an Ablution block at BOCCs Primary School",
     "Katondo", "Education", "Approved"],

    [11, "Procurement of a Hydraulic Tipper Truck",
     "All", "Transport", "Approved"],

    [12, "Proposed extension and rehabilitation of Waya market shelter",
     "Waya", "Commerce and Trade", "Approved"],

    [13, "Proposed Completion of Nakoli Market shelter",
     "Nakoli", "Commerce and Trade", "Approved"],

    [14, "Completion of Kabwe General Hospital's Relative waiting Shelter",
     "Luangwa", "Health", "Approved"],

    [15, "Construction of an Ablution Block at Mpima Prison Primary School",
     "MPIMA", "Education", "Approved"],

    [16, "Procurement of 500 ordinary desks and 40 special desks",
     "ALL", "Education", "Approved"],

    [17, "Additional Roads Funding for roads",
     "ALL", "Transport", "Approved"],

    [18, "Additional funding for Njanii Market structure",
     "NJANI", "Commerce and Trade", "Approved"],

    [19, "Additional funding for Kasanda Market",
     "Justin Kabwe", "Commerce and Trade", "Approved"],

    [20, "Proposed Construction of a 1x4 Classroom block (CRB) at Mpima Dairy School",
     "Mpima", "Education",
     "Not approved because the ward already got an allocation for one project, hence under the principal of equity, resources had to be evenly distributed to cater for all projects"],

    [21, "Proposed Completion of a 1x2 Science Laboratory at Mine Secondary School",
     "Justine Kabwe", "Education",
     "Not approved due to Insufficient funds to cater for all projects and priority was given to projects with wider community benefit or urgent needs"],

    [22, "Proposed Construction of a 1x3 Classroom block (CRB) at Kabwe Trust Primary School",
     "Luangwa", "Education",
     "Not approved due to Insufficient funds to cater for all projects"],

    [23, "Proposed Completion of the double storey classroom block at Family Future Community School",
     "Lukanga", "Education",
     "Not approved due to Insufficient funds to cater for all projects"],

    [24, "Proposed Construction of an Ablution block at Gombe Secondary School",
     "Waya", "Education",
     "Not approved as Waya ward had a project under educational sector which was approved, since resources had to be evenly distributed in all the 14 wards."],

    [25, "Proposed Construction of an Ablution block at Kakama Primary School",
     "Kalonga", "Education",
     "A project was already approved under this ward, and for equity sake, other wards had to be considered"],

    [26, "Proposed Construction of an Ablution block at Kamushanga Market shelter",
     "Kalonga", "Water and Sanitation",
     "Kalonga has had a one or two projects approved, hence the need to consider other wards amongst the 14."],

    [27, "Supply and installation of a Micro Burn Unit at Nakoli Clinic",
     "Nakoli", "Health",
     "Not approved due to Insufficient funds and furthermore, CDF prioritizes primary health care needs like OPD, Maternity annexes and no records showed a high incidence of burn cases in the catchment area"],

    [28, "Completion of Chindwin barrack school wall fence",
     "KAPUTULA", "Education",
     "Not approved due to Insufficient funds and furthermore, the need is not priority compared to other needs, as the school is next to a defense unit that currently provides security."],

    [29, "Procurement of equipment for Katondo Maternity Annex",
     "Katondo", "Health",
     "Not approved as the ward was already considered for a project under the educational sector"],

    [30, "Procurement of equipment for Magandanyama Maternity Annex and laboratory",
     "David Ramusho", "Health",
     "Not approved due to Insufficient funds to cater for all projects"],

    [31, "Procurement of equipment for Mpima Maternity Annex",
     "Mpima", "Health",
     "Not approved as another project under the health sector was already approved in this ward"],

    [32, "Procurement of equipment for Kamakuti Maternity Annex",
     "Waya", "Health",
     "Not approved due to Insufficient funds to cater for all projects - There was no needs assessment report from the department of Health and usage data to justify the equipment requested"],

    [33, "Proposed Construction of a 1x4 Classroom block (CRB) at David Ramusho Secondary School",
     "David Ramusho", "Education",
     "Not approved due to Insufficient funds and also this ward has a project approved already, hence the need to consider other wards"],
]

verified_df = pd.DataFrame(
    verified_1_33,
    columns=["no", "project", "ward", "sector", "comment"]
)

display(verified_df)

,no,project,ward,sector,comment
0,1,Proposed Construction of a standard Maternity ...,Mpima,Health,Approved
1,2,Proposed Construction of a 1x4 Classroom block...,High ridge,Education,Approved
2,3,Proposed Construction of a 1x3 Classroom block...,Chirwa,Education,Approved
3,4,Proposed Construction and installation of 02 S...,Waya,Water and Sanitation,Approved
4,5,Proposed Construction and installation of 02 S...,Kalonga,Water and Sanitation,Approved
5,6,Proposed Construction and installation of 02 S...,Kaputula,Education,Approved
6,7,Proposed Construction and installation of 02 S...,Njanji,Education,Approved
7,8,Proposed Construction and installation of 02 S...,Waya,Health,Approved
8,9,Proposed Construction of an Ablution block at ...,Kaputula,Education,Approved
9,10,Proposed Construction of an Ablution block at ...,Katondo,Education,Approved


## 75. Clean and Standardise the Verified 2025 CDF Dataset

The manually verified 2025 CDF project records are copied into a separate DataFrame for final cleaning and standardisation.

The preprocessing includes:

- Removing unnecessary whitespace from text fields.
- Standardising ward names to a consistent representation.
- Correcting common transcription variations in sector names.
- Preserving the verified project information while improving consistency across records.

The cleaning rules are applied consistently to the relevant text fields so that the resulting dataset is suitable for analysis and export.

Only a preview of the cleaned dataset is displayed to keep the notebook output concise.

In [ ]:
# Standardize the verified 2025 CDF dataset

clean_2025_df = verified_df.copy()

# Clean whitespace in all text columns
text_columns = ["project", "ward", "sector", "comment"]

for column in text_columns:
    clean_2025_df[column] = (
        clean_2025_df[column]
        .astype(str)
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
    )

# Standardize ward names
ward_mapping = {
    "ALL": "ALL",
    "All": "ALL",
    "MPIMA": "Mpima",
    "NJANI": "Njanji",
    "KAPUTULA": "Kaputula",
}

clean_2025_df["ward"] = clean_2025_df["ward"].replace(ward_mapping)

# Standardize common sector transcription variations
sector_mapping = {
    "Commerce ai Trade": "Commerce and Trade",
    "Commerce al Trade": "Commerce and Trade",
    "Commerce = Trade": "Commerce and Trade",
}

clean_2025_df["sector"] = clean_2025_df["sector"].replace(sector_mapping)

# Display a preview of the cleaned dataset
clean_2025_df.head()

,no,project,ward,sector,comment
0,1,Proposed Construction of a standard Maternity ...,Mpima,Health,Approved
1,2,Proposed Construction of a 1x4 Classroom block...,High ridge,Education,Approved
2,3,Proposed Construction of a 1x3 Classroom block...,Chirwa,Education,Approved
3,4,Proposed Construction and installation of 02 S...,Waya,Water and Sanitation,Approved
4,5,Proposed Construction and installation of 02 S...,Kalonga,Water and Sanitation,Approved
5,6,Proposed Construction and installation of 02 S...,Kaputula,Education,Approved
6,7,Proposed Construction and installation of 02 S...,Njanji,Education,Approved
7,8,Proposed Construction and installation of 02 S...,Waya,Health,Approved
8,9,Proposed Construction of an Ablution block at ...,Kaputula,Education,Approved
9,10,Proposed Construction of an Ablution block at ...,Katondo,Education,Approved


## 76. Validate the Cleaned 2025 CDF Dataset

A set of data-quality checks is performed on the cleaned 2025 CDF project dataset.

The validation checks:

- The number of records and columns.
- Column names and overall structure.
- Missing values in each field.
- Empty text values in important descriptive fields.
- Exact duplicate records.
- Duplicate project numbers.
- The range of project numbers captured from the source.
- The distribution of projects across sectors.
- The distribution of projects across wards.

These checks help confirm that the verified project records were preserved correctly during cleaning and that no obvious duplication or completeness problems were introduced during preprocessing.

The validation results provide evidence that the dataset is structurally consistent and ready for final export.

In [ ]:
# Validate the cleaned 2025 CDF dataset

print("Rows:", len(clean_2025_df))
print("Columns:", len(clean_2025_df.columns))

print("\nColumn names:")
print(clean_2025_df.columns.tolist())

print("\nMissing values:")
print(clean_2025_df.isna().sum())

print("\nEmpty text values:")
for column in ["project", "ward", "sector", "comment"]:
    empty_count = (
        clean_2025_df[column]
        .astype(str)
        .str.strip()
        .eq("")
        .sum()
    )
    print(f"{column}: {empty_count}")

duplicate_rows = clean_2025_df.duplicated().sum()
print("\nDuplicate rows:", duplicate_rows)

duplicate_project_numbers = clean_2025_df["no"].duplicated(keep=False).sum()
print("Duplicate project numbers:", duplicate_project_numbers)

if duplicate_project_numbers > 0:
    print("\nDuplicate project numbers found:")
    print(
        clean_2025_df[
            clean_2025_df["no"].duplicated(keep=False)
        ][["no", "project"]]
    )

print("\nProject number range:")
print(
    f"{clean_2025_df['no'].min()} to "
    f"{clean_2025_df['no'].max()}"
)

print("\nSector values:")
print(clean_2025_df["sector"].value_counts())

print("\nWard values:")
print(clean_2025_df["ward"].value_counts())

Rows: 33
Columns: 5

Column names:
['no', 'project', 'ward', 'sector', 'comment']

Missing values:
no         0
project    0
ward       0
sector     0
comment    0
dtype: int64

Empty text values:
project: 0
ward: 0
sector: 0
comment: 0

Duplicate rows: 0

Duplicate project numbers:
Empty DataFrame
Columns: [no, project]
Index: []

Project numbers present:
[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33]

Sector values:
sector
Education               16
Health                   8
Commerce and Trade       4
Water and Sanitation     3
Transport                2
Name: count, dtype: int64

Ward values:
ward
Waya             5
Mpima            4
Kalonga          3
Kaputula         3
ALL              3
Njanji           2
Katondo          2
Nakoli           2
Luangwa          2
David Ramusho    2
High ridge       1
Chirwa           1
Justin Kabwe     1
Justine Kabwe    1
Lukanga          1
Name: count, dtype: int64


## 77. Add Metadata to the 2025 CDF Dataset

Metadata describing the year, constituency, and source of the records is added to the cleaned 2025 CDF project dataset.

The `year` and `constituency` fields provide important contextual information about the project records.

The `source_url` field stores the official Kabwe Municipal Council document URL, while `source_document` identifies the specific publication used to obtain the records.

Including source metadata improves the traceability and reproducibility of the dataset and allows users of the final data to identify the original source document.

A preview of the final dataset structure is displayed, while the total number of rows and columns is reported as a final structural check.

In [ ]:
# Add metadata to the verified 2025 CDF dataset

final_2025_df = clean_2025_df.copy()

final_2025_df.insert(1, "year", 2025)
final_2025_df.insert(2, "constituency", "Kabwe Central")

final_2025_df["source_url"] = (
    "https://www.kabwecouncil.gov.zm/"
    "wp-content/uploads/2025/08/"
    "Proposed-2025-CDF-projects.pdf"
)

final_2025_df["source_document"] = (
    "Proposed 2025 CDF Projects"
)

# Display a preview of the final structure
display(final_2025_df.head())

print("Rows:", len(final_2025_df))
print("Columns:", len(final_2025_df.columns))

,no,year,constituency,project,ward,sector,comment,source_url,source_document
0,1,2025,Kabwe Central,Proposed Construction of a standard Maternity ...,Mpima,Health,Approved,https://www.kabwecouncil.gov.zm/wp-content/upl...,Proposed 2025 CDF Projects
1,2,2025,Kabwe Central,Proposed Construction of a 1x4 Classroom block...,High ridge,Education,Approved,https://www.kabwecouncil.gov.zm/wp-content/upl...,Proposed 2025 CDF Projects
2,3,2025,Kabwe Central,Proposed Construction of a 1x3 Classroom block...,Chirwa,Education,Approved,https://www.kabwecouncil.gov.zm/wp-content/upl...,Proposed 2025 CDF Projects
3,4,2025,Kabwe Central,Proposed Construction and installation of 02 S...,Waya,Water and Sanitation,Approved,https://www.kabwecouncil.gov.zm/wp-content/upl...,Proposed 2025 CDF Projects
4,5,2025,Kabwe Central,Proposed Construction and installation of 02 S...,Kalonga,Water and Sanitation,Approved,https://www.kabwecouncil.gov.zm/wp-content/upl...,Proposed 2025 CDF Projects
5,6,2025,Kabwe Central,Proposed Construction and installation of 02 S...,Kaputula,Education,Approved,https://www.kabwecouncil.gov.zm/wp-content/upl...,Proposed 2025 CDF Projects
6,7,2025,Kabwe Central,Proposed Construction and installation of 02 S...,Njanji,Education,Approved,https://www.kabwecouncil.gov.zm/wp-content/upl...,Proposed 2025 CDF Projects
7,8,2025,Kabwe Central,Proposed Construction and installation of 02 S...,Waya,Health,Approved,https://www.kabwecouncil.gov.zm/wp-content/upl...,Proposed 2025 CDF Projects
8,9,2025,Kabwe Central,Proposed Construction of an Ablution block at ...,Kaputula,Education,Approved,https://www.kabwecouncil.gov.zm/wp-content/upl...,Proposed 2025 CDF Projects
9,10,2025,Kabwe Central,Proposed Construction of an Ablution block at ...,Katondo,Education,Approved,https://www.kabwecouncil.gov.zm/wp-content/upl...,Proposed 2025 CDF Projects


Rows: 33
Columns: 9


## 78. Perform Final Quality Checks

Final quality-control checks are performed on the completed 2025 CDF dataset before export.

The checks confirm:

- The final number of records and columns.
- The final column structure.
- Missing values across all fields.
- The absence of exact duplicate records.
- The completeness and uniqueness of project numbers 1–33.
- The year and constituency values assigned to the dataset.
- Potential inconsistencies in ward names that may require attention.

The project-number check is particularly important because the 2025 project records were recovered from a scanned PDF using OCR and subsequently manually verified.

The ward-name check is used to identify possible inconsistencies in the source data. These values are not automatically changed unless they can be confirmed from the original source.

These checks provide a final quality-assurance step before the dataset is exported.

In [ ]:
# Final quality checks for the 2025 CDF dataset

print("Rows:", len(final_2025_df))
print("Columns:", len(final_2025_df.columns))

print("\nColumns:")
print(final_2025_df.columns.tolist())

print("\nMissing values:")
print(final_2025_df.isna().sum())

print("\nDuplicate rows:", final_2025_df.duplicated().sum())

# Verify that project numbers 1–33 are present exactly once
expected_numbers = list(range(1, 34))
actual_numbers = final_2025_df["no"].tolist()

project_numbers_complete = (
    sorted(actual_numbers) == expected_numbers
    and final_2025_df["no"].is_unique
)

print("\nProject numbers 1–33 complete:", project_numbers_complete)

print("\nYear values:")
print(final_2025_df["year"].unique())

print("\nConstituency values:")
print(final_2025_df["constituency"].unique())

print("\nPotentially inconsistent ward names:")
print(
    final_2025_df[
        final_2025_df["ward"].str.contains(
            "Justin|Justine",
            case=False,
            na=False
        )
    ][["no", "project", "ward"]]
)

Rows: 33
Columns: 9

Columns:
['no', 'year', 'constituency', 'project', 'ward', 'sector', 'comment', 'source_url', 'source_document']

Missing values:
no                 0
year               0
constituency       0
project            0
ward               0
sector             0
comment            0
source_url         0
source_document    0
dtype: int64

Duplicate rows: 0

Project numbers complete:
True

Years:
[2025]

Constituency:
<StringArray>
['Kabwe Central']
Length: 1, dtype: str

Ward names:
['ALL', 'Chirwa', 'David Ramusho', 'High ridge', 'Justin Kabwe', 'Justine Kabwe', 'Kalonga', 'Kaputula', 'Katondo', 'Luangwa', 'Lukanga', 'Mpima', 'Nakoli', 'Njanji', 'Waya']

Potentially inconsistent ward names:
    no                                            project           ward
18  19              Additional funding for Kasanda Market   Justin Kabwe
20  21  Proposed Completion of a 1x2 Science Laborator...  Justine Kabwe


## 79. Export the Final 2025 CDF Dataset

The final verified and cleaned 2025 CDF project dataset is exported as a CSV file using the pipe (`|`) delimiter required by the assignment.

The filename follows the prescribed naming convention:

`db-unza26-csc4792-kabwe_cdf_projects_2025.csv`

The dataset is saved in the `data/processed/CDF` directory, while the original source PDF and intermediate OCR files remain preserved separately.

After export, the CSV is read back into pandas using the same pipe delimiter. This provides an additional validation that the file was written successfully and that the expected number of records and columns was preserved.

The final processed dataset contains the structured 2025 CDF project records together with contextual and source metadata.

In [ ]:
# Export the final 2025 CDF dataset as a pipe-delimited CSV

output_path = (
    PROCESSED_DIR /
    "db-unza26-csc4792-kabwe_cdf_projects_2025.csv"
)

final_2025_df.to_csv(
    output_path,
    sep="|",
    index=False,
    encoding="utf-8"
)

# Reload the exported file to verify the saved structure
exported_2025_df = pd.read_csv(
    output_path,
    sep="|"
)

print("CSV exported successfully.")
print("File:", output_path)
print("Rows:", len(exported_2025_df))
print("Columns:", len(exported_2025_df.columns))
print("Delimiter: |")

CSV exported successfully.
File: ../data/processed/db-unza26-csc4792-kabwe_cdf_projects_2025.csv
Rows: 33
Columns: 9


## 80. Inspect the Combined 2024 CDF Project Dataset

The combined 2024 CDF project dataset is inspected after integrating the cleaned records from Bwacha and Kabwe Central constituencies.

The inspection confirms:

- The total number of project records and columns.
- The final column structure.
- The distribution of records between the two constituencies.
- The year represented in the integrated dataset.
- A sample of the combined records.

This provides an initial structural check that both constituency datasets were successfully integrated into the final 2024 CDF project dataset.

In [ ]:
# Inspect the combined 2024 CDF projects dataset

print("Rows:", len(kabwe_cdf_projects))
print("Columns:", len(kabwe_cdf_projects.columns))

print("\nColumns:")
print(kabwe_cdf_projects.columns.tolist())

print("\nConstituency distribution:")
print(kabwe_cdf_projects["constituency"].value_counts())

print("\nYear distribution:")
print(kabwe_cdf_projects["year"].value_counts())

print("\nFirst 5 rows:")
display(kabwe_cdf_projects.head())

NameError: name 'kabwe_cdf_projects' is not defined

## 81. Extract 2024 Bwacha CDF Project Records

The 2024 Bwacha CDF project document is processed using `pdfplumber` to extract the tables containing the project records.

The extraction process:

1. Opens the original PDF source document.
2. Processes each page and extracts detected tables.
3. Collects the extracted table rows.
4. Filters the rows to retain records whose first field contains a numeric project number.
5. Converts the validated project rows into a pandas DataFrame using descriptive column names.

Filtering based on the project number helps remove table headers and other non-project rows that may be returned during PDF table extraction.

The resulting DataFrame provides the structured project records that will be cleaned and enriched with year, constituency, and source metadata in subsequent steps.

In [ ]:
import pdfplumber
import pandas as pd

# Extract tables from the 2024 Bwacha CDF PDF

all_rows_bwacha = []

with pdfplumber.open(bwacha_path) as pdf:
    for page in pdf.pages:
        tables = page.extract_tables()

        for table in tables:
            for row in table:
                if row:
                    all_rows_bwacha.append(row)

# Keep rows whose first cell contains a project number
project_rows_bwacha = []

for row in all_rows_bwacha:
    if (
        row[0] is not None
        and str(row[0]).strip().isdigit()
    ):
        project_rows_bwacha.append(row)

print("Project rows extracted:", len(project_rows_bwacha))

# Create DataFrame
bwacha_df = pd.DataFrame(
    project_rows_bwacha,
    columns=[
        "project_number",
        "project_name",
        "project_description",
        "ward",
        "project_site",
        "application_amount",
        "engineers_estimate",
        "approved_amount",
        "contract_amount",
        "status"
    ]
)

display(bwacha_df.head())

Number of pages: 5
Page 1: 1 table(s) found
Page 2: 1 table(s) found
Page 3: 1 table(s) found
Page 4: 1 table(s) found
Page 5: 1 table(s) found

Project rows extracted: 36


,project_number,project_name,project_description,ward,project_site,application_amount,engineers_estimate,approved_amount,contract_amount,status
0,1,Construction of 1X3 Classroom\nBlock and Teach...,Construction of 1X3\nClassroom Block and\nTeac...,Kangomba,Kangomba\nward,,,,,
1,2,Construction of 1X4 Classroom\nBlock at Kagomb...,Construction of 1X4\nClassroom Block at Kagomb...,Kangomba,Kangomba\nPrimary\nSchool,,,,,
2,3,Construction of 1X4 Classroom\nBlock at Kagomb...,Construction of 1X4\nClassroom Block at Kagomb...,Kangomba,Kangomba\nPrimary\nSchool,,,,,
3,4,Construction of 1X3 Classroom\nBlock at Mine P...,Construction of 1X3\nClassroom Block at Mine\n...,Kangomba,Mine Primary\nSchool,,,,,
4,5,"Repairing of the Mono-Pump,\nConstruction of T...","Repairing of the Mono-Pump,\nConstruction of T...",Kangomba,Mary Chidgey\nCommunity\nSchool,,,,,


## 82. Validate the Extracted Bwacha Dataset Structure

The structure of the extracted Bwacha CDF dataset is checked before proceeding to data cleaning.

The validation confirms the number of extracted project records, the number of fields, and the column names assigned during the extraction process.

This provides an initial structural check that the PDF table was converted into the expected tabular format.

In [ ]:
print("Rows:", len(bwacha_df))
print("Columns:", len(bwacha_df.columns))

print("\nColumn names:")
print(bwacha_df.columns.tolist())

Rows: 36
Columns: 10

Column names:
['project_number', 'project_name', 'project_description', 'ward', 'project_site', 'application_amount', 'engineers_estimate', 'approved_amount', 'contract_amount', 'status']

First 10 rows:


,project_number,project_name,project_description,ward,project_site,application_amount,engineers_estimate,approved_amount,contract_amount,status
0,1,Construction of 1X3 Classroom\nBlock and Teach...,Construction of 1X3\nClassroom Block and\nTeac...,Kangomba,Kangomba\nward,,,,,
1,2,Construction of 1X4 Classroom\nBlock at Kagomb...,Construction of 1X4\nClassroom Block at Kagomb...,Kangomba,Kangomba\nPrimary\nSchool,,,,,
2,3,Construction of 1X4 Classroom\nBlock at Kagomb...,Construction of 1X4\nClassroom Block at Kagomb...,Kangomba,Kangomba\nPrimary\nSchool,,,,,
3,4,Construction of 1X3 Classroom\nBlock at Mine P...,Construction of 1X3\nClassroom Block at Mine\n...,Kangomba,Mine Primary\nSchool,,,,,
4,5,"Repairing of the Mono-Pump,\nConstruction of T...","Repairing of the Mono-Pump,\nConstruction of T...",Kangomba,Mary Chidgey\nCommunity\nSchool,,,,,
5,6,Construction of a Primary\nSchool in Kawama ward,Construction of a Primary\nSchool in Kawama ward,Kawama,Kawama ward,,,,,
6,7,Construction of 10 Teachers\nHouses at Chitaka...,Construction of 10 Teachers\nHouses at Chitaka...,Muwowo\nEast,Chitakata\nCommnuity\nSchool,,,,,
7,8,Construction of 1x4 Classroom\nBlock at Mukobe...,Construction of 1x4 Classroom\nBlock at Mukobe...,Muwowo\nEast,Mukobeko\nCorrectional\nDay\nSecondary\nSchool,,,,,
8,9,Construction of Youth\nResource Centre,Construction of Youth\nResource Centre,Bwacha,Bwacha ward,,,,,
9,10,Construction of a School Hall at\nRapheal Komb...,Construction of a School Hall\nat Rapheal Komb...,Chimaniman\ni,Rapheal\nKombe\nSecondary\nSchool,,,,,



Last 5 rows:


,project_number,project_name,project_description,ward,project_site,application_amount,engineers_estimate,approved_amount,contract_amount,status
31,5,Procurement of 8 Skip Bins in\nNgungu ward,Procurement of 8 Skip Bins in\nNgungu ward,Ngungu,Ngungu ward,,,,,
32,1,Rehabilitation of Muleya\nStadium,Rehabilitation of Muleya\nStadium,Bwacha,Bwacha ward,,,,,
33,1,Procurement of 2 Megaphones\nand 500 Garden Ch...,Procurement of 2\nMegaphones and 500 Garden\nC...,Chimaniman\ni,Chimanimani\nward,,,,,
34,2,Procurement of 4 Tents for\nFuneral Occassions...,Procurement of 4 Tents for\nFuneral Occassions...,Kawama,Kawama ward,,,,,
35,3,Construction of Council Office\n(For Councilor...,Construction of Council Office\n(For Councilor...,Kawama,Kawama ward,,,,,


# Make a copy so the original extracted dataframe remains unchanged

In [ ]:

clean_bwacha_df = bwacha_df.copy()

# Text columns extracted from the PDF
text_columns = [
    "project_name",
    "project_description",
    "ward",
    "project_site",
    "status"
]

# Remove line breaks, tabs, repeated spaces, and surrounding whitespace
for column in text_columns:
    clean_bwacha_df[column] = (
        clean_bwacha_df[column]
        .astype("string")
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
    )

print("Rows:", len(clean_bwacha_df))
print("Columns:", len(clean_bwacha_df.columns))

display(clean_bwacha_df.head(10))

Rows: 36
Columns: 10


,project_number,project_name,project_description,ward,project_site,application_amount,engineers_estimate,approved_amount,contract_amount,status
0,1,Construction of 1X3 Classroom Block and Teachers Houses in Kangomba ward,Construction of 1X3 Classroom Block and Teachers Houses in Kangomba ward - K...,Kangomba,Kangomba ward,,,,,
1,2,Construction of 1X4 Classroom Block at Kagomba Primary School,Construction of 1X4 Classroom Block at Kagomba Primary School,Kangomba,Kangomba Primary School,,,,,
2,3,Construction of 1X4 Classroom Block at Kagomba Primary School,Construction of 1X4 Classroom Block at Kagomba Primary School,Kangomba,Kangomba Primary School,,,,,
3,4,Construction of 1X3 Classroom Block at Mine Primary School,Construction of 1X3 Classroom Block at Mine Primary School - Mutwewansofu zone,Kangomba,Mine Primary School,,,,,
4,5,"Repairing of the Mono-Pump, Construction of Toilets for Pre- School Pupils, ...","Repairing of the Mono-Pump, Construction of Toilets for Pre- School Pupils, ...",Kangomba,Mary Chidgey Community School,,,,,
5,6,Construction of a Primary School in Kawama ward,Construction of a Primary School in Kawama ward,Kawama,Kawama ward,,,,,
6,7,Construction of 10 Teachers Houses at Chitakata Community School,Construction of 10 Teachers Houses at Chitakata Community School,Muwowo East,Chitakata Commnuity School,,,,,
7,8,Construction of 1x4 Classroom Block at Mukobeko Correctional Day Secondary S...,Construction of 1x4 Classroom Block at Mukobeko Correctional Day Secondary S...,Muwowo East,Mukobeko Correctional Day Secondary School,,,,,
8,9,Construction of Youth Resource Centre,Construction of Youth Resource Centre,Bwacha,Bwacha ward,,,,,
9,10,Construction of a School Hall at Rapheal Kombe Secondary School,Construction of a School Hall at Rapheal Kombe Secondary School,Chimaniman i,Rapheal Kombe Secondary School,,,,,


## 83. Clean and Standardise Financial Values

The financial fields extracted from the 2024 Bwacha CDF project document are cleaned and converted into numeric values.

The reusable `clean_amount()` function applies consistent preprocessing rules to the financial columns:

- Removes leading and trailing whitespace.
- Treats blank values and placeholders such as `N/A` and `-` as missing values.
- Removes currency symbols, commas, spaces, and other non-numeric characters.
- Converts valid financial values to numeric `float` values.

Missing values are preserved as missing rather than being replaced with zero. This is important because an unavailable value in the source document does not necessarily mean that the corresponding amount was zero.

The data types of the cleaned financial fields are then checked to confirm that the conversion was successful.

In [ ]:
financial_columns = [
    "application_amount",
    "engineers_estimate",
    "approved_amount",
    "contract_amount"
]

def clean_amount(value):
    value = str(value).strip()

    # Treat empty/missing values as missing data
    if value == "" or value.lower() in ["nan", "none", "n/a", "na", "-"]:
        return pd.NA

    # Remove currency symbols, commas, spaces and other non-numeric characters
    value = re.sub(r"[^0-9.\-]", "", value)

    if value == "":
        return pd.NA

    return float(value)


# Apply the cleaning function to all financial columns
for column in financial_columns:
    clean_bwacha_df[column] = clean_bwacha_df[column].apply(clean_amount)

print("Financial columns cleaned.")

# Check for remaining missing financial values
print("\nMissing financial values:")
print(clean_bwacha_df[financial_columns].isna().sum())

# Verify financial data types
print("\nFinancial data types:")
print(clean_bwacha_df[financial_columns].dtypes)

Financial columns cleaned.


,project_number,project_name,application_amount,engineers_estimate,approved_amount,contract_amount
0,1,Construction of 1X3 Classroom Block and Teachers Houses in Kangomba ward,<NA>,<NA>,<NA>,<NA>
1,2,Construction of 1X4 Classroom Block at Kagomba Primary School,<NA>,<NA>,<NA>,<NA>
2,3,Construction of 1X4 Classroom Block at Kagomba Primary School,<NA>,<NA>,<NA>,<NA>
3,4,Construction of 1X3 Classroom Block at Mine Primary School,<NA>,<NA>,<NA>,<NA>
4,5,"Repairing of the Mono-Pump, Construction of Toilets for Pre- School Pupils, ...",<NA>,<NA>,<NA>,<NA>
5,6,Construction of a Primary School in Kawama ward,<NA>,<NA>,<NA>,<NA>
6,7,Construction of 10 Teachers Houses at Chitakata Community School,<NA>,<NA>,<NA>,<NA>
7,8,Construction of 1x4 Classroom Block at Mukobeko Correctional Day Secondary S...,<NA>,<NA>,<NA>,<NA>
8,9,Construction of Youth Resource Centre,<NA>,<NA>,<NA>,<NA>
9,10,Construction of a School Hall at Rapheal Kombe Secondary School,<NA>,<NA>,<NA>,<NA>


## 84. Validate Bwacha Project Number Sequence

The project numbers in the cleaned Bwacha CDF dataset are checked to determine whether the extracted records form a complete sequential sequence.

The expected sequence is generated from 1 to the total number of extracted projects and compared with the project numbers in the dataset.

This check helps identify possible missing, duplicated, or incorrectly extracted project records before the dataset proceeds to further cleaning and integration.

In [ ]:
print("Number of projects:", len(clean_bwacha_df))

expected_numbers = list(range(1, len(clean_bwacha_df) + 1))

project_numbers_sequential = (
    clean_bwacha_df["project_number"].astype(int).tolist()
    == expected_numbers
)

print(
    "Project numbers sequential:",
    project_numbers_sequential
)

Project numbers:
['1', '2', '3', '4', '5', '6', '7', '8', '9', '10', '11', '12', '1', '2', '1', '1', '1', '2', '3', '4', '5', '6', '7', '8', '9', '1', '2', '1', '2', '3', '4', '5', '1', '1', '2', '3']

Number of projects: 36
Project numbers sequential: False


## 85. Check for Duplicate Bwacha Project Records

The cleaned Bwacha CDF dataset is checked for duplicate records as part of the data-quality validation process.

Two types of duplication are examined:

- **Complete duplicate rows**, where all field values are identical.
- **Duplicate project numbers**, where the same project identifier occurs more than once.

These checks help identify records that may have been duplicated during PDF extraction or subsequent data processing before the dataset is finalised.

In [ ]:
print("Duplicate complete rows:",
      clean_bwacha_df.duplicated().sum())

print("Duplicate project numbers:",
      clean_bwacha_df["project_number"].duplicated().sum())

Duplicate complete rows: 0
Duplicate project numbers: 24


## 86. Check for Empty and Missing Text Values

The cleaned Bwacha CDF dataset is checked for missing or empty values in the main text fields.

Both standard missing values (`NaN`) and empty strings are checked because they represent incomplete text information in different ways.

Leading and trailing whitespace is removed before checking for empty strings, ensuring that values containing only spaces are also identified.

This validation helps confirm the completeness of important descriptive fields before the dataset is finalised.

In [ ]:
for column in text_columns:
    empty_count = (
        clean_bwacha_df[column]
        .isna()
        .sum()
        +
        (clean_bwacha_df[column].astype("string").str.strip() == "").sum()
    )

    print(f"{column}: {empty_count} empty/missing")

project_name: 0 empty/missing
project_description: 0 empty/missing
ward: 0 empty/missing
project_site: 1 empty/missing
status: 36 empty/missing


In [ ]:
final_bwacha_df = clean_bwacha_df.copy()

final_bwacha_df.insert(1, "year", 2024)
final_bwacha_df.insert(2, "constituency", "Bwacha")

final_bwacha_df["source_url"] = (
    "https://www.kabwecouncil.gov.zm/"
    "wp-content/uploads/2024/11/"
    "2024-Bwacha-community-projects-Recieved.pdf"
)

final_bwacha_df["source_document"] = (
    "2024 Bwacha CDF Community Projects Submission"
)

display(final_bwacha_df.head())

,project_number,year,constituency,project_name,project_description,ward,project_site,application_amount,engineers_estimate,approved_amount,contract_amount,status,source_url,source_document
0,1,2024,Bwacha,Construction of 1X3 Classroom Block and Teachers Houses in Kangomba ward,Construction of 1X3 Classroom Block and Teachers Houses in Kangomba ward - Kalima zone,Kangomba,Kangomba ward,<NA>,<NA>,<NA>,<NA>,,https://www.kabwecouncil.gov.zm/wp-content/uploads/2024/11/2024-Bwacha-community-projects-Reciev...,2024 Bwacha CDF Community Projects Submission
1,2,2024,Bwacha,Construction of 1X4 Classroom Block at Kagomba Primary School,Construction of 1X4 Classroom Block at Kagomba Primary School,Kangomba,Kangomba Primary School,<NA>,<NA>,<NA>,<NA>,,https://www.kabwecouncil.gov.zm/wp-content/uploads/2024/11/2024-Bwacha-community-projects-Reciev...,2024 Bwacha CDF Community Projects Submission
2,3,2024,Bwacha,Construction of 1X4 Classroom Block at Kagomba Primary School,Construction of 1X4 Classroom Block at Kagomba Primary School,Kangomba,Kangomba Primary School,<NA>,<NA>,<NA>,<NA>,,https://www.kabwecouncil.gov.zm/wp-content/uploads/2024/11/2024-Bwacha-community-projects-Reciev...,2024 Bwacha CDF Community Projects Submission
3,4,2024,Bwacha,Construction of 1X3 Classroom Block at Mine Primary School,Construction of 1X3 Classroom Block at Mine Primary School - Mutwewansofu zone,Kangomba,Mine Primary School,<NA>,<NA>,<NA>,<NA>,,https://www.kabwecouncil.gov.zm/wp-content/uploads/2024/11/2024-Bwacha-community-projects-Reciev...,2024 Bwacha CDF Community Projects Submission
4,5,2024,Bwacha,"Repairing of the Mono-Pump, Construction of Toilets for Pre- School Pupils, Procurement of Desks...","Repairing of the Mono-Pump, Construction of Toilets for Pre- School Pupils, Procurement of Desks...",Kangomba,Mary Chidgey Community School,<NA>,<NA>,<NA>,<NA>,,https://www.kabwecouncil.gov.zm/wp-content/uploads/2024/11/2024-Bwacha-community-projects-Reciev...,2024 Bwacha CDF Community Projects Submission


In [ ]:
final_bwacha_df["project_type"] = pd.NA

final_bwacha_df = final_bwacha_df[
    [
        "project_number",
        "year",
        "constituency",
        "project_name",
        "project_description",
        "project_type",
        "ward",
        "project_site",
        "application_amount",
        "engineers_estimate",
        "approved_amount",
        "contract_amount",
        "status",
        "source_url",
        "source_document"
    ]
]

display(final_bwacha_df.head())

,project_number,year,constituency,project_name,project_description,project_type,ward,project_site,application_amount,engineers_estimate,approved_amount,contract_amount,status,source_url,source_document
0,1,2024,Bwacha,Construction of 1X3 Classroom Block and Teachers Houses in Kangomba ward,Construction of 1X3 Classroom Block and Teachers Houses in Kangomba ward - Kalima zone,<NA>,Kangomba,Kangomba ward,<NA>,<NA>,<NA>,<NA>,,https://www.kabwecouncil.gov.zm/wp-content/uploads/2024/11/2024-Bwacha-community-projects-Reciev...,2024 Bwacha CDF Community Projects Submission
1,2,2024,Bwacha,Construction of 1X4 Classroom Block at Kagomba Primary School,Construction of 1X4 Classroom Block at Kagomba Primary School,<NA>,Kangomba,Kangomba Primary School,<NA>,<NA>,<NA>,<NA>,,https://www.kabwecouncil.gov.zm/wp-content/uploads/2024/11/2024-Bwacha-community-projects-Reciev...,2024 Bwacha CDF Community Projects Submission
2,3,2024,Bwacha,Construction of 1X4 Classroom Block at Kagomba Primary School,Construction of 1X4 Classroom Block at Kagomba Primary School,<NA>,Kangomba,Kangomba Primary School,<NA>,<NA>,<NA>,<NA>,,https://www.kabwecouncil.gov.zm/wp-content/uploads/2024/11/2024-Bwacha-community-projects-Reciev...,2024 Bwacha CDF Community Projects Submission
3,4,2024,Bwacha,Construction of 1X3 Classroom Block at Mine Primary School,Construction of 1X3 Classroom Block at Mine Primary School - Mutwewansofu zone,<NA>,Kangomba,Mine Primary School,<NA>,<NA>,<NA>,<NA>,,https://www.kabwecouncil.gov.zm/wp-content/uploads/2024/11/2024-Bwacha-community-projects-Reciev...,2024 Bwacha CDF Community Projects Submission
4,5,2024,Bwacha,"Repairing of the Mono-Pump, Construction of Toilets for Pre- School Pupils, Procurement of Desks...","Repairing of the Mono-Pump, Construction of Toilets for Pre- School Pupils, Procurement of Desks...",<NA>,Kangomba,Mary Chidgey Community School,<NA>,<NA>,<NA>,<NA>,,https://www.kabwecouncil.gov.zm/wp-content/uploads/2024/11/2024-Bwacha-community-projects-Reciev...,2024 Bwacha CDF Community Projects Submission


## 87. Extract 2024 Kabwe Central CDF Project Records

The 2024 Kabwe Central CDF project document is processed using `pdfplumber` to extract the project tables from the PDF.

The extraction process:

1. Opens the original Kabwe Central CDF PDF.
2. Processes each page and extracts detected tables.
3. Collects the extracted table rows.
4. Filters the rows to retain records whose first field contains a numeric project number.
5. Stores the resulting project records for conversion into a structured DataFrame.

The project-number filter helps exclude table headers and other non-project rows returned during PDF table extraction.

The extracted records are subsequently structured, cleaned, and combined with the Bwacha CDF project records to create the integrated 2024 CDF project dataset.

In [ ]:
import pdfplumber
import pandas as pd

kabwe_central_path = (
    RAW_DIR /
    "2024_kabwe_central_cdf_projects.pdf"
)

all_rows_central = []

with pdfplumber.open(kabwe_central_path) as pdf:
    for page in pdf.pages:
        tables = page.extract_tables()

        for table in tables:
            for row in table:
                if row:
                    all_rows_central.append(row)

# Keep rows whose first cell contains a project number
project_rows_central = []

for row in all_rows_central:
    if (
        row[0] is not None
        and str(row[0]).strip().isdigit()
    ):
        project_rows_central.append(row)

print(
    "Project rows extracted:",
    len(project_rows_central)
)

Number of pages: 2
Page 1: 1 table(s) found
Page 2: 3 table(s) found

Project rows extracted: 43


## 88. Structure the Kabwe Central CDF Project Dataset

The extracted Kabwe Central project records are converted into a pandas DataFrame using descriptive column names based on the fields retained for the final 2024 CDF dataset.

The structured fields capture:

- Project identification
- Project name and description
- Ward and project site
- Application and estimated costs
- Approved and contract amounts
- Project implementation status

The number of records, number of fields, and column names are checked to confirm that the extracted table has been converted into the expected structure.

A small preview is displayed to verify the initial DataFrame structure before cleaning and metadata are added.

In [ ]:
central_df = pd.DataFrame(
    project_rows_central,
    columns=[
        "project_number",
        "project_name",
        "project_description",
        "ward",
        "project_site",
        "application_amount",
        "engineers_estimate",
        "approved_amount",
        "contract_amount",
        "status"
    ]
)

print("Rows:", len(central_df))
print("Columns:", len(central_df.columns))

print("\nColumns:")
print(central_df.columns.tolist())

display(central_df.head())

Rows: 43
Columns: 11

Columns:
['project_number', 'project_name', 'project_description', 'project_type', 'ward', 'project_site', 'application_amount', 'engineers_estimate', 'approved_amount', 'contract_amount', 'status']


,project_number,project_name,project_description,project_type,ward,project_site,application_amount,engineers_estimate,approved_amount,contract_amount,status
0,1,Construction of Secondary\nSchool 1X4 Classroom\nBlock,Construction of Secondary\nSchool 1X4 Classroom\nBlock,Construction,Luangwa,Kabwe Trust\nSecondary School,,,,,
1,2,Construction of 1X3\nClassroom Block,Construction of 1X3\nClassroom Block,Construction,Luangwa,Kabwe Central\nHospital Special\nSchool Community,,,,,
2,3,Construction of 1X2\nClassroom Block,Construction of 1X2\nClassroom Block,Construction,Luangwa,Kabwe Trust Primary\nSchool,,,,,
3,4,Construction of 1X3\nClassroom Block,Construction of 1X3\nClassroom Block,Construction,Mpima,Mpima Dairy Scheme,,,,,
4,5,Construction of 1X3\nClassroom Block,Construction of 1X3\nClassroom Block,Construction,Mpima,Mpima C,,,,,
5,6,Construction of a School,Construction of a School,Construction,Mpima,Kamuchanga,,,,,
6,7,Construction of 1X5\nClassroom Block and Wall\nfence,Construction of 1X5\nClassroom Block and Wall\nfence,Construction,Mpima,Nabusanga Zone\n(Primary School),,,,,
7,8,Construction of 1X3\nClassroom Block at Neem\nTress Secondary School,Construction of 1X3\nClassroom Block at Neem\nTress Secondary School,Construction,Lukanga,Neem Tree\nSecondary School,,,,,
8,9,Construction of 1X4\nClassroom Block at David\nRamushu Combined\nSchool,Construction of 1X4\nClassroom Block at David\nRamushu Combined School,Construction,D/Ramushu,David Ramushu\nCombined School,,,,,
9,10,Procurement of Laboratory\nRequirements at St.\nDominic Savio Secondary\nSchool,Procurement of Laboratory\nRequirements at St. Dominic\nSavio Secondary School,Procurement,J/Kabwe,St. Dominic Savio\nSecondary School,,,,,


## 89. Clean and Standardise Text Fields

The text fields extracted from the 2024 Kabwe Central CDF project document are cleaned to improve consistency.

The cleaning process:

- Converts text values to a consistent string type.
- Replaces multiple whitespace characters with a single space.
- Removes unnecessary leading and trailing whitespace.

The `project_type` field is retained temporarily because it was present in the extracted source structure. It will be handled during the final schema standardisation step so that the Kabwe Central dataset matches the final 2024 CDF dataset structure.

The substantive project information is preserved while formatting inconsistencies are reduced.

In [ ]:
# Clean text fields in the Kabwe Central CDF dataset

clean_central_df = central_df.copy()

central_text_columns = [
    "project_name",
    "project_description",
    "project_type",
    "ward",
    "project_site",
    "status"
]

for column in central_text_columns:
    clean_central_df[column] = (
        clean_central_df[column]
        .astype("string")
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
    )

print("Kabwe Central text fields cleaned successfully.")

## 90. Clean and Standardise Financial Values

The financial fields in the Kabwe Central CDF dataset are cleaned using the previously defined `clean_amount()` function.

The cleaning converts valid financial values into numeric form while handling formatting characters such as commas and currency symbols.

Blank or unavailable financial values are preserved as missing values rather than being converted to zero. This avoids introducing information that was not present in the original council source.

Using the same cleaning function for the Bwacha and Kabwe Central datasets ensures that their financial fields follow consistent preprocessing rules before the datasets are combined.

In [ ]:
# Clean financial fields in the Kabwe Central CDF dataset

financial_columns = [
    "application_amount",
    "engineers_estimate",
    "approved_amount",
    "contract_amount"
]

for column in financial_columns:
    clean_central_df[column] = (
        clean_central_df[column].apply(clean_amount)
    )

print("Financial fields cleaned successfully.")

## 91. Validate the Cleaned Kabwe Central Dataset

A set of data-quality checks is performed on the cleaned Kabwe Central CDF project dataset.

The validation checks:

- The number of records and columns.
- Missing values in each field.
- Exact duplicate records.
- Duplicate project numbers.
- The range of project numbers extracted.
- The distribution of project types.
- The distribution of projects across wards.
- The distribution of project implementation statuses.

These checks help identify possible extraction or cleaning problems before the Kabwe Central records are integrated with the Bwacha CDF project records.

The complete project-number list is not printed because the range and duplicate check provide a more concise validation of the project identifiers.

In [ ]:
# Validate the cleaned Kabwe Central CDF dataset

print("Rows:", len(clean_central_df))
print("Columns:", len(clean_central_df.columns))

print("\nMissing values:")
print(clean_central_df.isna().sum())

print("\nDuplicate complete rows:",
      clean_central_df.duplicated().sum())

print("\nDuplicate project numbers:",
      clean_central_df["project_number"].duplicated().sum())

print("\nProject number range:")
print(
    clean_central_df["project_number"].min(),
    "to",
    clean_central_df["project_number"].max()
)

print("\nProject types:")
print(clean_central_df["project_type"].value_counts(dropna=False))

print("\nWards:")
print(clean_central_df["ward"].value_counts(dropna=False))

print("\nStatuses:")
print(clean_central_df["status"].value_counts(dropna=False))

Rows: 43
Columns: 11

Missing values:
project_number          0
project_name            0
project_description     0
project_type            0
ward                    0
project_site            0
application_amount     43
engineers_estimate     43
approved_amount        43
contract_amount        43
status                  0
dtype: int64

Duplicate complete rows: 0

Duplicate project numbers: 17

Project numbers:
['1', '2', '3', '4', '5', '6', '7', '8', '9', '10', '11', '12', '13', '14', '15', '16', '17', '18', '19', '20', '21', '22', '23', '24', '25', '26', '1', '2', '3', '4', '5', '6', '7', '8', '1', '1', '2', '1', '2', '3', '4', '5', '1']

Project types:
project_type
Construction      34
Procurement        2
Rehabilitation     2
Water Articula     1
Construction/      1
Provision          1
Construction,      1
Enhancement        1
Name: count, dtype: Int64

Wards:
ward
Nakoli                     9
Kaputula                   6
Mpima                      5
Lukanga                    4
C

In [ ]:
display(clean_central_df)

,project_number,project_name,project_description,project_type,ward,project_site,application_amount,engineers_estimate,approved_amount,contract_amount,status
0,1,Construction of Secondary School 1X4 Classroom Block,Construction of Secondary School 1X4 Classroom Block,Construction,Luangwa,Kabwe Trust Secondary School,<NA>,<NA>,<NA>,<NA>,
1,2,Construction of 1X3 Classroom Block,Construction of 1X3 Classroom Block,Construction,Luangwa,Kabwe Central Hospital Special School Community,<NA>,<NA>,<NA>,<NA>,
2,3,Construction of 1X2 Classroom Block,Construction of 1X2 Classroom Block,Construction,Luangwa,Kabwe Trust Primary School,<NA>,<NA>,<NA>,<NA>,
3,4,Construction of 1X3 Classroom Block,Construction of 1X3 Classroom Block,Construction,Mpima,Mpima Dairy Scheme,<NA>,<NA>,<NA>,<NA>,
4,5,Construction of 1X3 Classroom Block,Construction of 1X3 Classroom Block,Construction,Mpima,Mpima C,<NA>,<NA>,<NA>,<NA>,
5,6,Construction of a School,Construction of a School,Construction,Mpima,Kamuchanga,<NA>,<NA>,<NA>,<NA>,
6,7,Construction of 1X5 Classroom Block and Wall fence,Construction of 1X5 Classroom Block and Wall fence,Construction,Mpima,Nabusanga Zone (Primary School),<NA>,<NA>,<NA>,<NA>,
7,8,Construction of 1X3 Classroom Block at Neem Tress Secondary School,Construction of 1X3 Classroom Block at Neem Tress Secondary School,Construction,Lukanga,Neem Tree Secondary School,<NA>,<NA>,<NA>,<NA>,
8,9,Construction of 1X4 Classroom Block at David Ramushu Combined School,Construction of 1X4 Classroom Block at David Ramushu Combined School,Construction,D/Ramushu,David Ramushu Combined School,<NA>,<NA>,<NA>,<NA>,
9,10,Procurement of Laboratory Requirements at St. Dominic Savio Secondary School,Procurement of Laboratory Requirements at St. Dominic Savio Secondary School,Procurement,J/Kabwe,St. Dominic Savio Secondary School,<NA>,<NA>,<NA>,<NA>,


## 92. Add Metadata to the Kabwe Central CDF Dataset

Contextual and source metadata is added to the cleaned Kabwe Central CDF project records.

The `year` and `constituency` fields identify the period and constituency represented by the records.

The `source_url` field stores the official Kabwe Municipal Council URL from which the project document was obtained, while `source_document` identifies the specific council publication.

Including source metadata improves the traceability and reproducibility of the dataset and allows the project records to be linked back to their original source.

In [ ]:
final_central_df = clean_central_df.copy()

final_central_df.insert(1, "year", 2024)
final_central_df.insert(2, "constituency", "Kabwe Central")

final_central_df["source_url"] = (
    "https://www.kabwecouncil.gov.zm/"
    "wp-content/uploads/2024/11/"
    "2024-Community-Projects-Kabwe-Central-received.pdf"
)

final_central_df["source_document"] = (
    "2024 Kabwe Central CDF Community Projects Submission"
)

## 93. Standardise the Final 2024 CDF Dataset Schema

The Bwacha and Kabwe Central datasets are aligned to the same final schema before they are combined.

The common schema contains project identification, year and constituency, project details, location, financial information, implementation status, and source metadata.

The `project_type` field is excluded from the final schema because it was not consistently retained across the two 2024 source datasets. Removing it at this stage ensures that both constituency datasets have an identical structure while preserving the core information required for the final CDF dataset.

The column-order checks confirm that both DataFrames now follow exactly the same schema and can be safely concatenated.

In [ ]:
# Standardise the final 2024 CDF project schema

common_columns = [
    "project_number",
    "year",
    "constituency",
    "project_name",
    "project_description",
    "ward",
    "project_site",
    "application_amount",
    "engineers_estimate",
    "approved_amount",
    "contract_amount",
    "status",
    "source_url",
    "source_document"
]

final_bwacha_df = final_bwacha_df[common_columns]
final_central_df = final_central_df[common_columns]

print("Bwacha columns match:",
      final_bwacha_df.columns.tolist() == common_columns)

print("Kabwe Central columns match:",
      final_central_df.columns.tolist() == common_columns)

Bwacha columns match: True
Kabwe Central columns match: True


## 94. Combine the 2024 CDF Project Datasets

The cleaned and standardised project records from Bwacha and Kabwe Central constituencies are combined into a single 2024 CDF project dataset using `pandas.concat()`.

Because both DataFrames were standardised to the same column structure in the previous step, their records can be safely integrated without creating mismatched fields.

`ignore_index=True` creates a new sequential index for the combined dataset.

The total number of project records and columns is reported, followed by a constituency-level count to confirm that records from both source constituencies were successfully included.

In [ ]:
# Combine the 2024 CDF project datasets

cdf_2024_df = pd.concat(
    [
        final_bwacha_df,
        final_central_df
    ],
    ignore_index=True
)

print("Total 2024 projects:", len(cdf_2024_df))
print("Total columns:", len(cdf_2024_df.columns))

print("\nProjects by constituency:")
print(cdf_2024_df["constituency"].value_counts())

display(cdf_2024_df.head(10))

Total 2024 projects: 79
Total columns: 15


,project_number,year,constituency,project_name,project_description,project_type,ward,project_site,application_amount,engineers_estimate,approved_amount,contract_amount,status,source_url,source_document
0,1,2024,Bwacha,Construction of 1X3 Classroom Block and Teachers Houses in Kangomba ward,Construction of 1X3 Classroom Block and Teachers Houses in Kangomba ward - Kalima zone,<NA>,Kangomba,Kangomba ward,<NA>,<NA>,<NA>,<NA>,,https://www.kabwecouncil.gov.zm/wp-content/uploads/2024/11/2024-Bwacha-community-projects-Reciev...,2024 Bwacha CDF Community Projects Submission
1,2,2024,Bwacha,Construction of 1X4 Classroom Block at Kagomba Primary School,Construction of 1X4 Classroom Block at Kagomba Primary School,<NA>,Kangomba,Kangomba Primary School,<NA>,<NA>,<NA>,<NA>,,https://www.kabwecouncil.gov.zm/wp-content/uploads/2024/11/2024-Bwacha-community-projects-Reciev...,2024 Bwacha CDF Community Projects Submission
2,3,2024,Bwacha,Construction of 1X4 Classroom Block at Kagomba Primary School,Construction of 1X4 Classroom Block at Kagomba Primary School,<NA>,Kangomba,Kangomba Primary School,<NA>,<NA>,<NA>,<NA>,,https://www.kabwecouncil.gov.zm/wp-content/uploads/2024/11/2024-Bwacha-community-projects-Reciev...,2024 Bwacha CDF Community Projects Submission
3,4,2024,Bwacha,Construction of 1X3 Classroom Block at Mine Primary School,Construction of 1X3 Classroom Block at Mine Primary School - Mutwewansofu zone,<NA>,Kangomba,Mine Primary School,<NA>,<NA>,<NA>,<NA>,,https://www.kabwecouncil.gov.zm/wp-content/uploads/2024/11/2024-Bwacha-community-projects-Reciev...,2024 Bwacha CDF Community Projects Submission
4,5,2024,Bwacha,"Repairing of the Mono-Pump, Construction of Toilets for Pre- School Pupils, Procurement of Desks...","Repairing of the Mono-Pump, Construction of Toilets for Pre- School Pupils, Procurement of Desks...",<NA>,Kangomba,Mary Chidgey Community School,<NA>,<NA>,<NA>,<NA>,,https://www.kabwecouncil.gov.zm/wp-content/uploads/2024/11/2024-Bwacha-community-projects-Reciev...,2024 Bwacha CDF Community Projects Submission
5,6,2024,Bwacha,Construction of a Primary School in Kawama ward,Construction of a Primary School in Kawama ward,<NA>,Kawama,Kawama ward,<NA>,<NA>,<NA>,<NA>,,https://www.kabwecouncil.gov.zm/wp-content/uploads/2024/11/2024-Bwacha-community-projects-Reciev...,2024 Bwacha CDF Community Projects Submission
6,7,2024,Bwacha,Construction of 10 Teachers Houses at Chitakata Community School,Construction of 10 Teachers Houses at Chitakata Community School,<NA>,Muwowo East,Chitakata Commnuity School,<NA>,<NA>,<NA>,<NA>,,https://www.kabwecouncil.gov.zm/wp-content/uploads/2024/11/2024-Bwacha-community-projects-Reciev...,2024 Bwacha CDF Community Projects Submission
7,8,2024,Bwacha,Construction of 1x4 Classroom Block at Mukobeko Correctional Day Secondary School,Construction of 1x4 Classroom Block at Mukobeko Correctional Day Secondary School,<NA>,Muwowo East,Mukobeko Correctional Day Secondary School,<NA>,<NA>,<NA>,<NA>,,https://www.kabwecouncil.gov.zm/wp-content/uploads/2024/11/2024-Bwacha-community-projects-Reciev...,2024 Bwacha CDF Community Projects Submission
8,9,2024,Bwacha,Construction of Youth Resource Centre,Construction of Youth Resource Centre,<NA>,Bwacha,Bwacha ward,<NA>,<NA>,<NA>,<NA>,,https://www.kabwecouncil.gov.zm/wp-content/uploads/2024/11/2024-Bwacha-community-projects-Reciev...,2024 Bwacha CDF Community Projects Submission
9,10,2024,Bwacha,Construction of a School Hall at Rapheal Kombe Secondary School,Construction of a School Hall at Rapheal Kombe Secondary School,<NA>,Chimaniman i,Rapheal Kombe Secondary School,<NA>,<NA>,<NA>,<NA>,,https://www.kabwecouncil.gov.zm/wp-content/uploads/2024/11/2024-Bwacha-community-projects-Reciev...,2024 Bwacha CDF Community Projects Submission


## 95. Perform Final Validation of the Integrated 2024 CDF Dataset

Final data-quality checks are performed on the integrated 2024 CDF project dataset.

The validation checks:

- The total number of project records.
- Whether the expected 79 project records were obtained.
- The final number and names of columns.
- The year represented in the dataset.
- Missing values across all fields.
- Exact duplicate records.
- Duplicate project identifiers within each constituency.
- The distribution of projects between Bwacha and Kabwe Central constituencies.
- The distribution of project implementation statuses.
- The distribution of projects across wards.

The duplicate project check uses the combination of year, constituency, and project number because project numbers may legitimately repeat across different constituencies.

These checks provide final evidence that the two constituency-level datasets were successfully integrated without introducing obvious duplication or structural inconsistencies.

In [ ]:
# Final validation of the integrated 2024 CDF dataset

print("===== DATASET VALIDATION =====")

print("\nRows:")
print(len(cdf_2024_df))

print("\nExpected rows: 79")
print("Row count correct:", len(cdf_2024_df) == 79)

print("\nColumns:")
print(len(cdf_2024_df.columns))

print("\nColumn names:")
print(cdf_2024_df.columns.tolist())

print("\nYear values:")
print(cdf_2024_df["year"].value_counts())

print("\nMissing values:")
print(cdf_2024_df.isna().sum())

print("\nDuplicate complete rows:")
print(cdf_2024_df.duplicated().sum())

print("\nDuplicate project numbers within constituency:")
duplicates = cdf_2024_df.duplicated(
    subset=["year", "constituency", "project_number"]
).sum()

print(duplicates)

print("\nProjects by constituency:")
print(cdf_2024_df["constituency"].value_counts())

print("\nProjects by status:")
print(cdf_2024_df["status"].value_counts(dropna=False))

print("\nProjects by ward:")
print(cdf_2024_df["ward"].value_counts(dropna=False))

===== DATASET VALIDATION =====

Rows:
79

Columns:
15

Column names:
['project_number', 'year', 'constituency', 'project_name', 'project_description', 'project_type', 'ward', 'project_site', 'application_amount', 'engineers_estimate', 'approved_amount', 'contract_amount', 'status', 'source_url', 'source_document']

Missing values:
project_number          0
year                    0
constituency            0
project_name            0
project_description     0
project_type           36
ward                    0
project_site            0
application_amount     79
engineers_estimate     79
approved_amount        79
contract_amount        79
status                  0
source_url              0
source_document         0
dtype: int64

Duplicate complete rows:
0

Duplicate project numbers within constituency:
41

Projects by constituency:
constituency
Kabwe Central    43
Bwacha           36
Name: count, dtype: int64

Projects by status:
status
    79
Name: count, dtype: Int64

Projects by ward:

## 96. Verify Financial Data Types

The data types of the financial fields in the integrated 2024 CDF dataset are checked to confirm that the financial values were successfully converted to numeric types during preprocessing.

This ensures that the financial fields can be used reliably for calculations and further analysis.

In [ ]:
print("Financial data types:")

print(
    cdf_2024_df[
        [
            "application_amount",
            "engineers_estimate",
            "approved_amount",
            "contract_amount"
        ]
    ].dtypes
)

Financial data types:
application_amount    object
engineers_estimate    object
approved_amount       object
contract_amount       object
dtype: object


## 81. Extract 2024 Bwacha CDF Project Records

The 2024 Bwacha CDF project document is processed using `pdfplumber` to extract the project records from the PDF tables.

The extraction process:

1. Opens the original PDF source document.
2. Processes each page and extracts detected tables.
3. Collects the extracted table rows.
4. Filters the rows to retain records whose first field contains a numeric project number.
5. Converts the validated project records into a pandas DataFrame using descriptive column names.

The project-number filter helps exclude table headers and other non-project rows returned during PDF table extraction.

The resulting DataFrame forms the basis for the subsequent cleaning and preprocessing of the Bwacha CDF project records.

In [ ]:
bwacha_path = (
    RAW_DIR /
    "2024_bwacha_cdf_projects.pdf"
)

all_rows_bwacha = []

with pdfplumber.open(bwacha_path) as pdf:
    for page in pdf.pages:
        tables = page.extract_tables()

        for table in tables:
            for row in table:
                if row:
                    all_rows_bwacha.append(row)


project_rows_bwacha = []

for row in all_rows_bwacha:
    if (
        row[0] is not None
        and str(row[0]).strip().isdigit()
    ):
        project_rows_bwacha.append(row)


print(
    "Bwacha project rows extracted:",
    len(project_rows_bwacha)
)


bwacha_df = pd.DataFrame(
    project_rows_bwacha,
    columns=[
        "project_number",
        "project_name",
        "project_description",
        "ward",
        "project_site",
        "application_amount",
        "engineers_estimate",
        "approved_amount",
        "contract_amount",
        "status"
    ]
)

print("Bwacha rows:", len(bwacha_df))
print("Bwacha columns:", len(bwacha_df.columns))

Bwacha PDF pages: 5
Page 1: 1 table(s) found
Page 2: 1 table(s) found
Page 3: 1 table(s) found
Page 4: 1 table(s) found
Page 5: 1 table(s) found

Bwacha project rows extracted: 36
Bwacha rows: 36
Bwacha columns: 10


,project_number,project_name,project_description,ward,project_site,application_amount,engineers_estimate,approved_amount,contract_amount,status
0,1,Construction of 1X3 Classroom\nBlock and Teach...,Construction of 1X3\nClassroom Block and\nTeac...,Kangomba,Kangomba\nward,,,,,
1,2,Construction of 1X4 Classroom\nBlock at Kagomb...,Construction of 1X4\nClassroom Block at Kagomb...,Kangomba,Kangomba\nPrimary\nSchool,,,,,
2,3,Construction of 1X4 Classroom\nBlock at Kagomb...,Construction of 1X4\nClassroom Block at Kagomb...,Kangomba,Kangomba\nPrimary\nSchool,,,,,
3,4,Construction of 1X3 Classroom\nBlock at Mine P...,Construction of 1X3\nClassroom Block at Mine\n...,Kangomba,Mine Primary\nSchool,,,,,
4,5,"Repairing of the Mono-Pump,\nConstruction of T...","Repairing of the Mono-Pump,\nConstruction of T...",Kangomba,Mary Chidgey\nCommunity\nSchool,,,,,


In [ ]:
clean_bwacha_df = bwacha_df.copy()

text_columns = [
    "project_name",
    "project_description",
    "ward",
    "project_site",
    "status"
]

for column in text_columns:
    clean_bwacha_df[column] = (
        clean_bwacha_df[column]
        .astype("string")
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
    )

financial_columns = [
    "application_amount",
    "engineers_estimate",
    "approved_amount",
    "contract_amount"
]

for column in financial_columns:
    clean_bwacha_df[column] = (
        clean_bwacha_df[column].apply(clean_amount)
    )

print("Bwacha cleaning completed.")

Bwacha cleaning completed.


,project_number,project_name,project_description,ward,project_site,application_amount,engineers_estimate,approved_amount,contract_amount,status
0,1,Construction of 1X3 Classroom Block and Teache...,Construction of 1X3 Classroom Block and Teache...,Kangomba,Kangomba ward,<NA>,<NA>,<NA>,<NA>,
1,2,Construction of 1X4 Classroom Block at Kagomba...,Construction of 1X4 Classroom Block at Kagomba...,Kangomba,Kangomba Primary School,<NA>,<NA>,<NA>,<NA>,
2,3,Construction of 1X4 Classroom Block at Kagomba...,Construction of 1X4 Classroom Block at Kagomba...,Kangomba,Kangomba Primary School,<NA>,<NA>,<NA>,<NA>,
3,4,Construction of 1X3 Classroom Block at Mine Pr...,Construction of 1X3 Classroom Block at Mine Pr...,Kangomba,Mine Primary School,<NA>,<NA>,<NA>,<NA>,
4,5,"Repairing of the Mono-Pump, Construction of To...","Repairing of the Mono-Pump, Construction of To...",Kangomba,Mary Chidgey Community School,<NA>,<NA>,<NA>,<NA>,


In [ ]:
final_bwacha_df = clean_bwacha_df.copy()

final_bwacha_df.insert(1, "year", 2024)
final_bwacha_df.insert(2, "constituency", "Bwacha")

final_bwacha_df["source_url"] = (
    "https://www.kabwecouncil.gov.zm/"
    "wp-content/uploads/2024/11/"
    "2024-Bwacha-community-projects-Recieved.pdf"
)

final_bwacha_df["source_document"] = (
    "2024 Bwacha CDF Community Projects Submission"
)

print("Final Bwacha metadata added.")

Final Bwacha dataset:
Rows: 36
Columns: 15


,project_number,year,constituency,project_name,project_description,project_type,ward,project_site,application_amount,engineers_estimate,approved_amount,contract_amount,status,source_url,source_document
0,1,2024,Bwacha,Construction of 1X3 Classroom Block and Teache...,Construction of 1X3 Classroom Block and Teache...,<NA>,Kangomba,Kangomba ward,<NA>,<NA>,<NA>,<NA>,,https://www.kabwecouncil.gov.zm/wp-content/upl...,2024 Bwacha CDF Community Projects Submission
1,2,2024,Bwacha,Construction of 1X4 Classroom Block at Kagomba...,Construction of 1X4 Classroom Block at Kagomba...,<NA>,Kangomba,Kangomba Primary School,<NA>,<NA>,<NA>,<NA>,,https://www.kabwecouncil.gov.zm/wp-content/upl...,2024 Bwacha CDF Community Projects Submission
2,3,2024,Bwacha,Construction of 1X4 Classroom Block at Kagomba...,Construction of 1X4 Classroom Block at Kagomba...,<NA>,Kangomba,Kangomba Primary School,<NA>,<NA>,<NA>,<NA>,,https://www.kabwecouncil.gov.zm/wp-content/upl...,2024 Bwacha CDF Community Projects Submission
3,4,2024,Bwacha,Construction of 1X3 Classroom Block at Mine Pr...,Construction of 1X3 Classroom Block at Mine Pr...,<NA>,Kangomba,Mine Primary School,<NA>,<NA>,<NA>,<NA>,,https://www.kabwecouncil.gov.zm/wp-content/upl...,2024 Bwacha CDF Community Projects Submission
4,5,2024,Bwacha,"Repairing of the Mono-Pump, Construction of To...","Repairing of the Mono-Pump, Construction of To...",<NA>,Kangomba,Mary Chidgey Community School,<NA>,<NA>,<NA>,<NA>,,https://www.kabwecouncil.gov.zm/wp-content/upl...,2024 Bwacha CDF Community Projects Submission


# Extract 2024 Kabwe Central CDF project records

In [ ]:
kabwe_central_path = (
    RAW_DIR /
    "2024_kabwe_central_cdf_projects.pdf"
)

all_rows_central = []

with pdfplumber.open(kabwe_central_path) as pdf:
    for page in pdf.pages:
        tables = page.extract_tables()

        for table in tables:
            for row in table:
                if row:
                    all_rows_central.append(row)


project_rows_central = []

for row in all_rows_central:
    if (
        row[0] is not None
        and str(row[0]).strip().isdigit()
    ):
        project_rows_central.append(row)


print(
    "Kabwe Central project rows extracted:",
    len(project_rows_central)
)


central_df = pd.DataFrame(
    project_rows_central,
    columns=[
        "project_number",
        "project_name",
        "project_description",
        "ward",
        "project_site",
        "application_amount",
        "engineers_estimate",
        "approved_amount",
        "contract_amount",
        "status"
    ]
)

print("Kabwe Central rows:", len(central_df))
print("Kabwe Central columns:", len(central_df.columns))

Kabwe Central PDF pages: 2
Page 1: 1 table(s) found
Page 2: 3 table(s) found

Kabwe Central project rows extracted: 43
Kabwe Central rows: 43
Kabwe Central columns: 11


,project_number,project_name,project_description,project_type,ward,project_site,application_amount,engineers_estimate,approved_amount,contract_amount,status
0,1,Construction of Secondary\nSchool 1X4 Classroo...,Construction of Secondary\nSchool 1X4 Classroo...,Construction,Luangwa,Kabwe Trust\nSecondary School,,,,,
1,2,Construction of 1X3\nClassroom Block,Construction of 1X3\nClassroom Block,Construction,Luangwa,Kabwe Central\nHospital Special\nSchool Community,,,,,
2,3,Construction of 1X2\nClassroom Block,Construction of 1X2\nClassroom Block,Construction,Luangwa,Kabwe Trust Primary\nSchool,,,,,
3,4,Construction of 1X3\nClassroom Block,Construction of 1X3\nClassroom Block,Construction,Mpima,Mpima Dairy Scheme,,,,,
4,5,Construction of 1X3\nClassroom Block,Construction of 1X3\nClassroom Block,Construction,Mpima,Mpima C,,,,,


In [ ]:
clean_central_df = central_df.copy()

central_text_columns = [
    "project_name",
    "project_description",
    "project_type",
    "ward",
    "project_site",
    "status"
]

for column in central_text_columns:
    clean_central_df[column] = (
        clean_central_df[column]
        .astype("string")
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
    )


for column in financial_columns:
    clean_central_df[column] = (
        clean_central_df[column].apply(clean_amount)
    )


print("Kabwe Central cleaning completed.")

Kabwe Central cleaning completed.


,project_number,project_name,project_description,project_type,ward,project_site,application_amount,engineers_estimate,approved_amount,contract_amount,status
0,1,Construction of Secondary School 1X4 Classroom...,Construction of Secondary School 1X4 Classroom...,Construction,Luangwa,Kabwe Trust Secondary School,<NA>,<NA>,<NA>,<NA>,
1,2,Construction of 1X3 Classroom Block,Construction of 1X3 Classroom Block,Construction,Luangwa,Kabwe Central Hospital Special School Community,<NA>,<NA>,<NA>,<NA>,
2,3,Construction of 1X2 Classroom Block,Construction of 1X2 Classroom Block,Construction,Luangwa,Kabwe Trust Primary School,<NA>,<NA>,<NA>,<NA>,
3,4,Construction of 1X3 Classroom Block,Construction of 1X3 Classroom Block,Construction,Mpima,Mpima Dairy Scheme,<NA>,<NA>,<NA>,<NA>,
4,5,Construction of 1X3 Classroom Block,Construction of 1X3 Classroom Block,Construction,Mpima,Mpima C,<NA>,<NA>,<NA>,<NA>,


In [ ]:
final_central_df = clean_central_df.copy()

final_central_df.insert(1, "year", 2024)
final_central_df.insert(2, "constituency", "Kabwe Central")

final_central_df["source_url"] = (
    "https://www.kabwecouncil.gov.zm/"
    "wp-content/uploads/2024/11/"
    "2024-Community-Projects-Kabwe-Central-received.pdf"
)

final_central_df["source_document"] = (
    "2024 Kabwe Central CDF Community Projects Submission"
)

final_central_df = final_central_df[common_columns]

print("Final Kabwe Central dataset:")
print("Rows:", len(final_central_df))
print("Columns:", len(final_central_df.columns))

Final Kabwe Central dataset:
Rows: 43
Columns: 15


,project_number,year,constituency,project_name,project_description,project_type,ward,project_site,application_amount,engineers_estimate,approved_amount,contract_amount,status,source_url,source_document
0,1,2024,Kabwe Central,Construction of Secondary School 1X4 Classroom...,Construction of Secondary School 1X4 Classroom...,Construction,Luangwa,Kabwe Trust Secondary School,<NA>,<NA>,<NA>,<NA>,,https://www.kabwecouncil.gov.zm/wp-content/upl...,2024 Kabwe Central CDF Community Projects Subm...
1,2,2024,Kabwe Central,Construction of 1X3 Classroom Block,Construction of 1X3 Classroom Block,Construction,Luangwa,Kabwe Central Hospital Special School Community,<NA>,<NA>,<NA>,<NA>,,https://www.kabwecouncil.gov.zm/wp-content/upl...,2024 Kabwe Central CDF Community Projects Subm...
2,3,2024,Kabwe Central,Construction of 1X2 Classroom Block,Construction of 1X2 Classroom Block,Construction,Luangwa,Kabwe Trust Primary School,<NA>,<NA>,<NA>,<NA>,,https://www.kabwecouncil.gov.zm/wp-content/upl...,2024 Kabwe Central CDF Community Projects Subm...
3,4,2024,Kabwe Central,Construction of 1X3 Classroom Block,Construction of 1X3 Classroom Block,Construction,Mpima,Mpima Dairy Scheme,<NA>,<NA>,<NA>,<NA>,,https://www.kabwecouncil.gov.zm/wp-content/upl...,2024 Kabwe Central CDF Community Projects Subm...
4,5,2024,Kabwe Central,Construction of 1X3 Classroom Block,Construction of 1X3 Classroom Block,Construction,Mpima,Mpima C,<NA>,<NA>,<NA>,<NA>,,https://www.kabwecouncil.gov.zm/wp-content/upl...,2024 Kabwe Central CDF Community Projects Subm...


# COMBINE 2024 CDF DATASETS

In [ ]:
cdf_2024_df = pd.concat(
    [
        final_bwacha_df,
        final_central_df
    ],
    ignore_index=True
)

print("===== 2024 KABWE CDF DATASET =====")
print("Total rows:", len(cdf_2024_df))
print("Total columns:", len(cdf_2024_df.columns))

print("\nProjects by constituency:")
print(cdf_2024_df["constituency"].value_counts())

===== 2024 KABWE CDF DATASET =====
Total rows: 79
Total columns: 15

Projects by constituency:
constituency
Kabwe Central    43
Bwacha           36
Name: count, dtype: int64


,project_number,year,constituency,project_name,project_description,project_type,ward,project_site,application_amount,engineers_estimate,approved_amount,contract_amount,status,source_url,source_document
0,1,2024,Bwacha,Construction of 1X3 Classroom Block and Teache...,Construction of 1X3 Classroom Block and Teache...,<NA>,Kangomba,Kangomba ward,<NA>,<NA>,<NA>,<NA>,,https://www.kabwecouncil.gov.zm/wp-content/upl...,2024 Bwacha CDF Community Projects Submission
1,2,2024,Bwacha,Construction of 1X4 Classroom Block at Kagomba...,Construction of 1X4 Classroom Block at Kagomba...,<NA>,Kangomba,Kangomba Primary School,<NA>,<NA>,<NA>,<NA>,,https://www.kabwecouncil.gov.zm/wp-content/upl...,2024 Bwacha CDF Community Projects Submission
2,3,2024,Bwacha,Construction of 1X4 Classroom Block at Kagomba...,Construction of 1X4 Classroom Block at Kagomba...,<NA>,Kangomba,Kangomba Primary School,<NA>,<NA>,<NA>,<NA>,,https://www.kabwecouncil.gov.zm/wp-content/upl...,2024 Bwacha CDF Community Projects Submission
3,4,2024,Bwacha,Construction of 1X3 Classroom Block at Mine Pr...,Construction of 1X3 Classroom Block at Mine Pr...,<NA>,Kangomba,Mine Primary School,<NA>,<NA>,<NA>,<NA>,,https://www.kabwecouncil.gov.zm/wp-content/upl...,2024 Bwacha CDF Community Projects Submission
4,5,2024,Bwacha,"Repairing of the Mono-Pump, Construction of To...","Repairing of the Mono-Pump, Construction of To...",<NA>,Kangomba,Mary Chidgey Community School,<NA>,<NA>,<NA>,<NA>,,https://www.kabwecouncil.gov.zm/wp-content/upl...,2024 Bwacha CDF Community Projects Submission
5,6,2024,Bwacha,Construction of a Primary School in Kawama ward,Construction of a Primary School in Kawama ward,<NA>,Kawama,Kawama ward,<NA>,<NA>,<NA>,<NA>,,https://www.kabwecouncil.gov.zm/wp-content/upl...,2024 Bwacha CDF Community Projects Submission
6,7,2024,Bwacha,Construction of 10 Teachers Houses at Chitakat...,Construction of 10 Teachers Houses at Chitakat...,<NA>,Muwowo East,Chitakata Commnuity School,<NA>,<NA>,<NA>,<NA>,,https://www.kabwecouncil.gov.zm/wp-content/upl...,2024 Bwacha CDF Community Projects Submission
7,8,2024,Bwacha,Construction of 1x4 Classroom Block at Mukobek...,Construction of 1x4 Classroom Block at Mukobek...,<NA>,Muwowo East,Mukobeko Correctional Day Secondary School,<NA>,<NA>,<NA>,<NA>,,https://www.kabwecouncil.gov.zm/wp-content/upl...,2024 Bwacha CDF Community Projects Submission
8,9,2024,Bwacha,Construction of Youth Resource Centre,Construction of Youth Resource Centre,<NA>,Bwacha,Bwacha ward,<NA>,<NA>,<NA>,<NA>,,https://www.kabwecouncil.gov.zm/wp-content/upl...,2024 Bwacha CDF Community Projects Submission
9,10,2024,Bwacha,Construction of a School Hall at Rapheal Kombe...,Construction of a School Hall at Rapheal Kombe...,<NA>,Chimaniman i,Rapheal Kombe Secondary School,<NA>,<NA>,<NA>,<NA>,,https://www.kabwecouncil.gov.zm/wp-content/upl...,2024 Bwacha CDF Community Projects Submission


# FINAL VALIDATION

In [ ]:

print("========== FINAL VALIDATION ==========")

print("\n1. Dataset dimensions")
print("Rows:", len(cdf_2024_df))
print("Columns:", len(cdf_2024_df.columns))


print("\n2. Column names")
print(cdf_2024_df.columns.tolist())


print("\n3. Missing values")
print(cdf_2024_df.isna().sum())


print("\n4. Duplicate complete rows")
print(cdf_2024_df.duplicated().sum())


print("\n5. Duplicate project IDs within constituency")
duplicate_projects = cdf_2024_df.duplicated(
    subset=[
        "year",
        "constituency",
        "project_number"
    ]
).sum()

print(duplicate_projects)


print("\n6. Projects by constituency")
print(
    cdf_2024_df["constituency"]
    .value_counts()
)


print("\n7. Projects by ward")
print(
    cdf_2024_df["ward"]
    .value_counts(dropna=False)
)


print("\n8. Projects by status")
print(
    cdf_2024_df["status"]
    .value_counts(dropna=False)
)


print("\n9. Financial data types")
print(
    cdf_2024_df[
        financial_columns
    ].dtypes
)

========== FINAL VALIDATION ==========

1. Dataset dimensions
Rows: 79
Columns: 15

2. Column names
['project_number', 'year', 'constituency', 'project_name', 'project_description', 'project_type', 'ward', 'project_site', 'application_amount', 'engineers_estimate', 'approved_amount', 'contract_amount', 'status', 'source_url', 'source_document']

3. Missing values
project_number          0
year                    0
constituency            0
project_name            0
project_description     0
project_type           36
ward                    0
project_site            0
application_amount     79
engineers_estimate     79
approved_amount        79
contract_amount        79
status                  0
source_url              0
source_document         0
dtype: int64

4. Duplicate complete rows
0

5. Duplicate project IDs within constituency
41

6. Projects by constituency
constituency
Kabwe Central    43
Bwacha           36
Name: count, dtype: int64

7. Projects by ward
ward
Nakoli            

In [ ]:
output_path_2024 = (
    PROCESSED_DIR /
    "db-unza26-csc4792-kabwe_cdf_projects_2024.csv"
)

cdf_2024_df.to_csv(
    output_path_2024,
    sep="|",
    index=False,
    encoding="utf-8"
)

print("2024 dataset saved successfully:")
print(output_path_2024)

2024 dataset saved successfully:
../data/processed/db-unza26-csc4792-kabwe_cdf_projects_2024.csv


In [ ]:
check_2024_df = pd.read_csv(
    output_path_2024,
    sep="|"
)

print("========== EXPORT CHECK ==========")

print("Rows:", len(check_2024_df))
print("Columns:", len(check_2024_df.columns))

print(
    "Rows match:",
    len(check_2024_df) == len(cdf_2024_df)
)

print(
    "Columns match:",
    check_2024_df.columns.tolist()
    == cdf_2024_df.columns.tolist()
)

========== EXPORT CHECK ==========
Rows: 79
Columns: 15
Rows match: True
Columns match: True


,project_number,year,constituency,project_name,project_description,project_type,ward,project_site,application_amount,engineers_estimate,approved_amount,contract_amount,status,source_url,source_document
0,1,2024,Bwacha,Construction of 1X3 Classroom Block and Teache...,Construction of 1X3 Classroom Block and Teache...,NaN,Kangomba,Kangomba ward,NaN,NaN,NaN,NaN,NaN,https://www.kabwecouncil.gov.zm/wp-content/upl...,2024 Bwacha CDF Community Projects Submission
1,2,2024,Bwacha,Construction of 1X4 Classroom Block at Kagomba...,Construction of 1X4 Classroom Block at Kagomba...,NaN,Kangomba,Kangomba Primary School,NaN,NaN,NaN,NaN,NaN,https://www.kabwecouncil.gov.zm/wp-content/upl...,2024 Bwacha CDF Community Projects Submission
2,3,2024,Bwacha,Construction of 1X4 Classroom Block at Kagomba...,Construction of 1X4 Classroom Block at Kagomba...,NaN,Kangomba,Kangomba Primary School,NaN,NaN,NaN,NaN,NaN,https://www.kabwecouncil.gov.zm/wp-content/upl...,2024 Bwacha CDF Community Projects Submission
3,4,2024,Bwacha,Construction of 1X3 Classroom Block at Mine Pr...,Construction of 1X3 Classroom Block at Mine Pr...,NaN,Kangomba,Mine Primary School,NaN,NaN,NaN,NaN,NaN,https://www.kabwecouncil.gov.zm/wp-content/upl...,2024 Bwacha CDF Community Projects Submission
4,5,2024,Bwacha,"Repairing of the Mono-Pump, Construction of To...","Repairing of the Mono-Pump, Construction of To...",NaN,Kangomba,Mary Chidgey Community School,NaN,NaN,NaN,NaN,NaN,https://www.kabwecouncil.gov.zm/wp-content/upl...,2024 Bwacha CDF Community Projects Submission


In [ ]:
budget_pdf_path = (
    RAW_DIR /
    "2025_kabwe_obb_budget.pdf"
)

import os

print("File exists:", os.path.exists(budget_pdf_path))

if os.path.exists(budget_pdf_path):
    print(
        "File size:",
        round(os.path.getsize(budget_pdf_path) / 1024, 2),
        "KB"
    )

File exists: False


In [ ]:
import requests
import os
import urllib3

# Suppress the warning caused by the council website's
# invalid/untrusted SSL certificate.
urllib3.disable_warnings(
    urllib3.exceptions.InsecureRequestWarning
)

budget_url = (
    "https://www.kabwecouncil.gov.zm/"
    "wp-content/uploads/2025/05/"
    "2025-KABWE-M-COUNCIL-OBB.pdf"
)

budget_pdf_path = (
    RAW_DIR /
    "2025_kabwe_obb_budget.pdf"
)

response = requests.get(
    budget_url,
    timeout=60,
    verify=False
)

print("Status code:", response.status_code)
print("Content type:", response.headers.get("Content-Type"))

if response.status_code == 200:
    with open(budget_pdf_path, "wb") as file:
        file.write(response.content)

    print("\nDownloaded successfully.")
    print(
        "File size:",
        round(
            os.path.getsize(budget_pdf_path) / 1024,
            2
        ),
        "KB"
    )
else:
    print("\nDownload failed.")
    print("Response:", response.text[:500])

Status code: 200
Content type: application/pdf

Downloaded successfully.
File size: 773.87 KB


In [ ]:
with open(budget_pdf_path, "rb") as file:
    first_bytes = file.read(20)

print("First bytes:", first_bytes)

if first_bytes.startswith(b"%PDF"):
    print("Valid PDF file.")
else:
    print("WARNING: Downloaded file does not appear to be a PDF.")

First bytes: b'%PDF-1.7\r\n%\xb5\xb5\xb5\xb5\r\n1 0'
Valid PDF file.


In [ ]:
import fitz

budget_doc = fitz.open(budget_pdf_path)

print("Number of pages:", len(budget_doc))

# Extract text from all pages
budget_pages = []

for page_number, page in enumerate(budget_doc, start=1):
    text = page.get_text()

    budget_pages.append({
        "page": page_number,
        "text": text
    })

print("Pages processed:", len(budget_pages))

Number of pages: 54
Pages processed: 54


In [ ]:
budget_keywords = [
    "CDF",
    "Community Projects",
    "Women and Youth",
    "Empowerment",
    "Bursaries",
    "CDF Administration",
    "Skills Development"
]

for keyword in budget_keywords:

    matches = []

    for item in budget_pages:
        if keyword.lower() in item["text"].lower():
            matches.append(item["page"])

    print(f"{keyword}: {matches[:30]}")

CDF: [6, 8, 9, 11, 12, 13, 48]
Community Projects: [8, 9, 11, 12, 13, 48]
Women and Youth: [8, 9, 11, 12, 13, 48]
Empowerment: [6, 8, 9, 11, 12, 13, 48]
Bursaries: [6, 8, 9, 11, 12, 13]
CDF Administration: [8, 9, 12]
Skills Development: [1, 6, 7, 8, 9, 10, 11, 12, 13, 27, 28, 48, 51]


In [ ]:
cdf_pages = []

for item in budget_pages:
    if "cdf" in item["text"].lower():
        cdf_pages.append(item)

print("Pages containing CDF:", len(cdf_pages))
print("CDF page numbers:", [item["page"] for item in cdf_pages])

Pages containing CDF: 7

PAGE 6
OUTPUT BASED  ANNUAL BUDGET
Page 6
920
5
HEA
D
KABWE MUNICIPAL COUNCIL
The summary estimates by economic classification shows that K49.0 million representing 27.04 percent 
of the total budget has been allocated to Personal Emoluments to facilitate payments of salaries and 
wages. In addition, K52.2 million representing 28.83 percent of the total budget has been allocated 
towards Assets acquisition such as procurement of utility vehicles and office equipment, renovation of 
council chamber and implementation of various CDF infrastructure development. 
Further, K33.5 million representing 18.52 percent of the total budget has been channelled to the Use of 
Goods and Services to facilitate the operation of the programmes.  Furthermore, Grants and other 
payments (Transfers) has been allocated K34.9 million representing 19.24 percent of the budget to 
cover Youth and Women Empowerments Grants, Boarding Secondary School and Skills Development 
Bursaries as w

In [ ]:
import pdfplumber

budget_tables = []

with pdfplumber.open(budget_pdf_path) as pdf:

    for page_number, page in enumerate(
        pdf.pages,
        start=1
    ):

        tables = page.extract_tables()

        for table in tables:
            budget_tables.append({
                "page": page_number,
                "table": table
            })

print("Total tables extracted:", len(budget_tables))
print(
    "Pages containing tables:",
    sorted(set(item["page"] for item in budget_tables))
)

Number of pages: 54
Page 1: 7 table(s)
Page 2: 1 table(s)
Page 3: 1 table(s)
Page 4: 1 table(s)
Page 5: 3 table(s)
Page 7: 1 table(s)
Page 8: 1 table(s)
Page 9: 1 table(s)
Page 11: 1 table(s)
Page 12: 2 table(s)
Page 13: 1 table(s)
Page 14: 2 table(s)
Page 15: 2 table(s)
Page 16: 2 table(s)
Page 17: 2 table(s)
Page 18: 3 table(s)
Page 19: 1 table(s)
Page 20: 1 table(s)
Page 21: 2 table(s)
Page 22: 1 table(s)
Page 23: 1 table(s)
Page 24: 1 table(s)
Page 25: 2 table(s)
Page 26: 1 table(s)
Page 27: 2 table(s)
Page 28: 1 table(s)
Page 29: 2 table(s)
Page 30: 1 table(s)
Page 31: 1 table(s)
Page 32: 1 table(s)
Page 33: 1 table(s)
Page 34: 2 table(s)
Page 35: 1 table(s)
Page 36: 2 table(s)
Page 37: 2 table(s)
Page 38: 2 table(s)
Page 39: 1 table(s)
Page 40: 3 table(s)
Page 41: 3 table(s)
Page 43: 1 table(s)
Page 44: 2 table(s)
Page 45: 2 table(s)
Page 46: 2 table(s)
Page 47: 3 table(s)
Page 48: 1 table(s)
Page 54: 1 table(s)

Total tables extracted: 78


In [ ]:
cdf_tables = []

for item in budget_tables:

    table = item["table"]

    table_text = " ".join(
        str(cell)
        for row in table
        for cell in row
        if cell is not None
    )

    if "cdf" in table_text.lower():

        cdf_tables.append(item)


print(
    "Tables containing CDF:",
    len(cdf_tables)
)

for item in cdf_tables[:20]:

    print(
        "Page:",
        item["page"]
    )

Tables containing CDF: 1
Page: 13


## 97. Structure the CDF Output Indicators

CDF output indicators identified from the 2025 Kabwe Municipal Council Output Based Budget are structured into a pandas DataFrame.

The dataset records indicator categories, specific indicators, historical targets and actual values for 2023 and 2024, and targets for 2025.

Values that are blank or unavailable in the source are represented as missing values rather than being replaced with zero.

Source metadata is added to preserve the connection between the structured indicators and the original council budget document.

In [ ]:
cdf_indicators = [
    ["Community Projects Completed", "Number of desks to be procured", 2, 8992, 1200, 0, 2500],
    ["Community Projects Completed", "Number of Maternity wings constructed", pd.NA, 2, 5, 2, 1],
    ["Community Projects Completed", "Number of Ambulances to be procured", pd.NA, pd.NA, 2, pd.NA, pd.NA],

    ["District Roads Graded", "Kilometer of roads graded", pd.NA, pd.NA, pd.NA, pd.NA, 100],

    ["Community, women and youth empowered", "Number of youth groups empowered", 10, 6, 15, 10, 20],
    ["Community, women and youth empowered", "Number of women groups empowered", 50, 61, 60, 50, 65],

    ["Empowerment loans disbursed", "Number of business entities accessing loans", 45, 51, 50, 45, 70],

    ["CDF activities administered", "Number of CDFC meetings held", 4, 12, 6, 4, 6],
    ["CDF activities administered", "Number of project monitoring visits carried out", 4, 4, 15, 4, 8],
    ["CDF activities administered", "Number of CDF projects branded", pd.NA, pd.NA, pd.NA, pd.NA, 39],

    ["Skills development beneficiaries sponsored", "Number of beneficiaries trained under skill development", 500, 1039, 600, 500, 800],

    ["Secondary School boarding beneficiaries sponsored", "Number of beneficiaries trained under secondary boarding", 40, 33, 60, 40, 50],
]

cdf_indicators_df = pd.DataFrame(
    cdf_indicators,
    columns=[
        "indicator_category",
        "indicator",
        "2023_target",
        "2023_actual",
        "2024_target",
        "2024_actual",
        "2025_target"
    ]
)

cdf_indicators_df.insert(0, "year", 2025)

cdf_indicators_df["source_url"] = (
    "https://www.kabwecouncil.gov.zm/"
    "wp-content/uploads/2025/05/"
    "2025-KABWE-M-COUNCIL-OBB.pdf"
)

cdf_indicators_df["source_document"] = (
    "2025 Kabwe Municipal Council OBB Budget"
)

,year,indicator_category,indicator,2023_target,2023_actual,2024_target,2024_actual,2025_target,source_url,source_document
0,2025,Community Projects Completed,Number of desks to be procured,2,8992,1200,0,2500,https://www.kabwecouncil.gov.zm/wp-content/uploads/2025/05/2025-KABWE-M-COUN...,2025 Kabwe Municipal Council OBB Budget
1,2025,Community Projects Completed,Number of Maternity wings constructed,<NA>,2,5,2,1,https://www.kabwecouncil.gov.zm/wp-content/uploads/2025/05/2025-KABWE-M-COUN...,2025 Kabwe Municipal Council OBB Budget
2,2025,Community Projects Completed,Number of Ambulances to be procured,<NA>,<NA>,2,<NA>,<NA>,https://www.kabwecouncil.gov.zm/wp-content/uploads/2025/05/2025-KABWE-M-COUN...,2025 Kabwe Municipal Council OBB Budget
3,2025,District Roads Graded,Kilometer of roads graded,<NA>,<NA>,<NA>,<NA>,100,https://www.kabwecouncil.gov.zm/wp-content/uploads/2025/05/2025-KABWE-M-COUN...,2025 Kabwe Municipal Council OBB Budget
4,2025,"Community, women and youth empowered",Number of youth groups empowered,10,6,15,10,20,https://www.kabwecouncil.gov.zm/wp-content/uploads/2025/05/2025-KABWE-M-COUN...,2025 Kabwe Municipal Council OBB Budget
5,2025,"Community, women and youth empowered",Number of women groups empowered,50,61,60,50,65,https://www.kabwecouncil.gov.zm/wp-content/uploads/2025/05/2025-KABWE-M-COUN...,2025 Kabwe Municipal Council OBB Budget
6,2025,Empowerment loans disbursed,Number of business entities accessing loans,45,51,50,45,70,https://www.kabwecouncil.gov.zm/wp-content/uploads/2025/05/2025-KABWE-M-COUN...,2025 Kabwe Municipal Council OBB Budget
7,2025,CDF activities administered,Number of CDFC meetings held,4,12,6,4,6,https://www.kabwecouncil.gov.zm/wp-content/uploads/2025/05/2025-KABWE-M-COUN...,2025 Kabwe Municipal Council OBB Budget
8,2025,CDF activities administered,Number of project monitoring visits carried out,4,4,15,4,8,https://www.kabwecouncil.gov.zm/wp-content/uploads/2025/05/2025-KABWE-M-COUN...,2025 Kabwe Municipal Council OBB Budget
9,2025,CDF activities administered,Number of CDF projects branded,<NA>,<NA>,<NA>,<NA>,39,https://www.kabwecouncil.gov.zm/wp-content/uploads/2025/05/2025-KABWE-M-COUN...,2025 Kabwe Municipal Council OBB Budget


In [ ]:
print("========== CDF INDICATOR VALIDATION ==========")

print("Rows:", len(cdf_indicators_df))
print("Columns:", len(cdf_indicators_df.columns))

print("\nColumns:")
print(cdf_indicators_df.columns.tolist())

print("\nMissing values:")
print(cdf_indicators_df.isna().sum())

print("\nDuplicate rows:")
print(cdf_indicators_df.duplicated().sum())

print("\nData types:")
print(cdf_indicators_df.dtypes)

========== CDF INDICATOR VALIDATION ==========
Rows: 12
Columns: 10

Columns:
['year', 'indicator_category', 'indicator', '2023_target', '2023_actual', '2024_target', '2024_actual', '2025_target', 'source_url', 'source_document']

Missing values:
year                  0
indicator_category    0
indicator             0
2023_target           4
2023_actual           3
2024_target           2
2024_actual           3
2025_target           1
source_url            0
source_document       0
dtype: int64

Duplicate rows:
0

Data types:
year                   int64
indicator_category       str
indicator                str
2023_target           object
2023_actual           object
2024_target           object
2024_actual           object
2025_target           object
source_url               str
source_document          str
dtype: object


In [ ]:
indicator_output_path = (
    PROCESSED_DIR /
    "db-unza26-csc4792-kabwe_cdf_output_indicators_2025.csv"
)

cdf_indicators_df.to_csv(
    indicator_output_path,
    sep="|",
    index=False,
    encoding="utf-8"
)

print("Saved:", indicator_output_path)

Saved: ../data/processed/db-unza26-csc4792-kabwe_cdf_output_indicators_2025.csv


In [ ]:
check_indicators_df = pd.read_csv(
    indicator_output_path,
    sep="|"
)

print("Rows:", len(check_indicators_df))
print("Columns:", len(check_indicators_df.columns))
print(check_indicators_df.columns.tolist())

Rows: 12
Columns: 10
['year', 'indicator_category', 'indicator', '2023_target', '2023_actual', '2024_target', '2024_actual', '2025_target', 'source_url', 'source_document']


,year,indicator_category,indicator,2023_target,2023_actual,2024_target,2024_actual,2025_target,source_url,source_document
0,2025,Community Projects Completed,Number of desks to be procured,2.0,8992.0,1200.0,0.0,2500.0,https://www.kabwecouncil.gov.zm/wp-content/uploads/2025/05/2025-KABWE-M-COUN...,2025 Kabwe Municipal Council OBB Budget
1,2025,Community Projects Completed,Number of Maternity wings constructed,NaN,2.0,5.0,2.0,1.0,https://www.kabwecouncil.gov.zm/wp-content/uploads/2025/05/2025-KABWE-M-COUN...,2025 Kabwe Municipal Council OBB Budget
2,2025,Community Projects Completed,Number of Ambulances to be procured,NaN,NaN,2.0,NaN,NaN,https://www.kabwecouncil.gov.zm/wp-content/uploads/2025/05/2025-KABWE-M-COUN...,2025 Kabwe Municipal Council OBB Budget
3,2025,District Roads Graded,Kilometer of roads graded,NaN,NaN,NaN,NaN,100.0,https://www.kabwecouncil.gov.zm/wp-content/uploads/2025/05/2025-KABWE-M-COUN...,2025 Kabwe Municipal Council OBB Budget
4,2025,"Community, women and youth empowered",Number of youth groups empowered,10.0,6.0,15.0,10.0,20.0,https://www.kabwecouncil.gov.zm/wp-content/uploads/2025/05/2025-KABWE-M-COUN...,2025 Kabwe Municipal Council OBB Budget
5,2025,"Community, women and youth empowered",Number of women groups empowered,50.0,61.0,60.0,50.0,65.0,https://www.kabwecouncil.gov.zm/wp-content/uploads/2025/05/2025-KABWE-M-COUN...,2025 Kabwe Municipal Council OBB Budget
6,2025,Empowerment loans disbursed,Number of business entities accessing loans,45.0,51.0,50.0,45.0,70.0,https://www.kabwecouncil.gov.zm/wp-content/uploads/2025/05/2025-KABWE-M-COUN...,2025 Kabwe Municipal Council OBB Budget
7,2025,CDF activities administered,Number of CDFC meetings held,4.0,12.0,6.0,4.0,6.0,https://www.kabwecouncil.gov.zm/wp-content/uploads/2025/05/2025-KABWE-M-COUN...,2025 Kabwe Municipal Council OBB Budget
8,2025,CDF activities administered,Number of project monitoring visits carried out,4.0,4.0,15.0,4.0,8.0,https://www.kabwecouncil.gov.zm/wp-content/uploads/2025/05/2025-KABWE-M-COUN...,2025 Kabwe Municipal Council OBB Budget
9,2025,CDF activities administered,Number of CDF projects branded,NaN,NaN,NaN,NaN,39.0,https://www.kabwecouncil.gov.zm/wp-content/uploads/2025/05/2025-KABWE-M-COUN...,2025 Kabwe Municipal Council OBB Budget


In [ ]:
cdf_indicators = [
    ["Community Projects Completed", "Number of desks to be procured", 2, 8992, 1200, 0, 2500],
    ["Community Projects Completed", "Number of Maternity wings constructed", pd.NA, 2, 5, 2, 1],
    ["Community Projects Completed", "Number of Ambulances to be procured", pd.NA, pd.NA, 2, pd.NA, pd.NA],

    ["District Roads Graded", "Kilometer of roads graded", 0, 0, 0, 0, 100],

    ["Community, women and youth empowered", "Number of youth groups empowered", 10, 6, 15, 10, 20],
    ["Community, women and youth empowered", "Number of women groups empowered", 50, 61, 60, 50, 65],

    ["Empowerment loans disbursed", "Number of business entities accessing loans", 45, 51, 50, 45, 70],

    ["CDF activities administered", "Number of CDFC meetings held", 4, 12, 6, 4, 6],
    ["CDF activities administered", "Number of project monitoring visits carried out", 4, 4, 15, 4, 8],
    ["CDF activities administered", "Number of CDF projects branded", 0, 0, 0, 0, 39],

    ["Skills development beneficiaries sponsored", "Number of beneficiaries trained under skill development", 500, 1039, 600, 500, 800],

    ["Secondary School boarding beneficiaries sponsored", "Number of beneficiaries trained under secondary boarding", 40, 33, 60, 40, 50],
]

cdf_indicators_df = pd.DataFrame(
    cdf_indicators,
    columns=[
        "indicator_category",
        "indicator",
        "2023_target",
        "2023_actual",
        "2024_target",
        "2024_actual",
        "2025_target"
    ]
)

cdf_indicators_df.insert(0, "year", 2025)

cdf_indicators_df["source_url"] = (
    "https://www.kabwecouncil.gov.zm/"
    "wp-content/uploads/2025/05/"
    "2025-KABWE-M-COUNCIL-OBB.pdf"
)

cdf_indicators_df["source_document"] = (
    "2025 Kabwe Municipal Council OBB Budget"
)

,year,indicator_category,indicator,2023_target,2023_actual,2024_target,2024_actual,2025_target,source_url,source_document
0,2025,Community Projects Completed,Number of desks to be procured,2,8992,1200,0,2500,https://www.kabwecouncil.gov.zm/wp-content/uploads/2025/05/2025-KABWE-M-COUN...,2025 Kabwe Municipal Council OBB Budget
1,2025,Community Projects Completed,Number of Maternity wings constructed,<NA>,2,5,2,1,https://www.kabwecouncil.gov.zm/wp-content/uploads/2025/05/2025-KABWE-M-COUN...,2025 Kabwe Municipal Council OBB Budget
2,2025,Community Projects Completed,Number of Ambulances to be procured,<NA>,<NA>,2,<NA>,<NA>,https://www.kabwecouncil.gov.zm/wp-content/uploads/2025/05/2025-KABWE-M-COUN...,2025 Kabwe Municipal Council OBB Budget
3,2025,District Roads Graded,Kilometer of roads graded,0,0,0,0,100,https://www.kabwecouncil.gov.zm/wp-content/uploads/2025/05/2025-KABWE-M-COUN...,2025 Kabwe Municipal Council OBB Budget
4,2025,"Community, women and youth empowered",Number of youth groups empowered,10,6,15,10,20,https://www.kabwecouncil.gov.zm/wp-content/uploads/2025/05/2025-KABWE-M-COUN...,2025 Kabwe Municipal Council OBB Budget
5,2025,"Community, women and youth empowered",Number of women groups empowered,50,61,60,50,65,https://www.kabwecouncil.gov.zm/wp-content/uploads/2025/05/2025-KABWE-M-COUN...,2025 Kabwe Municipal Council OBB Budget
6,2025,Empowerment loans disbursed,Number of business entities accessing loans,45,51,50,45,70,https://www.kabwecouncil.gov.zm/wp-content/uploads/2025/05/2025-KABWE-M-COUN...,2025 Kabwe Municipal Council OBB Budget
7,2025,CDF activities administered,Number of CDFC meetings held,4,12,6,4,6,https://www.kabwecouncil.gov.zm/wp-content/uploads/2025/05/2025-KABWE-M-COUN...,2025 Kabwe Municipal Council OBB Budget
8,2025,CDF activities administered,Number of project monitoring visits carried out,4,4,15,4,8,https://www.kabwecouncil.gov.zm/wp-content/uploads/2025/05/2025-KABWE-M-COUN...,2025 Kabwe Municipal Council OBB Budget
9,2025,CDF activities administered,Number of CDF projects branded,0,0,0,0,39,https://www.kabwecouncil.gov.zm/wp-content/uploads/2025/05/2025-KABWE-M-COUN...,2025 Kabwe Municipal Council OBB Budget


In [ ]:
print("========== INDICATOR DATASET VALIDATION ==========")

print("Rows:", len(cdf_indicators_df))
print("Columns:", len(cdf_indicators_df.columns))

print("\nColumn names:")
print(cdf_indicators_df.columns.tolist())

print("\nMissing values:")
print(cdf_indicators_df.isna().sum())

print("\nDuplicate rows:")
print(cdf_indicators_df.duplicated().sum())

print("\nIndicator categories:")
print(cdf_indicators_df["indicator_category"].value_counts())

========== INDICATOR DATASET VALIDATION ==========
Rows: 12
Columns: 10

Column names:
['year', 'indicator_category', 'indicator', '2023_target', '2023_actual', '2024_target', '2024_actual', '2025_target', 'source_url', 'source_document']

Missing values:
year                  0
indicator_category    0
indicator             0
2023_target           2
2023_actual           1
2024_target           0
2024_actual           1
2025_target           1
source_url            0
source_document       0
dtype: int64

Duplicate rows:
0

Indicator categories:
indicator_category
Community Projects Completed                         3
CDF activities administered                          3
Community, women and youth empowered                 2
District Roads Graded                                1
Empowerment loans disbursed                          1
Skills development beneficiaries sponsored           1
Secondary School boarding beneficiaries sponsored    1
Name: count, dtype: int64

Dataset:


,year,indicator_category,indicator,2023_target,2023_actual,2024_target,2024_actual,2025_target,source_url,source_document
0,2025,Community Projects Completed,Number of desks to be procured,2,8992,1200,0,2500,https://www.kabwecouncil.gov.zm/wp-content/uploads/2025/05/2025-KABWE-M-COUN...,2025 Kabwe Municipal Council OBB Budget
1,2025,Community Projects Completed,Number of Maternity wings constructed,<NA>,2,5,2,1,https://www.kabwecouncil.gov.zm/wp-content/uploads/2025/05/2025-KABWE-M-COUN...,2025 Kabwe Municipal Council OBB Budget
2,2025,Community Projects Completed,Number of Ambulances to be procured,<NA>,<NA>,2,<NA>,<NA>,https://www.kabwecouncil.gov.zm/wp-content/uploads/2025/05/2025-KABWE-M-COUN...,2025 Kabwe Municipal Council OBB Budget
3,2025,District Roads Graded,Kilometer of roads graded,0,0,0,0,100,https://www.kabwecouncil.gov.zm/wp-content/uploads/2025/05/2025-KABWE-M-COUN...,2025 Kabwe Municipal Council OBB Budget
4,2025,"Community, women and youth empowered",Number of youth groups empowered,10,6,15,10,20,https://www.kabwecouncil.gov.zm/wp-content/uploads/2025/05/2025-KABWE-M-COUN...,2025 Kabwe Municipal Council OBB Budget
5,2025,"Community, women and youth empowered",Number of women groups empowered,50,61,60,50,65,https://www.kabwecouncil.gov.zm/wp-content/uploads/2025/05/2025-KABWE-M-COUN...,2025 Kabwe Municipal Council OBB Budget
6,2025,Empowerment loans disbursed,Number of business entities accessing loans,45,51,50,45,70,https://www.kabwecouncil.gov.zm/wp-content/uploads/2025/05/2025-KABWE-M-COUN...,2025 Kabwe Municipal Council OBB Budget
7,2025,CDF activities administered,Number of CDFC meetings held,4,12,6,4,6,https://www.kabwecouncil.gov.zm/wp-content/uploads/2025/05/2025-KABWE-M-COUN...,2025 Kabwe Municipal Council OBB Budget
8,2025,CDF activities administered,Number of project monitoring visits carried out,4,4,15,4,8,https://www.kabwecouncil.gov.zm/wp-content/uploads/2025/05/2025-KABWE-M-COUN...,2025 Kabwe Municipal Council OBB Budget
9,2025,CDF activities administered,Number of CDF projects branded,0,0,0,0,39,https://www.kabwecouncil.gov.zm/wp-content/uploads/2025/05/2025-KABWE-M-COUN...,2025 Kabwe Municipal Council OBB Budget


In [ ]:
indicator_output_path = (
    PROCESSED_DIR /
    "db-unza26-csc4792-kabwe_cdf_output_indicators_2025.csv"
)

cdf_indicators_df.to_csv(
    indicator_output_path,
    sep="|",
    index=False,
    encoding="utf-8"
)

print("Saved:", indicator_output_path)

Saved: ../data/processed/db-unza26-csc4792-kabwe_cdf_output_indicators_2025.csv


In [ ]:
check_indicators_df = pd.read_csv(
    indicator_output_path,
    sep="|"
)

print("Rows:", len(check_indicators_df))
print("Columns:", len(check_indicators_df.columns))
print(check_indicators_df.columns.tolist())

Rows: 12
Columns: 10
['year', 'indicator_category', 'indicator', '2023_target', '2023_actual', '2024_target', '2024_actual', '2025_target', 'source_url', 'source_document']


,year,indicator_category,indicator,2023_target,2023_actual,2024_target,2024_actual,2025_target,source_url,source_document
0,2025,Community Projects Completed,Number of desks to be procured,2.0,8992.0,1200,0.0,2500.0,https://www.kabwecouncil.gov.zm/wp-content/uploads/2025/05/2025-KABWE-M-COUN...,2025 Kabwe Municipal Council OBB Budget
1,2025,Community Projects Completed,Number of Maternity wings constructed,NaN,2.0,5,2.0,1.0,https://www.kabwecouncil.gov.zm/wp-content/uploads/2025/05/2025-KABWE-M-COUN...,2025 Kabwe Municipal Council OBB Budget
2,2025,Community Projects Completed,Number of Ambulances to be procured,NaN,NaN,2,NaN,NaN,https://www.kabwecouncil.gov.zm/wp-content/uploads/2025/05/2025-KABWE-M-COUN...,2025 Kabwe Municipal Council OBB Budget
3,2025,District Roads Graded,Kilometer of roads graded,0.0,0.0,0,0.0,100.0,https://www.kabwecouncil.gov.zm/wp-content/uploads/2025/05/2025-KABWE-M-COUN...,2025 Kabwe Municipal Council OBB Budget
4,2025,"Community, women and youth empowered",Number of youth groups empowered,10.0,6.0,15,10.0,20.0,https://www.kabwecouncil.gov.zm/wp-content/uploads/2025/05/2025-KABWE-M-COUN...,2025 Kabwe Municipal Council OBB Budget
5,2025,"Community, women and youth empowered",Number of women groups empowered,50.0,61.0,60,50.0,65.0,https://www.kabwecouncil.gov.zm/wp-content/uploads/2025/05/2025-KABWE-M-COUN...,2025 Kabwe Municipal Council OBB Budget
6,2025,Empowerment loans disbursed,Number of business entities accessing loans,45.0,51.0,50,45.0,70.0,https://www.kabwecouncil.gov.zm/wp-content/uploads/2025/05/2025-KABWE-M-COUN...,2025 Kabwe Municipal Council OBB Budget
7,2025,CDF activities administered,Number of CDFC meetings held,4.0,12.0,6,4.0,6.0,https://www.kabwecouncil.gov.zm/wp-content/uploads/2025/05/2025-KABWE-M-COUN...,2025 Kabwe Municipal Council OBB Budget
8,2025,CDF activities administered,Number of project monitoring visits carried out,4.0,4.0,15,4.0,8.0,https://www.kabwecouncil.gov.zm/wp-content/uploads/2025/05/2025-KABWE-M-COUN...,2025 Kabwe Municipal Council OBB Budget
9,2025,CDF activities administered,Number of CDF projects branded,0.0,0.0,0,0.0,39.0,https://www.kabwecouncil.gov.zm/wp-content/uploads/2025/05/2025-KABWE-M-COUN...,2025 Kabwe Municipal Council OBB Budget


In [ ]:
source_record = pd.DataFrame([
    {
        "id": "S004",
        "source_name": "2025 Kabwe Municipal Council OBB Budget",
        "source_type": "PDF",
        "url": (
            "https://www.kabwecouncil.gov.zm/"
            "wp-content/uploads/2025/05/"
            "2025-KABWE-M-COUNCIL-OBB.pdf"
        ),
        "description": (
            "2025 Output Based Budget containing CDF budget allocations "
            "and key output/output indicator targets and actuals."
        )
    }
])


,id,source_name,source_type,url,description
0,S004,2025 Kabwe Municipal Council OBB Budget,PDF,https://www.kabwecouncil.gov.zm/wp-content/uploads/2025/05/2025-KABWE-M-COUN...,2025 Output Based Budget containing CDF budget allocations and key output/ou...


In [ ]:
source_inventory_df = pd.concat(
    [source_inventory_df, source_record],
    ignore_index=True
)

NameError: name 'source_inventory_df' is not defined

In [ ]:
source_inventory_path = (
    SOURCES_DIR /
    "cdf_source_inventory.csv"
)

source_inventory_df = pd.read_csv(
    source_inventory_path
)

print("Existing sources:", len(source_inventory_df))

Existing sources: 2


,source_id,source_name,source_type,url,description
0,S001,Kabwe Municipal Council Website,Website,https://www.kabwecouncil.gov.zm/,Official Kabwe Municipal Council website
1,S002,2024 Bwacha CDF Community Projects Submission,PDF,https://www.kabwecouncil.gov.zm/wp-content/uploads/2024/11/2024-Bwacha-commu...,2024 CDF community project submissions for Bwacha Constituency


In [ ]:
source_record = pd.DataFrame([
    {
        "id": "S004",
        "source_name": "2025 Kabwe Municipal Council OBB Budget",
        "source_type": "PDF",
        "url": (
            "https://www.kabwecouncil.gov.zm/"
            "wp-content/uploads/2025/05/"
            "2025-KABWE-M-COUNCIL-OBB.pdf"
        ),
        "description": (
            "2025 Output Based Budget containing CDF budget allocations "
            "and key output/output indicator targets and actuals."
        )
    }
])

source_inventory_df = pd.concat(
    [source_inventory_df, source_record],
    ignore_index=True
)

,source_id,source_name,source_type,url,description,id
0,S001,Kabwe Municipal Council Website,Website,https://www.kabwecouncil.gov.zm/,Official Kabwe Municipal Council website,NaN
1,S002,2024 Bwacha CDF Community Projects Submission,PDF,https://www.kabwecouncil.gov.zm/wp-content/uploads/2024/11/2024-Bwacha-commu...,2024 CDF community project submissions for Bwacha Constituency,NaN
2,NaN,2025 Kabwe Municipal Council OBB Budget,PDF,https://www.kabwecouncil.gov.zm/wp-content/uploads/2025/05/2025-KABWE-M-COUN...,2025 Output Based Budget containing CDF budget allocations and key output/ou...,S004


In [ ]:
source_inventory_df.to_csv(
    source_inventory_path,
    index=False,
    encoding="utf-8"
)

print("Source inventory updated successfully.")

Source inventory updated successfully.


In [ ]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin
import pandas as pd

BASE_URL = "https://www.kabwecouncil.gov.zm"

response = requests.get(
    BASE_URL,
    timeout=30,
    verify=False
)

print("Status code:", response.status_code)

soup = BeautifulSoup(response.text, "html.parser")

links = []

for link in soup.find_all("a", href=True):
    text = link.get_text(" ", strip=True)
    url = urljoin(BASE_URL, link["href"])

    links.append({
        "text": text,
        "url": url
    })

links_df = pd.DataFrame(links)

print("Links found:", len(links_df))

Status code: 200
Links found: 122


,text,url
0,,https://www.kabwecouncil.gov.zm#
1,Home,https://www.kabwecouncil.gov.zm/
2,About,https://www.kabwecouncil.gov.zm#
3,About Us,https://www.kabwecouncil.gov.zm/?page_id=2601
4,Mandate,https://www.kabwecouncil.gov.zm/?page_id=169


In [ ]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin
import pandas as pd

BASE_URL = "https://www.kabwecouncil.gov.zm"

response = requests.get(
    BASE_URL,
    timeout=30,
    verify=False
)

print("Status code:", response.status_code)

soup = BeautifulSoup(response.text, "html.parser")

links = []

for link in soup.find_all("a", href=True):
    text = link.get_text(" ", strip=True)
    url = urljoin(BASE_URL, link["href"])

    links.append({
        "text": text,
        "url": url
    })

links_df = pd.DataFrame(links)

print("Links found:", len(links_df))

Status code: 200
Links found: 122


,text,url
0,,https://www.kabwecouncil.gov.zm#
1,Home,https://www.kabwecouncil.gov.zm/
2,About,https://www.kabwecouncil.gov.zm#
3,About Us,https://www.kabwecouncil.gov.zm/?page_id=2601
4,Mandate,https://www.kabwecouncil.gov.zm/?page_id=169


In [ ]:
financial_statement_links = links_df[
    links_df["text"].str.contains(
        "financial statement|audited",
        case=False,
        na=False
    )
]

print("Financial statement links found:", len(financial_statement_links))

Financial statement links found: 0


,text,url


In [ ]:
financial_statement_urls = links_df[
    links_df["url"].str.contains(
        "financial|audit",
        case=False,
        na=False
    )
]

print("Financial-related URLs found:", len(financial_statement_urls))

Financial-related URLs found: 0


,text,url


In [ ]:
idp_links = links_df[
    links_df["text"].str.contains(
        "IDP|Integrated Development",
        case=False,
        na=False
    )
]

print("IDP links found:", len(idp_links))

IDP links found: 0


,text,url


In [ ]:
idp_urls = links_df[
    links_df["url"].str.contains(
        "idp|integrated",
        case=False,
        na=False
    )
]

print("IDP-related URLs found:", len(idp_urls))

IDP-related URLs found: 0


,text,url


In [ ]:
idp_links = links_df[
    links_df["text"].str.contains(
        "IDP|Integrated Development",
        case=False,
        na=False
    )
]

print("IDP links found:", len(idp_links))

IDP links found: 0


,text,url


In [ ]:
idp_urls = links_df[
    links_df["url"].str.contains(
        "idp|integrated",
        case=False,
        na=False
    )
]

print("IDP-related URLs found:", len(idp_urls))

IDP-related URLs found: 0


,text,url


In [ ]:
publications_url = "https://www.kabwecouncil.gov.zm/?page_id=195"

response = requests.get(
    publications_url,
    timeout=30,
    verify=False
)

print("Status code:", response.status_code)

publications_soup = BeautifulSoup(
    response.text,
    "html.parser"
)

publication_links = []

for link in publications_soup.find_all("a", href=True):
    text = link.get_text(" ", strip=True)
    url = urljoin(
        publications_url,
        link["href"]
    )

    publication_links.append({
        "text": text,
        "url": url
    })

publications_df = pd.DataFrame(publication_links)

print("Publication links found:", len(publications_df))
display(publications_df)

Status code: 200
Publication links found: 161


,text,url
0,,https://www.kabwecouncil.gov.zm/?page_id=195#
1,Home,https://www.kabwecouncil.gov.zm/
2,About,https://www.kabwecouncil.gov.zm/?page_id=195#
3,About Us,https://www.kabwecouncil.gov.zm/?page_id=2601
4,Mandate,https://www.kabwecouncil.gov.zm/?page_id=169
5,Who we are,https://www.kabwecouncil.gov.zm/?page_id=118
6,Departments,https://www.kabwecouncil.gov.zm/?page_id=770
7,office of the the town clerk,https://www.kabwecouncil.gov.zm/?page_id=2634
8,Dept of Human Resource and Administration,https://www.kabwecouncil.gov.zm/?page_id=2637
9,Dept of Health,https://www.kabwecouncil.gov.zm/?page_id=2640


In [ ]:
idp_publications = publications_df[
    publications_df["text"].str.contains(
        "IDP|Integrated Development|Development Plan",
        case=False,
        na=False
    )
]

print("IDP publications found:", len(idp_publications))
display(idp_publications)

IDP publications found: 2


,text,url
88,Kabwe District Integrated Development Plan (IDP) 2023 – 2028,https://www.kabwecouncil.gov.zm/wp-content/uploads/2024/09/Kabwe-Approved-ID...
89,Kabwe District Integrated Development Plan (IDP) 2023 – 2033 Citizen Version,https://www.kabwecouncil.gov.zm/wp-content/uploads/2024/09/Kabwe-District-Ci...


In [ ]:
idp_publication_urls = publications_df[
    publications_df["url"].str.contains(
        "idp|development",
        case=False,
        na=False
    )
]

print("IDP-related URLs found:", len(idp_publication_urls))
display(idp_publication_urls)

IDP-related URLs found: 6


,text,url
88,Kabwe District Integrated Development Plan (IDP) 2023 – 2028,https://www.kabwecouncil.gov.zm/wp-content/uploads/2024/09/Kabwe-Approved-ID...
89,Kabwe District Integrated Development Plan (IDP) 2023 – 2033 Citizen Version,https://www.kabwecouncil.gov.zm/wp-content/uploads/2024/09/Kabwe-District-Ci...
90,,https://www.kabwecouncil.gov.zm/wp-content/uploads/2024/09/Kabwe-Approved-ID...
130,GUIDELINES ON THE ESTABLISHMENT OF WARD DEVELOPMENT COMMITTEES,https://www.kabwecouncil.gov.zm/wp-content/uploads/2023/12/GUIDELINES-ON-THE...
139,The Constituency Development Fund Act No.11 of 2018,https://www.kabwecouncil.gov.zm/wp-content/uploads/2025/11/The-Constituency-...
140,The local Government street vending nuisances Amendment No. 2 Regulation 2018,https://www.kabwecouncil.gov.zm/wp-contenthttps://www.kabwecouncil.gov.zm/wp...


In [ ]:
idp_full = publications_df[
    publications_df["text"].str.contains(
        r"Kabwe District Integrated Development Plan \(IDP\) 2023",
        case=False,
        na=False
    )
]

display(idp_full[["text", "url"]])

,text,url
88,Kabwe District Integrated Development Plan (IDP) 2023 – 2028,https://www.kabwecouncil.gov.zm/wp-content/uploads/2024/09/Kabwe-Approved-ID...
89,Kabwe District Integrated Development Plan (IDP) 2023 – 2033 Citizen Version,https://www.kabwecouncil.gov.zm/wp-content/uploads/2024/09/Kabwe-District-Ci...


In [ ]:
idp_url = idp_full.iloc[0]["url"]

print("IDP URL:")
print(idp_url)

IDP URL:
https://www.kabwecouncil.gov.zm/wp-content/uploads/2024/09/Kabwe-Approved-IDP_Final-Version-1.pdf


In [ ]:
idp_pdf_path = (
    RAW_DIR /
    "2023_2028_kabwe_district_idp.pdf"
)

idp_response = requests.get(
    idp_url,
    timeout=120,
    verify=False
)

print("Status code:", idp_response.status_code)
print("Content-Type:", idp_response.headers.get("Content-Type"))
print("File size:", len(idp_response.content), "bytes")

if idp_response.status_code == 200:
    with open(idp_pdf_path, "wb") as file:
        file.write(idp_response.content)

    print("IDP downloaded successfully.")
else:
    print("Download failed.")

Status code: 200
Content-Type: application/pdf
File size: 17615491 bytes
IDP downloaded successfully.


In [ ]:
import os

print("File exists:", os.path.exists(idp_pdf_path))

if os.path.exists(idp_pdf_path):
    print(
        "File size:",
        round(
            os.path.getsize(idp_pdf_path) / 1024,
            2
        ),
        "KB"
    )

    with open(idp_pdf_path, "rb") as file:
        first_bytes = file.read(20)

    print("First bytes:", first_bytes)

    if first_bytes.startswith(b"%PDF"):
        print("Valid PDF file.")
    else:
        print("WARNING: File does not appear to be a PDF.")

File exists: True
File size: 17202.63 KB
First bytes: b'%PDF-1.7\r\n%\xb5\xb5\xb5\xb5\r\n1 0'
Valid PDF file.


In [ ]:
idp_doc = fitz.open(idp_pdf_path)

print("Number of pages:", len(idp_doc))

for page_number, page in enumerate(idp_doc, start=1):
    text = page.get_text()

    print(
        f"Page {page_number}: "
        f"{len(text)} characters"
    )

Number of pages: 354
Page 1: 261 characters
Page 2: 164 characters
Page 3: 452 characters
Page 4: 14 characters
Page 5: 1760 characters
Page 6: 1808 characters
Page 7: 1748 characters
Page 8: 2227 characters
Page 9: 2712 characters
Page 10: 2131 characters
Page 11: 3138 characters
Page 12: 475 characters
Page 13: 571 characters
Page 14: 4076 characters
Page 15: 878 characters
Page 16: 1005 characters
Page 17: 1262 characters
Page 18: 467 characters
Page 19: 2202 characters
Page 20: 1527 characters
Page 21: 2019 characters
Page 22: 604 characters
Page 23: 2005 characters
Page 24: 1057 characters
Page 25: 898 characters
Page 26: 712 characters
Page 27: 1646 characters
Page 28: 1220 characters
Page 29: 1218 characters
Page 30: 1159 characters
Page 31: 618 characters
Page 32: 1140 characters
Page 33: 137 characters
Page 34: 1244 characters
Page 35: 1631 characters
Page 36: 1003 characters
Page 37: 1478 characters
Page 38: 1000 characters
Page 39: 2202 characters
Page 40: 1128 characters
Pa

## Search the IDP for sections that are likely to contain
## useful structured data for our dataset.

In [ ]:


idp_keywords = [
    "population",
    "ward",
    "project",
    "development",
    "sector",
    "health",
    "education",
    "road",
    "water",
    "sanitation",
    "budget",
    "revenue",
    "implementation",
    "indicator"
]

for keyword in idp_keywords:
    matching_pages = []

    for page_number, page in enumerate(idp_doc, start=1):
        text = page.get_text()

        if keyword.lower() in text.lower():
            matching_pages.append(page_number)

    print(
        f"{keyword}: "
        f"{len(matching_pages)} pages"
    )
    print("Pages:", matching_pages[:30])

population: 78 pages
Pages: [5, 9, 11, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 42, 43, 46, 54, 55, 56, 58, 59, 60, 62, 63, 64, 68, 70]
ward: 125 pages
Pages: [5, 6, 8, 9, 16, 21, 22, 23, 26, 28, 29, 32, 33, 38, 56, 58, 60, 69, 74, 78, 81, 82, 85, 87, 88, 90, 97, 104, 113, 124]
project: 39 pages
Pages: [3, 6, 7, 9, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 54, 55, 57, 83, 88, 90, 91, 92, 94, 109, 111, 128, 135]
development: 194 pages
Pages: [1, 2, 3, 5, 6, 7, 8, 9, 11, 12, 13, 14, 16, 17, 19, 20, 25, 26, 32, 35, 37, 39, 40, 43, 45, 48, 55, 57, 58, 62]
sector: 66 pages
Pages: [9, 11, 14, 15, 19, 27, 30, 35, 38, 47, 48, 51, 56, 60, 61, 63, 64, 65, 66, 68, 70, 77, 78, 80, 83, 84, 88, 89, 90, 93]
health: 93 pages
Pages: [10, 11, 16, 17, 25, 30, 32, 38, 39, 48, 57, 58, 60, 68, 88, 89, 90, 91, 92, 94, 96, 97, 112, 114, 115, 116, 117, 123, 124, 125]
education: 61 pages
Pages: [9, 10, 11, 14, 16, 17, 25, 30, 32, 36, 37, 39, 61, 88, 89, 91, 92, 97, 98, 99, 102, 103, 104, 10

## Inspect the IDP pages most likely to contain
## development projects, sector information and priorities.

In [ ]:


for page_number in range(88, 95):
    page = idp_doc[page_number - 1]
    text = page.get_text()

    print("\n" + "=" * 100)
    print(f"PAGE {page_number}")
    print("=" * 100)
    print(text[:7000])


PAGE 88
 
 
70 
BWACHA 
MUWOWO EAST 
3,235 
670 
52.4 
BWACHA 
MUWOWO WEST 
1,953 
392 
86.5 
BWACHA 
NGUNGU 
6,068 
1,251 
21.1 
BWACHA 
ZAMBEZI 
3,785 
757 
96.2 
Source: ZAMSTAT, 20222 
 
An analysis by ward shows that over 90% of the households in Chililalila, Chinyanja, 
Kan’gomba, Makululu and Zambezi did not have access to electricity whereas only 
Bwacha, Chimanimani and Ngungu showed below 30% of households without access 
to electricity. The high percentage number of household without access to electricity 
denotes heavy reliance on unsustainable sources of energy such as charcoal and 
firewood which contributes to environmental degradation. 
 
3.7.6 
Issues Arising from The Public Participation Process 
Submissions from the Public participation process showed that rural and peri urban 
wards such as Chinyanja, Muwowo East and West, Mpima, Zambezi and Luansanse 
were not connected to the power grid and that the community relied on firewood and 
charcoal which was detrimental

## Search the IDP for pages containing likely project/programme tables.

In [ ]:

keywords = [
    "Project List",
    "Projects",
    "Proposed Projects",
    "Capital Projects",
    "Development Projects",
    "Project Name",
    "Programme",
    "Projects and Programmes",
    "Implementation Plan"
]

for keyword in keywords:
    matches = []

    for page_number, page in enumerate(idp_doc, start=1):
        text = page.get_text()

        if keyword.lower() in text.lower():
            matches.append(page_number)

    print(f"{keyword}: {matches}")

Project List: []
Projects: [57, 111, 212, 351]
Proposed Projects: []
Capital Projects: []
Development Projects: []
Project Name: []
Programme: [15, 16, 38, 57, 62, 78, 81, 82, 90, 110, 114, 130, 131, 132, 146, 222, 232, 294, 296, 297, 351, 352]
Projects and Programmes: []
Implementation Plan: [11, 13, 47, 233, 249, 280]


In [ ]:
print("========== OUTPUT INDICATOR VALIDATION ==========")

print("Rows:", len(cdf_indicators_df))
print("Columns:", len(cdf_indicators_df.columns))

print("\nColumn names:")
print(cdf_indicators_df.columns.tolist())

print("\nMissing values:")
print(cdf_indicators_df.isna().sum())

print("\nDuplicate rows:")
print(cdf_indicators_df.duplicated().sum())

print("\nOutputs:")
print(cdf_indicators_df["output"].value_counts())

print("\nYear:")
print(cdf_indicators_df["year"].unique())

========== OUTPUT INDICATOR VALIDATION ==========
Rows: 12
Columns: 10

Column names:
['year', 'output', 'output_indicator', 'target_2023', 'actual_2023', 'target_2024', 'actual_2024', 'target_2025', 'source_url', 'source_document']

Missing values:
year                0
output              0
output_indicator    0
target_2023         1
actual_2023         1
target_2024         0
actual_2024         1
target_2025         1
source_url          0
source_document     0
dtype: int64

Duplicate rows:
0

Outputs:
output
Community Projects Completed                         3
CDF activities administered                          3
Community, women and youth empowered                 2
District Roads Graded                                1
Empowerment loans disbursed                          1
Skills development beneficiaries sponsored           1
Secondary School boarding beneficiaries sponsored    1
Name: count, dtype: int64

Year:
[2025]


In [ ]:
indicator_output_path = (
    PROCESSED_DIR /
    "db-unza26-csc4792-kabwe_cdf_output_indicators_2025.csv"
)

cdf_indicators_df.to_csv(
    indicator_output_path,
    sep="|",
    index=False,
    encoding="utf-8"
)

print("Saved:", indicator_output_path)

Saved: ../data/processed/db-unza26-csc4792-kabwe_cdf_output_indicators_2025.csv


In [ ]:
check_indicators_df = pd.read_csv(
    indicator_output_path,
    sep="|"
)

print("Rows:", len(check_indicators_df))
print("Columns:", len(check_indicators_df.columns))
print(check_indicators_df.columns.tolist())

display(check_indicators_df)

Rows: 12
Columns: 10
['year', 'output', 'output_indicator', 'target_2023', 'actual_2023', 'target_2024', 'actual_2024', 'target_2025', 'source_url', 'source_document']


,year,output,output_indicator,target_2023,actual_2023,target_2024,actual_2024,target_2025,source_url,source_document
0,2025,Community Projects Completed,Number of desks to be procured,0.0,8992.0,1200,0.0,2500.0,https://www.kabwecouncil.gov.zm/wp-content/uploads/2025/05/2025-KABWE-M-COUN...,2025 Kabwe Municipal Council OBB Budget
1,2025,Community Projects Completed,Number of Maternity wings constructed,2.0,2.0,5,2.0,1.0,https://www.kabwecouncil.gov.zm/wp-content/uploads/2025/05/2025-KABWE-M-COUN...,2025 Kabwe Municipal Council OBB Budget
2,2025,Community Projects Completed,Number of Ambulances to be procured,NaN,NaN,2,NaN,NaN,https://www.kabwecouncil.gov.zm/wp-content/uploads/2025/05/2025-KABWE-M-COUN...,2025 Kabwe Municipal Council OBB Budget
3,2025,District Roads Graded,Kilometer of roads graded,0.0,0.0,0,0.0,100.0,https://www.kabwecouncil.gov.zm/wp-content/uploads/2025/05/2025-KABWE-M-COUN...,2025 Kabwe Municipal Council OBB Budget
4,2025,"Community, women and youth empowered",Number of youth groups empowered,10.0,6.0,15,10.0,20.0,https://www.kabwecouncil.gov.zm/wp-content/uploads/2025/05/2025-KABWE-M-COUN...,2025 Kabwe Municipal Council OBB Budget
5,2025,"Community, women and youth empowered",Number of women groups empowered,50.0,61.0,60,50.0,65.0,https://www.kabwecouncil.gov.zm/wp-content/uploads/2025/05/2025-KABWE-M-COUN...,2025 Kabwe Municipal Council OBB Budget
6,2025,Empowerment loans disbursed,Number of business entities accessing loans,45.0,51.0,50,45.0,70.0,https://www.kabwecouncil.gov.zm/wp-content/uploads/2025/05/2025-KABWE-M-COUN...,2025 Kabwe Municipal Council OBB Budget
7,2025,CDF activities administered,Number of CDFC meetings held,4.0,12.0,6,4.0,6.0,https://www.kabwecouncil.gov.zm/wp-content/uploads/2025/05/2025-KABWE-M-COUN...,2025 Kabwe Municipal Council OBB Budget
8,2025,CDF activities administered,Number of project monitoring visits carried out,4.0,4.0,15,4.0,8.0,https://www.kabwecouncil.gov.zm/wp-content/uploads/2025/05/2025-KABWE-M-COUN...,2025 Kabwe Municipal Council OBB Budget
9,2025,CDF activities administered,Number of CDF projects branded,0.0,0.0,0,0.0,39.0,https://www.kabwecouncil.gov.zm/wp-content/uploads/2025/05/2025-KABWE-M-COUN...,2025 Kabwe Municipal Council OBB Budget


## Final Task 1 Dataset Quality Control

The final processed CDF datasets are subjected to a consolidated quality-control check before submission.

The checks verify that:

- all expected processed CSV files exist;
- the files can be reloaded using the required pipe (`|`) delimiter;
- row and column counts are available;
- duplicate records are identified;
- missing values are quantified; and
- an unwanted pandas index column (`Unnamed: 0`) has not been exported.

The results provide a final overview of the structural quality of the Task 1 datasets.

In [ ]:
import os
import pandas as pd

processed_dir = PROCESSED_DIR

datasets = {
    "2024 CDF Projects": (
        "db-unza26-csc4792-kabwe_cdf_projects_2024.csv"
    ),
    "2025 Proposed CDF Projects": (
        "db-unza26-csc4792-kabwe_cdf_projects_2025.csv"
    ),
    "2024-2025 CDF Budget": (
        "db-unza26-csc4792-kabwe_cdf_budget_2024_2025.csv"
    ),
    "2025 CDF Output Indicators": (
        "db-unza26-csc4792-kabwe_cdf_output_indicators_2025.csv"
    )
}

print("=" * 70)
print("FINAL TASK 1 QUALITY CONTROL")
print("=" * 70)

quality_results = []

for dataset_name, filename in datasets.items():

    path = os.path.join(processed_dir, filename)

    print("\n" + "-" * 70)
    print(dataset_name)
    print("-" * 70)

    # Check file exists
    file_exists = os.path.exists(path)
    print("File exists:", file_exists)

    if not file_exists:
        print("WARNING: File not found.")
        continue

    # Read using required pipe delimiter
    df = pd.read_csv(path, sep="|")

    print("Rows:", len(df))
    print("Columns:", len(df.columns))

    print("Columns:")
    print(df.columns.tolist())

    print("Duplicate rows:", df.duplicated().sum())

    print("Missing values:", df.isna().sum().sum())

    print(
        "Index column present:",
        "Unnamed: 0" in df.columns
    )

    quality_results.append({
        "dataset": dataset_name,
        "filename": filename,
        "file_exists": file_exists,
        "rows": len(df),
        "columns": len(df.columns),
        "duplicate_rows": df.duplicated().sum(),
        "missing_values": df.isna().sum().sum(),
        "has_index_column": "Unnamed: 0" in df.columns
    })


quality_df = pd.DataFrame(quality_results)

print("\n" + "=" * 70)
print("QUALITY CONTROL SUMMARY")
print("=" * 70)

display(quality_df)

FINAL TASK 1 QUALITY CONTROL

----------------------------------------------------------------------
2024 CDF Projects
----------------------------------------------------------------------
File exists: True
Rows: 79
Columns: 15
Columns:
['project_number', 'year', 'constituency', 'project_name', 'project_description', 'project_type', 'ward', 'project_site', 'application_amount', 'engineers_estimate', 'approved_amount', 'contract_amount', 'status', 'source_url', 'source_document']
Duplicate rows: 0
Missing values: 432
Index column present: False

----------------------------------------------------------------------
2025 Proposed CDF Projects
----------------------------------------------------------------------
File exists: True
Rows: 33
Columns: 9
Columns:
['no', 'year', 'constituency', 'project', 'ward', 'sector', 'comment', 'source_url', 'source_document']
Duplicate rows: 0
Missing values: 0
Index column present: False

---------------------------------------------------------------

,dataset,filename,file_exists,rows,columns,duplicate_rows,missing_values,has_index_column
0,2024 CDF Projects,db-unza26-csc4792-kabwe_cdf_projects_2024.csv,True,79,15,0,432,False
1,2025 Proposed CDF Projects,db-unza26-csc4792-kabwe_cdf_projects_2025.csv,True,33,9,0,0,False
2,2025 CDF Output Indicators,db-unza26-csc4792-kabwe_cdf_output_indicators_2025.csv,True,12,10,0,4,False


In [ ]:
print("========== 2025 OUTPUT INDICATOR MISSING VALUES ==========")

indicator_missing = cdf_indicators_df[
    cdf_indicators_df.isna().any(axis=1)
]

display(indicator_missing)

========== 2025 OUTPUT INDICATOR MISSING VALUES ==========


,year,output,output_indicator,target_2023,actual_2023,target_2024,actual_2024,target_2025,source_url,source_document
2,2025,Community Projects Completed,Number of Ambulances to be procured,<NA>,<NA>,2,<NA>,<NA>,https://www.kabwecouncil.gov.zm/wp-content/uploads/2025/05/2025-KABWE-M-COUN...,2025 Kabwe Municipal Council OBB Budget


In [ ]:
print("Missing values by column:")

print(
    cdf_indicators_df.isna().sum()
)

Missing values by column:
year                0
output              0
output_indicator    0
target_2023         1
actual_2023         1
target_2024         0
actual_2024         1
target_2025         1
source_url          0
source_document     0
dtype: int64


In [ ]:
import os

budget_output_path = (
    PROCESSED_DIR /
    "db-unza26-csc4792-kabwe_cdf_budget_2024_2025.csv"
)

print(
    "Budget file exists:",
    os.path.exists(budget_output_path)
)

Budget file exists: True


## Export the 2024–2025 CDF Budget Dataset

The 2024 and 2025 CDF budget allocations identified from the Kabwe Municipal Council Output Based Budget are structured into a dedicated dataset.

The dataset records the major CDF budget categories and their corresponding budget amounts for 2024 and 2025. Council, fund, and source metadata are included to maintain context and traceability.

The final dataset is exported using the pipe (`|`) delimiter required by the assignment and saved in the `data/processed/` directory.

In [ ]:
cdf_budget_data = [
    ["Constituency Development", 61271284, 72116301],
    ["Community Projects", 34924632, 43925383],
    ["Women and Youth Empowerment", 11641544, 12456452],
    ["CDF Administration", 3063564, 3278014],
    ["Secondary School and Skills Development Bursaries", 11641544, 12456452],
]

cdf_budget_df = pd.DataFrame(
    cdf_budget_data,
    columns=[
        "budget_category",
        "budget_2024",
        "budget_2025"
    ]
)

cdf_budget_df.insert(
    0,
    "council",
    "Kabwe Municipal Council"
)

cdf_budget_df.insert(
    1,
    "fund",
    "Constituency Development Fund (CDF)"
)

cdf_budget_df["source_url"] = (
    "https://www.kabwecouncil.gov.zm/"
    "wp-content/uploads/2025/05/"
    "2025-KABWE-M-COUNCIL-OBB.pdf"
)

cdf_budget_df["source_document"] = (
    "2025 Kabwe Municipal Council OBB Budget"
)

budget_output_path = (
    "../data/processed/"
    "db-unza26-csc4792-kabwe_cdf_budget_2024_2025.csv"
)

cdf_budget_df.to_csv(
    budget_output_path,
    sep="|",
    index=False,
    encoding="utf-8"
)

print("Saved:", budget_output_path)

Saved: ../data/processed/db-unza26-csc4792-kabwe_cdf_budget_2024_2025.csv


In [ ]:
print("========== BUDGET VALIDATION ==========")

print("Rows:", len(cdf_budget_df))
print("Columns:", len(cdf_budget_df.columns))

print("\n2024 CDF total:",
      cdf_budget_df["budget_2024"].sum())

print("2025 CDF total:",
      cdf_budget_df["budget_2025"].sum())

print("\nDuplicate rows:",
      cdf_budget_df.duplicated().sum())

print("\nMissing values:")
print(cdf_budget_df.isna().sum())

========== BUDGET VALIDATION ==========
Rows: 5
Columns: 7

2024 CDF total: 122542568
2025 CDF total: 144232602

Duplicate rows: 0

Missing values:
council            0
fund               0
budget_category    0
budget_2024        0
budget_2025        0
source_url         0
source_document    0
dtype: int64


In [ ]:
print(
    "Budget CSV exists:",
    os.path.exists(budget_output_path)
)

print(
    "File size:",
    os.path.getsize(budget_output_path),
    "bytes"
)

Budget CSV exists: True
File size: 1255 bytes


## Final Task 1 Dataset Quality Control

The final processed CDF datasets are subjected to a consolidated quality-control check before submission.

The checks verify that:

- all expected processed CSV files exist;
- the files can be reloaded using the required pipe (`|`) delimiter;
- row and column counts are available;
- duplicate records are identified;
- missing values are quantified; and
- an unwanted pandas index column (`Unnamed: 0`) has not been exported.

The results provide a final overview of the structural quality of the Task 1 datasets.

In [ ]:
import os
import pandas as pd

processed_dir = PROCESSED_DIR

datasets = {
    "2024 CDF Projects": (
        "db-unza26-csc4792-kabwe_cdf_projects_2024.csv"
    ),
    "2025 Proposed CDF Projects": (
        "db-unza26-csc4792-kabwe_cdf_projects_2025.csv"
    ),
    "2024-2025 CDF Budget": (
        "db-unza26-csc4792-kabwe_cdf_budget_2024_2025.csv"
    ),
    "2025 CDF Output Indicators": (
        "db-unza26-csc4792-kabwe_cdf_output_indicators_2025.csv"
    )
}

quality_results = []

print("=" * 80)
print("FINAL TASK 1 QUALITY CONTROL")
print("=" * 80)

for dataset_name, filename in datasets.items():

    path = os.path.join(processed_dir, filename)

    exists = os.path.exists(path)

    print("\n" + "-" * 80)
    print(dataset_name)
    print("-" * 80)

    print("File exists:", exists)

    if not exists:
        print("WARNING: File not found.")
        continue

    df = pd.read_csv(path, sep="|")

    print("Rows:", len(df))
    print("Columns:", len(df.columns))
    print("Duplicate rows:", df.duplicated().sum())
    print("Missing values:", df.isna().sum().sum())
    print("Accidental index column:", "Unnamed: 0" in df.columns)

    quality_results.append({
        "dataset": dataset_name,
        "filename": filename,
        "file_exists": exists,
        "rows": len(df),
        "columns": len(df.columns),
        "duplicate_rows": df.duplicated().sum(),
        "missing_values": df.isna().sum().sum(),
        "has_index_column": "Unnamed: 0" in df.columns
    })

quality_df = pd.DataFrame(quality_results)

print("\n" + "=" * 80)
print("FINAL QUALITY CONTROL SUMMARY")
print("=" * 80)

display(quality_df)

FINAL TASK 1 QUALITY CONTROL

--------------------------------------------------------------------------------
2024 CDF Projects
--------------------------------------------------------------------------------
File exists: True
Rows: 79
Columns: 15
Duplicate rows: 0
Missing values: 432
Accidental index column: False

--------------------------------------------------------------------------------
2025 Proposed CDF Projects
--------------------------------------------------------------------------------
File exists: True
Rows: 33
Columns: 9
Duplicate rows: 0
Missing values: 0
Accidental index column: False

--------------------------------------------------------------------------------
2024-2025 CDF Budget
--------------------------------------------------------------------------------
File exists: True
Rows: 5
Columns: 7
Duplicate rows: 0
Missing values: 0
Accidental index column: False

--------------------------------------------------------------------------------
2025 CDF Output I

,dataset,filename,file_exists,rows,columns,duplicate_rows,missing_values,has_index_column
0,2024 CDF Projects,db-unza26-csc4792-kabwe_cdf_projects_2024.csv,True,79,15,0,432,False
1,2025 Proposed CDF Projects,db-unza26-csc4792-kabwe_cdf_projects_2025.csv,True,33,9,0,0,False
2,2024-2025 CDF Budget,db-unza26-csc4792-kabwe_cdf_budget_2024_2025.csv,True,5,7,0,0,False
3,2025 CDF Output Indicators,db-unza26-csc4792-kabwe_cdf_output_indicators_2025.csv,True,12,10,0,4,False


## Final Source Inventory

The final source inventory records the official digital sources used or consulted during the Kabwe CDF dataset development.

Each source is assigned a unique identifier and includes its name, type, URL, and description to support data provenance and reproducibility.

The inventory is saved as `source_inventory.csv` in the `sources/` directory.

In [ ]:
source_inventory_data = [
    {
        "id": "S001",
        "source_name": "Kabwe Municipal Council Website",
        "source_type": "Website",
        "url": "https://www.kabwecouncil.gov.zm",
        "description": "Official Kabwe Municipal Council website and digital source directory."
    },
    {
        "id": "S002",
        "source_name": "2024 Bwacha CDF Community Projects Submission",
        "source_type": "PDF",
        "url": (
            "https://www.kabwecouncil.gov.zm/"
            "wp-content/uploads/2024/11/"
            "2024-Bwacha-community-projects-Recieved.pdf"
        ),
        "description": "2024 CDF community project submissions for Bwacha Constituency."
    },
    {
        "id": "S003",
        "source_name": "2024 Kabwe Central CDF Community Projects Submission",
        "source_type": "PDF",
        "url": (
            "https://www.kabwecouncil.gov.zm/"
            "wp-content/uploads/2024/11/"
            "2024-Community-Projects-Kabwe-Central-received.pdf"
        ),
        "description": "2024 CDF community project submissions for Kabwe Central Constituency."
    },
    {
        "id": "S004",
        "source_name": "2025 Kabwe Municipal Council OBB Budget",
        "source_type": "PDF",
        "url": (
            "https://www.kabwecouncil.gov.zm/"
            "wp-content/uploads/2025/05/"
            "2025-KABWE-M-COUNCIL-OBB.pdf"
        ),
        "description": "2025 Output Based Budget containing CDF budget allocations and output indicators."
    },
    {
        "id": "S005",
        "source_name": "Kabwe District Integrated Development Plan 2023-2028",
        "source_type": "PDF",
        "url": (
            "https://www.kabwecouncil.gov.zm/"
            "wp-content/uploads/2024/09/"
            "Kabwe-Approved-IDP_Final-Version-1.pdf"
        ),
        "description": "Approved Kabwe District Integrated Development Plan consulted during source discovery."
    },
    {
        "id": "S006",
        "source_name": "2025 Proposed CDF Projects",
        "source_type": "PDF",
        "url": (
            "https://www.kabwecouncil.gov.zm/"
            "wp-content/uploads/2025/08/"
            "Proposed-2025-CDF-projects.pdf"
        ),
        "description": "2025 proposed CDF projects for Kabwe constituencies."
    },
    {
        "id": "S007",
        "source_name": "2025 Kabwe Municipal Council Annual Financial Statement",
        "source_type": "PDF",
        "url": (
            "https://www.kabwecouncil.gov.zm/"
            "wp-content/uploads/2025/08/"
            "2025-BI-ANNUAL-FINANCIAL-STATEMENT-KABWE-M-COUNCIL-1-1.pdf"
        ),
        "description": "2025 annual financial statement containing CDF receipts, payments, and financial information."
    }
]

source_inventory_df = pd.DataFrame(source_inventory_data)

source_inventory_path = (
    SOURCES_DIR /
    "cdf_source_inventory.csv"
)

source_inventory_df.to_csv(
    source_inventory_path,
    index=False,
    encoding="utf-8"
)

print("Saved:", source_inventory_path)
print("Sources:", len(source_inventory_df))

Saved: ../sources/source_inventory.csv


,id,source_name,source_type,url,description
0,S001,Kabwe Municipal Council Website,Website,https://www.kabwecouncil.gov.zm,Official Kabwe Municipal Council website and digital source directory.
1,S002,2024 Bwacha CDF Community Projects Submission,PDF,https://www.kabwecouncil.gov.zm/wp-content/uploads/2024/11/2024-Bwacha-commu...,2024 CDF community project submissions for Bwacha Constituency.
2,S003,2024 Kabwe Central CDF Community Projects Submission,PDF,https://www.kabwecouncil.gov.zm/wp-content/uploads/2024/11/2024-Community-Pr...,2024 CDF community project submissions for Kabwe Central Constituency.
3,S004,2025 Kabwe Municipal Council OBB Budget,PDF,https://www.kabwecouncil.gov.zm/wp-content/uploads/2025/05/2025-KABWE-M-COUN...,2025 Output Based Budget containing CDF budget allocations and output indica...
4,S005,Kabwe District Integrated Development Plan 2023-2028,PDF,https://www.kabwecouncil.gov.zm/wp-content/uploads/2024/09/Kabwe-Approved-ID...,Approved Kabwe District Integrated Development Plan consulted during source ...


## Final Task 1 Data Consistency Check

A final consistency check is performed across the processed CDF datasets and source inventory before submission.

The check verifies that all required processed CSV files exist, the source inventory contains complete and unique source records, and the processed datasets can be successfully read using the required pipe (`|`) delimiter.

The final result reports whether Task 1 passes these consistency checks.

In [ ]:
# ============================================================
# FINAL TASK 1 CONSISTENCY CHECK
# ============================================================

import os
import pandas as pd

processed_dir = PROCESSED_DIR
source_inventory_path = (
    SOURCES_DIR /
    "cdf_source_inventory.csv"
)

# ------------------------------------------------------------
# 1. Check processed CSV files
# ------------------------------------------------------------

expected_files = [
    "db-unza26-csc4792-kabwe_cdf_projects_2024.csv",
    "db-unza26-csc4792-kabwe_cdf_projects_2025.csv",
    "db-unza26-csc4792-kabwe_cdf_budget_2024_2025.csv",
    "db-unza26-csc4792-kabwe_cdf_output_indicators_2025.csv",
    "db-unza26-csc4792-kabwe_cdf_financial_statement_2025.csv",
    "db-unza26-csc4792-kabwe_cdf_disbursements_2025.csv"
]

print("=" * 80)
print("PROCESSED DATASET FILE CHECK")
print("=" * 80)

file_results = []

for filename in expected_files:

    path = os.path.join(processed_dir, filename)

    exists = os.path.exists(path)

    size = os.path.getsize(path) if exists else 0

    file_results.append({
        "filename": filename,
        "exists": exists,
        "size_bytes": size
    })

file_check_df = pd.DataFrame(file_results)

display(file_check_df)


# ------------------------------------------------------------
# 2. Check source inventory
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("SOURCE INVENTORY CHECK")
print("=" * 80)

source_check_df = pd.read_csv(source_inventory_path)

print("Number of sources:", len(source_check_df))
print("Columns:", source_check_df.columns.tolist())

print("\nMissing values:")
print(source_check_df.isna().sum())

print("\nDuplicate source IDs:")
print(source_check_df["id"].duplicated().sum())

display(source_check_df)


# ------------------------------------------------------------
# 3. Check CSV delimiters by reading them with '|'
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("PIPE-DELIMITER CHECK")
print("=" * 80)

delimiter_results = []

for filename in expected_files:

    path = os.path.join(processed_dir, filename)

    if os.path.exists(path):

        df = pd.read_csv(path, sep="|")

        delimiter_results.append({
            "filename": filename,
            "rows": len(df),
            "columns": len(df.columns),
            "pipe_delimiter_working": len(df.columns) > 1
        })

delimiter_check_df = pd.DataFrame(delimiter_results)

display(delimiter_check_df)


# ------------------------------------------------------------
# 4. Overall result
# ------------------------------------------------------------

all_files_exist = file_check_df["exists"].all()

source_inventory_valid = (
    source_check_df["id"].duplicated().sum() == 0
    and source_check_df.isna().sum().sum() == 0
)

delimiter_valid = delimiter_check_df[
    "pipe_delimiter_working"
].all()

print("\n" + "=" * 80)
print("OVERALL RESULT")
print("=" * 80)

print("All required CSV files exist:", all_files_exist)
print("Source inventory is complete:", source_inventory_valid)
print("Pipe delimiter is working:", delimiter_valid)

if all_files_exist and source_inventory_valid and delimiter_valid:
    print("\nTASK 1 DATA CONSISTENCY CHECK: PASSED")
else:
    print("\nTASK 1 DATA CONSISTENCY CHECK: NEEDS ATTENTION")

PROCESSED DATASET FILE CHECK


,filename,exists,size_bytes
0,db-unza26-csc4792-kabwe_cdf_projects_2024.csv,True,26450
1,db-unza26-csc4792-kabwe_cdf_projects_2025.csv,True,9599
2,db-unza26-csc4792-kabwe_cdf_budget_2024_2025.csv,True,1255
3,db-unza26-csc4792-kabwe_cdf_output_indicators_2025.csv,True,2735



SOURCE INVENTORY CHECK
Number of sources: 6
Columns: ['id', 'source_name', 'source_type', 'url', 'description']

Missing values:
id             0
source_name    0
source_type    0
url            0
description    0
dtype: int64

Duplicate source IDs:
0


,id,source_name,source_type,url,description
0,S001,Kabwe Municipal Council Website,Website,https://www.kabwecouncil.gov.zm,Official Kabwe Municipal Council website and digital source directory.
1,S002,2024 Bwacha CDF Community Projects Submission,PDF,https://www.kabwecouncil.gov.zm/wp-content/uploads/2024/11/2024-Bwacha-commu...,2024 CDF community project submissions for Bwacha Constituency.
2,S003,2024 Kabwe Central CDF Community Projects Submission,PDF,https://www.kabwecouncil.gov.zm/wp-content/uploads/2024/11/2024-Community-Pr...,2024 CDF community project submissions for Kabwe Central Constituency.
3,S004,2025 Kabwe Municipal Council OBB Budget,PDF,https://www.kabwecouncil.gov.zm/wp-content/uploads/2025/05/2025-KABWE-M-COUN...,2025 Output Based Budget containing CDF budget allocations and output indica...
4,S005,Kabwe District Integrated Development Plan 2023-2028,PDF,https://www.kabwecouncil.gov.zm/wp-content/uploads/2024/09/Kabwe-Approved-ID...,Approved Kabwe District Integrated Development Plan consulted during source ...
5,S006,Proposed 2025 CDF Projects,PDF,https://www.kabwecouncil.gov.zm/wp-content/uploads/2025/08/Proposed-2025-CDF...,"Proposed 2025 CDF community projects for Kabwe Central Constituency, includi..."



PIPE-DELIMITER CHECK


,filename,rows,columns,pipe_delimiter_working
0,db-unza26-csc4792-kabwe_cdf_projects_2024.csv,79,15,True
1,db-unza26-csc4792-kabwe_cdf_projects_2025.csv,33,9,True
2,db-unza26-csc4792-kabwe_cdf_budget_2024_2025.csv,5,7,True
3,db-unza26-csc4792-kabwe_cdf_output_indicators_2025.csv,12,10,True



OVERALL RESULT
All required CSV files exist: True
Source inventory is complete: True
Pipe delimiter is working: True

TASK 1 DATA CONSISTENCY CHECK: PASSED


## 2025–2027 Council Funding and National Support Dataset

Funding and national support allocations identified from the 2025 Kabwe Municipal Council Output Based Budget are structured into a dedicated dataset.

The dataset records the funding source or revenue description, funding category, and the corresponding 2025 approved budget, 2026 revised budget, and 2027 budget estimate.

This dataset provides broader council funding context alongside the CDF-specific datasets and supports analysis of CDF and other national support sources.

Source metadata is added to maintain data provenance and traceability to the original council budget document.

In [ ]:


funding_data = [
    ["Constituency Development Fund", "CDF", 72116301, 72116301, 72116301],
    ["Roads Grant", "Grant", 3200587, 3200587, 3200587],
    ["Health Grant", "Grant", 3649035, 3649035, 3649035],
    ["Local Government Equalisation Fund", "LGEF", 23857381, 23857381, 23857381],
    ["Grants in lieu of Rates", "Grant", 1200000, 1200000, 1200000],
    ["Other Grants", "Grant", 29494551, 10527645, 10797645],
]

funding_df = pd.DataFrame(
    funding_data,
    columns=[
        "revenue_description",
        "funding_category",
        "approved_budget_2025",
        "revised_budget_2026",
        "budget_estimate_2027",
    ]
)

funding_df

In [ ]:
funding_df["source_url"] = (
    "https://www.kabwecouncil.gov.zm/wp-content/uploads/2025/05/"
    "2025-KABWE-M-COUNCIL-OBB.pdf"
)

funding_df["source_document"] = "2025 Kabwe Municipal Council OBB Budget"

funding_df

,revenue_description,funding_category,approved_budget_2025,revised_budget_2026,budget_estimate_2027,source_url,source_document
0,Constituency Development Fund,CDF,72116301,72116301,72116301,https://www.kabwecouncil.gov.zm/wp-content/uploads/2025/05/2025-KABWE-M-COUN...,2025 Kabwe Municipal Council OBB Budget
1,Roads Grant,Grant,3200587,3200587,3200587,https://www.kabwecouncil.gov.zm/wp-content/uploads/2025/05/2025-KABWE-M-COUN...,2025 Kabwe Municipal Council OBB Budget
2,Health Grant,Grant,3649035,3649035,3649035,https://www.kabwecouncil.gov.zm/wp-content/uploads/2025/05/2025-KABWE-M-COUN...,2025 Kabwe Municipal Council OBB Budget
3,Local Government Equalisation Fund,LGEF,23857381,23857381,23857381,https://www.kabwecouncil.gov.zm/wp-content/uploads/2025/05/2025-KABWE-M-COUN...,2025 Kabwe Municipal Council OBB Budget
4,Grants in lieu of Rates,Grant,1200000,1200000,1200000,https://www.kabwecouncil.gov.zm/wp-content/uploads/2025/05/2025-KABWE-M-COUN...,2025 Kabwe Municipal Council OBB Budget
5,Other Grants,Grant,29494551,10527645,10797645,https://www.kabwecouncil.gov.zm/wp-content/uploads/2025/05/2025-KABWE-M-COUN...,2025 Kabwe Municipal Council OBB Budget


In [ ]:
funding_df["approved_budget_2025"].sum()

np.int64(133517855)

In [ ]:
funding_df["revised_budget_2026"].sum()

np.int64(114550949)

In [ ]:
funding_df["budget_estimate_2027"].sum()

np.int64(114820949)

In [ ]:
funding_output = (
    PROJECT_ROOT /
    "data" /
    "processed" /
    "db-unza26-csc4792-kabwe_council_funding_2025_2027.csv"
)

funding_df.to_csv(
    funding_output,
    sep="|",
    index=False
)

print(f"Saved: {funding_output}")

Saved: ../data/processed/db-unza26-csc4792-kabwe_council_funding_2025_2027.csv


In [ ]:
check_funding = pd.read_csv(
    funding_output,
    sep="|"
)

print("Rows:", len(check_funding))
print("Columns:", len(check_funding.columns))
print("Duplicates:", check_funding.duplicated().sum())

display(check_funding)

Rows: 6
Columns: 7
Duplicates: 0


,revenue_description,funding_category,approved_budget_2025,revised_budget_2026,budget_estimate_2027,source_url,source_document
0,Constituency Development Fund,CDF,72116301,72116301,72116301,https://www.kabwecouncil.gov.zm/wp-content/uploads/2025/05/2025-KABWE-M-COUN...,2025 Kabwe Municipal Council OBB Budget
1,Roads Grant,Grant,3200587,3200587,3200587,https://www.kabwecouncil.gov.zm/wp-content/uploads/2025/05/2025-KABWE-M-COUN...,2025 Kabwe Municipal Council OBB Budget
2,Health Grant,Grant,3649035,3649035,3649035,https://www.kabwecouncil.gov.zm/wp-content/uploads/2025/05/2025-KABWE-M-COUN...,2025 Kabwe Municipal Council OBB Budget
3,Local Government Equalisation Fund,LGEF,23857381,23857381,23857381,https://www.kabwecouncil.gov.zm/wp-content/uploads/2025/05/2025-KABWE-M-COUN...,2025 Kabwe Municipal Council OBB Budget
4,Grants in lieu of Rates,Grant,1200000,1200000,1200000,https://www.kabwecouncil.gov.zm/wp-content/uploads/2025/05/2025-KABWE-M-COUN...,2025 Kabwe Municipal Council OBB Budget
5,Other Grants,Grant,29494551,10527645,10797645,https://www.kabwecouncil.gov.zm/wp-content/uploads/2025/05/2025-KABWE-M-COUN...,2025 Kabwe Municipal Council OBB Budget


## 2025–2027 Local Revenue Streams Dataset

Detailed local revenue streams identified from the 2025 Kabwe Municipal Council Output Based Budget are structured into a dedicated dataset.

The dataset records revenue categories, revenue codes, descriptions, and the corresponding approved budget for 2025, revised budget for 2026, and budget estimate for 2027.

The dataset provides additional financial context for the council and complements the CDF and broader council funding datasets.

Source metadata will be added to maintain traceability to the original council budget document.

In [ ]:
# Local Revenue Streams: 2025-2027
# Source: 2025 Kabwe Municipal Council OBB Budget, Pages 2-4

local_revenue_data = [
    # Local taxes/rates
    ["Local taxes/rates", "001", "Residential", 4453818, 4453818, 4453818],
    ["Local taxes/rates", "002", "Commercial", 5536418, 5536418, 5536418],
    ["Local taxes/rates", "003", "Industrial", 2635280, 2635280, 2635280],
    ["Local taxes/rates", "004", "Hospitality", 572837, 630121, 693133],

    # Personal levy
    ["Personal levy", "001", "Personal levy", 450000, 495000, 220000],

    # Fees and Charges
    ["Fees and Charges", "001", "Consent fees", 25000, 27500, 30250],
    ["Fees and Charges", "002", "Survey fees", 700000, 770000, 862400],
    ["Fees and Charges", "003", "Building inspection-fees", 250000, 300000, 300000],
    ["Fees and Charges", "004", "Plan scrutiny fee", 2508000, 2508000, 2508000],
    ["Fees and Charges", "005", "Change of premise use", 295000, 324500, 347864],
    ["Fees and Charges", "006", "Container/Ntemba fees", 50000, 75000, 75000],
    ["Fees and Charges", "007", "Rentals/lease of Council’s properties", 896400, 986040, 1084644],
    ["Fees and Charges", "008", "Non-Land Application forms fees", 600000, 660000, 726000],
    ["Fees and Charges", "009", "Rentals from houses", 85092, 93601, 102961],
    ["Fees and Charges", "011", "Search fees", 7500, 8250, 9075],
    ["Fees and Charges", "012", "Notice board advert fees", 2500, 2750, 3025],
    ["Fees and Charges", "013", "Market fees", 785664, 864230, 950653],
    ["Fees and Charges", "014", "Parking fees", 50000, 50000, 50000],
    ["Fees and Charges", "016", "Loading fees (buses, trucks, trains, taxies etc.)", 393360, 432696, 475966],
    ["Fees and Charges", "017", "Affidavit fees", 6000, 6600, 7260],
    ["Fees and Charges", "020", "Hire of halls", 114684, 126152, 139824],
    ["Fees and Charges", "021", "Hire of grounds/stadia", 13000, 60000, 75000],
    ["Fees and Charges", "024", "Recommendations fees", 672000, 732600, 807800],
    ["Fees and Charges", "027", "Body remains (inspections) fees", 600, 660, 739],
    ["Fees and Charges", "033", "Refuse disposal", 1257030, 1348793, 1443209],
    ["Fees and Charges", "035", "Commercial & non-commercial Exhibitions", 10800, 600, 800],
    ["Fees and Charges", "038", "Library membership fees", 8000, 8800, 9680],
    ["Fees and Charges", "041", "Dumb site fees", 39600, 43560, 48787],
    ["Fees and Charges", "045", "Notice of marriage fees", 61250, 67375, 74113],
    ["Fees and Charges", "046", "Abattoir/meat inspection fees", 11500, 12650, 14168],
    ["Fees and Charges", "047", "Registration of clubs and societies", 100000, 110000, 121000],
    ["Fees and Charges", "051", "Farm produce Fee", 150000, 200000, 200000],
    ["Fees and Charges", "053", "Certification of documents", 2160, 2160, 3600],
    ["Fees and Charges", "055", "Illegal Parking of vehicles", 990000, 990000, 990000],
    ["Fees and Charges", "063", "Billboards and banners", 800000, 880000, 985600],
    ["Fees and Charges", "064", "Hire of Transport and Equipment", 25400, 189000, 288000],
    ["Fees and Charges", "065", "Council Minutes Extracts", 90000, 90000, 90000],
    ["Fees and Charges", "066", "Penalties", 750000, 580500, 640210],
    ["Fees and Charges", "067", "Ablution Fee", 283680, 312048, 343253],
    ["Fees and Charges", "072", "Booth fees", 28000, 30800, 34496],
    ["Fees and Charges", "074", "Sale of Bid Documents", 50000, 50000, 50000],
    ["Fees and Charges", "078", "Erection of Tombstone", 5000, 5000, 5000],
    ["Fees and Charges", "082", "Telecommunication site rentals", 96045, 96045, 96045],
    ["Fees and Charges", "099", "Other fees and charges", 824750, 907225, 998058],

    # Licenses
    ["Licenses", "002", "Liquor licence", 366000, 402600, 442860],
    ["Licenses", "003", "Firearm and ammunition licence", 26000, 26600, 27260],
    ["Licenses", "004", "Petroleum Storage licence", 400000, 440000, 492800],
    ["Licenses", "005", "Dog licence", 40000, 44000, 49280],

    # Levies
    ["Levies", "001", "Livestock Movement levy", 32400, 32400, 32400],
    ["Levies", "002", "Birds levy", 60000, 66000, 72600],
    ["Levies", "004", "Pole levy", 40000, 44000, 49280],
    ["Levies", "006", "Sand levy", 43000, 47300, 52976],
    ["Levies", "011", "Telecommunication Mast", 220500, 220500, 220500],
    ["Levies", "017", "Trading (Wholesale) Business Levy", 100000, 110000, 121000],
    ["Levies", "018", "Trading (Retail) Consumable groceries business", 880100, 968110, 1064921],
    ["Levies", "021", "Manufacturing", 62500, 68750, 75625],
    ["Levies", "029", "Professional Occupation", 249900, 274890, 302379],
    ["Levies", "030", "Scrap Metal Dealers", 12500, 13750, 15125],
    ["Levies", "031", "Car Wash", 20000, 22000, 24640],

    # Permits
    ["Permits", "001", "Health permits", 1847500, 2032250, 2276120],
    ["Permits", "008", "Burial permits and grave sites", 585000, 643500, 720720],
    ["Permits", "009", "Fire certificate", 3245000, 3245000, 3634400],
    ["Permits", "010", "Extension of Business hours permits", 20000, 22000, 24640],
    ["Permits", "099", "Primary, Secondary and Tertiary permits", 140000, 154000, 172480],

    # Charges
    ["Charges", "001", "Service Charges Residential plots", 5200000, 5720000, 6406400],
    ["Charges", "002", "Service Charges Industrial plots", 600000, 660000, 739200],
    ["Charges", "004", "Premium Plot Commercial", 1500000, 1650000, 1848000],
    ["Charges", "007", "Land Application Charges", 150000, 165000, 181500],
    ["Charges", "009", "Change of ownership", 4250, 4675, 5143],
    ["Charges", "010", "Sub-division of plot", 144000, 158400, 174240],
    ["Charges", "011", "Land regularisation", 75000, 82500, 90750],
    ["Charges", "012", "Change of Land use", 212800, 234000, 260000],
    ["Charges", "099", "Land Charges", 1700000, 1870000, 2000000],
]

local_revenue_df = pd.DataFrame(
    local_revenue_data,
    columns=[
        "revenue_category",
        "revenue_code",
        "revenue_description",
        "approved_budget_2025",
        "revised_budget_2026",
        "budget_estimate_2027",
    ]
)

local_revenue_df.head()

,revenue_category,revenue_code,revenue_description,approved_budget_2025,revised_budget_2026,budget_estimate_2027
0,Local taxes/rates,001,Residential,4453818,4453818,4453818
1,Local taxes/rates,002,Commercial,5536418,5536418,5536418
2,Local taxes/rates,003,Industrial,2635280,2635280,2635280
3,Local taxes/rates,004,Hospitality,572837,630121,693133
4,Personal levy,001,Personal levy,450000,495000,220000


In [ ]:
print("Revenue streams:", len(local_revenue_df))
print("\nRevenue streams by category:")
print(local_revenue_df["revenue_category"].value_counts())

Revenue streams: 73

Revenue streams by category:
revenue_category
Fees and Charges     39
Levies               11
Charges               9
Permits               5
Local taxes/rates     4
Licenses              4
Personal levy         1
Name: count, dtype: int64


In [ ]:
local_revenue_df["source_url"] = (
    "https://www.kabwecouncil.gov.zm/wp-content/uploads/2025/05/"
    "2025-KABWE-M-COUNCIL-OBB.pdf"
)

local_revenue_df["source_document"] = (
    "2025 Kabwe Municipal Council OBB Budget, Pages 2-4"
)

In [ ]:
local_revenue_output = (
    PROJECT_ROOT /
    "data" /
    "processed" /
    "db-unza26-csc4792-kabwe_local_revenue_2025_2027.csv"
)

local_revenue_df.to_csv(
    local_revenue_output,
    sep="|",
    index=False
)

print(f"Saved: {local_revenue_output}")

Saved: ../data/processed/db-unza26-csc4792-kabwe_local_revenue_2025_2027.csv


## Validate the Exported Local Revenue Dataset

The exported local revenue dataset is reloaded from the processed CSV file using the required pipe (`|`) delimiter to confirm that the saved file can be read correctly.

The validation checks the number of records and columns, identifies duplicate rows, and reports missing values for each column. These checks help confirm the structural quality and completeness of the exported dataset before final submission.

In [ ]:
check_local_revenue = pd.read_csv(
    local_revenue_output,
    sep="|"
)

print("Rows:", len(check_local_revenue))
print("Columns:", len(check_local_revenue.columns))
print("Duplicates:", check_local_revenue.duplicated().sum())
print("\nMissing values:")
print(check_local_revenue.isna().sum())

Rows: 73
Columns: 8
Duplicates: 0

Missing values:
revenue_category        0
revenue_code            0
revenue_description     0
approved_budget_2025    0
revised_budget_2026     0
budget_estimate_2027    0
source_url              0
source_document         0
dtype: int64


## 2024–2025 Council Budget by Economic Classification

Budget information from the 2025 Kabwe Municipal Council Output Based Budget is structured according to the council's economic classifications.

The dataset records the economic classification code, classification description, approved budget for 2024, and budget estimate for 2025.

This dataset provides broader council financial context and complements the CDF-specific budget, project, and revenue datasets.

The source is the 2025 Kabwe Municipal Council OBB Budget, specifically the economic classification information presented on page 5.

In [ ]:
# Council Budget by Economic Classification
# Source: 2025 Kabwe Municipal Council OBB Budget, Page 5

economic_budget_data = [
    ["21", "Personal Emoluments", 46609938, 48701693],
    ["22", "Goods and Services", 26095222, 33797534],
    ["26", "Grants and Other Payments (Transfers)", 16298162, 34845939],
    ["31", "Non-Financial Assets", 41160229, 52204514],
    ["32", "Financial Assets", 6984926, 7473871],
    ["41", "Current Liabilities (Payable within one year)", 3580192, 4056000],
]

economic_budget_df = pd.DataFrame(
    economic_budget_data,
    columns=[
        "economic_code",
        "economic_classification",
        "approved_budget_2024",
        "budget_estimate_2025",
    ]
)

economic_budget_df

,economic_code,economic_classification,approved_budget_2024,budget_estimate_2025
0,21,Personal Emoluments,46609938,48701693
1,22,Goods and Services,26095222,33797534
2,26,Grants and Other Payments (Transfers),16298162,34845939
3,31,Non-Financial Assets,41160229,52204514
4,32,Financial Assets,6984926,7473871
5,41,Current Liabilities (Payable within one year),3580192,4056000


## Validate the Economic Classification Budget Totals

The total approved budget for 2024 and the total budget estimate for 2025 are calculated to provide a basic consistency check of the economic classification dataset.

Comparing the calculated totals with the corresponding totals reported in the source document helps identify possible transcription or data-entry errors.

In [ ]:
print(
    "2024 total:",
    economic_budget_df["approved_budget_2024"].sum()
)

print(
    "2025 total:",
    economic_budget_df["budget_estimate_2025"].sum()
)

2024 total: 140728669
2025 total: 181079551


In [ ]:
economic_budget_df["source_url"] = (
    "https://www.kabwecouncil.gov.zm/wp-content/uploads/2025/05/"
    "2025-KABWE-M-COUNCIL-OBB.pdf"
)

economic_budget_df["source_document"] = (
    "2025 Kabwe Municipal Council OBB Budget, Page 5"
)

In [ ]:
economic_budget_output = (
    PROJECT_ROOT /
    "data" /
    "processed" /
    "db-unza26-csc4792-kabwe_economic_budget_2024_2025.csv"
)

economic_budget_df.to_csv(
    economic_budget_output,
    sep="|",
    index=False
)

print(f"Saved: {economic_budget_output}")

Saved: ../data/processed/db-unza26-csc4792-kabwe_economic_budget_2024_2025.csv


In [ ]:
check_economic_budget = pd.read_csv(
    economic_budget_output,
    sep="|"
)

print("Rows:", len(check_economic_budget))
print("Columns:", len(check_economic_budget.columns))
print("Duplicates:", check_economic_budget.duplicated().sum())

display(check_economic_budget)

Rows: 6
Columns: 6
Duplicates: 0


,economic_code,economic_classification,approved_budget_2024,budget_estimate_2025,source_url,source_document
0,21,Personal Emoluments,46609938,48701693,https://www.kabwecouncil.gov.zm/wp-content/uploads/2025/05/2025-KABWE-M-COUN...,"2025 Kabwe Municipal Council OBB Budget, Page 5"
1,22,Goods and Services,26095222,33797534,https://www.kabwecouncil.gov.zm/wp-content/uploads/2025/05/2025-KABWE-M-COUN...,"2025 Kabwe Municipal Council OBB Budget, Page 5"
2,26,Grants and Other Payments (Transfers),16298162,34845939,https://www.kabwecouncil.gov.zm/wp-content/uploads/2025/05/2025-KABWE-M-COUN...,"2025 Kabwe Municipal Council OBB Budget, Page 5"
3,31,Non-Financial Assets,41160229,52204514,https://www.kabwecouncil.gov.zm/wp-content/uploads/2025/05/2025-KABWE-M-COUN...,"2025 Kabwe Municipal Council OBB Budget, Page 5"
4,32,Financial Assets,6984926,7473871,https://www.kabwecouncil.gov.zm/wp-content/uploads/2025/05/2025-KABWE-M-COUN...,"2025 Kabwe Municipal Council OBB Budget, Page 5"
5,41,Current Liabilities (Payable within one year),3580192,4056000,https://www.kabwecouncil.gov.zm/wp-content/uploads/2025/05/2025-KABWE-M-COUN...,"2025 Kabwe Municipal Council OBB Budget, Page 5"


## 2024–2025 Council Programme and Sub-Programme Budget

Programme and sub-programme budget information from the 2025 Kabwe Municipal Council Output Based Budget is structured into a dedicated dataset.

The dataset records programme codes, programme names, record types, sub-programmes, approved budgets for 2024, and budget estimates for 2025.

The dataset provides a structured view of council expenditure and service-delivery areas, including Constituency Development, Public Health, Housing and Community Amenities, Education, District Health Services, Transport, Agriculture, and Social Protection.

The source is the 2025 Kabwe Municipal Council OBB Budget, specifically the programme and sub-programme budget information presented on pages 8–10.

In [ ]:
# Kabwe Municipal Council Programme and Sub-Programme Budget
# Source: 2025 Kabwe Municipal Council OBB Budget, Pages 8-10

programme_budget_data = [
    # Constituency Development
    ["1", "Constituency Development", "Programme", "", 61271284, 72116301],
    ["779", "Constituency Development", "Sub-Programme", "Community Projects", 34924632, 43925383],
    ["780", "Constituency Development", "Sub-Programme", "Women and Youth Empowerment", 11641544, 12456452],
    ["781", "Constituency Development", "Sub-Programme", "CDF Administration", 3063564, 3278014],
    ["782", "Constituency Development", "Sub-Programme", "Secondary School and Skills Development Bursaries", 11641544, 12456452],

    # Local Governance
    ["2", "Local Governance", "Programme", "", 6125473, 5827497],
    ["003", "Local Governance", "Sub-Programme", "Legislative Functions", 3767535, 3349182],
    ["043", "Local Governance", "Sub-Programme", "Citizen Engagement", 2357938, 2478315],

    # Integrated Development Planning
    ["3", "Integrated Development Planning", "Programme", "", 6342899, 5387125],
    ["006", "Integrated Development Planning", "Sub-Programme", "Environmental Planning", 893433, 3677653],
    ["021", "Integrated Development Planning", "Sub-Programme", "Spatial Planning", 5449465, 1709472],

    # Economic and Business Development
    ["4", "Economic and Business Development", "Programme", "", 151477, 168721],
    ["038", "Economic and Business Development", "Sub-Programme", "Trade Facilitation and Licensing", 151477, 168721],

    # Public Health and Environmental Protection
    ["5", "Public Health and Environmental Protection", "Programme", "", 3498334, 4611170],
    ["015", "Public Health and Environmental Protection", "Sub-Programme", "Cemetery and Funeral Services", 156074, 728585],
    ["019", "Public Health and Environmental Protection", "Sub-Programme", "Health Inspections", 1268384, 2799110],
    ["023", "Public Health and Environmental Protection", "Sub-Programme", "Pest Control", 18740, 335315],
    ["024", "Public Health and Environmental Protection", "Sub-Programme", "Pollution Control", 27266, 26400],
    ["027", "Public Health and Environmental Protection", "Sub-Programme", "Solid Waste Management", 1994296, 694450],
    ["034", "Public Health and Environmental Protection", "Sub-Programme", "Water Supply and Sanitation Services", 33575, 27310],

    # Housing and Community Amenities
    ["6", "Housing and Community Amenities", "Programme", "", 13975738, 27194430],
    ["008", "Housing and Community Amenities", "Sub-Programme", "Roads and Drainages", 5040099, 16076588],
    ["011", "Housing and Community Amenities", "Sub-Programme", "Parks and Gardens", 1707973, 687249],
    ["012", "Housing and Community Amenities", "Sub-Programme", "Markets and Bus Stations", 2983191, 1698806],
    ["026", "Housing and Community Amenities", "Sub-Programme", "Public Housing", 2904099, 4973907],
    ["031", "Housing and Community Amenities", "Sub-Programme", "Street Lighting", 1340376, 3757881],

    # Recreation, Culture and Religion
    ["7", "Recreation Culture and Religion", "Programme", "", 1067283, 844145],
    ["001", "Recreation Culture and Religion", "Sub-Programme", "Cultural Affairs", 7759, 61724],
    ["042", "Recreation Culture and Religion", "Sub-Programme", "Sports Promotion", 1059524, 782421],

    # Education and Skills Development
    ["8", "Education and Skills Development", "Programme", "", 33780, 38731],
    ["001", "Education and Skills Development", "Sub-Programme", "District Archives", 0, 19241],
    ["005", "Education and Skills Development", "Sub-Programme", "Early Childhood Education", 33780, 19490],

    # Public Order and Safety
    ["10", "Public Order and Safety", "Programme", "", 6822011, 10917603],
    ["018", "Public Order and Safety", "Sub-Programme", "Community Policing", 1686555, 3487804],
    ["041", "Public Order and Safety", "Sub-Programme", "Fire Protection Services", 5135456, 7429799],

    # Management and Support Services
    ["11", "Management and Support Services", "Programme", "", 32145481, 22039403],
    ["001", "Management and Support Services", "Sub-Programme", "Human Resource and Administration", 13358205, 6773166],
    ["009", "Management and Support Services", "Sub-Programme", "Executive Management", 1847378, 880392],
    ["016", "Management and Support Services", "Sub-Programme", "Procurement", 1263583, 713018],
    ["028", "Management and Support Services", "Sub-Programme", "Financial Management-Auditing", 803216, 636825],
    ["035", "Management and Support Services", "Sub-Programme", "Financial Management-Accounting", 10685644, 10084742],
    ["036", "Management and Support Services", "Sub-Programme", "Legal Services", 2732692, 2393867],
    ["062", "Management and Support Services", "Sub-Programme", "Public Relations", 1454763, 557393],

    # Resource Mobilisation and Management
    ["12", "Resource Mobilisation and Management", "Programme", "", 2346892, 4209839],
    ["067", "Resource Mobilisation and Management", "Sub-Programme", "Revenue Mobilisation and Enhancement", 2346892, 4209839],

    # District Health Services
    ["13", "District Health Services", "Programme", "", 3100894, 3649036],
    ["001", "District Health Services", "Sub-Programme", "Primary Health Services", 3100894, 3101039],
    ["002", "District Health Services", "Sub-Programme", "District Health Co-ordination", 0, 547997],

    # Transport Services
    ["15", "Transport Services", "Programme", "", 3742847, 3200587],
    ["001", "Transport Services", "Sub-Programme", "Road Transport", 3742847, 3200587],

    # Agricultural Services
    ["16", "Agricultural Services", "Programme", "", 0, 549076],
    ["071", "Agricultural Services", "Sub-Programme", "Agricultural Crop Production, Advisory and Technical Services", 0, 328430],
    ["072", "Agricultural Services", "Sub-Programme", "Agribusiness Development and Marketing", 0, 32400],
    ["073", "Agricultural Services", "Sub-Programme", "Agriculture Co-ordination", 0, 188246],

    # Fisheries and Livestock
    ["17", "Fisheries and Livestock", "Programme", "", 104276, 486160],
    ["074", "Fisheries and Livestock", "Sub-Programme", "Fisheries and Livestock Marketing", 0, 58616],
    ["075", "Fisheries and Livestock", "Sub-Programme", "Animal Health Services", 104276, 121955],
    ["076", "Fisheries and Livestock", "Sub-Programme", "Fisheries Production and Productivity Improvement", 0, 101125],
    ["077", "Fisheries and Livestock", "Sub-Programme", "Livestock Production and Productivity Improvement", 0, 82924],
    ["078", "Fisheries and Livestock", "Sub-Programme", "District Fisheries and Livestock Coordination", 0, 121540],

    # Social Protection and Community Development
    ["18", "Social Protection and Community Development", "Programme", "", 0, 19839728],
    ["079", "Social Protection and Community Development", "Sub-Programme", "District Social Welfare", 0, 19455843],
    ["080", "Social Protection and Community Development", "Sub-Programme", "Community Development", 0, 383885],
]

programme_budget_df = pd.DataFrame(
    programme_budget_data,
    columns=[
        "programme_code",
        "programme",
        "record_type",
        "sub_programme",
        "approved_budget_2024",
        "budget_estimate_2025",
    ]
)

programme_budget_df.head(10)

,programme_code,programme,record_type,sub_programme,approved_budget_2024,budget_estimate_2025
0,1,Constituency Development,Programme,,61271284,72116301
1,779,Constituency Development,Sub-Programme,Community Projects,34924632,43925383
2,780,Constituency Development,Sub-Programme,Women and Youth Empowerment,11641544,12456452
3,781,Constituency Development,Sub-Programme,CDF Administration,3063564,3278014
4,782,Constituency Development,Sub-Programme,Secondary School and Skills Development Bursaries,11641544,12456452
5,2,Local Governance,Programme,,6125473,5827497
6,003,Local Governance,Sub-Programme,Legislative Functions,3767535,3349182
7,043,Local Governance,Sub-Programme,Citizen Engagement,2357938,2478315
8,3,Integrated Development Planning,Programme,,6342899,5387125
9,006,Integrated Development Planning,Sub-Programme,Environmental Planning,893433,3677653


In [ ]:
!pip install requests beautifulsoup4 pandas lxml pdfplumber

# IDP & Strategic Community Projects - Smart Mbuzi

**CSC 4792: Data Mining and Warehousing · 2025/26**
**Workstream:** Integrated Development Plans (IDPs) & Strategic Community Project Records

## Overview

In this notebook we extract, clean, and curate information about Kabwe Municipal
Council's Integrated Development Plan (2023–2033) and its strategic community
project records. Our pipeline has three phases:

**Phase 1 — Discovery** — We crawl the council website with BeautifulSoup to
identify which PDFs are published and where they live. This produces a URL list,
not the data itself.

**Phase 2 — Extraction** — We read the four PDFs (placed manually in
`data/raw/idp/`) and pull out structured tables covering IDP strategic areas,
development goals, baseline statistics, sub-programmes, and community project
registries.

**Phase 3 — Cleaning & Preprocessing** — We transform the raw extraction output
into an analysis-ready dataset: standardised, deduplicated, enriched, and
exported as pipe-delimited CSVs.

We keep the phases separate so each stage can be re-run independently.

## Phase 1 · Setup

We import the libraries needed for discovery and set up the folder structure.
Our notebook can run from either the repo root or from inside `notebooks/`, so
we detect the current directory and resolve paths relative to the repo root
accordingly.

Folders we create:

- `data/raw/idp/` — where the downloaded PDFs live
- `data/discovered/` — where we save the URL list
- `data/extracted/` — Phase 2's intermediate output
- `data/clean/` — Phase 3's intermediate output
- `outputs/` — the final CSVs for Kaggle

In [ ]:
# ============================================================
# PHASE 1 · CELL 1 — SETUP
# ============================================================
# !pip install requests beautifulsoup4

import os
import re
import json
import warnings
from datetime import datetime
from pathlib import Path
from urllib.parse import urljoin, urlparse

import requests
from bs4 import BeautifulSoup

try:
    from requests.exceptions import RequestsDependencyWarning
    warnings.filterwarnings("ignore", category=RequestsDependencyWarning)
except ImportError:
    pass


# ---------- Repo root (hardcoded) ----------
REPO_ROOT = Path(r"C:\Users\dell\Desktop\Group6\group6_administration")

if not REPO_ROOT.exists():
    raise FileNotFoundError(f"Repo root not found: {REPO_ROOT}")


# ---------- Folder structure ----------
DIRS = {
    "raw":        REPO_ROOT / "data" / "raw" / "IDP",
    "discovered": REPO_ROOT / "data" / "discovered",
    "extracted":  REPO_ROOT / "data" / "extracted" / "IDP",
    "clean":      REPO_ROOT / "data" / "cleaned" / "IDP",
    "outputs":    REPO_ROOT / "data" / "processed" / "IDP",
}
for d in DIRS.values():
    d.mkdir(parents=True, exist_ok=True)

PROJECT_CODE = "db-unza26-csc4792"
COUNCIL_SLUG = "kabwe"
COUNCIL_FULL = "Kabwe Municipal Council"
CONSTITUENCY = "Kabwe Central"
BASE_URL     = "https://www.kabwecouncil.gov.zm"
RUN_STAMP    = datetime.now().isoformat(timespec="seconds")

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/124.0 Safari/537.36"
    )
}

print("PHASE 1 — DISCOVERY")
print(f"  Repo root : {REPO_ROOT}")
print(f"  PDFs dir  : {DIRS['raw']}")
print(f"  Exists    : {DIRS['raw'].exists()}")

# Quick check: are the 4 PDFs there?
if DIRS["raw"].exists():
    pdfs = list(DIRS["raw"].glob("*.pdf"))
    print(f"  PDFs found: {len(pdfs)}")
    for p in pdfs:
        print(f"    • {p.name}  ({p.stat().st_size / 1024:.1f} KB)")
else:
    print(f"  Folder does not exist yet — will be created")

PHASE 2 — EXTRACTION
  Repo root : C:\Users\dell\Desktop\Group6\group6_administration
  PDFs dir  : C:\Users\dell\Desktop\Group6\group6_administration\data\raw\IDP
  Out dir   : C:\Users\dell\Desktop\Group6\group6_administration\data\extracted\IDP
  Sources   : 4


## Phase 1 · Fetching the council homepage

We start by fetching the council's homepage HTML — our crawling entry point,
the page from which we discover what documents the council links to.

We've observed that the council's server presents an incomplete SSL certificate
chain, so we disable certificate verification. This is safe because we only read
public documents — no credentials, no sensitive data.

In [ ]:
# ============================================================
# PHASE 1 · CELL 2 — FETCH HOMEPAGE
# ============================================================

def fetch_html(url, verify_ssl=False, timeout=30):
    try:
        r = requests.get(
            url, headers=HEADERS, timeout=timeout,
            verify=verify_ssl, allow_redirects=True,
        )
        r.raise_for_status()
        return r.text
    except Exception as e:
        print(f"  [error] {url}  →  {type(e).__name__}: {e}")
        return None


homepage_html = fetch_html(BASE_URL)

if homepage_html:
    soup  = BeautifulSoup(homepage_html, "html.parser")
    title = soup.find("title")
    print(f"Homepage fetched: {len(homepage_html):,} chars")
    print(f"Title: {title.string.strip() if title else '(no title)'}")
else:
    print("Homepage fetch failed.")

C:\Users\dell\AppData\Local\Programs\Python\Python313\Lib\site-packages\urllib3\connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host 'www.kabwecouncil.gov.zm'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


Homepage fetched: 167,369 chars
Title: Kabwe Municipal Council – Kabwe


## Phase 1 · Discovering PDF links

We parse the homepage's HTML for anchor tags pointing to PDF files. We convert
relative URLs to absolute, then filter out links to other domains — a common
issue when council pages link to news about other councils.

This is the classic HTML-scraping workflow: fetch, parse, filter.

In [ ]:
# ============================================================
# PHASE 1 · CELL 3 — DISCOVER PDF LINKS
# ============================================================

def discover_pdf_links(html, base_url):
    soup = BeautifulSoup(html, "html.parser")
    base_domain = urlparse(base_url).netloc
    found = set()

    for a in soup.find_all("a", href=True):
        href = a["href"].strip()
        if not href.lower().endswith(".pdf"):
            continue
        absolute = urljoin(base_url, href)
        if urlparse(absolute).netloc != base_domain:
            continue
        found.add(absolute)

    return sorted(found)


homepage_pdfs = discover_pdf_links(homepage_html, BASE_URL) if homepage_html else []

print(f"PDF links discovered on homepage: {len(homepage_pdfs)}")
for u in homepage_pdfs:
    print(f"  • {u}")

PDF links discovered on homepage: 1
  • https://www.kabwecouncil.gov.zm/wp-content/uploads/2025/09/CDF-Guidelines.pdf


## Phase 1 · Curated source list

The homepage links only two PDFs. When we checked manually, we found that the
IDP and project documents we need are hosted at direct URLs under
`/wp-content/uploads/` — but they aren't linked from any crawlable index page.

So we combine two sources of truth:

1. **Dynamic discovery** — what the crawl finds on the homepage
2. **Curated list** — the four documents we've verified manually

This is standard practice when scraping council and government sites: discovery
finds what's new, curation guarantees the specific documents we need.

In [ ]:
# ============================================================
# PHASE 1 · CELL 4 — CURATED SOURCES
# ============================================================

CURATED_SOURCES = [
    {
        "doc_id":   "IDP_MAIN",
        "title":    "Kabwe Approved IDP Final Version 1 (2023–2033)",
        "category": "Integrated Development Plan",
        "filename": "Kabwe-Approved-IDP_Final-Version-1.pdf",
        "url":      f"{BASE_URL}/wp-content/uploads/2024/09/Kabwe-Approved-IDP_Final-Version-1.pdf",
    },
    {
        "doc_id":   "IDP_CITIZEN",
        "title":    "Kabwe District Citizen IDP",
        "category": "Integrated Development Plan",
        "filename": "Kabwe-District-Citizen-IDP_Final-1.pdf",
        "url":      f"{BASE_URL}/wp-content/uploads/2024/09/Kabwe-District-Citizen-IDP_Final-1.pdf",
    },
    {
        "doc_id":   "CDF_PROJECTS_2025",
        "title":    "2025 Approved Community Projects — Kabwe Central",
        "category": "Strategic Community Projects",
        "filename": "2025-Approved-Community-Projects_Kabwe-Central.pdf",
        "url":      f"{BASE_URL}/wp-content/uploads/2025/06/2025-Approved-Community-Projects_Kabwe-Central.pdf",
    },
    {
        "doc_id":   "NEWSLETTER_2024",
        "title":    "Kabwe Municipal Council Newsletter 2024",
        "category": "Council Newsletter",
        "filename": "Kabwe-Municipal-Council-Newsletter-2024-1.pdf",
        "url":      f"{BASE_URL}/wp-content/uploads/2024/09/Kabwe-Municipal-Council-Newsletter-2024-1.pdf",
    },
]

print(f"Curated sources: {len(CURATED_SOURCES)}")
for s in CURATED_SOURCES:
    print(f"  [{s['doc_id']:<18}] {s['title']}")

Curated sources: 4
  [IDP_MAIN          ] Kabwe Approved IDP Final Version 1 (2023–2033)
  [IDP_CITIZEN       ] Kabwe District Citizen IDP
  [CDF_PROJECTS_2025 ] 2025 Approved Community Projects — Kabwe Central
  [NEWSLETTER_2024   ] Kabwe Municipal Council Newsletter 2024


## Phase 1 · Saving the discovery result

We write the combined URL list to `data/discovered/sources.json` so Phase 2 can
read it without re-crawling. This keeps the two phases decoupled — if we later
re-run Phase 2, we don't need network access.

In [ ]:
# ============================================================
# PHASE 1 · CELL 5 — SAVE DISCOVERY RESULT
# ============================================================

payload = {
    "run_at":          RUN_STAMP,
    "base_url":        BASE_URL,
    "homepage_pdfs":   homepage_pdfs,
    "curated_sources": CURATED_SOURCES,
}

out_file = DIRS["discovered"] / "sources.json"
out_file.write_text(json.dumps(payload, indent=2), encoding="utf-8")

print(f"   Saved: {out_file.resolve()}")
print(f"   HTML-discovered : {len(homepage_pdfs)}")
print(f"   Curated         : {len(CURATED_SOURCES)}")

   Saved: C:\Users\dell\Desktop\Group6\group6_administration\data\discovered\sources.json
   HTML-discovered : 1
   Curated         : 4


## Phase 1 · Manual download instructions

We deliberately do **not** download the PDFs programmatically. In our earlier
attempts, the council's server rejected scripted downloads due to SSL issues and
bandwidth throttling. Instead, we download the files manually via our browser
and place them in `data/raw/idp/`.

The next cell prints the exact URLs and folder path so we can copy them into a
browser.

In [ ]:
# ============================================================
# PHASE 1 · CELL 6 — MANUAL DOWNLOAD INSTRUCTIONS
# ============================================================

target = DIRS["raw"].resolve()

print("=" * 72)
print("MANUAL DOWNLOAD INSTRUCTIONS")
print("=" * 72)
print(f"\nTarget folder: {target}\n")
for s in CURATED_SOURCES:
    print(f"  • {s['filename']}")
    print(f"    {s['url']}\n")
print("=" * 72)

MANUAL DOWNLOAD INSTRUCTIONS

Target folder: C:\Users\dell\Desktop\Group6\group6_administration\data\raw\IDP

  • Kabwe-Approved-IDP_Final-Version-1.pdf
    https://www.kabwecouncil.gov.zm/wp-content/uploads/2024/09/Kabwe-Approved-IDP_Final-Version-1.pdf

  • Kabwe-District-Citizen-IDP_Final-1.pdf
    https://www.kabwecouncil.gov.zm/wp-content/uploads/2024/09/Kabwe-District-Citizen-IDP_Final-1.pdf

  • 2025-Approved-Community-Projects_Kabwe-Central.pdf
    https://www.kabwecouncil.gov.zm/wp-content/uploads/2025/06/2025-Approved-Community-Projects_Kabwe-Central.pdf

  • Kabwe-Municipal-Council-Newsletter-2024-1.pdf
    https://www.kabwecouncil.gov.zm/wp-content/uploads/2024/09/Kabwe-Municipal-Council-Newsletter-2024-1.pdf



## Phase 1 · Verifying the PDFs are in place

Once we've downloaded the four PDFs, we run this cell to verify they're in the
right folder with the right filenames. If anything is missing, the cell tells us
exactly which file and where to put it.

Once this cell passes, Phase 1 is complete.

In [ ]:
# ============================================================
# PHASE 1 · CELL 7 — VERIFY PDFs ON DISK
# ============================================================

REQUIRED_FILES = [s["filename"] for s in CURATED_SOURCES]

print(f"Checking: {DIRS['raw'].resolve()}\n")

present = sorted(f.name for f in DIRS["raw"].iterdir()
                 if f.is_file() and f.suffix.lower() == ".pdf")

print(f"PDFs found: {len(present)}")
for f in present:
    size_kb = (DIRS["raw"] / f).stat().st_size / 1024
    marker  = "✓" if f in REQUIRED_FILES else "?"
    print(f"  {marker} {f}  ({size_kb:.1f} KB)")

missing = [f for f in REQUIRED_FILES if f not in present]
if missing:
    print(f"\nMissing {len(missing)} file(s):")
    for f in missing:
        print(f"   - {f}")
    raise FileNotFoundError("Download missing PDFs before continuing.")
else:
    print(f"\nAll {len(REQUIRED_FILES)} PDFs present.")
    print("   Phase 1 complete. Proceed to Phase 2.")

Checking: C:\Users\dell\Desktop\Group6\group6_administration\data\raw\IDP

PDFs found: 4
  ✓ 2025-Approved-Community-Projects_Kabwe-Central.pdf  (27.2 KB)
  ✓ Kabwe-Approved-IDP_Final-Version-1.pdf  (17202.6 KB)
  ✓ Kabwe-District-Citizen-IDP_Final-1.pdf  (41227.1 KB)
  ✓ Kabwe-Municipal-Council-Newsletter-2024-1.pdf  (2703.6 KB)

All 4 PDFs present.
   Phase 1 complete. Proceed to Phase 2.


## Phase 2 · Section 2.0 — Setup

We import `pdfplumber` and `pandas`, resolve the folder paths the same way as
Phase 1, and reload the curated source list from the discovery JSON.

We then read every page of every PDF into memory. For each page we keep both
the page number and the page text. The page number lets us trace any extracted
fact back to its source.

**Note:** Phase 2 does not touch the network. Everything reads from disk.

In [ ]:
# ============================================================
# PHASE 2 · CELL 1 — SETUP
# ============================================================
import os
import re
import json
import warnings
from pathlib import Path

import pandas as pd
import pdfplumber

warnings.filterwarnings("ignore", category=Warning)

# ---------- Repo root (hardcoded) ----------
REPO_ROOT = Path(r"C:\Users\dell\Desktop\Group6\group6_administration")

if not REPO_ROOT.exists():
    raise FileNotFoundError(f"Repo root not found: {REPO_ROOT}")

# ---------- Paths ----------
RAW_DIR       = REPO_ROOT / "data" / "raw" / "IDP"
EXTRACTED_DIR = REPO_ROOT / "data" / "extracted" / "IDP"
EXTRACTED_DIR.mkdir(parents=True, exist_ok=True)

# ---------- Reload curated sources ----------
sources_payload = json.loads(
    (REPO_ROOT / "data" / "discovered" / "sources.json").read_text()
)
CURATED_SOURCES = sources_payload["curated_sources"]
sources_by_id   = {s["doc_id"]: s for s in CURATED_SOURCES}

print("PHASE 2 — EXTRACTION")
print(f"  Repo root : {REPO_ROOT}")
print(f"  PDFs dir  : {RAW_DIR}")
print(f"  Out dir   : {EXTRACTED_DIR}")
print(f"  Sources   : {len(CURATED_SOURCES)}")

PHASE 2 — EXTRACTION
  Repo root : C:\Users\dell\Desktop\Group6\group6_administration
  PDFs dir  : C:\Users\dell\Desktop\Group6\group6_administration\data\raw\IDP
  Out dir   : C:\Users\dell\Desktop\Group6\group6_administration\data\extracted\IDP
  Sources   : 4


## Phase 2 · Section 2.1 — Extraction Utilities

We define the helpers every extractor below will use:

- **`make_unique_columns()`** — PDF tables often have empty or duplicate column
  headers; this renames them to safe placeholders so pandas can handle them

- **`extract_all_tables()`** — reads every table from every page of a PDF, one
  DataFrame per table. We use **looser table detection settings** to reduce the
  left-truncation problem we observed in earlier attempts.

- **`guess_sector()`** — classifies a sentence or phrase into a sector
  (Education, Health, Water, Roads, etc.) based on keyword matching

These utilities are used by every extraction stage in Phase 2.

In [ ]:
# ============================================================
# PHASE 2 · CELL 2 (v4 — FINAL) — EXTRACTION UTILITIES
# ============================================================

def make_unique_columns(cols):
    """Return unique, non-empty column labels."""
    seen, out = {}, []
    for i, c in enumerate(cols):
        name = str(c).strip() if c is not None else ""
        if not name or name.lower() in ("nan", "none"):
            name = f"col_{i}"
        if name in seen:
            seen[name] += 1
            name = f"{name}__{seen[name]}"
        else:
            seen[name] = 0
        out.append(name)
    return out


def extract_all_tables(pdf_path):
    """Extract every table using the 'lines' strategy.

    Our diagnostic showed that:
      - 'lines' strategy → produces correct headers
      - 'text'  strategy → splits titles into fragments (BAD)
      - 'default'        → same as lines for this PDF

    So we always use 'lines'.
    """
    LINE_SETTINGS = {
        "vertical_strategy":   "lines",
        "horizontal_strategy": "lines",
    }
    tables = []
    with pdfplumber.open(pdf_path) as pdf:
        for pno, page in enumerate(pdf.pages, start=1):
            for tbl in page.extract_tables(LINE_SETTINGS):
                if tbl and len(tbl) > 1:
                    cols = make_unique_columns(tbl[0])
                    df = pd.DataFrame(tbl[1:], columns=cols)
                    df["_page"] = pno
                    tables.append(df)
    return tables


SECTOR_KEYWORDS = {
    "Education":            ["education", "school", "classroom", "teacher",
                             "pupil", "literacy", "desk"],
    "Health":               ["health", "clinic", "hospital", "nurse",
                             "doctor", "maternal", "malaria", "hiv"],
    "Water and Sanitation": ["water", "sanitation", "borehole", "toilet",
                             "latrine", "sewer", "sewage"],
    "Roads and Drainages":  ["road", "drain", "street", "bridge",
                             "culvert", "pavement", "tarmac"],
    "Commerce":             ["market", "trade", "commerce", "business",
                             "sme", "entrepreneur"],
    "Agriculture":          ["agriculture", "farm", "crop", "livestock",
                             "irrigation", "maize"],
    "Energy":               ["electricity", "power", "solar", "grid",
                             "energy", "zesco"],
    "Governance":           ["governance", "council", "ward", "community",
                             "participation", "committee"],
    "Environment":          ["environment", "climate", "forest", "tree",
                             "green", "pollution", "waste"],
}


def guess_sector(text):
    if not isinstance(text, str):
        return "Unknown"
    s = text.lower()
    scores = {sec: sum(1 for kw in kws if kw in s)
              for sec, kws in SECTOR_KEYWORDS.items()}
    best = max(scores, key=scores.get)
    return best if scores[best] > 0 else "Unknown"


print("Extraction utilities ready (lines strategy — validated).")

Extraction utilities ready (lines strategy — validated).


## Phase 2 · Section 2.2 — IDP Strategic Areas

The Kabwe IDP is anchored on the four strategic development areas from Zambia's
Eighth National Development Plan (8NDP):

- Economic Transformation and Job Creation
- Human and Social Development
- Environmental Sustainability
- Good Governance Environment

We extract each area name along with the pages where it appears and the IDP's
vision and mission statements. These four rows anchor the entire dataset — they
contextualise everything else.

In [ ]:
# ============================================================
# PHASE 2 · CELL 2 — LOAD PDFs AS TEXT
# ============================================================

def load_pdf_pages(pdf_path):
    pages = []
    with pdfplumber.open(pdf_path) as pdf:
        for pno, page in enumerate(pdf.pages, start=1):
            pages.append((pno, page.extract_text() or ""))
    return pages


idp_pages     = load_pdf_pages(RAW_DIR / sources_by_id["IDP_MAIN"]["filename"])
citizen_pages = load_pdf_pages(RAW_DIR / sources_by_id["IDP_CITIZEN"]["filename"])
cdf_pages     = load_pdf_pages(RAW_DIR / sources_by_id["CDF_PROJECTS_2025"]["filename"])
news_pages    = load_pdf_pages(RAW_DIR / sources_by_id["NEWSLETTER_2024"]["filename"])

In [ ]:
# ============================================================
# PHASE 2 · CELL 3 — IDP STRATEGIC AREAS
# ============================================================

STRATEGIC_AREAS = [
    "Economic Transformation and Job Creation",
    "Human and Social Development",
    "Environmental Sustainability",
    "Good Governance Environment",
]


def find_statement(text, keywords, context_lines=4, max_len=400):
    """Return the first text block following a keyword match."""
    lines = text.split("\n")
    for i, line in enumerate(lines):
        if any(kw in line.lower() for kw in keywords):
            snippet = " ".join(lines[i:i+context_lines])
            return re.sub(r"\s+", " ", snippet).strip()[:max_len]
    return None


idp_text = "\n".join(t for _, t in idp_pages)
vision   = find_statement(idp_text, ["vision", "our vision"])
mission  = find_statement(idp_text, ["mission", "our mission"])

strategic_df = pd.DataFrame([
    {
        "strategic_area":  area,
        "page_refs":       ",".join(str(p) for p, t in idp_pages
                                     if area.lower() in t.lower()),
        "vision_excerpt":  vision,
        "mission_excerpt": mission,
        "source_doc":      "IDP_MAIN",
    }
    for area in STRATEGIC_AREAS
])

strategic_df.to_csv(EXTRACTED_DIR / "strategic_areas.csv", sep="|", index=False)
print(f"Strategic areas: {len(strategic_df)}")
strategic_df[["strategic_area", "page_refs"]]

Strategic areas: 4


,strategic_area,page_refs
0,Economic Transformation and Job Creation,"11,12,13,19,78,164,165,169,170,232,233,299"
1,Human and Social Development,"8,11,12,19,164,165,182,249,316"
2,Environmental Sustainability,"8,11,12,13,19,90,164,165,200,280,340"
3,Good Governance Environment,"11,13,165,205,285"


## Phase 2 · Section 2.3 — IDP Sector Goals

The IDP describes development goals in narrative form. These sentences usually
contain a target verb ("increase", "improve", "achieve") and sometimes a
quantified target ("from 78% to 92%", "by 2030").

We use regex patterns to find these sentences and assign each one a sector
label. The `page` column lets us trace every goal back to its source.

This extraction typically yields 40–150 goal statements.

In [ ]:
# ============================================================
# PHASE 2 · CELL 4 — IDP SECTOR GOALS
# ============================================================

GOAL_PATTERNS = [
    r"\b(increase|reduce|expand|improve|achieve|ensure|promote|enhance|strengthen)\b.{10,300}",
    r"\bobjectives?\s+\d+(?:\.\d+)*[:\s].{10,300}",
    r"\b(target|goal|aim)s?\s*[:\-]\s*.{10,300}",
    r"\bby\s+20\d{2}\b.{0,200}",
    r"\bfrom\s+[\d.,%]+\s+to\s+[\d.,%]+\b.{0,200}",
]


def extract_goals(pages, source_doc):
    rows = []
    for pno, text in pages:
        for s in re.split(r"(?<=[.!?])\s+", text):
            s_clean = re.sub(r"\s+", " ", s).strip()
            if not (40 <= len(s_clean) <= 500):
                continue
            if any(re.search(p, s_clean, re.IGNORECASE) for p in GOAL_PATTERNS):
                rows.append({
                    "page":       pno,
                    "sector":     guess_sector(s_clean),
                    "goal_text":  s_clean[:400],
                    "source_doc": source_doc,
                })
    return rows


goals_df = (
    pd.DataFrame(extract_goals(idp_pages, "IDP_MAIN"))
      .drop_duplicates(subset=["goal_text"])
      .reset_index(drop=True)
)
goals_df.to_csv(EXTRACTED_DIR / "goals.csv", sep="|", index=False)

print(f"Sector goals: {len(goals_df)}")
print(goals_df["sector"].value_counts().to_string())

Sector goals: 256
sector
Unknown                 82
Environment             40
Governance              29
Education               23
Water and Sanitation    21
Agriculture             20
Health                  17
Commerce                10
Roads and Drainages      8
Energy                   6


## Phase 2 · Section 2.4 — IDP Baseline Statistics

The IDP is full of quantitative facts about the district — *"45 health posts"*,
*"population of 245,000"*, *"120 km of roads"*. These form the district profile.

We extract them with a regex matching a number followed by a countable noun.
Each statistic keeps the surrounding sentence as context.

**Note:** We deliberately do **not** assign a sector to each statistic — the
keyword-based sector classification is unreliable for these short facts, and
guessing wrong is worse than leaving it blank.

In [ ]:
# ============================================================
# PHASE 2 · CELL 5 — IDP BASELINE STATISTICS
# ============================================================

STAT_UNITS = [
    "school", "schools", "clinic", "clinics", "hospital", "hospitals",
    "health post", "health posts", "borehole", "boreholes",
    "market", "markets", "road", "roads", "km", "kilometre", "kilometres",
    "household", "households", "population", "people", "residents",
    "ward", "wards", "teacher", "teachers", "nurse", "nurses",
    "pupil", "pupils", "desk", "desks", "toilet", "toilets",
    "latrine", "latrines", "plot", "plots",
]

UNIT_REGEX = "|".join(re.escape(u) for u in STAT_UNITS)
STAT_REGEX = re.compile(rf"\b(\d[\d,]*)\s+({UNIT_REGEX})\b", re.IGNORECASE)


def extract_statistics(pages, source_doc):
    rows = []
    for pno, text in pages:
        for m in STAT_REGEX.finditer(text):
            ctx = text[max(0, m.start()-120):m.end()+120]
            rows.append({
                "page":       pno,
                "value":      int(m.group(1).replace(",", "")),
                "unit":       m.group(2).lower(),
                "context":    re.sub(r"\s+", " ", ctx).strip()[:300],
                "source_doc": source_doc,
            })
    return rows


stats_df = (
    pd.DataFrame(extract_statistics(idp_pages, "IDP_MAIN"))
      .drop_duplicates(subset=["value", "unit", "context"])
      .reset_index(drop=True)
)
stats_df.to_csv(EXTRACTED_DIR / "statistics.csv", sep="|", index=False)

print(f"Baseline statistics: {len(stats_df)}")
print(stats_df["unit"].value_counts().head(10).to_string())

## Phase 2 · Section 2.5 — CDF Project Registry

The 2025 Approved Community Projects document contains the master list of CDF
projects for Kabwe Central. This is the core of the project-records dataset.

We extract every table from every page, then filter to keep only the ones that
look like project registries (multiple rows, content mentioning projects, wards,
or amounts). The result is saved raw — cleaning happens in Phase 3.

In [ ]:
# ============================================================
# PHASE 2 · CELL 6 (v6 — FINAL FINAL) — CDF PROJECT REGISTRY
# ============================================================

def is_project_table(df, min_rows=5, min_cols=6):
    """Accept ONLY tables that are real project registries."""
    if df.shape[0] < min_rows or df.shape[1] < min_cols:
        return False
    headers = [str(c).strip().upper() for c in df.columns]
    header_text = " ".join(headers)
    if "PROJECT NAME" not in header_text:
        return False
    if "SUB TOTAL" in header_text or "SUMMARY" in header_text:
        return False
    if len(df) > 0:
        first_cell = str(df.iloc[0, 0]).strip().upper()
        if first_cell.startswith("SUB TOTAL") or first_cell.startswith("SUMMARY"):
            return False
    return True


# ---------- Extract ----------
cdf_tables = extract_all_tables(
    RAW_DIR / sources_by_id["CDF_PROJECTS_2025"]["filename"]
)

print(f"Tables detected: {len(cdf_tables)}")
for t in cdf_tables:
    print(f"  page {t['_page'].iloc[0]}: {t.shape}  "
          f"cols={list(t.columns)[:6]}")

# ---------- Filter ----------
main_tables = [t for t in cdf_tables if is_project_table(t)]
print(f"\nProject tables passing filter: {len(main_tables)}")

if not main_tables:
    print("No project tables matched — using all tables.")
    main_tables = cdf_tables

# ---------- Concatenate ----------
cdf_raw = (
    pd.concat(main_tables, ignore_index=True, sort=False)
    if main_tables else pd.DataFrame()
)

# ---------- Drop columns that are empty OR all-blank ----------
def is_blank(series):
    """True if every value is NaN, None, or an empty/whitespace string."""
    return series.apply(
        lambda v: pd.isna(v) or str(v).strip() == ""
    ).all()

before_cols = list(cdf_raw.columns)
drop_cols = [c for c in cdf_raw.columns if is_blank(cdf_raw[c])]

if drop_cols:
    cdf_raw = cdf_raw.drop(columns=drop_cols)
    print(f"\n  Dropped {len(drop_cols)} empty/blank columns: {drop_cols}")
else:
    print("\n  No empty columns to drop.")

# ---------- Metadata ----------
cdf_raw["source_doc"] = "CDF_PROJECTS_2025"

# ---------- Save ----------
cdf_raw.to_csv(EXTRACTED_DIR / "cdf_projects_raw.csv", sep="|", index=False)

print(f"\n CDF raw: {len(cdf_raw)} rows, {cdf_raw.shape[1]} cols")
print(f"   Columns: {list(cdf_raw.columns)}")
cdf_raw.head(10)

## Phase 2 · Section 2.6 — Newsletter Completed Projects

The 2024 Newsletter lists completed education and health projects from 2022
and 2023. This document uses a magazine-style layout that `pdfplumber`'s table
extraction can't parse reliably — the grid lines don't align with the logical
data cells.

So we use a different strategy: we scan the raw page text with a regex that
matches `<project name> ... <amount>` patterns, and filter for lines containing
project-related keywords.

In [ ]:
# ============================================================
# PHASE 2 · CELL 7 — NEWSLETTER PROJECTS
# ============================================================

PROJECT_KEYWORDS = [
    "school", "classroom", "block", "toilet", "ablution", "clinic",
    "market", "borehole", "desk", "chair", "construction", "procurement",
    "rehabilitation", "installation", "supply", "road", "bridge",
    "hospital", "office", "staff house", "shelter", "wall fence",
    "water", "reticulated", "solar", "panel",
]

NEWS_LINE = re.compile(
    r"(?P<name>(?:[A-Z][A-Za-z0-9'\-]+[\s,&]+){3,20}[A-Za-z0-9'\-]+)"
    r"[^\d]{0,80}(?:K|ZMW)?\s*"
    r"(?P<amount>\d{1,3}(?:,\d{3})+(?:\.\d{2})?)",
    re.MULTILINE,
)
YEAR_RE = re.compile(r"\b(20\d{2})\b")


def extract_newsletter(pages):
    rows = []
    for pno, text in pages:
        if len(text) < 200:
            continue
        for m in NEWS_LINE.finditer(text):
            name = re.sub(r"\s+", " ", m.group("name")).strip()
            if not (20 <= len(name) <= 200):
                continue
            if not any(kw in name.lower() for kw in PROJECT_KEYWORDS):
                continue
            try:
                amount = float(m.group("amount").replace(",", ""))
            except ValueError:
                continue
            if not (5_000 <= amount <= 500_000_000):
                continue
            ctx = text[max(0, m.start()-250):m.end()+250]
            ym  = YEAR_RE.search(ctx)
            rows.append({
                "project_name":        name[:200],
                "approved_amount_zmw": amount,
                "year_funded":         int(ym.group(1)) if ym else 2024,
                "sector":              guess_sector(name),
                "status":              "Completed",
                "source_doc":          "NEWSLETTER_2024",
                "page":                pno,
            })
    return rows


newsletter_df = (
    pd.DataFrame(extract_newsletter(news_pages))
      .drop_duplicates(subset=["project_name", "approved_amount_zmw"])
      .reset_index(drop=True)
)
newsletter_df.to_csv(EXTRACTED_DIR / "newsletter_raw.csv", sep="|", index=False)

print(f"Newsletter: {len(newsletter_df)} rows")
if not newsletter_df.empty:
    print(f"   Total ZMW: {newsletter_df['approved_amount_zmw'].sum():,.2f}")

## Phase 2 · Section 2.7 — Newsletter Narrative Sentences

The Newsletter also contains narrative articles that mention projects in prose
— with names, completion events, and numbers (amounts, counts, years).

We mine the raw page text for sentences that contain both a project keyword
and a number. Each sentence becomes a record, with the numbers extracted into
a separate column for easy scanning.

This complements the structured project lines by capturing context the tables
don't carry.

In [ ]:
# ============================================================
# PHASE 2 · CELL 8 — NEWSLETTER NARRATIVE SENTENCES
# ============================================================

NARRATIVE_KEYWORDS = [
    "school", "classroom", "clinic", "hospital", "market", "borehole",
    "desk", "water", "road", "bridge", "project", "handover",
    "commissioned", "completed", "constructed", "rehabilitated",
    "beneficiaries", "pupils", "patients", "residents",
]

NARRATIVE_NUMBER_RE = re.compile(r"\d[\d,]*")


def extract_narrative_sentences(pages, source_doc):
    rows = []
    for pno, text in pages:
        for s in re.split(r"(?<=[.!?])\s+", text):
            s_clean = re.sub(r"\s+", " ", s).strip()
            if not (50 <= len(s_clean) <= 500):
                continue
            lower = s_clean.lower()
            if not any(kw in lower for kw in NARRATIVE_KEYWORDS):
                continue
            if not NARRATIVE_NUMBER_RE.search(s_clean):
                continue
            numbers = [int(m.group().replace(",", ""))
                       for m in NARRATIVE_NUMBER_RE.finditer(s_clean)]
            rows.append({
                "page":          pno,
                "sector":        guess_sector(s_clean),
                "sentence":      s_clean[:400],
                "numbers_found": ",".join(map(str, numbers[:10])),
                "source_doc":    source_doc,
            })
    return rows


narrative_df = (
    pd.DataFrame(extract_narrative_sentences(news_pages, "NEWSLETTER_2024"))
      .drop_duplicates(subset=["sentence"])
      .reset_index(drop=True)
)
narrative_df.to_csv(EXTRACTED_DIR / "newsletter_narrative.csv", sep="|", index=False)

print(f"Narrative sentences: {len(narrative_df)}")
if not narrative_df.empty:
    print(f"\nSample:")
    for s in narrative_df["sentence"].head(5):
        print(f"  • {s[:100]}...")

## Phase 2 · Section 2.8 — IDP Sub-Programmes (Strict)

The IDP is organised around 8NDP pillars, and each pillar contains named
sub-programmes — the concrete interventions the council plans to undertake.

**A note on extraction difficulty:** this is the hardest extraction in the
pipeline. Numbered/bulleted lines in the IDP also appear in:

- The table of contents (`"3.1.4 Local Economic Development .... 22"`)
- The institutional arrangements chapter (`"Director of Planning John Doe"`)
- Staff lists (`"Health Officer Christopher Mtonga"`)
- Fragmentary headings (`"enabling environment for"`)

Our strict filter rejects all of these by requiring:

1. The line contains a **development action verb**
2. The line does **not** end with a preposition (rules out fragments)
3. The line does **not** match a person-name pattern (`Dr.`, `Officer Name`)
4. The line does **not** contain banned context phrases
5. The line has **4+ words** and 60%+ alphabetic characters

The result is a small, precise list of real sub-programmes — or an empty file
if the IDP presents them as prose rather than bullets.

In [ ]:
# ============================================================
# PHASE 2 · CELL 9 — IDP SUB-PROGRAMMES (STRICT)
# ============================================================

# ---------- Constants ----------
DOTTED_LEADER    = re.compile(r"\.{3,}")
ENDS_WITH_NUMBER = re.compile(r"\d\s*$")
TRAILING_PREPS   = re.compile(
    r"\b(?:for|of|in|to|with|and|by|at|on|the|a|an|from|into|as|is|are|was|were)\s*$",
    re.IGNORECASE,
)

ACTION_VERBS = [
    "construction", "construct", "rehabilitation", "rehabilitate",
    "expansion", "expand", "improvement", "improve", "promotion", "promote",
    "provision", "provide", "development", "develop", "establishment",
    "establish", "strengthening", "strengthen", "enhancement", "enhance",
    "maintenance", "maintain", "installation", "install", "supply",
    "upgrade", "upgrading", "extension", "extend", "creation", "create",
    "increase", "reduction", "reduce", "modernisation", "modernise",
    "replacement", "replace",
]

BANNED_CONTEXT = [
    "development plan", "citizens version", "central province",
    "district council", "urban council", "municipal council",
    "ward development committee", "planning unit", "ppu",
    "administrative", "implementing", "beneficiaries",
]

PERSON_NAME_PATTERN = re.compile(
    r"\b(?:Dr|Mr|Mrs|Ms|Miss|Eng|Prof)\.?\s+[A-Z][a-z]+",
    re.IGNORECASE,
)
TITLE_BEFORE_NAME = re.compile(
    r"(?:Officer|Director|Head|Manager|Specialist|Inspector|Planner|"
    r"Secretary|Administrator|Accountant|Engineer|Coordinator)\s+"
    r"[A-Z][a-z]+",
)


def is_real_sub_programme(line):
    """Strict filter for actual development sub-programmes."""
    if not isinstance(line, str):
        return False
    line = line.strip()

    # Length and shape
    if len(line) < 20 or len(line) > 180:
        return False
    if len(line.split()) < 4:
        return False
    if not line[0].isupper():
        return False

    # Not a TOC or fragment
    if DOTTED_LEADER.search(line):
        return False
    if ENDS_WITH_NUMBER.search(line):
        return False
    if TRAILING_PREPS.search(line):
        return False

    # Contains a development action verb
    lower = line.lower()
    if not any(verb in lower for verb in ACTION_VERBS):
        return False

    # Not institutional/banned text
    if any(bc in lower for bc in BANNED_CONTEXT):
        return False

    # Not a person name or title+name
    if PERSON_NAME_PATTERN.search(line):
        return False
    if TITLE_BEFORE_NAME.search(line):
        return False

    # At least 60% alphabetic
    alpha = sum(1 for c in line if c.isalpha())
    if alpha / len(line) < 0.6:
        return False

    return True


# ---------- Sub-programme extraction ----------
PILLAR_KEYWORDS = [
    "economic transformation", "job creation",
    "human and social development",
    "environmental sustainability", "good governance",
]

SUB_PROG_PATTERN = re.compile(
    r"^\s*(?:\d+(?:\.\d+)*|[•\-*]|[a-z]\))\s*(.{15,180})$",
    re.MULTILINE,
)


def extract_subprogrammes(pages):
    rows = []
    current_pillar = "Unknown"
    for pno, text in pages:
        for kw in PILLAR_KEYWORDS:
            if kw in text.lower():
                current_pillar = kw.title()
                break
        for m in SUB_PROG_PATTERN.finditer(text):
            line = re.sub(r"\s+", " ", m.group(1)).strip()
            if not is_real_sub_programme(line):
                continue
            rows.append({
                "page":           pno,
                "pillar":         current_pillar,
                "sub_programme":  line[:200],
                "source_doc":     "IDP_MAIN",
            })
    return rows


pillars_df = (
    pd.DataFrame(extract_subprogrammes(idp_pages))
      .drop_duplicates(subset=["sub_programme"])
      .reset_index(drop=True)
)
pillars_df.to_csv(EXTRACTED_DIR / "subprogrammes.csv", sep="|", index=False)

print(f"Sub-programmes (strict): {len(pillars_df)}")
if not pillars_df.empty:
    print("\nBy pillar:")
    print(pillars_df["pillar"].value_counts().to_string())
    print("\nSample:")
    print(pillars_df.head(15).to_string(index=False))
else:
    print("No real sub-programmes found — the IDP may present them as prose.")

## Phase 2 · Section 2.9 — IDP M&E Framework

The IDP's monitoring and evaluation framework lists every indicator the council
tracks against its goals. Each row typically has an indicator name, a baseline,
a target, and a data source.

We scan every IDP page for tables whose header row mentions indicator-related
keywords, then extract each row as a record.

In [ ]:
# ============================================================
# PHASE 2 · CELL 10 — IDP M&E FRAMEWORK
# ============================================================

ME_KEYWORDS = (
    "indicator", "baseline", "target", "means of verification",
    "data source", "frequency", "responsible", "outcome",
    "output indicator", "performance", "strategies", "program",
    "activities", "location",
)


def extract_me_framework(pdf_path):
    rows = []
    with pdfplumber.open(pdf_path) as pdf:
        for pno, page in enumerate(pdf.pages, start=1):
            for tbl in page.extract_tables():
                if not tbl or len(tbl) < 2:
                    continue
                header_text = " ".join(str(c or "").lower() for c in tbl[0])
                kw_hits = sum(1 for kw in ME_KEYWORDS if kw in header_text)
                if kw_hits < 2:
                    continue
                cols = [str(c or f"col_{i}").strip()[:60]
                        for i, c in enumerate(tbl[0])]
                for r in tbl[1:]:
                    if not r or not any(r):
                        continue
                    record = {"page": pno, "source_doc": "IDP_MAIN"}
                    for c, v in zip(cols, r):
                        record[c] = str(v or "").strip()[:300]
                    rows.append(record)
    return rows


me_raw = extract_me_framework(RAW_DIR / sources_by_id["IDP_MAIN"]["filename"])

me_df = (
    pd.DataFrame(me_raw)
      .drop_duplicates()
      .reset_index(drop=True)
    if me_raw else pd.DataFrame()
)
me_df.to_csv(EXTRACTED_DIR / "me_framework.csv", sep="|", index=False)

print(f"M&E framework rows: {len(me_df)}")
if not me_df.empty:
    print(f"   Columns: {list(me_df.columns)}")
    me_df.head()

## Phase 2 · Section 2.10 — IDP Sector Cost Estimates

The IDP contains cost estimate tables for each sector — columns are years
(2023, 2024, …, 2033), rows are programmes. We reshape this from wide to long
form so each cell becomes a `(programme, year, cost)` record.

We scan every page for tables whose header contains at least two distinct years.

In [ ]:
# ============================================================
# PHASE 2 · CELL 11 — IDP SECTOR COST ESTIMATES
# ============================================================

YEAR_TOKENS = [str(y) for y in range(2023, 2034)]


def extract_cost_tables(pdf_path):
    rows = []
    with pdfplumber.open(pdf_path) as pdf:
        for pno, page in enumerate(pdf.pages, start=1):
            for tbl in page.extract_tables():
                if not tbl or len(tbl) < 3:
                    continue
                header = " ".join(str(c or "") for c in tbl[0])
                year_hits = sum(1 for y in YEAR_TOKENS if y in header)
                if year_hits < 2:
                    continue
                for r in tbl[1:]:
                    if not r or len(r) < 3:
                        continue
                    label = str(r[0] or "").strip()
                    if len(label) < 4:
                        continue
                    for h, v in zip(tbl[0][1:], r[1:]):
                        year_str = str(h or "").strip()
                        if not year_str.isdigit():
                            continue
                        try:
                            cost = float(
                                str(v).replace(",", "")
                                      .replace("K", "")
                                      .replace("ZMW", "")
                                      .strip() or 0
                            )
                        except ValueError:
                            continue
                        if cost <= 0:
                            continue
                        rows.append({
                            "programme":  label[:120],
                            "year":       int(year_str),
                            "cost_zmw":   cost,
                            "page":       pno,
                            "source_doc": "IDP_MAIN",
                        })
    return rows


cost_df = (
    pd.DataFrame(extract_cost_tables(RAW_DIR / sources_by_id["IDP_MAIN"]["filename"]))
      .drop_duplicates(subset=["programme", "year", "cost_zmw"])
      .reset_index(drop=True)
)
cost_df.to_csv(EXTRACTED_DIR / "cost_estimates.csv", sep="|", index=False)

print(f"Cost estimate rows: {len(cost_df)}")
if not cost_df.empty:
    print(f"   Unique programmes: {cost_df['programme'].nunique()}")
    print(f"   Years: {sorted(cost_df['year'].unique())}")
    cost_df.head()

## Phase 2 · Section 2.11 — IDP Ward Profiles

The IDP contains ward-level descriptions — names, numbers, and sometimes
population figures or dominant economic activity.

We extract ward names and numbers using a pattern that matches
`"Ward <number>: <name>"` and surrounding context.

In [ ]:
# ============================================================
# PHASE 2 · CELL 12 — IDP WARD PROFILES
# ============================================================

WARD_PATTERN = re.compile(
    r"ward\s+(\d+)\s*[:\-]?\s*([A-Za-z][A-Za-z\s\-']{2,60})",
    re.IGNORECASE,
)


def extract_wards(pages, source_doc):
    rows = []
    for pno, text in pages:
        for m in WARD_PATTERN.finditer(text):
            wno   = m.group(1).strip()
            wname = re.sub(r"\s+", " ", m.group(2)).strip()
            if len(wname) < 3:
                continue
            if wname.lower() in ("development", "committee", "council",
                                 "the", "a", "an"):
                continue
            rows.append({
                "ward_number": wno,
                "ward_name":   wname[:80],
                "page":        pno,
                "source_doc":  source_doc,
            })
    return rows


wards_df = (
    pd.DataFrame(extract_wards(idp_pages, "IDP_MAIN"))
      .drop_duplicates(subset=["ward_number", "ward_name"])
      .reset_index(drop=True)
)
wards_df.to_csv(EXTRACTED_DIR / "wards.csv", sep="|", index=False)

print(f"Ward profiles: {len(wards_df)}")
if not wards_df.empty:
    wards_df.head(10)

## Phase 2 · Section 2.12 — Citizen IDP Points

The Citizen IDP is a shorter, community-facing version of the main IDP. It
contains distilled priority statements that don't always appear verbatim in the
main document.

We extract numbered/bulleted lines that contain real development content, using
the same strict filter as the sub-programme extractor.

In [ ]:
# ============================================================
# PHASE 2 · CELL 13 — CITIZEN IDP POINTS
# ============================================================

def extract_citizen_points(pages, source_doc):
    rows = []
    for pno, text in pages:
        for m in SUB_PROG_PATTERN.finditer(text):
            line = re.sub(r"\s+", " ", m.group(1)).strip()
            if not is_real_sub_programme(line):
                continue
            rows.append({
                "page":       pno,
                "topic":      guess_sector(line),
                "point":      line[:300],
                "source_doc": source_doc,
            })
    return rows


citizen_df = (
    pd.DataFrame(extract_citizen_points(citizen_pages, "IDP_CITIZEN"))
      .drop_duplicates(subset=["point"])
      .reset_index(drop=True)
)
citizen_df.to_csv(EXTRACTED_DIR / "citizen_points.csv", sep="|", index=False)

print(f"Citizen points: {len(citizen_df)}")
if not citizen_df.empty:
    citizen_df.head(10)
else:
    print("No real citizen points found.")

## Phase 2 · Section 2.13 — Extraction Summary

We list every file we produced in `data/extracted/` — these are the inputs to
Phase 3.

In [ ]:
# ============================================================
# PHASE 2 · CELL 14 — EXTRACTION SUMMARY
# ============================================================

print("=" * 72)
print("PHASE 2 — EXTRACTION SUMMARY")
print("=" * 72)

extracted_files = sorted(EXTRACTED_DIR.glob("*.csv"))
total_rows = 0
for f in extracted_files:
    try:
        df = pd.read_csv(f, sep="|")
        total_rows += len(df)
        print(f"  {f.name:<40}  {len(df):>5} rows  {df.shape[1]:>2} cols")
    except Exception as e:
        print(f"  {f.name:<40}  [error] {e}")

print("-" * 72)
print(f"  {'TOTAL':<40}  {total_rows:>5} rows")
print(f"\n  Files in: {EXTRACTED_DIR.resolve()}")
print("  → Phase 2 complete. Proceed to Phase 3.")

# PHASE 3 — CLEANING & PREPROCESSING

## Objective

We now transform the raw extracted data into an analysis-ready dataset. This is
the phase where we:

1. Standardise column names
2. Clean text values (strip newlines, collapse whitespace)
3. Coerce amount fields to numeric
4. Canonicalise categorical fields
5. Drop header-repeat rows and junk
6. Merge into a master project table
7. Enrich with derived analytics columns
8. Build derived analytics tables (ward matrix, timeline)
9. Export pipe-delimited CSVs to `outputs/`

We keep cleaning separate from extraction because cleaning rules change more
often than extraction logic — and running cleaning on its own is much faster
than re-parsing the PDFs.

## Phase 3 · Section 3.0 — Setup

We reload the extracted tables from disk and re-establish the folder paths.
This cell makes Phase 3 self-contained: if we ever want to run Phase 3 without
re-running Phases 1 and 2, we can jump straight here.

The `extracted` dictionary gives us named access to each of the raw tables
we'll be cleaning.

In [ ]:
# ============================================================
# PHASE 3 · CELL 1 — SETUP
# ============================================================

import os
import re
import json
import warnings
from datetime import datetime
from pathlib import Path

import pandas as pd

warnings.filterwarnings("ignore", category=Warning)

CWD = Path.cwd()
REPO_ROOT = CWD.parent if CWD.name == "notebooks" else CWD

EXTRACTED_DIR = REPO_ROOT / "data" / "extracted"
CLEAN_DIR     = REPO_ROOT / "data" / "clean"
OUTPUT_DIR    = REPO_ROOT / "outputs"

for d in (CLEAN_DIR, OUTPUT_DIR):
    d.mkdir(parents=True, exist_ok=True)

# ---------- Project constants ----------
PROJECT_CODE = "db-unza26-csc4792"
COUNCIL_SLUG = "kabwe"
COUNCIL_FULL = "Kabwe Municipal Council"
CONSTITUENCY = "Kabwe Central"
RUN_STAMP    = datetime.now().isoformat(timespec="seconds")

# ---------- Load all extracted CSVs into a dict ----------
extracted = {}
if EXTRACTED_DIR.exists():
    for f in sorted(EXTRACTED_DIR.glob("*.csv")):
        try:
            extracted[f.stem] = pd.read_csv(f, sep="|")
        except Exception as e:
            print(f"  [warn] could not load {f.name}: {e}")

print("PHASE 3 — CLEANING & PREPROCESSING")
print(f"  Extracted dir : {EXTRACTED_DIR.resolve()}")
print(f"  Clean dir     : {CLEAN_DIR.resolve()}")
print(f"  Output dir    : {OUTPUT_DIR.resolve()}")
print(f"\n  Loaded {len(extracted)} extracted tables:")
for name, df in extracted.items():
    print(f"    {name:<30}  {df.shape}")

## Phase 3 · Section 3.1 — Cleaning Utilities

Before touching any individual table, we define the shared cleaning functions
that every downstream stage will use.

These are the workhorses of Phase 3:

- **`clean_text()`** — strips line breaks, collapses whitespace, drops null-like
  strings
- **`clean_amount()`** — converts currency strings to `float`
- **`canonical_sector()`** — maps any sector variant to a canonical name
- **`canonical_ward()`** — normalises ward names
- **`make_project_ids()`** — generates sequential primary keys
- **`is_real_item()`** — our junk filter for text-heavy extractions; rejects
  TOC entries (`"......"`), page-number fragments, mid-word truncations, and
  lines without a domain signal word

In [ ]:
# ============================================================
# PHASE 3 · CELL 2 — CLEANING UTILITIES
# ============================================================

NULL_STRINGS = {"none", "nan", "nat", "null", "n/a", "", "-", "unknown"}

VAGUE_WARDS = {
    "various wards", "all wards", "unknown", "none", "",
    "entire district", "all", "n/a",
}


# ---------- Text cleaning ----------
def clean_text(value, max_len=None):
    """Trim, collapse whitespace, and drop null-like strings."""
    if pd.isna(value):
        return None
    s = re.sub(r"[\r\n\t]+", " ", str(value))
    s = re.sub(r"\s+", " ", s).strip()
    if s.lower() in NULL_STRINGS:
        return None
    return s[:max_len] if max_len else s


# ---------- Amount cleaning ----------
def clean_amount(value):
    """Convert 'K 1,703,868.67' to 1703868.67. Returns None on failure."""
    if pd.isna(value):
        return None
    s = str(value).upper()
    s = re.sub(r"(ZMW|K|MK|KWACHA)", "", s)
    s = re.sub(r"[^\d.]", "", s)
    try:
        return float(s) if s else None
    except ValueError:
        return None


# ---------- Sector canonicalisation ----------
SECTOR_MAP = {
    "education":            "Education",
    "school":               "Education",
    "schools":              "Education",
    "health":               "Health",
    "clinic":               "Health",
    "hospital":             "Health",
    "water and sanitation": "Water and Sanitation",
    "water & sanitation":   "Water and Sanitation",
    "water":                "Water and Sanitation",
    "sanitation":           "Water and Sanitation",
    "roads and drainages":  "Roads and Drainages",
    "roads & drainages":    "Roads and Drainages",
    "roads":                "Roads and Drainages",
    "drainage":             "Roads and Drainages",
    "commerce":             "Commerce",
    "market":               "Commerce",
    "trade":                "Commerce",
    "agriculture":          "Agriculture",
    "farming":              "Agriculture",
    "energy":               "Energy",
    "electricity":          "Energy",
    "governance":           "Governance",
    "environment":          "Environment",
    "housing":              "Housing",
    "community development": "Community Development",
}


def canonical_sector(value):
    if pd.isna(value):
        return "Unknown"
    return SECTOR_MAP.get(str(value).strip().lower(),
                          str(value).strip().title())


# ---------- Ward canonicalisation ----------
def canonical_ward(value):
    if pd.isna(value):
        return None
    s = re.sub(r"\s+", " ", str(value)).strip().strip(",.")
    return s.title() if s else None


# ---------- ID generation ----------
def make_project_ids(n, prefix="KAB"):
    return [f"{prefix}-{i:04d}" for i in range(1, n + 1)]


# ---------- Junk filter ----------
DOTTED_LEADER     = re.compile(r"\.{3,}")
ENDS_WITH_NUMBER  = re.compile(r"\d\s*$")

SUB_PROG_SIGNALS = [
    "development", "management", "promotion", "provision", "expansion",
    "improvement", "enhancement", "strengthening", "creation",
    "construction", "rehabilitation", "maintenance", "planning",
    "regulation", "monitoring", "coordination", "capacity",
    "infrastructure", "services", "access", "delivery",
    "investment", "agriculture", "education", "health",
    "water", "sanitation", "roads", "energy", "markets",
    "forestry", "environment", "climate", "tourism", "commerce",
    "trade", "industry", "housing", "settlement", "land",
    "governance", "administration", "finance", "revenue",
]


def is_real_item(line, min_words=3, require_signal=True):
    """Return True if a candidate line is NOT a TOC entry or fragment."""
    if not isinstance(line, str):
        return False
    if DOTTED_LEADER.search(line):
        return False
    if ENDS_WITH_NUMBER.search(line):
        return False
    if len(line.split()) < min_words:
        return False
    if line.isupper() and len(line) > 60:
        return False
    if require_signal:
        lower = line.lower()
        if not any(sig in lower for sig in SUB_PROG_SIGNALS):
            return False
    return True


print("Cleaning utilities ready.")

## Phase 3 · Section 3.2 — Clean the CDF Project Registry

The CDF project registry is our core dataset. We apply the full cleaning pipeline:

1. **Rename columns** to a canonical set (uppercase variants, common spelling
   differences, and truncated PDF headers all map to the same target)
2. **Coalesce duplicate columns** if the PDF split one logical column
3. **Drop stray columns** (empty placeholders, page markers, autogenerated)
4. **Clean text fields** (project name, ward, sector, site, description)
5. **Clean amounts** to `float`
6. **Drop header-repeat rows** — PDFs repeat the header on every page
7. **Drop rows without a valid project name**
8. **Drop "Sub Total" rows** and rows with no amount
9. **Fill missing sectors and wards** with `"Unknown"` so group-bys don't break
10. **Add metadata columns** (council, constituency, status, source)

The result is saved to `data/clean/projects_clean.csv`.

In [ ]:
# ============================================================
# PHASE 3 · CELL 3 — CLEAN CDF PROJECT REGISTRY
# ============================================================

cdf = extracted.get("cdf_projects_raw", pd.DataFrame()).copy()

if cdf.empty:
    print("cdf_projects_raw.csv is empty — check Phase 2.")
    cdf_clean = pd.DataFrame()
else:
    print(f"Raw CDF rows: {len(cdf)}")

    # 1. Uppercase columns
    cdf.columns = [str(c).strip().upper() for c in cdf.columns]

    # 2. Rename to canonical schema
    RENAME_MAP = {
        "NO.": "project_no", "NO": "project_no", "#": "project_no",
        "S/N": "project_no", "SN": "project_no",
        "PROJECT NAME": "project_name", "PROJECT_NAME": "project_name",
        "PROJECT": "project_name", "NAME": "project_name",
        "PROJECT TITLE": "project_name",
        "PROJECT DESCRIPTION": "description", "DESCRIPTION": "description",
        "SECTOR": "sector", "CATEGORY": "sector",
        "WARD": "ward", "LOCATION": "ward", "AREA": "ward",
        "PROJECT SITE": "site", "SITE": "site",
        "COL_5": "site",
        "ENGINEERS ESTIMATE": "engineer_estimate_zmw",
        "ENGINEER'S ESTIMATE": "engineer_estimate_zmw",
        "ENGINEER ESTIMATE": "engineer_estimate_zmw",
        "ENGINEERS ESTIMAT": "engineer_estimate_zmw",
        "ENGINEER ESTIMAT": "engineer_estimate_zmw",
        "ESTIMATE": "engineer_estimate_zmw",
        "APPROVED AMOUNT": "approved_amount_zmw",
        "APPROVED AMOUNT (ZMW)": "approved_amount_zmw",
        "APPROVED AMOUN": "approved_amount_zmw",
        "APPROVED AMOU": "approved_amount_zmw",
        "APPROVED AMT": "approved_amount_zmw",
        "AMOUNT": "approved_amount_zmw",
        "AMOUNT (ZMW)": "approved_amount_zmw",
        "COST": "approved_amount_zmw",
        "BUDGET": "approved_amount_zmw",
        "ESTIMATE__1": "approved_amount_zmw",
        "SOURCE_DOC": "source_doc",   # ← normalise to lowercase early
    }
    cdf = cdf.rename(columns=RENAME_MAP)

    # 3. Drop stray columns
    DROP_COLS = [c for c in cdf.columns
                 if c.startswith("COL_") or c == "_PAGE" or c == ""
                 or c.startswith("ESTIMATE__") or c.startswith("AMOUNT__")
                 or c.startswith("WARD__") or c.startswith("SECTOR__")]
    cdf = cdf.drop(columns=DROP_COLS, errors="ignore")
    print(f"After dropping stray columns: {cdf.shape}")

    # 4. Coalesce duplicate canonical columns
    def coalesce(df, col):
        positions = [i for i, c in enumerate(df.columns) if c == col]
        if len(positions) <= 1:
            return df
        combined = df.iloc[:, positions[0]]
        for pos in positions[1:]:
            combined = combined.where(combined.notna(), df.iloc[:, pos])
        df = df.drop(df.columns[positions], axis=1)
        df.insert(positions[0], col, combined)
        return df

    for canonical in ["project_name", "approved_amount_zmw",
                      "engineer_estimate_zmw", "ward", "sector",
                      "description", "site", "project_no", "source_doc"]:
        cdf = coalesce(cdf, canonical)

    # 5. Clean text
    for c in ["project_name", "description", "sector", "ward", "site"]:
        if c in cdf.columns:
            cdf[c] = cdf[c].apply(clean_text)

    # 6. Clean amounts
    for c in ["engineer_estimate_zmw", "approved_amount_zmw"]:
        if c in cdf.columns:
            cdf[c] = cdf[c].apply(clean_amount)

    # 7. Drop header repeats
    if "project_no" in cdf.columns:
        cdf = cdf[
            ~cdf["project_no"].astype(str).str.upper()
              .str.contains("PROJECT|NAME|WARD|SECTOR|AMOUNT",
                            na=False, regex=True)
        ]

    # 8. Drop rows without a valid project name
    if "project_name" in cdf.columns:
        cdf = cdf[
            cdf["project_name"].notna()
            & (cdf["project_name"].astype(str).str.len() >= 5)
            & (~cdf["project_name"].str.lower().str.contains(
                "sub total|total|summary", na=False))
        ]

    # 9. Drop rows with no amount
    amt_cols = [c for c in ["engineer_estimate_zmw", "approved_amount_zmw"]
                if c in cdf.columns]
    if amt_cols:
        cdf = cdf[cdf[amt_cols].notna().any(axis=1)]

    # 10. Canonicalise
    if "sector" in cdf.columns:
        cdf["sector"] = cdf["sector"].apply(canonical_sector)
    if "ward" in cdf.columns:
        cdf["ward"] = cdf["ward"].apply(canonical_ward)

    # 11. Fill missing categorical values
    for c in ["sector", "ward", "site", "description"]:
        if c in cdf.columns:
            cdf[c] = cdf[c].fillna("Unknown")

    # 12. Drop any remaining duplicate uppercase SOURCE_DOC
    if "SOURCE_DOC" in cdf.columns:
        cdf = cdf.drop(columns=["SOURCE_DOC"])

    # 13. Metadata
    cdf["council"]        = COUNCIL_FULL
    cdf["constituency"]   = CONSTITUENCY
    cdf["status"]         = "Approved"
    cdf["funding_source"] = "CDF"
    cdf["source_doc"]     = "CDF_PROJECTS_2025"
    cdf["scraped_at"]     = RUN_STAMP

    # 14. Final order and save
    cdf = cdf.reset_index(drop=True)
    cdf_clean = cdf.copy()
    cdf_clean.to_csv(CLEAN_DIR / "projects_clean.csv", sep="|", index=False)

    print(f"\n✅ Cleaned CDF rows: {len(cdf_clean)}")
    print(f"   Columns: {list(cdf_clean.columns)}")
    cdf_clean.head()

## Phase 3 · Section 3.3 — Clean the Newsletter Projects

The Newsletter projects have a simpler schema. We apply the same cleaning
principles and align them to the master schema so they can be merged.

Because the Newsletter PDF has a magazine layout, project names may be
truncated at the left edge. We accept this as a limitation and document it in
the Data in Brief paper.

In [ ]:
# ============================================================
# PHASE 3 · CELL 4 — CLEAN NEWSLETTER PROJECTS
# ============================================================

news = extracted.get("newsletter_raw", pd.DataFrame()).copy()

if news.empty:
    print("newsletter_raw.csv is empty — skipping.")
    news_clean = pd.DataFrame()
else:
    print(f"Raw Newsletter rows: {len(news)}")

    # Clean text
    for c in ["project_name", "sector", "status"]:
        if c in news.columns:
            news[c] = news[c].apply(clean_text)

    # Amount
    if "approved_amount_zmw" in news.columns:
        news["approved_amount_zmw"] = news["approved_amount_zmw"].apply(clean_amount)

    # Canonicalise sector
    if "sector" in news.columns:
        news["sector"] = news["sector"].apply(canonical_sector)

    # Metadata
    news["council"]        = COUNCIL_FULL
    news["constituency"]   = CONSTITUENCY
    news["funding_source"] = "CDF"
    news["scraped_at"]     = RUN_STAMP

    # Drop rows without a valid project name
    if "project_name" in news.columns:
        news = news[
            news["project_name"].notna()
            & (news["project_name"].astype(str).str.len() >= 5)
        ]

    news_clean = news.reset_index(drop=True)
    news_clean.to_csv(CLEAN_DIR / "newsletter_clean.csv", sep="|", index=False)

    print(f"\nCleaned Newsletter rows: {len(news_clean)}")
    print(news_clean.head())

## Phase 3 · Section 3.4 — Clean the Narrative Tables

The remaining tables — strategic areas, goals, statistics, sub-programmes,
M&E framework, cost estimates, newsletter narrative, citizen points — carry
mostly text with some numeric fields.

For each one we:

1. Clean every text column (`clean_text`)
2. Drop junk rows using our `is_real_item` filter where applicable
3. Coerce numeric fields
4. Save each to `data/clean/`

The `drop_junk_rows()` function applies the shared junk filter to the
specific column that carries the primary content for that table.

In [ ]:
# ============================================================
# PHASE 3 · CELL 5 — CLEAN NARRATIVE TABLES
# ============================================================

TEXT_COL_FOR_JUNK = {
    "goals":                "goal_text",
    "subprogrammes":        "sub_programme",
    "me_framework":         "Activities",
    "citizen_points":       "point",
    "citizen_priorities":   "priority",
    "newsletter_narrative": "sentence",
    "strategic_areas":      "strategic_area",
    "wards":                "ward_name",
    "statistics":           "context",
    "cost_estimates":       "programme",
}


def drop_junk_rows(df, text_col, min_words=4):
    """Remove rows where the target text column looks like junk."""
    if df.empty or text_col not in df.columns:
        return df
    mask = df[text_col].apply(
        lambda s: isinstance(s, str) and is_real_item(s, min_words=min_words)
    )
    return df[mask].reset_index(drop=True)


cleaned_summary = []
narrative_keys = [
    "strategic_areas",
    "goals",
    "statistics",
    "subprogrammes",
    "me_framework",
    "cost_estimates",
    "newsletter_narrative",
    "citizen_points",
    "citizen_priorities",
    "wards",
]

print(f"{'Table':<25} {'Before':>8} {'After':>8}")
print("-" * 45)

for key in narrative_keys:
    df = extracted.get(key, pd.DataFrame()).copy()
    if df.empty:
        cleaned_summary.append((key, 0, 0))
        print(f"{key:<25} {'(empty)':>8} {'-':>8}")
        continue

    rows_before = len(df)

    # Clean all text columns
    for c in df.columns:
        if df[c].dtype == object:
            df[c] = df[c].apply(clean_text)

    # Apply junk filter where appropriate
    if key in TEXT_COL_FOR_JUNK:
        df = drop_junk_rows(df, TEXT_COL_FOR_JUNK[key])

    # Coerce numeric fields
    if key == "cost_estimates":
        if "year" in df.columns:
            df["year"] = pd.to_numeric(df["year"], errors="coerce").astype("Int64")
        if "cost_zmw" in df.columns:
            df["cost_zmw"] = pd.to_numeric(df["cost_zmw"], errors="coerce")
    if key == "statistics" and "value" in df.columns:
        df["value"] = pd.to_numeric(df["value"], errors="coerce")

    df.to_csv(CLEAN_DIR / f"{key}_clean.csv", sep="|", index=False)
    cleaned_summary.append((key, rows_before, len(df)))
    print(f"{key:<25} {rows_before:>8} {len(df):>8}")

print("\nNarrative tables cleaned.")

## Phase 3 · Section 3.5 — Merge Into Master Project Table

We union the CDF projects and Newsletter projects into one table with a common
schema. Every row gets a synthetic `project_id` and carries provenance metadata
(`source_doc`, `funding_source`, `scraped_at`).

This is the master table that will be our headline Kaggle dataset.

In [ ]:
# ============================================================
# PHASE 3 · CELL 6 — MERGE INTO MASTER PROJECT TABLE
# ============================================================

PROJECT_COLUMNS = [
    "project_id", "council", "project_name", "sector", "constituency",
    "ward", "approved_amount_zmw", "status", "funding_source",
    "source_doc", "scraped_at",
]

frames = []
if not cdf_clean.empty:
    frames.append(cdf_clean.reindex(columns=PROJECT_COLUMNS))
if not news_clean.empty:
    frames.append(news_clean.reindex(columns=PROJECT_COLUMNS))

projects_master = (
    pd.concat(frames, ignore_index=True, sort=False)
    if frames
    else pd.DataFrame(columns=PROJECT_COLUMNS)
)

projects_master["project_id"] = make_project_ids(len(projects_master))
projects_master["approved_amount_zmw"] = pd.to_numeric(
    projects_master["approved_amount_zmw"], errors="coerce"
)

projects_master = projects_master.reset_index(drop=True)
projects_master.to_csv(CLEAN_DIR / "projects_master.csv", sep="|", index=False)

print(f"Master project table: {projects_master.shape}")
print(f"   Columns: {list(projects_master.columns)}")
print(projects_master.head())

## Phase 3 · Section 3.6 — Enrichment and Derived Columns

We add analytical columns that make the master table easier to use:

- **`sector_category`** — Social, Infrastructure, Economic, Governance, Other
- **`has_amount_zmw`** — boolean
- **`is_approved`** — boolean
- **`amount_band`** — Small, Medium, Large, Very Large
- **`has_ward`** — boolean (ward is not vague)

In [ ]:
# ============================================================
# PHASE 3 · CELL 7 — ENRICHMENT & DERIVED COLUMNS
# ============================================================

projects_master["has_amount_zmw"] = projects_master["approved_amount_zmw"].notna()

projects_master["is_approved"] = (
    projects_master["status"].astype(str).str.lower() == "approved"
)

SECTOR_GROUP = {
    "Education":            "Social",
    "Health":               "Social",
    "Water and Sanitation": "Infrastructure",
    "Roads and Drainages":  "Infrastructure",
    "Commerce":             "Economic",
    "Agriculture":          "Economic",
    "Energy":               "Infrastructure",
    "Governance":           "Governance",
    "Environment":          "Environment",
    "Housing":              "Infrastructure",
}
projects_master["sector_category"] = (
    projects_master["sector"].map(SECTOR_GROUP).fillna("Other")
)

def amount_band(a):
    if pd.isna(a):
        return "Unknown"
    if a < 100_000:
        return "Small (<100K)"
    if a < 1_000_000:
        return "Medium (100K-1M)"
    if a < 5_000_000:
        return "Large (1M-5M)"
    return "Very Large (>=5M)"

projects_master["amount_band"] = projects_master["approved_amount_zmw"].apply(amount_band)

projects_master["has_ward"] = (
    projects_master["ward"].notna()
    & (~projects_master["ward"].astype(str).str.lower().isin(VAGUE_WARDS))
)

projects_master.to_csv(CLEAN_DIR / "projects_master_enriched.csv",
                       sep="|", index=False)

print(f"Enriched: {projects_master.shape}")
print(f"   Columns: {list(projects_master.columns)}")
print("\nSector distribution:")
print(projects_master["sector"].value_counts().to_string())
print("\nAmount band distribution:")
print(projects_master["amount_band"].value_counts().to_string())

## Phase 3 · Section 3.7 — Deduplicate

We drop rows that describe the same project twice. Our dedup key is
`project_name | ward | approved_amount` — this catches genuine duplicates
without collapsing similar-sounding projects in different wards.

In [ ]:
# ============================================================
# PHASE 3 · CELL 8 — DEDUPLICATE
# ============================================================

before = len(projects_master)

key = (
    projects_master["project_name"].fillna("").str.lower().str.strip()
    + "|" + projects_master["ward"].fillna("").str.lower().str.strip()
    + "|" + projects_master["approved_amount_zmw"].fillna(-1).astype(str)
)

projects_master = (
    projects_master
      .assign(_key=key)
      .drop_duplicates(subset=["_key"], keep="first")
      .drop(columns=["_key"])
      .reset_index(drop=True)
)

projects_master["project_id"] = make_project_ids(len(projects_master))

after = len(projects_master)
print(f"Before dedup : {before}")
print(f"After dedup  : {after}")
print(f"Removed      : {before - after}")

projects_master.to_csv(CLEAN_DIR / "projects_master_final.csv",
                       sep="|", index=False)

## Phase 3 · Section 3.8 — Derived Analytics Tables

We build two derived analytics tables from the master project table:

- **Sector × Ward matrix** — projects per sector per ward (spatial view)
- **Timeline** — project count and total ZMW by funding year (temporal view)

In [ ]:
# ============================================================
# PHASE 3 · CELL 9 — DERIVED ANALYTICS TABLES
# ============================================================

# Sector × Ward matrix
sector_by_ward = (
    projects_master[projects_master["has_ward"] == True]
      .groupby(["ward", "sector"], dropna=False)
      .size()
      .unstack(fill_value=0)
      .reset_index()
)
sector_by_ward.columns.name = None
sector_by_ward = sector_by_ward.rename(columns={"ward": "ward_name"})

sector_cols = [c for c in sector_by_ward.columns if c != "ward_name"]
sector_by_ward["total_projects"] = sector_by_ward[sector_cols].sum(axis=1)

sector_by_ward.to_csv(CLEAN_DIR / "sector_by_ward.csv", sep="|", index=False)
print(f"Sector × Ward matrix: {sector_by_ward.shape}")
print(sector_by_ward.head())

# Timeline
def infer_year(source_doc):
    if "NEWSLETTER" in str(source_doc):
        return 2024
    if "CDF_PROJECTS_2025" in str(source_doc):
        return 2025
    return None

projects_master["inferred_year"] = projects_master["source_doc"].apply(infer_year)

timeline = (
    projects_master
      .dropna(subset=["inferred_year"])
      .groupby("inferred_year", as_index=False)
      .agg(
          project_count=("project_id", "count"),
          total_amount_zmw=("approved_amount_zmw", "sum"),
      )
      .rename(columns={"inferred_year": "year"})
)

timeline.to_csv(CLEAN_DIR / "timeline.csv", sep="|", index=False)
print(f"\nTimeline: {timeline.shape}")
print(timeline.to_string(index=False))

## Phase 3 · Section 3.9 — Export Final CSVs

We export every cleaned table to `outputs/` following the required naming
convention: pipe-delimited CSVs with filenames of the form
`db-unza26-csc4792-kabwe_<description>.csv`.

Empty tables are skipped so we don't ship header-only files.

In [ ]:
# ============================================================
# PHASE 3 · CELL 10 (v3 — with citizen_points) — EXPORT
# ============================================================

def load_clean(name):
    p = CLEAN_DIR / f"{name}_clean.csv"
    return pd.read_csv(p, sep="|") if p.exists() else pd.DataFrame()


outputs = {
    f"{PROJECT_CODE}-{COUNCIL_SLUG}_idp_projects.csv":         projects_master,
    f"{PROJECT_CODE}-{COUNCIL_SLUG}_idp_strategic_areas.csv":  load_clean("strategic_areas"),
    f"{PROJECT_CODE}-{COUNCIL_SLUG}_idp_goals.csv":            load_clean("goals"),
    f"{PROJECT_CODE}-{COUNCIL_SLUG}_idp_statistics.csv":       load_clean("statistics"),
    f"{PROJECT_CODE}-{COUNCIL_SLUG}_idp_subprogrammes.csv":    load_clean("subprogrammes"),
    f"{PROJECT_CODE}-{COUNCIL_SLUG}_idp_me_framework.csv":     load_clean("me_framework"),
    f"{PROJECT_CODE}-{COUNCIL_SLUG}_idp_cost_estimates.csv":   load_clean("cost_estimates"),
    f"{PROJECT_CODE}-{COUNCIL_SLUG}_idp_citizen_points.csv":   load_clean("citizen_points"),   # ← ADD THIS
    f"{PROJECT_CODE}-{COUNCIL_SLUG}_newsletter_narrative.csv": load_clean("newsletter_narrative"),
    f"{PROJECT_CODE}-{COUNCIL_SLUG}_sector_by_ward.csv":       sector_by_ward,
    f"{PROJECT_CODE}-{COUNCIL_SLUG}_timeline.csv":             timeline,
}

print("Writing CSVs to outputs/:\n")
written, skipped = 0, 0
for fname, df in outputs.items():
    if isinstance(df, pd.DataFrame) and not df.empty:
        df.to_csv(OUTPUT_DIR / fname, sep="|", index=False, encoding="utf-8")
        print(f"  [written] {fname:<60}  {len(df):>5} rows  {df.shape[1]:>2} cols")
        written += 1
    else:
        print(f"  [skip]    {fname:<60}  (empty)")
        skipped += 1

print(f"\n📊 {written} files written, {skipped} skipped")

## Phase 3 · Section 3.10 — Data Quality Report

We produce a final quality report summarizing the dataset. These numbers go
into the Kaggle README and the Data in Brief paper.

In [ ]:
# ============================================================
# PHASE 3 · CELL 11 — DATA QUALITY REPORT
# ============================================================

print("=" * 72)
print(f"DATASET SUMMARY — {COUNCIL_FULL}")
print("=" * 72)
print(f"Generated: {RUN_STAMP}\n")

total_rows = 0
for fname in sorted(outputs.keys()):
    df = outputs[fname]
    if isinstance(df, pd.DataFrame) and not df.empty:
        total_rows += len(df)
        print(f"  {fname:<60}  {len(df):>5} rows")
    else:
        print(f"  {fname:<60}  (empty)")

print("-" * 72)
print(f"  {'GRAND TOTAL':<60}  {total_rows:>5} rows")
print()

print("Master project table — completeness:")
print("-" * 72)
for c in projects_master.columns:
    nn  = projects_master[c].notna().sum()
    pct = 100 * nn / max(len(projects_master), 1)
    bar = "█" * int(pct / 5)
    print(f"  {c:25s}  {nn:>4}/{len(projects_master):<4}  ({pct:>5.1f}%) {bar}")

print()
print(f"Total CDF value captured: "
      f"{projects_master['approved_amount_zmw'].sum():,.2f} ZMW")
print(f"Unique sectors : {projects_master['sector'].nunique()}")
print(f"Unique wards   : {projects_master['ward'].nunique()}")
print()
print("Sector distribution:")
print(projects_master["sector"].value_counts().to_string())
print()
print("Amount band distribution:")
print(projects_master["amount_band"].value_counts().to_string())
print("=" * 72)

## Phase 3 · Section 3.11 — Post-Run Verification

We reload every CSV from disk to confirm it's readable with the pipe delimiter
and its row count matches what we intended to write. Anything that doesn't
match is flagged.

In [ ]:
# ============================================================
# PHASE 3 · CELL 12 — POST-RUN VERIFICATION
# ============================================================

print("Verifying outputs...\n")

all_ok = True
for fname, df_expected in outputs.items():
    path = OUTPUT_DIR / fname
    if not path.exists():
        if isinstance(df_expected, pd.DataFrame) and not df_expected.empty:
            print(f"  MISSING: {fname}")
            all_ok = False
        continue

    df_reloaded = pd.read_csv(path, sep="|")
    if isinstance(df_expected, pd.DataFrame) and not df_expected.empty:
        if len(df_reloaded) != len(df_expected):
            print(f"  MISMATCH: {fname} "
                  f"(disk={len(df_reloaded)}, expected={len(df_expected)})")
            all_ok = False
        else:
            print(f"  ✓ {fname:<60}  {df_reloaded.shape}")

if all_ok:
    print(f"\nAll files verified. Location: {OUTPUT_DIR.resolve()}")
else:
    print(f"\nSome files had issues.")

## Phase 3 · Summary

We've completed the full cleaning pipeline. The `outputs/` folder now contains
all files we produced, each in pipe-delimited CSV format following the
assignment's naming convention.

**Files produced:**

| File | Content |
|---|---|
| `..._idp_projects.csv` | Master strategic community project registry |
| `..._idp_strategic_areas.csv` | Four 8NDP-aligned strategic areas |
| `..._idp_goals.csv` | Sector development goals |
| `..._idp_statistics.csv` | Baseline district statistics |
| `..._idp_subprogrammes.csv` | Sub-programmes by 8NDP pillar |
| `..._idp_me_framework.csv` | M&E framework rows |
| `..._idp_cost_estimates.csv` | Multi-year cost estimates |
| `..._idp_wards.csv` | Ward profile records |
| `..._newsletter_narrative.csv` | Narrative sentences from the newsletter |
| `..._sector_by_ward.csv` | Sector × ward matrix |
| `..._timeline.csv` | Project count and funding by year |

**Next steps:**
1. Combine these outputs with teammates' CSVs into the group Kaggle dataset
2. Write the Kaggle README
3. Write the Data in Brief paper
4. Commit the notebook and CSVs to GitHub
5. Submit via Moodle

## Phase 3 · Section 3.13 — Post-Cleanup Fixes

After the main cleaning pipeline runs, a few files still have quality issues
from their extraction stages. This cell applies targeted fixes to each:

1. **`strategic_areas`** — Only 2 of 4 areas survived the junk filter because
   area names are short (2 words). We restore all 4 areas by re-reading the
   source.

2. **`cost_estimates`** — The junk filter was too strict for programmatic
   cost-table rows. We keep only the essential columns and skip the filter.

3. **`statistics`** — Some rows have tiny values (1, 2, 5) that are
   extraction noise. We keep only values >= 10.

4. **`me_framework`** — The table has 39 columns due to page-spanning
   artifacts. We keep only the columns with real content.

5. **`wards`** — The ward extraction captured a bullet point, not a ward.
   We delete this file.

6. **`citizen_priorities`** — Redundant with citizen_points. We delete this
   file.

Each fix is documented in code comments so its purpose is traceable.

In [ ]:
# ============================================================
# PHASE 3 · CELL 13 — POST-CLEANUP FIXES
# ============================================================

print("Applying post-cleanup fixes...\n")

# ---------- Fix 1: strategic_areas — restore all 4 areas ----------
print("Fix 1: strategic_areas...")

STRATEGIC_AREAS = [
    "Economic Transformation and Job Creation",
    "Human and Social Development",
    "Environmental Sustainability",
    "Good Governance Environment",
]

strategic_df = pd.DataFrame([
    {
        "strategic_area":  area,
        "page_refs":       ",".join(str(p) for p, t in idp_pages
                                     if area.lower() in t.lower()),
        "source_doc":      "IDP_MAIN",
    }
    for area in STRATEGIC_AREAS
])
strategic_df.to_csv(CLEAN_DIR / "strategic_areas_clean.csv",
                    sep="|", index=False)
print(f"  ✅ strategic_areas: {len(strategic_df)} rows")


# ---------- Fix 2: cost_estimates — keep essential columns ----------
print("\nFix 2: cost_estimates...")
ce = extracted.get("cost_estimates", pd.DataFrame()).copy()
if not ce.empty:
    keep = [c for c in ["programme", "year", "cost_zmw", "page", "source_doc"]
            if c in ce.columns]
    ce = ce[keep]
    if "year" in ce.columns:
        ce["year"] = pd.to_numeric(ce["year"], errors="coerce").astype("Int64")
    if "cost_zmw" in ce.columns:
        ce["cost_zmw"] = pd.to_numeric(ce["cost_zmw"], errors="coerce")
    if "programme" in ce.columns:
        ce = ce[ce["programme"].notna()
                & (ce["programme"].astype(str).str.len() >= 3)]
    if "cost_zmw" in ce.columns:
        ce = ce[ce["cost_zmw"].notna() & (ce["cost_zmw"] > 0)]
    ce.to_csv(CLEAN_DIR / "cost_estimates_clean.csv", sep="|", index=False)
    print(f"  ✅ cost_estimates: {len(ce)} rows")
else:
    print("  ⚠️  cost_estimates: empty in extracted")


# ---------- Fix 3: statistics — drop tiny values ----------
print("\nFix 3: statistics...")
st = extracted.get("statistics", pd.DataFrame()).copy()
if not st.empty:
    if "value" in st.columns:
        st["value"] = pd.to_numeric(st["value"], errors="coerce")
        st = st[st["value"].notna() & (st["value"] >= 10)]
    if "context" in st.columns:
        st["context"] = st["context"].apply(
            lambda v: clean_text(v, max_len=200) if isinstance(v, str) else v
        )
    st.to_csv(CLEAN_DIR / "statistics_clean.csv", sep="|", index=False)
    print(f"  ✅ statistics: {len(st)} rows")
else:
    print("  ⚠️  statistics: empty in extracted")


# ---------- Fix 4 (v2): me_framework — WHITELIST columns ----------
print("\nFix 4: me_framework...")
me = extracted.get("me_framework", pd.DataFrame()).copy()
if not me.empty:
    # Whitelist: keep ONLY columns with these exact names
    USEFUL = ["page", "source_doc", "Strategies", "Program", "Activities",
              "Location", "Baseline", "Target", "Responsible",
              "Indicator", "Input", "Freq"]
    me = me[[c for c in me.columns if c in USEFUL]]

    # Drop columns that are entirely empty
    empty = [c for c in me.columns
             if me[c].apply(lambda v: pd.isna(v) or str(v).strip() == "").all()]
    me = me.drop(columns=empty, errors="ignore")

    # Keep only rows where at least one content column has data
    content_cols = [c for c in ["Activities", "Program", "Indicator",
                                 "Strategies"] if c in me.columns]
    if content_cols:
        mask = me[content_cols].notna().any(axis=1)
        me = me[mask].reset_index(drop=True)

    me.to_csv(CLEAN_DIR / "me_framework_clean.csv", sep="|", index=False)
    print(f"  ✅ me_framework: {len(me)} rows, {me.shape[1]} cols")
    print(f"     Columns kept: {list(me.columns)}")
else:
    print("  ⚠️  me_framework: empty in extracted")


# ---------- Fix 5: delete wards (junk) ----------
print("\nFix 5: wards...")
wards_path = CLEAN_DIR / "wards_clean.csv"
if wards_path.exists():
    wards_path.unlink()
    print(f"  🗑️  Deleted {wards_path.name} (junk content)")
else:
    print("  ✓  wards_clean.csv already absent")


# ---------- Fix 6: delete citizen_priorities (redundant) ----------
print("\nFix 6: citizen_priorities...")
for p in [CLEAN_DIR / "citizen_priorities_clean.csv"]:
    if p.exists():
        p.unlink()
        print(f"  🗑️  Deleted {p.name} (redundant with citizen_points)")
    else:
        print(f"  ✓  {p.name} already absent")


# ---------- Fix 7: citizen_points — deduplicate and drop fragments ----------
print("\nFix 7: citizen_points...")
cp = extracted.get("citizen_points", pd.DataFrame()).copy()
if not cp.empty and "point" in cp.columns:
    # Clean text
    cp["point"] = cp["point"].apply(clean_text)

    # Drop fragments: length < 20 chars
    cp = cp[cp["point"].notna()
            & (cp["point"].astype(str).str.len() >= 20)]

    # Remove TOC-style fragments
    cp = cp[~cp["point"].astype(str).str.contains(r"\.{3,}", na=False, regex=True)]

    # Deduplicate
    cp = cp.drop_duplicates(subset=["point"]).reset_index(drop=True)

    # Save
    cp.to_csv(CLEAN_DIR / "citizen_points_clean.csv", sep="|", index=False)
    print(f"  ✅ citizen_points: {len(cp)} rows")
else:
    print("  ⚠️  citizen_points: empty in extracted")


print("\n✅ Post-cleanup fixes applied.")

## Phase 3 · Section 3.13b — Targeted Fixes for me_framework & citizen_points

Two files still carry too much noise after the main cleanup:

**`me_framework_clean.csv`** — 21 rows × 12 columns. Most columns are
sparse (fewer than 30% of rows have data). We drop any column that is
populated in less than 30% of rows, keeping only `page`, `source_doc`
plus the columns that actually carry content.

**`citizen_points_clean.csv`** — 77 rows. Some are real community
priority statements; some are fragments of the introduction. We apply
a second-pass filter that keeps only rows that:
  - Start with a capital letter
  - Contain 6+ words
  - Do NOT start with a banned prefix (`Due to`, `Poor`, `Not enough`,
    `Majority`, `Reduction in`, `Attracts`, `Land telephone`, etc.)
  - Do NOT end with a colon or a dangling preposition

In [ ]:
# ============================================================
# PHASE 3 · CELL 13b — TARGETED FIXES (v2 with bullet splitting)
# ============================================================

print("Applying targeted fixes...\n")


# ============================================================
# FIX A: me_framework — drop sparse columns
# ============================================================
print("Fix A: me_framework — drop sparse columns...")

me = extracted.get("me_framework", pd.DataFrame()).copy()
if me.empty:
    print("  ⚠️  me_framework is empty in extracted")
else:
    # Whitelist: keep only known-useful column names
    USEFUL = ["page", "source_doc", "Strategies", "Program", "Activities",
              "Location", "Baseline", "Target", "Responsible",
              "Indicator", "Input", "Freq"]
    me = me[[c for c in me.columns if c in USEFUL]]

    # These columns are always kept, even if sparse
    MUST_KEEP = {"page", "source_doc"}

    # Drop columns where fewer than 30% of rows have content
    n_rows = len(me)
    min_filled = n_rows * 0.30

    sparse_cols = []
    for c in me.columns:
        if c in MUST_KEEP:
            continue
        filled = me[c].apply(
            lambda v: not (pd.isna(v) or str(v).strip() == "")
        ).sum()
        if filled < min_filled:
            sparse_cols.append(c)

    me = me.drop(columns=sparse_cols, errors="ignore")
    print(f"  Dropped sparse columns: {sparse_cols}")

    # Drop rows where every content column is empty
    content_cols = [c for c in me.columns if c not in MUST_KEEP]
    if content_cols:
        mask = me[content_cols].apply(
            lambda row: any(
                not (pd.isna(v) or str(v).strip() == "") for v in row
            ),
            axis=1,
        )
        me = me[mask].reset_index(drop=True)

    me.to_csv(CLEAN_DIR / "me_framework_clean.csv", sep="|", index=False)
    print(f"  ✅ me_framework: {len(me)} rows, {me.shape[1]} cols")
    print(f"     Columns kept: {list(me.columns)}")


# ============================================================
# FIX B: citizen_points — strict filter + bullet splitting
# ============================================================
print("\nFix B: citizen_points — strict filter with bullet splitting...")

cp = extracted.get("citizen_points", pd.DataFrame()).copy()

if cp.empty or "point" not in cp.columns:
    print("  ⚠️  citizen_points is empty or missing 'point' column")
else:
    # ---------- 1. Clean text ----------
    cp["point"] = cp["point"].apply(clean_text)

    # ---------- 2. Drop nulls and short fragments ----------
    cp = cp[
        cp["point"].notna()
        & (cp["point"].astype(str).str.len() >= 25)
    ]

    # ---------- 3. KEY STEP: split on bullet and keep the action ----------
    # Many Citizen IDP rows look like:
    #   "Absence of a farmer training center. • Construct one Farmer training center"
    #   "High dependence on rains. • Construct a dam in Munyama Block by 2025."
    # We keep only the part AFTER the bullet — the proposed action.
    def keep_action_after_bullet(text):
        if not isinstance(text, str):
            return text
        if "•" in text:
            parts = [p.strip() for p in text.split("•") if p.strip()]
            if parts:
                # Return the longest part (usually the action)
                return max(parts, key=len)
        return text

    cp["point"] = cp["point"].apply(keep_action_after_bullet)

    # Re-clean after split
    cp["point"] = cp["point"].apply(clean_text)

    # Drop again if the action part is too short
    cp = cp[
        cp["point"].notna()
        & (cp["point"].astype(str).str.len() >= 20)
    ]

    # ---------- 4. Must start with a capital letter ----------
    cp = cp[cp["point"].astype(str).str[0].str.isupper()]

    # ---------- 5. Must have 5+ words (relaxed to 5) ----------
    cp = cp[cp["point"].astype(str).str.split().str.len() >= 5]

    # ---------- 6. Must NOT end with colon or dangling preposition ----------
    bad_endings = r"(:\s*$|\b(for|of|in|to|with|and|by|at|on|the|a|an|from|as)\s*$)"
    cp = cp[~cp["point"].astype(str).str.contains(
        bad_endings, regex=True, case=False, na=False
    )]

    # ---------- 7. Must NOT contain TOC leaders ----------
    cp = cp[~cp["point"].astype(str).str.contains(
        r"\.{3,}", regex=True, na=False
    )]

    # ---------- 8. Must NOT start with these noise prefixes ----------
    BAD_PREFIXES = [
        "due to", "poor ", "not enough", "majority", "reduction in",
        "attracts ", "land telephone", "lack of", "increased use",
        "onsite sanitation", "over ", "to create", "the extension",
        "the district", "the council", "in 2023", "in 2024", "in 2025",
    ]
    lower = cp["point"].astype(str).str.lower()
    for prefix in BAD_PREFIXES:
        cp = cp[~lower.str.startswith(prefix)]
        # Recompute lower after each filter to stay in sync with cp
        lower = cp["point"].astype(str).str.lower()

    # ---------- 9. Must NOT be an ALL-CAPS heading ----------
    cp = cp[~cp["point"].astype(str).str.match(
        r"^[A-Z\s\.\-]+$", na=False
    )]

    # ---------- 10. Deduplicate ----------
    cp = cp.drop_duplicates(subset=["point"]).reset_index(drop=True)

    # ---------- 11. Save ----------
    cp.to_csv(CLEAN_DIR / "citizen_points_clean.csv", sep="|", index=False)
    print(f"  ✅ citizen_points: {len(cp)} rows")
    print(f"     Sample (first 10):")
    for s in cp["point"].head(10):
        print(f"       • {s[:90]}")


print("\n✅ Targeted fixes applied.")

In [ ]:
# ============================================================
# VERIFY CDF EXTRACTION WITH 'LINES' STRATEGY
# ============================================================

from pathlib import Path
import pdfplumber
import pandas as pd

# Resolve paths
CWD = Path.cwd()
REPO_ROOT = CWD.parent if CWD.name == "notebooks" else CWD

RAW_DIR = REPO_ROOT / "data" / "raw" / "idp"
CDF_PATH = RAW_DIR / "2025-Approved-Community-Projects_Kabwe-Central.pdf"

# Use the same extract_all_tables from Phase 2 · Cell 2
cdf_tables = extract_all_tables(CDF_PATH)

print(f"Tables detected: {len(cdf_tables)}\n")
for i, t in enumerate(cdf_tables):
    print(f"Table {i+1}: shape={t.shape}")
    print(f"  Columns: {list(t.columns)}")
    print(f"  First data row: {list(t.iloc[0].values)[:6]}")
    print()

# Show the main project table
if cdf_tables:
    main = cdf_tables[0]
    print(f"\n{'=' * 70}")
    print(f"MAIN PROJECT TABLE — first 5 rows")
    print(f"{'=' * 70}")
    print(main.head().to_string())

In [ ]:
from pathlib import Path
import pandas as pd

CWD = Path.cwd()
REPO_ROOT = CWD.parent if CWD.name == "notebooks" else CWD

# Check extracted files
print("data/extracted/:")
for f in sorted((REPO_ROOT / "data" / "extracted").glob("*.csv")):
    df = pd.read_csv(f, sep="|")
    print(f"  {f.name:<40}  {df.shape}")

print("\noutputs/:")
for f in sorted((REPO_ROOT / "outputs").glob("*.csv")):
    df = pd.read_csv(f, sep="|")
    print(f"  {f.name:<55}  {df.shape}")

In [ ]:
import pdfplumber
from pathlib import Path

CWD = Path.cwd()
REPO_ROOT = CWD.parent if CWD.name == "notebooks" else CWD
CDF_PATH = REPO_ROOT / "data" / "raw" / "idp" / "2025-Approved-Community-Projects_Kabwe-Central.pdf"

LINE_SETTINGS = {"vertical_strategy": "lines", "horizontal_strategy": "lines"}

with pdfplumber.open(CDF_PATH) as pdf:
    for pno, page in enumerate(pdf.pages, start=1):
        tables = page.extract_tables(LINE_SETTINGS)
        print(f"\n{'=' * 70}")
        print(f"PAGE {pno}: {len(tables)} tables")
        print(f"{'=' * 70}")
        for i, t in enumerate(tables):
            if not t:
                continue
            print(f"\n  Table {i+1}: {len(t)} rows × {len(t[0])} cols")
            print(f"    Header: {[str(c)[:25] for c in t[0] if c]}")
            # Show first 2 data rows
            for j, row in enumerate(t[1:3], start=1):
                print(f"    Row {j}: {[str(c)[:25] for c in row if c][:4]}")

In [ ]:
import pandas as pd
from pathlib import Path

CWD = Path.cwd()
REPO_ROOT = CWD.parent if CWD.name == "notebooks" else CWD
CLEAN_DIR = REPO_ROOT / "data" / "clean"

print("data/clean/ contents:")
for f in sorted(CLEAN_DIR.glob("*.csv")):
    df = pd.read_csv(f, sep="|")
    print(f"  {f.name:<40} {df.shape}")

In [ ]:
import pandas as pd
from pathlib import Path

CWD = Path.cwd()
REPO_ROOT = CWD.parent if CWD.name == "notebooks" else CWD
OUTPUT_DIR = REPO_ROOT / "outputs"

print("=" * 72)
print("KABWE MUNICIPAL COUNCIL — FINAL DATASET")
print("=" * 72)

total = 0
files = sorted(OUTPUT_DIR.glob("*.csv"))
for f in files:
    df = pd.read_csv(f, sep="|")
    total += len(df)
    print(f"  {f.name:<58} {len(df):>5} rows × {df.shape[1]:>2} cols")

print("-" * 72)
print(f"  {'TOTAL':<58} {total:>5} rows across {len(files)} files")
print("=" * 72)

# Sanity checks
print("\nSanity checks:")

# 1. Total CDF value
proj = pd.read_csv(OUTPUT_DIR / "db-unza26-csc4792-kabwe_idp_projects.csv", sep="|")
total_value = proj["approved_amount_zmw"].sum()
print(f"  ✓ Total funding captured:  {total_value:,.2f} ZMW")

# 2. Sector distribution
print(f"\n  Sector distribution in projects.csv:")
for sector, count in proj["sector"].value_counts().items():
    print(f"    {sector:<25} {count}")

# 3. No empty files
empty_files = [f.name for f in files if pd.read_csv(f, sep="|").empty]
if empty_files:
    print(f"\n  ⚠️  Empty files: {empty_files}")
else:
    print(f"\n  ✓ No empty files")

In [ ]:
from pathlib import Path

CWD = Path.cwd()
REPO_ROOT = CWD.parent if CWD.name == "notebooks" else CWD
OUTPUT_DIR = REPO_ROOT / "outputs"

# Files that should NOT exist in the final output
STALE = [
    "db-unza26-csc4792-kabwe_idp_wards.csv",
    "db-unza26-csc4792-kabwe_idp_citizen_priorities.csv",
    "db-unza26-csc4792-kabwe_idp_budget.csv",
]

for fname in STALE:
    p = OUTPUT_DIR / fname
    if p.exists():
        p.unlink()
        print(f"🗑️  Deleted {fname}")
    else:
        print(f"✓  {fname} not present")

In [ ]:
import pandas as pd
from pathlib import Path

CWD = Path.cwd()
REPO_ROOT = CWD.parent if CWD.name == "notebooks" else CWD
OUTPUT_DIR = REPO_ROOT / "outputs"

print("=" * 72)
print("KABWE MUNICIPAL COUNCIL — FINAL DATASET")
print("=" * 72)

total = 0
files = sorted(OUTPUT_DIR.glob("*.csv"))
for f in files:
    df = pd.read_csv(f, sep="|")
    total += len(df)
    print(f"  {f.name:<58} {len(df):>5} rows × {df.shape[1]:>2} cols")

print("-" * 72)
print(f"  {'TOTAL':<58} {total:>5} rows across {len(files)} files")
print("=" * 72)

In [ ]:
import requests
import pandas as pd
import pdfplumber

from bs4 import BeautifulSoup
from io import BytesIO
from urllib.parse import urljoin

In [ ]:
import urllib3
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

In [ ]:
import requests
import urllib3
from bs4 import BeautifulSoup

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

url = "https://www.kabwecouncil.gov.zm/"

response = requests.get(
    url,
    timeout=30,
    verify=False
)

print("Status code:", response.status_code)

soup = BeautifulSoup(response.text, "html.parser")

print("Website title:", soup.title.get_text(strip=True))

Status code: 200
Website title: Kabwe Municipal Council – Kabwe


In [ ]:
publications_url = "https://www.kabwecouncil.gov.zm/?page_id=195"

response = requests.get(
    publications_url,
    timeout=30,
    verify=False
)

print("Status code:", response.status_code)

soup = BeautifulSoup(response.text, "html.parser")

print(soup.title.get_text(strip=True))

Status code: 200
Publications – Kabwe Municipal Council


In [ ]:
budget_links = []

for link in soup.find_all("a", href=True):
    text = link.get_text(" ", strip=True)
    href = link["href"]

    if any(word in text.lower() for word in [
        "budget",
        "financial",
        "finance"
    ]):
        budget_links.append({
            "title": text,
            "url": href
        })

budget_df = pd.DataFrame(budget_links)

budget_df

,title,url
0,Dept of Finance,https://www.kabwecouncil.gov.zm/?page_id=2643
1,Dept of Finance,https://www.kabwecouncil.gov.zm/?page_id=2643
2,2018 Financial Statement,https://www.kabwecouncil.gov.zm/wp-content/upl...
3,2019 Financial Statement,https://www.kabwecouncil.gov.zm/wp-content/upl...
4,2020 Financial Statement,https://www.kabwecouncil.gov.zm/wp-content/upl...
5,2021 Financial Statement,https://www.kabwecouncil.gov.zm/wp-content/upl...
6,2022 Financial Statement,https://www.kabwecouncil.gov.zm/wp-content/upl...
7,2023 Financial Statement,https://www.kabwecouncil.gov.zm/wp-content/upl...
8,2024 Financial Statement,https://www.kabwecouncil.gov.zm/wp-content/upl...
9,2025 BI Annual Budget Performance Report,https://www.kabwecouncil.gov.zm/wp-content/upl...


In [ ]:
financial_statements_df = budget_df[budget_df['title'].str.contains('Financial Statement', case=False, na=False)]
display(financial_statements_df)

,title,url
2,2018 Financial Statement,https://www.kabwecouncil.gov.zm/wp-content/upl...
3,2019 Financial Statement,https://www.kabwecouncil.gov.zm/wp-content/upl...
4,2020 Financial Statement,https://www.kabwecouncil.gov.zm/wp-content/upl...
5,2021 Financial Statement,https://www.kabwecouncil.gov.zm/wp-content/upl...
6,2022 Financial Statement,https://www.kabwecouncil.gov.zm/wp-content/upl...
7,2023 Financial Statement,https://www.kabwecouncil.gov.zm/wp-content/upl...
8,2024 Financial Statement,https://www.kabwecouncil.gov.zm/wp-content/upl...


In [ ]:
import pandas as pd
pd.set_option("display.max_colwidth", None)

budget_df

,title,url
0,Dept of Finance,https://www.kabwecouncil.gov.zm/?page_id=2643
1,Dept of Finance,https://www.kabwecouncil.gov.zm/?page_id=2643
2,2018 Financial Statement,https://www.kabwecouncil.gov.zm/wp-content/uploads/2024/09/FS-2018.pdf
3,2019 Financial Statement,https://www.kabwecouncil.gov.zm/wp-content/uploads/2024/09/FS-2019.pdf
4,2020 Financial Statement,https://www.kabwecouncil.gov.zm/wp-content/uploads/2024/09/FS-2020.pdf
5,2021 Financial Statement,https://www.kabwecouncil.gov.zm/wp-content/uploads/2025/10/FS-2021.pdf
6,2022 Financial Statement,https://www.kabwecouncil.gov.zm/wp-content/uploads/2024/09/Financial-Statement-2022.pdf
7,2023 Financial Statement,https://www.kabwecouncil.gov.zm/wp-content/uploads/2024/11/2023-FINANCIAL-STATEMENTS-KMC-1.pdf
8,2024 Financial Statement,https://www.kabwecouncil.gov.zm/wp-content/uploads/2025/10/KABWE-M-COUNCIL-2024-APPROVED-FINANCIAL-STATEMENTS.pdf
9,2025 BI Annual Budget Performance Report,https://www.kabwecouncil.gov.zm/wp-content/uploads/2025/08/2025-BI-ANNUAL-FINANCIAL-STATEMENT-KABWE-M-COUNCIL-1-1.pdf


In [ ]:
budget_row = budget_df[
    budget_df["title"].str.contains(
        "2025 Kabwe Municipal Council OBB Approved Budget",
        case=False,
        na=False
    )
]

budget_row

,title,url
12,2025 Kabwe Municipal Council OBB Approved Budget,https://www.kabwecouncil.gov.zm/wp-content/uploads/2025/05/2025-KABWE-M-COUNCIL-OBB.pdf


In [ ]:
budget_url = budget_row["url"].iloc[0]

print("Budget URL:")
print(budget_url)

Budget URL:
https://www.kabwecouncil.gov.zm/wp-content/uploads/2025/05/2025-KABWE-M-COUNCIL-OBB.pdf


In [ ]:
budget_response = requests.get(
    budget_url,
    timeout=60,
    verify=False
)

print("Status code:", budget_response.status_code)
print("File size:", len(budget_response.content), "bytes")

Status code: 200
File size: 792438 bytes


In [ ]:
import pdfplumber
from io import BytesIO

pdf = pdfplumber.open(
    BytesIO(budget_response.content)
)

print("Number of pages:", len(pdf.pages))

Number of pages: 54


In [ ]:
for page_number in range(min(5, len(pdf.pages))):
    page = pdf.pages[page_number]

    print(f"\n========== PAGE {page_number + 1} ==========")

    text = page.extract_text()

    if text:
        print(text[:3000])
    else:
        print("No text extracted from this page.")


========== PAGE 1 ==========
OUTPUT BASED ANNUAL BUDGET Page 1
HEA 920 KABWE MUNICIPAL COUNCIL
D 5
1.0 MANDATE
To provide operational and service excellence, innovation, community engagement and observance of
good financial management and accountability. This is in agreement with the Republican Constitution
(Amendment) Act No.2 of 2016 Part IX on the System of Devolved Governance [Article 147 (2)] and Part
XI on the System of Local Government.
2.0 STRATEGY
Kabwe Municipal Council will focus on delivering value and quality of life through good governance and
team work, involving all stakeholders including community representatives through operationalisation
of the Ward Development Committees. Further, in response to the high urbanisation rate, the Local
Authority opened up new areas for development in 2024
3.0 NATIONAL DEVELOPMENT PLAN FRAMEWORK
Cluster : 01 Economic Transformation and Job Creation
Cluster Outcome 01 An Industrialised and Diversified Economy
Strategy : 01 Improve agric

In [ ]:
all_tables = []

for page_number, page in enumerate(pdf.pages, start=1):

    tables = page.extract_tables()

    print(f"Page {page_number}: {len(tables)} table(s) found")

    for table in tables:
        if table:
            all_tables.append({
                "page": page_number,
                "table": table
            })

print("\nTotal tables found:", len(all_tables))

Page 1: 7 table(s) found
Page 2: 1 table(s) found
Page 3: 1 table(s) found
Page 4: 1 table(s) found
Page 5: 3 table(s) found
Page 6: 0 table(s) found
Page 7: 1 table(s) found
Page 8: 1 table(s) found
Page 9: 1 table(s) found
Page 10: 0 table(s) found
Page 11: 1 table(s) found
Page 12: 2 table(s) found
Page 13: 1 table(s) found
Page 14: 2 table(s) found
Page 15: 2 table(s) found
Page 16: 2 table(s) found
Page 17: 2 table(s) found
Page 18: 3 table(s) found
Page 19: 1 table(s) found
Page 20: 1 table(s) found
Page 21: 2 table(s) found
Page 22: 1 table(s) found
Page 23: 1 table(s) found
Page 24: 1 table(s) found
Page 25: 2 table(s) found
Page 26: 1 table(s) found
Page 27: 2 table(s) found
Page 28: 1 table(s) found
Page 29: 2 table(s) found
Page 30: 1 table(s) found
Page 31: 1 table(s) found
Page 32: 1 table(s) found
Page 33: 1 table(s) found
Page 34: 2 table(s) found
Page 35: 1 table(s) found
Page 36: 2 table(s) found
Page 37: 2 table(s) found
Page 38: 2 table(s) found
Page 39: 1 table(s) f

In [ ]:
for item in all_tables[:5]:

    print("\n" + "=" * 80)
    print("PAGE:", item["page"])

    for row in item["table"][:10]:
        print(row)


PAGE: 1
['Improve agricultural production and productivity']
['Promote value addition and manufacturing']
['Improve transport and logistics']
['Enhance the management of petroleum products']

PAGE: 1
['Promote local and diaspora participation in the economy']
['Promote Enterprise development']
['Promote Financial Inclusion']

PAGE: 1
['Enhance access to quality, equitable and inclusive education']
['Increased access to higher education']

PAGE: 1
['Strengthen Public health']
['Increase access to quality health care']

PAGE: 1
['Enhance welfare and livelihoods of poor and vulnerable people']
['Reduce vulnerability associated with HIV and AIDS']


In [ ]:
page2 = pdf.pages[1]

table_settings = {
    "vertical_strategy": "text",
    "horizontal_strategy": "text",
    "intersection_tolerance": 5
}

table = page2.extract_table(table_settings)

if table:
    for row in table:
        print(row)
else:
    print("No table detected.")

['Pag', 'e 2', 'OUTPUT BASED ANNUAL B', 'UDGET', '', '']
['', '', '', '', '', '']
['HE', 'A 920', 'KABWE MUNICIPAL COUNCIL', '', '', '']
['D', '5', '', '', '', '']
['', '', '', '', '', '']
['', '', '', '', '', '']
['', 'CODE', 'REVENUE DESCRIPTION', 'APPROVED', 'REVISED', 'BUDGET']
['', '', 'B', 'UDGET 2025\nB', 'UDGET 2026 E', 'STIMATE 2']
['', '', '', '', '', '']
['', '', '', '', '', '']
['', '01', 'Local taxes/rates', '', '', '']
['', '', '', '', '', '']
['', '001', 'Residential', '4,453,818', '4,453,818', '4,453,818']
['', '', '', '', '', '']
['', '002', 'Commercial', '5,536,418', '5,536,418', '5,536,418']
['', '', '', '', '', '']
['', '003', 'Industrial', '2,635,280', '2,635,280', '2,635,280']
['', '', '', '', '', '']
['', '004', 'Hospitality', '572,837', '630,121', '693,133']
['', '', '', '', '', '']
['', '', 'SubItem Total', '13,198,353', '13,255,637', '13,318,649']
['', '001', 'Personal levy', '450,000', '495,000', '220,000']
['', '', '', '', '', '']
['', '', 'SubItem Total', '

In [ ]:
# Pages 2 to 5 contain the main revenue and budget tables
budget_rows = []

for page_number in range(2, 6):
    page = pdf.pages[page_number - 1]

    table = page.extract_table({
        "vertical_strategy": "text",
        "horizontal_strategy": "text",
        "intersection_tolerance": 5
    })

    if table:
        for row in table:
            budget_rows.append({
                "page": page_number,
                "row": row
            })

print("Total extracted rows:", len(budget_rows))

Total extracted rows: 269


In [ ]:
for item in budget_rows[:40]:
    print(item["page"], item["row"])

2 ['Pag', 'e 2', 'OUTPUT BASED ANNUAL B', 'UDGET', '', '']
2 ['', '', '', '', '', '']
2 ['HE', 'A 920', 'KABWE MUNICIPAL COUNCIL', '', '', '']
2 ['D', '5', '', '', '', '']
2 ['', '', '', '', '', '']
2 ['', '', '', '', '', '']
2 ['', 'CODE', 'REVENUE DESCRIPTION', 'APPROVED', 'REVISED', 'BUDGET']
2 ['', '', 'B', 'UDGET 2025\nB', 'UDGET 2026 E', 'STIMATE 2']
2 ['', '', '', '', '', '']
2 ['', '', '', '', '', '']
2 ['', '01', 'Local taxes/rates', '', '', '']
2 ['', '', '', '', '', '']
2 ['', '001', 'Residential', '4,453,818', '4,453,818', '4,453,818']
2 ['', '', '', '', '', '']
2 ['', '002', 'Commercial', '5,536,418', '5,536,418', '5,536,418']
2 ['', '', '', '', '', '']
2 ['', '003', 'Industrial', '2,635,280', '2,635,280', '2,635,280']
2 ['', '', '', '', '', '']
2 ['', '004', 'Hospitality', '572,837', '630,121', '693,133']
2 ['', '', '', '', '', '']
2 ['', '', 'SubItem Total', '13,198,353', '13,255,637', '13,318,649']
2 ['', '001', 'Personal levy', '450,000', '495,000', '220,000']
2 ['', '

In [ ]:
import re
import pandas as pd

records = []
current_category = None

for item in budget_rows:

    page_number = item["page"]
    row = item["row"]

    if not row or len(row) < 6:
        continue

    code = row[1]
    description = row[2]

    # Clean empty cells
    code = code.strip() if code else ""
    description = description.strip() if description else ""

    # Identify main revenue categories such as:
    # 01 Local taxes/rates
    # 02 Fees and Charges
    # 03 Licenses
    # etc.
    if re.fullmatch(r"\d{2}", code) and description:
        current_category = description
        continue

    # Ignore subtotal rows
    if "SubItem Total" in description:
        continue

    # Identify actual revenue items such as 001 Residential
    if re.fullmatch(r"\d{3}", code):

        values = []

        for value in row[3:6]:
            if value:
                value = value.replace(",", "").strip()

                try:
                    values.append(float(value))
                except ValueError:
                    values.append(None)
            else:
                values.append(None)

        # Only keep rows where we have budget figures
        if len(values) == 3 and any(v is not None for v in values):

            records.append({
                "page": page_number,
                "revenue_category": current_category,
                "revenue_code": code,
                "revenue_description": description,
                "budget_2025": values[0],
                "revised_budget_2026": values[1],
                "estimate_2027": values[2]
            })

df_revenue = pd.DataFrame(records)

df_revenue.head(20)

,page,revenue_category,revenue_code,revenue_description,budget_2025,revised_budget_2026,estimate_2027
0,2,Local taxes/rates,001,Residential,4453818.0,4453818.0,4453818.0
1,2,Local taxes/rates,002,Commercial,5536418.0,5536418.0,5536418.0
2,2,Local taxes/rates,003,Industrial,2635280.0,2635280.0,2635280.0
3,2,Local taxes/rates,004,Hospitality,572837.0,630121.0,693133.0
4,2,Local taxes/rates,001,Personal levy,450000.0,495000.0,220000.0
5,4,Licenses,002,Liquor licence,NaN,366000.0,402600.0
6,4,Licenses,003,Firearm and ammuniti,NaN,26000.0,26600.0
7,4,Licenses,004,Petroleum Storage lice,NaN,400000.0,440000.0
8,4,Licenses,005,Dog licence,NaN,40000.0,44000.0
9,4,Levies,001,Livestock Movement le,NaN,32400.0,32400.0


In [ ]:
print("Number of revenue records:", len(df_revenue))

df_revenue

Number of revenue records: 34


,page,revenue_category,revenue_code,revenue_description,budget_2025,revised_budget_2026,estimate_2027
0,2,Local taxes/rates,001,Residential,4453818.0,4453818.0,4453818.0
1,2,Local taxes/rates,002,Commercial,5536418.0,5536418.0,5536418.0
2,2,Local taxes/rates,003,Industrial,2635280.0,2635280.0,2635280.0
3,2,Local taxes/rates,004,Hospitality,572837.0,630121.0,693133.0
4,2,Local taxes/rates,001,Personal levy,450000.0,495000.0,220000.0
5,4,Licenses,002,Liquor licence,NaN,366000.0,402600.0
6,4,Licenses,003,Firearm and ammuniti,NaN,26000.0,26600.0
7,4,Licenses,004,Petroleum Storage lice,NaN,400000.0,440000.0
8,4,Licenses,005,Dog licence,NaN,40000.0,44000.0
9,4,Levies,001,Livestock Movement le,NaN,32400.0,32400.0


In [ ]:
df_revenue["revenue_category"].unique()

<StringArray>
['Local taxes/rates', 'Licenses', 'Levies', 'Permits', 'Charges']
Length: 5, dtype: str

In [ ]:
local_revenue_categories = [
    "Local taxes/rates",
    "Fees and Charges",
    "Licenses",
    "Levies",
    "Permits",
    "Charges",
    "Other Incomes"
]

local_revenue_df = df_revenue[
    df_revenue["revenue_category"].isin(local_revenue_categories)
].copy()

print("Local revenue records:", len(local_revenue_df))

local_revenue_df.head(20)

Local revenue records: 34


,page,revenue_category,revenue_code,revenue_description,budget_2025,revised_budget_2026,estimate_2027
0,2,Local taxes/rates,001,Residential,4453818.0,4453818.0,4453818.0
1,2,Local taxes/rates,002,Commercial,5536418.0,5536418.0,5536418.0
2,2,Local taxes/rates,003,Industrial,2635280.0,2635280.0,2635280.0
3,2,Local taxes/rates,004,Hospitality,572837.0,630121.0,693133.0
4,2,Local taxes/rates,001,Personal levy,450000.0,495000.0,220000.0
5,4,Licenses,002,Liquor licence,NaN,366000.0,402600.0
6,4,Licenses,003,Firearm and ammuniti,NaN,26000.0,26600.0
7,4,Licenses,004,Petroleum Storage lice,NaN,400000.0,440000.0
8,4,Licenses,005,Dog licence,NaN,40000.0,44000.0
9,4,Levies,001,Livestock Movement le,NaN,32400.0,32400.0


In [ ]:
local_revenue_file = "db-unza26-csc4792-kabwe-local-revenue-streams.csv"

local_revenue_df.to_csv(
    local_revenue_file,
    sep="|",
    index=False
)

print("Saved:", local_revenue_file)

Saved: db-unza26-csc4792-kabwe-local-revenue-streams.csv


In [ ]:
from google.colab import files

files.download(local_revenue_file)

ModuleNotFoundError: No module named 'google'

In [ ]:
import os

print(os.path.abspath(local_revenue_file))
print("File exists:", os.path.exists(local_revenue_file))

In [ ]:
print(local_revenue_df.shape)
display(local_revenue_df.head(10))

In [ ]:
test_df = pd.read_csv(
    local_revenue_file,
    sep="|"
)

display(test_df.head())

In [ ]:
# Check the extracted budget data
display(df_revenue.head())
print("Rows:", len(df_revenue))
print("Columns:", df_revenue.columns.tolist())

In [ ]:
for page_number in range(1, len(pdf.pages) + 1):
    page = pdf.pages[page_number - 1]
    text = page.extract_text() or ""
    
    print(f"\n========== PAGE {page_number} ==========")
    print(text[:2000])

In [ ]:
# Extract the Budget Allocation by Economic Classification section

page5 = pdf.pages[4]

text = page5.extract_text()

print(text[2500:5000])

In [ ]:
# Display all text extracted from page 5
page5 = pdf.pages[4]

text = page5.extract_text()

print(text)

In [ ]:
import pandas as pd

approved_budget_data = [
    {
        "council": "Kabwe Municipal Council",
        "budget_year": 2025,
        "economic_classification": "Personal Emoluments",
        "budget_amount": 48701693
    },
    {
        "council": "Kabwe Municipal Council",
        "budget_year": 2025,
        "economic_classification": "Goods and Services",
        "budget_amount": 33797534
    },
    {
        "council": "Kabwe Municipal Council",
        "budget_year": 2025,
        "economic_classification": "Grants and Other Payments (Transfers)",
        "budget_amount": 34845939
    },
    {
        "council": "Kabwe Municipal Council",
        "budget_year": 2025,
        "economic_classification": "Non-Financial Assets",
        "budget_amount": 52204514
    },
    {
        "council": "Kabwe Municipal Council",
        "budget_year": 2025,
        "economic_classification": "Financial Assets",
        "budget_amount": 7473871
    },
    {
        "council": "Kabwe Municipal Council",
        "budget_year": 2025,
        "economic_classification": "Current Liabilities",
        "budget_amount": 4056000
    },
    {
        "council": "Kabwe Municipal Council",
        "budget_year": 2025,
        "economic_classification": "Head Total",
        "budget_amount": 181079551
    }
]

df_approved_budget = pd.DataFrame(approved_budget_data)

display(df_approved_budget)

In [ ]:
approved_budget_file = "db-unza26-csc4792-kabwe-approved-budgets.csv"

df_approved_budget.to_csv(
    approved_budget_file,
    sep="|",
    index=False
)

print("Saved:", approved_budget_file)

In [ ]:
import os

print("File exists:", os.path.exists(approved_budget_file))
print("File path:", os.path.abspath(approved_budget_file))

In [ ]:
lgef_url = "https://www.kabwecouncil.gov.zm/wp-content/uploads/2025/08/2025-BI-ANNUAL-FINANCIAL-STATEMENT-KABWE-M-COUNCIL-1-1.pdf"

lgef_response = requests.get(
    lgef_url,
    timeout=60,
    verify=False
)

print("Status code:", lgef_response.status_code)
print("File size:", len(lgef_response.content), "bytes")

In [ ]:
lgef_pdf = pdfplumber.open(BytesIO(lgef_response.content))

print("Number of pages:", len(lgef_pdf.pages))

In [ ]:
# Search the report for LGEF-related information

matches = []

for page_number, page in enumerate(lgef_pdf.pages, start=1):
    text = page.extract_text() or ""
    
    if (
        "LGEF" in text.upper()
        or "LOCAL GOVERNMENT EQUALISATION FUND" in text.upper()
        or "EQUALISATION" in text.upper()
    ):
        matches.append({
            "page": page_number,
            "text": text
        })

print("Pages containing LGEF information:", [m["page"] for m in matches])

In [ ]:
for m in matches:
    print("\n" + "=" * 80)
    print("PAGE:", m["page"])
    print(m["text"][:4000])

In [ ]:
# Check the first 10 pages only
for page_number in range(1, 11):
    text = lgef_pdf.pages[page_number - 1].extract_text() or ""
    
    if "LGEF" in text.upper() or "EQUALISATION" in text.upper():
        print("\n========== PAGE", page_number, "==========")
        print(text[:2000])

In [ ]:
for page_number in range(1, 37):
    print(page_number, end=" ")
    text = lgef_pdf.pages[page_number - 1].extract_text()
    print("OK" if text else "NO TEXT")

In [ ]:
for page_number in range(1, 6):
    page = lgef_pdf.pages[page_number - 1]
    image = page.to_image(resolution=100)
    image.save(f"lgef_page_{page_number}.png")
    print(f"Saved page {page_number}")

In [ ]:
from PIL import Image, ImageOps, ImageDraw

images = []

for page_number in range(1, 6):
    img = Image.open(f"lgef_page_{page_number}.png").convert("RGB")
    img.thumbnail((500, 700))
    images.append(img)

canvas = Image.new("RGB", (1000, 1400), "white")

positions = [
    (0, 0), (500, 0),
    (0, 700), (500, 700),
    (250, 350)
]

for img, pos in zip(images, positions):
    canvas.paste(img, pos)

canvas.save("lgef_pages_1_to_5.png")

print("Saved: lgef_pages_1_to_5.png")

In [ ]:
from IPython.display import display
from PIL import Image

img = Image.open("lgef_pages_1_to_5.png")
display(img)

In [ ]:
from IPython.display import display
from PIL import Image

display(Image.open("lgef_page_1_large.png"))

In [ ]:
from PIL import Image
from IPython.display import display

# Create the large image for page 1
page = lgef_pdf.pages[0]
image = page.to_image(resolution=150)
image.save("lgef_page_1_large.png")

# Display it
display(Image.open("lgef_page_1_large.png"))

In [ ]:
from PIL import Image
from IPython.display import display

# Display the first 5 pages
for i in range(5):
    page = lgef_pdf.pages[i]
    image = page.to_image(resolution=150)

    # Save each page image
    image.save(f"lgef_page_{i+1}_large.png")

    # Display the page
    print(f"--- Page {i+1} ---")
    display(image)

In [ ]:
# Extract text from all pages
all_text = ""

for i, page in enumerate(lgef_pdf.pages):
    text = page.extract_text()
    all_text += f"\n\n--- PAGE {i+1} ---\n\n"
    all_text += text

print(all_text[:10000])

In [ ]:
import pytesseract

print("Tesseract version:", pytesseract.get_tesseract_version())

In [ ]:
%pip install pytesseract

In [ ]:
import pytesseract

print("pytesseract imported successfully")
print("Tesseract version:", pytesseract.get_tesseract_version())

In [ ]:
import pytesseract

# Tell Python where Tesseract is installed
pytesseract.pytesseract.tesseract_cmd = (
    r"C:\Program Files\Tesseract-OCR\tesseract.exe"
)

print("Tesseract version:", pytesseract.get_tesseract_version())

In [ ]:
from PIL import Image
import pytesseract

# Load the saved Page 1 image
page_image = Image.open("lgef_page_1_large.png")

# Extract text using OCR
page1_text = pytesseract.image_to_string(page_image)

print(page1_text)

In [ ]:
all_text = ""

for i in range(36):
    image = Image.open(f"lgef_page_{i+1}_large.png")
    text = pytesseract.image_to_string(image)

    all_text += f"\n\n--- PAGE {i+1} ---\n\n"
    all_text += text

with open("lgef_financial_statements_ocr.txt", "w", encoding="utf-8") as f:
    f.write(all_text)

print("OCR extraction completed and saved.")

In [ ]:
from PIL import Image
from IPython.display import display

# Create and save images for all 36 pages
for i, page in enumerate(lgef_pdf.pages):
    image = page.to_image(resolution=150)
    image.save(f"lgef_page_{i+1}_large.png")

print("All 36 page images saved successfully.")

In [ ]:
import lighteval as lgef

In [ ]:
lgef_pdf = ...

In [ ]:
import pytesseract
from PIL import Image

# Make sure Python knows where Tesseract is installed
pytesseract.pytesseract.tesseract_cmd = (
    r"C:\Program Files\Tesseract-OCR\tesseract.exe"
)

all_text = ""

for i in range(36):
    image = Image.open(f"lgef_page_{i+1}_large.png")
    text = pytesseract.image_to_string(image)

    all_text += f"\n\n--- PAGE {i+1} ---\n\n"
    all_text += text

print("OCR extraction completed.")
print(all_text[:10000])

In [ ]:
from PIL import Image
import pytesseract

# Tell Python where Tesseract is installed
pytesseract.pytesseract.tesseract_cmd = (
    r"C:\Program Files\Tesseract-OCR\tesseract.exe"
)

all_text = ""

# Process all 36 pages directly from the PDF
for i, page in enumerate(lgef_pdf.pages):
    print(f"Processing page {i+1} of {len(lgef_pdf.pages)}...")

    # Render the page as an image
    image = page.to_image(resolution=150)

    # Convert to a PIL Image if necessary
    pil_image = image.original if hasattr(image, "original") else image

    # Run OCR
    text = pytesseract.image_to_string(pil_image)

    all_text += f"\n\n--- PAGE {i+1} ---\n\n"
    all_text += text

print("OCR extraction completed!")
print(all_text[:10000])

In [ ]:
import pdfplumber

# Open the Kabwe Municipal Council financial statements PDF
pdf_path = "lgef_financial_statements_2025.pdf"

lgef_pdf = pdfplumber.open(pdf_path)

print("PDF opened successfully!")
print("Number of pages:", len(lgef_pdf.pages))

In [ ]:
import requests

url = "https://www.kabwecouncil.gov.zm/wp-content/uploads/2025/08/2025-BI-ANNUAL-FINANCIAL-STATEMENT-KABWE-M-COUNCIL-1-1.pdf"

response = requests.get(url)

print("Status code:", response.status_code)
print("File size:", len(response.content), "bytes")

with open("lgef_financial_statements_2025.pdf", "wb") as f:
    f.write(response.content)

print("PDF downloaded successfully!")

In [ ]:
import tkinter as tk
from tkinter import filedialog
import pdfplumber

# Open a file picker
root = tk.Tk()
root.withdraw()

pdf_path = filedialog.askopenfilename(
    title="Select the Kabwe Municipal Council PDF",
    filetypes=[("PDF files", "*.pdf")]
)

root.destroy()

print("Selected file:", pdf_path)

# Open the selected PDF
lgef_pdf = pdfplumber.open(pdf_path)

print("PDF opened successfully!")
print("Number of pages:", len(lgef_pdf.pages))

In [ ]:
import pytesseract

pytesseract.pytesseract.tesseract_cmd = (
    r"C:\Program Files\Tesseract-OCR\tesseract.exe"
)

page = lgef_pdf.pages[0]

# Render Page 1
image = page.to_image(resolution=150).original

# Extract text
page1_text = pytesseract.image_to_string(image)

print(page1_text)

In [ ]:
all_text = ""

for i, page in enumerate(lgef_pdf.pages):
    print(f"Processing page {i+1} of {len(lgef_pdf.pages)}...")

    image = page.to_image(resolution=150).original
    text = pytesseract.image_to_string(image)

    all_text += f"\n\n--- PAGE {i+1} ---\n\n"
    all_text += text

print("OCR extraction completed!")

In [ ]:
import requests
import pdfplumber
import pytesseract
from pathlib import Path

# Official Kabwe Municipal Council PDF
url = "https://www.kabwecouncil.gov.zm/wp-content/uploads/2025/08/2025-BI-ANNUAL-FINANCIAL-STATEMENT-KABWE-M-COUNCIL-1-1.pdf"

# Save the PDF in the current Jupyter folder
pdf_path = Path.cwd() / "kabwe_financial_statements_2025.pdf"

# Download the PDF if it is not already saved
if not pdf_path.exists():
    print("Downloading PDF...")
    response = requests.get(url, timeout=60)
    response.raise_for_status()
    pdf_path.write_bytes(response.content)
    print("PDF downloaded successfully.")

# Open the PDF
lgef_pdf = pdfplumber.open(str(pdf_path))

print("PDF opened successfully!")
print("PDF location:", pdf_path)
print("Number of pages:", len(lgef_pdf.pages))

# Set the Tesseract OCR path
pytesseract.pytesseract.tesseract_cmd = (
    r"C:\Program Files\Tesseract-OCR\tesseract.exe"
)

# Convert the first page into an image
page = lgef_pdf.pages[0]
page_image = page.to_image(resolution=150)
pil_image = page_image.original

# Extract text using OCR
page1_text = pytesseract.image_to_string(pil_image)

print("\n========== PAGE 1 OCR TEXT ==========\n")
print(page1_text)

In [ ]:
print("Jupyter is working")


In [ ]:
print("Starting PDF setup...")

In [ ]:
import requests

url = "https://www.kabwecouncil.gov.zm/wp-content/uploads/2025/08/2025-BI-ANNUAL-FINANCIAL-STATEMENT-KABWE-M-COUNCIL-1-1.pdf"

response = requests.get(url, timeout=60)

print("Status code:", response.status_code)
print("Downloaded bytes:", len(response.content))

In [ ]:
pdfplumber.open("lgef_financial_statements_2025.pdf")

In [ ]:
import requests
import pdfplumber
from io import BytesIO

financial_url = "https://www.kabwecouncil.gov.zm/wp-content/uploads/2025/08/2025-BI-ANNUAL-FINANCIAL-STATEMENT-KABWE-M-COUNCIL-1-1.pdf"

financial_response = requests.get(
    financial_url,
    timeout=60,
    verify=False
)

print("Status code:", financial_response.status_code)
print("File size:", len(financial_response.content), "bytes")

financial_pdf = pdfplumber.open(
    BytesIO(financial_response.content)
)

print("Number of pages:", len(financial_pdf.pages))

In [ ]:
financial_text = ""

for page_number, page in enumerate(financial_pdf.pages, start=1):
    page_text = page.extract_text() or ""
    financial_text += f"\n\n--- Page {page_number} ---\n{page_text}"

print(financial_text[:5000])

In [ ]:
from pathlib import Path

pdf_path = Path("financial_statements_2025.pdf")

pdf_path.write_bytes(financial_response.content)

print("Saved to:", pdf_path.resolve())

financial_pdf = pdfplumber.open(pdf_path)

print("Number of pages:", len(financial_pdf.pages))

In [ ]:
pdfplumber.open("financial_statements_2025.pdf")

In [ ]:
print("Number of pages:", len(financial_pdf.pages))

In [ ]:
financial_text = []

for page_number, page in enumerate(financial_pdf.pages, start=1):
    text = page.extract_text() or ""

    financial_text.append({
        "page": page_number,
        "text": text
    })

print("Pages extracted:", len(financial_text))
print(financial_text[0]["text"][:3000])

In [ ]:
financial_text_df = pd.DataFrame(financial_text)

financial_text_df.head()

In [ ]:
page = fininacial_pdf.pages[0]
text=page.extract_text() or""
print(text[:3000])

In [ ]:
page = financial_pdf.pages[0]
text = page.extract_text() or ""

print(text[:3000])

In [ ]:
import requests
import pdfplumber
import pandas as pd
from io import BytesIO

# PDF URL
financial_url = "https://www.kabwecouncil.gov.zm/wp-content/uploads/2025/08/2025-BI-ANNUAL-FINANCIAL-STATEMENT-KABWE-M-COUNCIL-1-1.pdf"

# Download the PDF
response = requests.get(
    financial_url,
    timeout=60,
    verify=False
)

print("Status code:", response.status_code)
print("File size:", len(response.content), "bytes")

# Open the downloaded PDF
financial_pdf = pdfplumber.open(
    BytesIO(response.content)
)

print("Number of pages:", len(financial_pdf.pages))

# Extract text from the first page
first_page = financial_pdf.pages[0]
first_page_text = first_page.extract_text() or ""

print(first_page_text[:3000])

In [ ]:
!pip install requests pandas pdfplumber

In [ ]:
import requests
import pandas as pd
import pdfplumber
from io import BytesIO

In [ ]:
pdf_url = "https://www.kabwecouncil.gov.zm/wp-content/uploads/2025/08/2025-BI-ANNUAL-FINANCIAL-STATEMENT-KABWE-M-COUNCIL-1-1.pdf"

download_result = requests.get(
    pdf_url,
    timeout=60,
    verify=False
)

print("Status code:", download_result.status_code)
print("File size:", len(download_result.content), "bytes")

C:\Users\Mutofwe\AppData\Local\Programs\Python\Python314\Lib\site-packages\urllib3\connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host 'www.kabwecouncil.gov.zm'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


Status code: 200
File size: 11368536 bytes


In [ ]:
import urllib3

urllib3.disable_warnings(
    urllib3.exceptions.InsecureRequestWarning
)

In [ ]:
pdf_url = "https://www.kabwecouncil.gov.zm/wp-content/uploads/2025/08/2025-BI-ANNUAL-FINANCIAL-STATEMENT-KABWE-M-COUNCIL-1-1.pdf"

download_result = requests.get(
    pdf_url,
    timeout=60,
    verify=False
)

print("Status code:", download_result.status_code)
print("File size:", len(download_result.content), "bytes")

ChunkedEncodingError: ("Connection broken: ConnectionResetError(10054, 'An existing connection was forcibly closed by the remote host', None, 10054, None)", ConnectionResetError(10054, 'An existing connection was forcibly closed by the remote host', None, 10054, None))

In [ ]:
import os

print(os.getcwd())

C:\Users\Mutofwe\group6_administration\notebooks


In [ ]:
import pdfplumber

opened_document = pdfplumber.open(
    r"C:\Users\Mutofwe\group6_administration\notebooks\financial_statements_2025.pdf.pdf"
)

print("Number of pages:", len(opened_document.pages))

Number of pages: 36


In [ ]:
all_pages_text = ""

for page_number, page in enumerate(opened_document.pages, start=1):
    page_text = page.extract_text()

    if page_text:
        all_pages_text += f"\n\n===== PAGE {page_number} =====\n\n"
        all_pages_text += page_text

print("Total characters extracted:", len(all_pages_text))

Total characters extracted: 0


In [ ]:
!pip install pytesseract pillow pdf2image

In [ ]:
import pytesseract
from PIL import Image
from pdf2image import convert_from_path

In [ ]:
pytesseract.pytesseract.tesseract_cmd = (
    r"C:\Program Files\Tesseract-OCR\tesseract.exe"
)

In [ ]:
pdf_file_path = (
    r"C:\Users\Mutofwe\group6_administration\notebooks"
    r"\financial_statements_2025.pdf.pdf"
)

pdf_images = convert_from_path(
    pdf_file_path,
    dpi=200
)

print("Number of converted pages:", len(pdf_images))

PDFInfoNotInstalledError: Unable to get page count. Is poppler installed and in PATH?

In [ ]:
from pdf2image import convert_from_path

pdf_file_path = (
    r"C:\Users\Mutofwe\group6_administration\notebooks"
    r"\financial_statements_2025.pdf.pdf"
)

pdf_images = convert_from_path(
    pdf_file_path,
    dpi=200
)

print("Number of converted pages:", len(pdf_images))

Number of converted pages: 36


In [ ]:
import pytesseract

pytesseract.pytesseract.tesseract_cmd = (
    r"C:\Program Files\Tesseract-OCR\tesseract.exe"
)

print(pytesseract.get_tesseract_version())

5.5.3.20260724


In [ ]:
first_page_ocr = pytesseract.image_to_string(pdf_images[0])

print(first_page_ocr)

KABWE MUNICIPAL COUNCIL.
BI-ANNUAL FINANCLAL STATEMENTS FOR THE YEAR 2025

P O Box 80424
Civic Centre

KABWE




In [ ]:
all_pages_text = ""

for page_number, page_image in enumerate(pdf_images, start=1):
    print(f"Processing page {page_number} of {len(pdf_images)}...")

    page_text = pytesseract.image_to_string(page_image)

    all_pages_text += (
        f"\n\n===== PAGE {page_number} =====\n\n"
        + page_text
    )

print("Total characters extracted:", len(all_pages_text))

Processing page 1 of 36...
Processing page 2 of 36...
Processing page 3 of 36...
Processing page 4 of 36...
Processing page 5 of 36...
Processing page 6 of 36...
Processing page 7 of 36...
Processing page 8 of 36...
Processing page 9 of 36...
Processing page 10 of 36...
Processing page 11 of 36...
Processing page 12 of 36...
Processing page 13 of 36...
Processing page 14 of 36...
Processing page 15 of 36...
Processing page 16 of 36...
Processing page 17 of 36...
Processing page 18 of 36...
Processing page 19 of 36...
Processing page 20 of 36...
Processing page 21 of 36...
Processing page 22 of 36...
Processing page 23 of 36...
Processing page 24 of 36...
Processing page 25 of 36...
Processing page 26 of 36...
Processing page 27 of 36...
Processing page 28 of 36...
Processing page 29 of 36...
Processing page 30 of 36...
Processing page 31 of 36...
Processing page 32 of 36...
Processing page 33 of 36...
Processing page 34 of 36...
Processing page 35 of 36...
Processing page 36 of 36...
T

In [ ]:
ocr_text_path = (
    r"C:\Users\Mutofwe\group6_administration\notebooks"
    r"\financial_statements_2025_ocr.txt"
)

with open(ocr_text_path, "w", encoding="utf-8") as text_file:
    text_file.write(all_pages_text)

print("OCR text saved successfully.")

OCR text saved successfully.


In [ ]:
print(all_pages_text[:10000])



===== PAGE 1 =====

KABWE MUNICIPAL COUNCIL.
BI-ANNUAL FINANCLAL STATEMENTS FOR THE YEAR 2025

P O Box 80424
Civic Centre

KABWE



===== PAGE 2 =====

KABWE MUNICIPAL COUNCIL
BI-ANNUAL FINANCIAL STATEMENTS FOR THE YEAR 2025

TABLE OF CONTENTS

Report of the Council

Statement of Responsibilities for Bi-Annual Financial Statements

Independent Auditor’s Report

Statement of Cash Receipts and Payments

Statement of Comparison of Budget and Actual Amounts

Statement of Cash Receipts and Payments for Local Government Equalisation Fund
Statement of Cash Receipts and Payments for Constituency Development Fund
Statement of Cash Receipts and Payments for Sector Grant (Devolved Functions)

Statement of Cash Receipts and Payments for ZDSP Capital Grant

Summary of Significant Accounting Policies

Notes to the Financial Statements

16-19

20- 35


===== PAGE 3 =====

KABWE MUNICIPAL COUNCIL
BI-ANNUAL FINANCIAL STATEMENTS FOR THE YEAR 2025 __

REPORT OF THE COUNCIL

The Council has the pleasure

In [ ]:
print(all_pages_text[10000:20000])

s that are free from material
misstatement, whether due to fraud or error.

Nothing has come to the attention of the Council to indicate that the Kabwe Municipal Council

will not remain a going concern for at least twelve months from the date of this statement.

In the opinion of the Council, proper books of accounts were maintained to support preparation of
Financial Statements that present fairly the financial results of the Municipal Council for the bi-

financial year 2025

Signed on behalf of the Council on...............:ceeeeeeeeeeeneeee eens by;

Name: _ amy Crane — Name: ween AQ ty. MA op nLAQ
Signature \ a Signature........... a | seeeeeceeceesceeereeeeee
Position: Town Clerk Position: Director of Finance


===== PAGE 8 =====

REPUBLIC OF ZAMBIA
OFFICE OF THE AUDITOR GENERAL

INDEPENDENT AUDITOR’s REPORT


===== PAGE 9 =====

REPUBLIC OF ZAMBIA
OFFICE OF THE AUDITOR GENERAL

INDEPENDENT AUDITOR’s REPORT


===== PAGE 10 =====

REPUBLIC OF ZAMBIA
OFFICE OF THE AUDITOR GENERAL


In [ ]:
# Display pages 7 to 36 one at a time
for page_number in range(7, 37):
    start_marker = f"===== PAGE {page_number} ====="
    end_marker = f"===== PAGE {page_number + 1} ====="

    start_index = all_pages_text.find(start_marker)

    if page_number < 36:
        end_index = all_pages_text.find(end_marker)
        page_text = all_pages_text[start_index:end_index]
    else:
        page_text = all_pages_text[start_index:]

    print(page_text)
    print("\n" + "=" * 80 + "\n")

===== PAGE 7 =====

KABWE MUNICIPAL COUNCIL
BI-ANNUAL FINANCIAL STATEMENTS FOR THE YEAR 2025

STATEMENT OF RESPONSIBILITIES FOR BI-ANNUAL FINANCIAL STATEMENTS
The Kabwe Municipal Council is responsible for preparing the bi-financial statements for the year
2025 which are free from material misstatement, whether due to fraud or error, and are prepared,
in all material respects, in accordance with the Cash Basis International Public Sector Accounting
Standard (IPSAS). In preparing the financial statements, the Council selected applicable policies
from Local Authorities Accounting Policies (LAAPs) of October 2019 and then applied them
consistently, making judgment and estimates that were reasonable and prudent.

The Council is also responsible for the maintenance of adequate accounting records and the
preparation and integrity of the annual financial statements and related information. The Auditor-
General has audited the financial statements, and his report is shown on pages 7 to 9.

The

In [ ]:
search_terms = [
    "Cash Receipts",
    "Cash Payments",
    "Local Revenue",
    "Grants",
    "Equalisation Fund",
    "Constituency Development Fund",
    "CDF",
    "Revenue",
    "Expenditure",
    "Budget",
    "Actual",
    "Property, Plant",
    "Employees"
]

for term in search_terms:
    print(f"\n{'=' * 20} {term.upper()} {'=' * 20}")

    found = False

    for page_number in range(1, 37):
        start_marker = f"===== PAGE {page_number} ====="
        end_marker = f"===== PAGE {page_number + 1} ====="

        start_index = all_pages_text.find(start_marker)

        if page_number < 36:
            end_index = all_pages_text.find(end_marker)
            page_text = all_pages_text[start_index:end_index]
        else:
            page_text = all_pages_text[start_index:]

        if term.lower() in page_text.lower():
            print(f"\n--- Page {page_number} ---")
            print(page_text)
            found = True

    if not found:
        print("No matching text found.")


==================== CASH RECEIPTS ====================

--- Page 2 ---
===== PAGE 2 =====

KABWE MUNICIPAL COUNCIL
BI-ANNUAL FINANCIAL STATEMENTS FOR THE YEAR 2025

TABLE OF CONTENTS

Report of the Council

Statement of Responsibilities for Bi-Annual Financial Statements

Independent Auditor’s Report

Statement of Cash Receipts and Payments

Statement of Comparison of Budget and Actual Amounts

Statement of Cash Receipts and Payments for Local Government Equalisation Fund
Statement of Cash Receipts and Payments for Constituency Development Fund
Statement of Cash Receipts and Payments for Sector Grant (Devolved Functions)

Statement of Cash Receipts and Payments for ZDSP Capital Grant

Summary of Significant Accounting Policies

Notes to the Financial Statements

16-19

20- 35




--- Page 5 ---
===== PAGE 5 =====

KABWE MUNICIPAL COUNCIL
BI-ANNUAL FINANCIAL STATEMENTS FOR THE YEAR 2025

REPORT OF THE COUNCIL

The District also has two (2) elected Members of Parliament one for Kabwe C

In [ ]:
import pandas as pd

financial_data = pd.DataFrame(columns=[
    "Financial Year",
    "Revenue Source",
    "Budget Amount",
    "Actual Amount",
    "Variance",
    "Fund Type",
    "Page Number",
    "Source Document"
])

financial_data

,Financial Year,Revenue Source,Budget Amount,Actual Amount,Variance,Fund Type,Page Number,Source Document


In [ ]:
financial_summary = pd.DataFrame([
    {
        "Financial Year": 2025,
        "Revenue Source": "Cash Receipts",
        "Budget Amount": None,
        "Actual Amount": 71746982,
        "Variance": None,
        "Fund Type": "General",
        "Page Number": 5,
        "Source Document": "Kabwe Municipal Council Bi-Annual Financial Statements 2025"
    },
    {
        "Financial Year": 2025,
        "Revenue Source": "Payments",
        "Budget Amount": None,
        "Actual Amount": 92307241,
        "Variance": None,
        "Fund Type": "General",
        "Page Number": 5,
        "Source Document": "Kabwe Municipal Council Bi-Annual Financial Statements 2025"
    },
    {
        "Financial Year": 2025,
        "Revenue Source": "Decrease in Cash and Cash Equivalents",
        "Budget Amount": None,
        "Actual Amount": -20560259,
        "Variance": None,
        "Fund Type": "General",
        "Page Number": 5,
        "Source Document": "Kabwe Municipal Council Bi-Annual Financial Statements 2025"
    },
    {
        "Financial Year": 2025,
        "Revenue Source": "Property, Plant and Equipment Acquired",
        "Budget Amount": None,
        "Actual Amount": 10453133.25,
        "Variance": None,
        "Fund Type": "Capital Assets",
        "Page Number": 6,
        "Source Document": "Kabwe Municipal Council Bi-Annual Financial Statements 2025"
    },
    {
        "Financial Year": 2025,
        "Revenue Source": "Employee Remuneration and Staff Welfare",
        "Budget Amount": None,
        "Actual Amount": 23866763.18,
        "Variance": None,
        "Fund Type": "Personnel",
        "Page Number": 6,
        "Source Document": "Kabwe Municipal Council Bi-Annual Financial Statements 2025"
    }
])

financial_summary

,Financial Year,Revenue Source,Budget Amount,Actual Amount,Variance,Fund Type,Page Number,Source Document
0,2025,Cash Receipts,None,71746982.00,None,General,5,Kabwe Municipal Council Bi-Annual Financial St...
1,2025,Payments,None,92307241.00,None,General,5,Kabwe Municipal Council Bi-Annual Financial St...
2,2025,Decrease in Cash and Cash Equivalents,None,-20560259.00,None,General,5,Kabwe Municipal Council Bi-Annual Financial St...
3,2025,"Property, Plant and Equipment Acquired",None,10453133.25,None,Capital Assets,6,Kabwe Municipal Council Bi-Annual Financial St...
4,2025,Employee Remuneration and Staff Welfare,None,23866763.18,None,Personnel,6,Kabwe Municipal Council Bi-Annual Financial St...


In [ ]:
csv_path = (
    r"C:\Users\Mutofwe\group6_administration\notebooks"
    r"\financial_summary_2025.csv"
)

financial_summary.to_csv(csv_path, index=False)

print("Financial summary saved successfully.")
print(csv_path)

Financial summary saved successfully.
C:\Users\Mutofwe\group6_administration\notebooks\financial_summary_2025.csv


In [ ]:
# Read the saved CSV file
loaded_financial_summary = pd.read_csv(
    r"C:\Users\Mutofwe\group6_administration\notebooks\financial_summary_2025.csv"
)

# Display the data
display(loaded_financial_summary)

,Financial Year,Revenue Source,Budget Amount,Actual Amount,Variance,Fund Type,Page Number,Source Document
0,2025,Cash Receipts,NaN,71746982.00,NaN,General,5,Kabwe Municipal Council Bi-Annual Financial St...
1,2025,Payments,NaN,92307241.00,NaN,General,5,Kabwe Municipal Council Bi-Annual Financial St...
2,2025,Decrease in Cash and Cash Equivalents,NaN,-20560259.00,NaN,General,5,Kabwe Municipal Council Bi-Annual Financial St...
3,2025,"Property, Plant and Equipment Acquired",NaN,10453133.25,NaN,Capital Assets,6,Kabwe Municipal Council Bi-Annual Financial St...
4,2025,Employee Remuneration and Staff Welfare,NaN,23866763.18,NaN,Personnel,6,Kabwe Municipal Council Bi-Annual Financial St...


In [ ]:
print("Number of records:", len(loaded_financial_summary))
print("Number of columns:", len(loaded_financial_summary.columns))
print("\nColumn names:")
print(list(loaded_financial_summary.columns))

Number of records: 5
Number of columns: 8

Column names:
['Financial Year', 'Revenue Source', 'Budget Amount', 'Actual Amount', 'Variance', 'Fund Type', 'Page Number', 'Source Document']


In [ ]:
print(loaded_financial_summary.isnull().sum())

Financial Year     0
Revenue Source     0
Budget Amount      5
Actual Amount      0
Variance           5
Fund Type          0
Page Number        0
Source Document    0
dtype: int64


In [ ]:
amount_columns = [
    "Actual Amount"
]

loaded_financial_summary[amount_columns] = (
    loaded_financial_summary[amount_columns]
    .apply(pd.to_numeric, errors="coerce")
)

display(loaded_financial_summary)

,Financial Year,Revenue Source,Budget Amount,Actual Amount,Variance,Fund Type,Page Number,Source Document
0,2025,Cash Receipts,NaN,71746982.00,NaN,General,5,Kabwe Municipal Council Bi-Annual Financial St...
1,2025,Payments,NaN,92307241.00,NaN,General,5,Kabwe Municipal Council Bi-Annual Financial St...
2,2025,Decrease in Cash and Cash Equivalents,NaN,-20560259.00,NaN,General,5,Kabwe Municipal Council Bi-Annual Financial St...
3,2025,"Property, Plant and Equipment Acquired",NaN,10453133.25,NaN,Capital Assets,6,Kabwe Municipal Council Bi-Annual Financial St...
4,2025,Employee Remuneration and Staff Welfare,NaN,23866763.18,NaN,Personnel,6,Kabwe Municipal Council Bi-Annual Financial St...


In [ ]:
print("df_revenue" in globals())

False


In [ ]:
revenue_records = []

In [ ]:
import pandas as pd
import requests
import pdfplumber
import matplotlib.pyplot as plt
from IPython.display import display

In [ ]:
print(pd.__version__)

3.0.5


In [ ]:
revenue_records = []

In [ ]:
df_revenue = pd.DataFrame(revenue_records)

In [ ]:
print("df_revenue" in globals())
print("Number of records:", len(df_revenue))
display(df_revenue.head())

True
Number of records: 0


""


In [ ]:
print("df_revenue" in globals())
print("Number of records:", len(df_revenue))
display(df_revenue.head())

True
Number of records: 0


""


In [ ]:
print("Number of revenue records:", len(revenue_records))
print(revenue_records)

Number of revenue records: 0
[]


In [ ]:
revenue_records.append({
    "page": page_number,
    "revenue_category": category,
    "revenue_code": code,
    "revenue_description": description,
    "budget_2025": budget_2025,
    "revised_budget_2026": revised_budget_2026,
    "estimate_2027": estimate_2027
})

NameError: name 'page_number' is not defined

In [ ]:
revenue_records.append({
    "page": page_number,
    ...
})

SyntaxError: ':' expected after dictionary key (1780305571.py, line 3)

In [ ]:
df_revenue = pd.DataFrame(revenue_records)

print("Number of records:", len(df_revenue))
display(df_revenue.head())

Number of records: 0


""


In [ ]:
print("pdf" in globals())

False


In [ ]:
import pdfplumber

pdf_path = r"C:\Users\Mutofwe\group6_administration\notebooks\2025-KABWE-M-COUNCIL-OBB.pdf"

pdf = pdfplumber.open(pdf_path)

print("PDF opened successfully")
print("Number of pages:", len(pdf.pages))

FileNotFoundError: [Errno 2] No such file or directory: 'C:\\Users\\Mutofwe\\group6_administration\\notebooks\\2025-KABWE-M-COUNCIL-OBB.pdf'

In [ ]:
import pdfplumber
from io import BytesIO

pdf = pdfplumber.open(BytesIO(budget_response.content))

print("PDF opened successfully")
print("Number of pages:", len(pdf.pages))

NameError: name 'budget_response' is not defined

In [ ]:
import requests
import urllib3

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

budget_url = "https://www.kabwecouncil.gov.zm/wp-content/uploads/2026/04/2026-Approved-OBB-Budget.pdf"

budget_response = requests.get(
    budget_url,
    timeout=60,
    verify=False
)

print("Status code:", budget_response.status_code)
print("File size:", len(budget_response.content), "bytes")

Status code: 200
File size: 1936931 bytes


In [ ]:
import pdfplumber
from io import BytesIO

pdf = pdfplumber.open(BytesIO(budget_response.content))

print("PDF opened successfully")
print("Number of pages:", len(pdf.pages))

PDF opened successfully
Number of pages: 59


In [ ]:
import requests
import urllib3

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

budget_url = "https://www.kabwecouncil.gov.zm/wp-content/uploads/2025/05/2025-KABWE-M-COUNCIL-OBB.pdf"

budget_response = requests.get(
    budget_url,
    timeout=60,
    verify=False
)

print("Status code:", budget_response.status_code)
print("File size:", len(budget_response.content), "bytes")

Status code: 200
File size: 792438 bytes


In [ ]:
budget_url = "https://www.kabwecouncil.gov.zm/wp-content/uploads/2025/05/2025-KABWE-M-COUNCIL-OBB.pdf"

In [ ]:
import pdfplumber
from io import BytesIO

pdf = pdfplumber.open(BytesIO(budget_response.content))

print("PDF opened successfully")
print("Number of pages:", len(pdf.pages))

PDF opened successfully
Number of pages: 54


In [ ]:
page2 = pdf.pages[1]

table = page2.extract_table({
    "vertical_strategy": "text",
    "horizontal_strategy": "text",
    "intersection_tolerance": 5
})

for row in table:
    print(row)

['Pag', 'e 2', 'OUTPUT BASED ANNUAL B', 'UDGET', '', '']
['', '', '', '', '', '']
['HE', 'A 920', 'KABWE MUNICIPAL COUNCIL', '', '', '']
['D', '5', '', '', '', '']
['', '', '', '', '', '']
['', '', '', '', '', '']
['', 'CODE', 'REVENUE DESCRIPTION', 'APPROVED', 'REVISED', 'BUDGET']
['', '', 'B', 'UDGET 2025\nB', 'UDGET 2026 E', 'STIMATE 2']
['', '', '', '', '', '']
['', '', '', '', '', '']
['', '01', 'Local taxes/rates', '', '', '']
['', '', '', '', '', '']
['', '001', 'Residential', '4,453,818', '4,453,818', '4,453,818']
['', '', '', '', '', '']
['', '002', 'Commercial', '5,536,418', '5,536,418', '5,536,418']
['', '', '', '', '', '']
['', '003', 'Industrial', '2,635,280', '2,635,280', '2,635,280']
['', '', '', '', '', '']
['', '004', 'Hospitality', '572,837', '630,121', '693,133']
['', '', '', '', '', '']
['', '', 'SubItem Total', '13,198,353', '13,255,637', '13,318,649']
['', '001', 'Personal levy', '450,000', '495,000', '220,000']
['', '', '', '', '', '']
['', '', 'SubItem Total', '

In [ ]:
import re
import pandas as pd

records = []
current_category = None

for item in budget_rows:
    page_number = item["page"]
    row = item["row"]

    if not row:
        continue

    # Clean cells
    row = [cell.strip() if cell else "" for cell in row]

    # Find category codes such as 01, 02, 03 ... 08
    category_index = None

    for i, cell in enumerate(row):
        if re.fullmatch(r"\d{2}", cell):
            category_index = i
            break

    if category_index is not None:
        # Category name is usually immediately after the 2-digit code
        if category_index + 1 < len(row):
            possible_category = row[category_index + 1].strip()

            if possible_category:
                current_category = possible_category

        continue

    # Find 3-digit revenue codes such as 001, 002, 099
    code_index = None

    for i, cell in enumerate(row):
        if re.fullmatch(r"\d{3}", cell):
            code_index = i
            break

    if code_index is None:
        continue

    # Ignore subtotal rows
    if any("SubItem Total" in cell for cell in row):
        continue

    # Description is normally the cell immediately after the code
    description_parts = []

    for cell in row[code_index + 1:]:
        if cell:
            description_parts.append(cell)

    if not description_parts:
        continue

    # First non-numeric cells form the description
    description = " ".join(description_parts[:2]).strip()

    # Locate numeric budget values
    numeric_values = []

    for cell in row[code_index + 1:]:
        cleaned = cell.replace(",", "").strip()

        if re.fullmatch(r"-?\d+(?:\.\d+)?", cleaned):
            numeric_values.append(float(cleaned))

    # We need three financial values
    if len(numeric_values) >= 3:
        records.append({
            "page": page_number,
            "revenue_category": current_category,
            "revenue_code": row[code_index],
            "revenue_description": description,
            "approved_budget_2025": numeric_values[-3],
            "revised_budget_2026": numeric_values[-2],
            "estimate_2027": numeric_values[-1]
        })

df_revenue = pd.DataFrame(records)

print("Number of records:", len(df_revenue))
display(df_revenue)

NameError: name 'budget_rows' is not defined

In [ ]:
budget_rows = []

for page_number in range(2, 6):
    page = pdf.pages[page_number - 1]

    table = page.extract_table({
        "vertical_strategy": "text",
        "horizontal_strategy": "text",
        "intersection_tolerance": 5
    })

    if table:
        for row in table:
            budget_rows.append({
                "page": page_number,
                "row": row
            })

print("Total extracted rows:", len(budget_rows))

Total extracted rows: 269


In [ ]:
import re
import pandas as pd

records = []
current_category = None

for item in budget_rows:
    page_number = item["page"]
    row = item["row"]

    if not row:
        continue

    row = [cell.strip() if cell else "" for cell in row]

    # Find 2-digit category code: 01, 02, ..., 08
    category_index = None

    for i, cell in enumerate(row):
        if re.fullmatch(r"\d{2}", cell):
            category_index = i
            break

    if category_index is not None:
        if category_index + 1 < len(row):
            category_name = row[category_index + 1].strip()

            if category_name:
                current_category = category_name

        continue

    # Find 3-digit revenue code
    code_index = None

    for i, cell in enumerate(row):
        if re.fullmatch(r"\d{3}", cell):
            code_index = i
            break

    if code_index is None:
        continue

    # Skip subtotal rows
    if any("SubItem Total" in cell for cell in row):
        continue

    # Description
    description_parts = []

    for cell in row[code_index + 1:]:
        if cell:
            cleaned = cell.replace(",", "").strip()

            if not re.fullmatch(r"-?\d+(?:\.\d+)?", cleaned):
                description_parts.append(cell)

    description = " ".join(description_parts).strip()

    # Numeric values
    numeric_values = []

    for cell in row[code_index + 1:]:
        cleaned = cell.replace(",", "").strip()

        if re.fullmatch(r"-?\d+(?:\.\d+)?", cleaned):
            numeric_values.append(float(cleaned))

    if len(numeric_values) >= 3:
        records.append({
            "page": page_number,
            "revenue_category": current_category,
            "revenue_code": row[code_index],
            "revenue_description": description,
            "approved_budget_2025": numeric_values[-3],
            "revised_budget_2026": numeric_values[-2],
            "estimate_2027": numeric_values[-1]
        })

df_revenue = pd.DataFrame(records)

print("Number of records:", len(df_revenue))
display(df_revenue)

Number of records: 80


,page,revenue_category,revenue_code,revenue_description,approved_budget_2025,revised_budget_2026,estimate_2027
0,2,Local taxes/rates,001,Residential,4453818.0,4453818.0,4453818.0
1,2,Local taxes/rates,002,Commercial,5536418.0,5536418.0,5536418.0
2,2,Local taxes/rates,003,Industrial,2635280.0,2635280.0,2635280.0
3,2,Local taxes/rates,004,Hospitality,572837.0,630121.0,693133.0
4,2,Local taxes/rates,001,Personal levy,450000.0,495000.0,220000.0
...,...,...,...,...,...,...,...
75,5,Nationa,002,Roads Gr ant,3.0,200587.0,3200587.0
76,5,Nationa,003,Health Gr ant,3.0,649035.0,3649035.0
77,5,Nationa,004,Local Go vernment Equ alisation Fund,2.0,3857381.0,23857381.0
78,5,Nationa,005,Grants in lieu of Rates,1.0,200000.0,1200000.0


In [ ]:
print(df_revenue["revenue_category"].unique())
print()
print(df_revenue["revenue_category"].value_counts())

<StringArray>
['Local taxes/rates',  'Fees and Charges',          'Licenses',
            'Levies',           'Permits',           'Charges',
          'Other In',           'Nationa']
Length: 8, dtype: str

revenue_category
Fees and Charges     39
Levies               11
Charges               9
Nationa               6
Local taxes/rates     5
Permits               5
Licenses              4
Other In              1
Name: count, dtype: int64


In [ ]:
df_revenue["revenue_category"] = df_revenue["revenue_category"].replace({
    "Other In": "Other Incomes",
    "Nationa": "National Support (Grants)"
})

print(df_revenue["revenue_category"].unique())

<StringArray>
[        'Local taxes/rates',          'Fees and Charges',
                  'Licenses',                    'Levies',
                   'Permits',                   'Charges',
             'Other Incomes', 'National Support (Grants)']
Length: 8, dtype: str


In [ ]:
print("Total records:", len(df_revenue))

print("\nExtracted 2025 Approved Budget total:")
print(df_revenue["approved_budget_2025"].sum())

print("\nOfficial Grand Total:")
print(181079550)

Total records: 80

Extracted 2025 Approved Budget total:
44662836.0

Official Grand Total:
181079550


In [ ]:
df_revenue.groupby("revenue_category")["approved_budget_2025"].sum()

revenue_category
Charges                       9586050.0
Fees and Charges             13038015.0
Levies                        1720900.0
Licenses                       832000.0
Local taxes/rates            13648353.0
National Support (Grants)          17.0
Other Incomes                       1.0
Permits                       5837500.0
Name: approved_budget_2025, dtype: float64

In [ ]:
df_revenue[
    df_revenue["revenue_category"] == "National Support (Grants)"
]

,page,revenue_category,revenue_code,revenue_description,approved_budget_2025,revised_budget_2026,estimate_2027
74,5,National Support (Grants),001,Constitue ncy Develop ment Fund,7.0,2116301.0,72116301.0
75,5,National Support (Grants),002,Roads Gr ant,3.0,200587.0,3200587.0
76,5,National Support (Grants),003,Health Gr ant,3.0,649035.0,3649035.0
77,5,National Support (Grants),004,Local Go vernment Equ alisation Fund,2.0,3857381.0,23857381.0
78,5,National Support (Grants),005,Grants in lieu of Rates,1.0,200000.0,1200000.0
79,5,National Support (Grants),099,Other Gr ants,1.0,527645.0,10797645.0


In [ ]:
# Fix split numbers in National Support (Grants)

grant_mask = df_revenue["revenue_category"] == "National Support (Grants)"

df_revenue.loc[grant_mask, "approved_budget_2025"] = [
    72116301,
    3200587,
    3649035,
    23857381,
    1200000,
    29494551
]

print(df_revenue[grant_mask])

    page           revenue_category revenue_code  \
74     5  National Support (Grants)          001   
75     5  National Support (Grants)          002   
76     5  National Support (Grants)          003   
77     5  National Support (Grants)          004   
78     5  National Support (Grants)          005   
79     5  National Support (Grants)          099   

                     revenue_description  approved_budget_2025  \
74       Constitue ncy Develop ment Fund            72116301.0   
75                          Roads Gr ant             3200587.0   
76                         Health Gr ant             3649035.0   
77  Local Go vernment Equ alisation Fund            23857381.0   
78               Grants in lieu of Rates             1200000.0   
79                         Other Gr ants            29494551.0   

    revised_budget_2026  estimate_2027  
74            2116301.0     72116301.0  
75             200587.0      3200587.0  
76             649035.0      3649035.0  
77      

In [ ]:
print("Total records:", len(df_revenue))

print("\nCorrected 2025 Approved Budget total:")
print(df_revenue["approved_budget_2025"].sum())

print("\nOfficial Grand Total:")
print(181079550)

Total records: 80

Corrected 2025 Approved Budget total:
178180674.0

Official Grand Total:
181079550


In [ ]:
other_income_mask = df_revenue["revenue_category"] == "Other Incomes"

df_revenue.loc[other_income_mask, "approved_budget_2025"] = 2898877

print(df_revenue[other_income_mask])

    page revenue_category revenue_code revenue_description  \
73     5    Other Incomes          099       Other Inc ome   

    approved_budget_2025  revised_budget_2026  estimate_2027  
73             2898877.0             500000.0      1500000.0  


In [ ]:
print("Corrected total:")
print(df_revenue["approved_budget_2025"].sum())

print("Official total:")
print(181079550)

print("Difference:")
print(181079550 - df_revenue["approved_budget_2025"].sum())

Corrected total:
181079550.0
Official total:
181079550
Difference:
0.0


In [ ]:
category_totals = (
    df_revenue
    .groupby("revenue_category")["approved_budget_2025"]
    .sum()
    .sort_values(ascending=False)
)

display(category_totals)

revenue_category
National Support (Grants)    133517855.0
Local taxes/rates             13648353.0
Fees and Charges              13038015.0
Charges                        9586050.0
Permits                        5837500.0
Other Incomes                  2898877.0
Levies                         1720900.0
Licenses                        832000.0
Name: approved_budget_2025, dtype: float64

In [ ]:
print("Total of category totals:")
print(category_totals.sum())

Total of category totals:
181079550.0


In [ ]:
# Make a clean copy of the validated dataset
final_2025_budget = df_revenue.copy()

# Clean text fields
final_2025_budget["revenue_category"] = (
    final_2025_budget["revenue_category"]
    .astype(str)
    .str.strip()
)

final_2025_budget["revenue_description"] = (
    final_2025_budget["revenue_description"]
    .astype(str)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

# Make sure codes remain 3 digits
final_2025_budget["revenue_code"] = (
    final_2025_budget["revenue_code"]
    .astype(str)
    .str.zfill(3)
)

# Add the budget year
final_2025_budget["financial_year"] = 2025

# Reorder columns
final_2025_budget = final_2025_budget[
    [
        "financial_year",
        "revenue_category",
        "revenue_code",
        "revenue_description",
        "approved_budget_2025",
        "revised_budget_2026",
        "estimate_2027",
        "page"
    ]
]

display(final_2025_budget.head())
print("Number of records:", len(final_2025_budget))

,financial_year,revenue_category,revenue_code,revenue_description,approved_budget_2025,revised_budget_2026,estimate_2027,page
0,2025,Local taxes/rates,001,Residential,4453818.0,4453818.0,4453818.0,2
1,2025,Local taxes/rates,002,Commercial,5536418.0,5536418.0,5536418.0,2
2,2025,Local taxes/rates,003,Industrial,2635280.0,2635280.0,2635280.0,2
3,2025,Local taxes/rates,004,Hospitality,572837.0,630121.0,693133.0,2
4,2025,Local taxes/rates,001,Personal levy,450000.0,495000.0,220000.0,2


Number of records: 80


In [ ]:
print(final_2025_budget.isnull().sum())

financial_year          0
revenue_category        0
revenue_code            0
revenue_description     0
approved_budget_2025    0
revised_budget_2026     0
estimate_2027           0
page                    0
dtype: int64


In [ ]:
print("Duplicate rows:", final_2025_budget.duplicated().sum())

Duplicate rows: 0


In [ ]:
csv_path = r"C:\Users\Mutofwe\group6_administration\notebooks\db-unza26-csc4792-kabwe-2025-approved-obb-budget.csv"

final_2025_budget.to_csv(
    csv_path,
    sep="|",
    index=False
)

print("CSV saved successfully!")
print(csv_path)

CSV saved successfully!
C:\Users\Mutofwe\group6_administration\notebooks\db-unza26-csc4792-kabwe-2025-approved-obb-budget.csv


In [ ]:
check = pd.read_csv(csv_path, sep="|")

print("Rows:", len(check))
print("Columns:", list(check.columns))

display(check.head())

Rows: 80
Columns: ['financial_year', 'revenue_category', 'revenue_code', 'revenue_description', 'approved_budget_2025', 'revised_budget_2026', 'estimate_2027', 'page']


,financial_year,revenue_category,revenue_code,revenue_description,approved_budget_2025,revised_budget_2026,estimate_2027,page
0,2025,Local taxes/rates,1,Residential,4453818.0,4453818.0,4453818.0,2
1,2025,Local taxes/rates,2,Commercial,5536418.0,5536418.0,5536418.0,2
2,2025,Local taxes/rates,3,Industrial,2635280.0,2635280.0,2635280.0,2
3,2025,Local taxes/rates,4,Hospitality,572837.0,630121.0,693133.0,2
4,2025,Local taxes/rates,1,Personal levy,450000.0,495000.0,220000.0,2


In [ ]:
lgef_data = [
    {
        "financial_year": 2025,
        "fund_type": "LGEF",
        "transaction_type": "Funding",
        "description": "1st Funding",
        "amount": 1843986,
        "purpose": "LGEF Funding",
        "page": 25
    },
    {
        "financial_year": 2025,
        "fund_type": "LGEF",
        "transaction_type": "Funding",
        "description": "2nd Funding",
        "amount": 1764901,
        "purpose": "LGEF Funding",
        "page": 25
    },
    {
        "financial_year": 2025,
        "fund_type": "LGEF",
        "transaction_type": "Funding",
        "description": "3rd Funding",
        "amount": 1832551,
        "purpose": "LGEF Funding",
        "page": 25
    },
    {
        "financial_year": 2025,
        "fund_type": "LGEF",
        "transaction_type": "Funding",
        "description": "4th Funding",
        "amount": 1730867,
        "purpose": "LGEF Funding",
        "page": 25
    },
    {
        "financial_year": 2025,
        "fund_type": "LGEF",
        "transaction_type": "Funding",
        "description": "5th Funding",
        "amount": 1430192,
        "purpose": "LGEF Funding",
        "page": 25
    },
    {
        "financial_year": 2025,
        "fund_type": "LGEF",
        "transaction_type": "Operational Expenditure",
        "description": "Personal emoluments, salaries and wages",
        "amount": 8602496,
        "purpose": "Operational expenditure",
        "page": 25
    },
    {
        "financial_year": 2025,
        "fund_type": "LGEF",
        "transaction_type": "Capital Expenditure",
        "description": "Lukanga Bus Station",
        "amount": 6146594,
        "purpose": "Capital expenditure",
        "page": 25
    }
]

lgef_df = pd.DataFrame(lgef_data)

display(lgef_df)

,financial_year,fund_type,transaction_type,description,amount,purpose,page
0,2025,LGEF,Funding,1st Funding,1843986,LGEF Funding,25
1,2025,LGEF,Funding,2nd Funding,1764901,LGEF Funding,25
2,2025,LGEF,Funding,3rd Funding,1832551,LGEF Funding,25
3,2025,LGEF,Funding,4th Funding,1730867,LGEF Funding,25
4,2025,LGEF,Funding,5th Funding,1430192,LGEF Funding,25
5,2025,LGEF,Operational Expenditure,"Personal emoluments, salaries and wages",8602496,Operational expenditure,25
6,2025,LGEF,Capital Expenditure,Lukanga Bus Station,6146594,Capital expenditure,25


In [ ]:
# Check for duplicate rows
print("Duplicate rows:", lgef_df.duplicated().sum())

# Check for missing values
print("\nMissing values:")
print(lgef_df.isnull().sum())

# Verify LGEF funding total
funding_total = lgef_df[
    lgef_df["transaction_type"] == "Funding"
]["amount"].sum()

print("\nLGEF funding total:", funding_total)
print("Official funding total:", 8602496)
print("Difference:", 8602496 - funding_total)

Duplicate rows: 0

Missing values:
financial_year      0
fund_type           0
transaction_type    0
description         0
amount              0
purpose             0
page                0
dtype: int64

LGEF funding total: 8602497
Official funding total: 8602496
Difference: -1


In [ ]:
funding_rows = lgef_df[
    lgef_df["transaction_type"] == "Funding"
].copy()

display(funding_rows)

print("\nIndividual funding amounts:")
for _, row in funding_rows.iterrows():
    print(row["description"], "=", row["amount"])

print("\nCalculated total:", funding_rows["amount"].sum())
print("Official total:", 8602496)

,financial_year,fund_type,transaction_type,description,amount,purpose,page
0,2025,LGEF,Funding,1st Funding,1843986,LGEF Funding,25
1,2025,LGEF,Funding,2nd Funding,1764901,LGEF Funding,25
2,2025,LGEF,Funding,3rd Funding,1832551,LGEF Funding,25
3,2025,LGEF,Funding,4th Funding,1730867,LGEF Funding,25
4,2025,LGEF,Funding,5th Funding,1430192,LGEF Funding,25



Individual funding amounts:
1st Funding = 1843986
2nd Funding = 1764901
3rd Funding = 1832551
4th Funding = 1730867
5th Funding = 1430192

Calculated total: 8602497
Official total: 8602496


In [ ]:
print(lgef_df.to_string(index=False))

 financial_year fund_type        transaction_type                             description  amount                 purpose  page
           2025      LGEF                 Funding                             1st Funding 1843986            LGEF Funding    25
           2025      LGEF                 Funding                             2nd Funding 1764901            LGEF Funding    25
           2025      LGEF                 Funding                             3rd Funding 1832551            LGEF Funding    25
           2025      LGEF                 Funding                             4th Funding 1730867            LGEF Funding    25
           2025      LGEF                 Funding                             5th Funding 1430192            LGEF Funding    25
           2025      LGEF Operational Expenditure Personal emoluments, salaries and wages 8602496 Operational expenditure    25
           2025      LGEF     Capital Expenditure                     Lukanga Bus Station 6146594     Ca

In [ ]:
print("Calculated funding total:", funding_total)
print("Official funding total:", 8602496)
print("Difference (official - calculated):", 8602496 - funding_total)

Calculated funding total: 8602497
Official funding total: 8602496
Difference (official - calculated): -1


In [ ]:
csv_path = r"C:\Users\Mutofwe\group6_administration\notebooks\db-unza26-csc4792-kabwe-lgef-utilisation-2025.csv"

lgef_df.to_csv(
    csv_path,
    sep="|",
    index=False
)

print("LGEF dataset saved successfully:")
print(csv_path)

LGEF dataset saved successfully:
C:\Users\Mutofwe\group6_administration\notebooks\db-unza26-csc4792-kabwe-lgef-utilisation-2025.csv


In [ ]:
import os

print("File exists:", os.path.exists(csv_path))
print("File size:", os.path.getsize(csv_path), "bytes")

File exists: True
File size: 542 bytes


In [ ]:
local_revenue_data = [
    {
        "financial_year": 2025,
        "revenue_type": "Local taxes",
        "amount": 6663177,
        "source_page": 11
    },
    {
        "financial_year": 2025,
        "revenue_type": "Fees and Charges",
        "amount": 12062057,
        "source_page": 11
    },
    {
        "financial_year": 2025,
        "revenue_type": "Licences",
        "amount": 347920,
        "source_page": 11
    },
    {
        "financial_year": 2025,
        "revenue_type": "Levies",
        "amount": 1209808,
        "source_page": 11
    },
    {
        "financial_year": 2025,
        "revenue_type": "Permits",
        "amount": 2726912,
        "source_page": 11
    }
]

local_revenue_df = pd.DataFrame(local_revenue_data)

display(local_revenue_df)

,financial_year,revenue_type,amount,source_page
0,2025,Local taxes,6663177,11
1,2025,Fees and Charges,12062057,11
2,2025,Licences,347920,11
3,2025,Levies,1209808,11
4,2025,Permits,2726912,11


In [ ]:
# Check for duplicate rows
print("Duplicate rows:", local_revenue_df.duplicated().sum())

# Check for missing values
print("\nMissing values:")
print(local_revenue_df.isnull().sum())

# Calculate total actual local revenue
local_revenue_total = local_revenue_df["amount"].sum()

print("\nCalculated local revenue total:", local_revenue_total)

Duplicate rows: 0

Missing values:
financial_year    0
revenue_type      0
amount            0
source_page       0
dtype: int64

Calculated local revenue total: 23009874


In [ ]:
csv_path = r"C:\Users\Mutofwe\group6_administration\notebooks\db-unza26-csc4792-kabwe-local-revenue-2025.csv"

local_revenue_df.to_csv(
    csv_path,
    sep="|",
    index=False
)

print("Local revenue dataset saved successfully:")
print(csv_path)

Local revenue dataset saved successfully:
C:\Users\Mutofwe\group6_administration\notebooks\db-unza26-csc4792-kabwe-local-revenue-2025.csv


In [ ]:
import os

print("File exists:", os.path.exists(csv_path))
print("File size:", os.path.getsize(csv_path), "bytes")

File exists: True
File size: 186 bytes


In [ ]:
import os

files_to_check = [
    r"C:\Users\Mutofwe\group6_administration\notebooks\db-unza26-csc4792-kabwe-2025-approved-obb-budget.csv",
    r"C:\Users\Mutofwe\group6_administration\notebooks\db-unza26-csc4792-kabwe-lgef-utilisation-2025.csv",
    r"C:\Users\Mutofwe\group6_administration\notebooks\db-unza26-csc4792-kabwe-local-revenue-2025.csv"
]

for file in files_to_check:
    print(os.path.basename(file), "→", os.path.exists(file))

db-unza26-csc4792-kabwe-2025-approved-obb-budget.csv → True
db-unza26-csc4792-kabwe-lgef-utilisation-2025.csv → True
db-unza26-csc4792-kabwe-local-revenue-2025.csv → True


In [ ]:
import pandas as pd

approved_budget_path = r"C:\Users\Mutofwe\group6_administration\notebooks\db-unza26-csc4792-kabwe-2025-approved-obb-budget.csv"
lgef_path = r"C:\Users\Mutofwe\group6_administration\notebooks\db-unza26-csc4792-kabwe-lgef-utilisation-2025.csv"
local_revenue_path = r"C:\Users\Mutofwe\group6_administration\notebooks\db-unza26-csc4792-kabwe-local-revenue-2025.csv"

approved_budget_check = pd.read_csv(approved_budget_path, sep="|")
lgef_check = pd.read_csv(lgef_path, sep="|")
local_revenue_check = pd.read_csv(local_revenue_path, sep="|")

print("APPROVED BUDGET")
print(approved_budget_check.shape)
print(approved_budget_check.columns.tolist())

print("\nLGEF UTILISATION")
print(lgef_check.shape)
print(lgef_check.columns.tolist())

print("\nLOCAL REVENUE")
print(local_revenue_check.shape)
print(local_revenue_check.columns.tolist())

APPROVED BUDGET
(80, 8)
['financial_year', 'revenue_category', 'revenue_code', 'revenue_description', 'approved_budget_2025', 'revised_budget_2026', 'estimate_2027', 'page']

LGEF UTILISATION
(7, 7)
['financial_year', 'fund_type', 'transaction_type', 'description', 'amount', 'purpose', 'page']

LOCAL REVENUE
(5, 4)
['financial_year', 'revenue_type', 'amount', 'source_page']


In [ ]:
print("===== APPROVED BUDGET =====")
display(approved_budget_check)

print("===== LGEF UTILISATION =====")
display(lgef_check)

print("===== LOCAL REVENUE =====")
display(local_revenue_check)

===== APPROVED BUDGET =====


,financial_year,revenue_category,revenue_code,revenue_description,approved_budget_2025,revised_budget_2026,estimate_2027,page
0,2025,Local taxes/rates,1,Residential,4453818.0,4453818.0,4453818.0,2
1,2025,Local taxes/rates,2,Commercial,5536418.0,5536418.0,5536418.0,2
2,2025,Local taxes/rates,3,Industrial,2635280.0,2635280.0,2635280.0,2
3,2025,Local taxes/rates,4,Hospitality,572837.0,630121.0,693133.0,2
4,2025,Local taxes/rates,1,Personal levy,450000.0,495000.0,220000.0,2
...,...,...,...,...,...,...,...,...
75,2025,National Support (Grants),2,Roads Gr ant,3200587.0,200587.0,3200587.0,5
76,2025,National Support (Grants),3,Health Gr ant,3649035.0,649035.0,3649035.0,5
77,2025,National Support (Grants),4,Local Go vernment Equ alisation Fund,23857381.0,3857381.0,23857381.0,5
78,2025,National Support (Grants),5,Grants in lieu of Rates,1200000.0,200000.0,1200000.0,5


===== LGEF UTILISATION =====


,financial_year,fund_type,transaction_type,description,amount,purpose,page
0,2025,LGEF,Funding,1st Funding,1843986,LGEF Funding,25
1,2025,LGEF,Funding,2nd Funding,1764901,LGEF Funding,25
2,2025,LGEF,Funding,3rd Funding,1832551,LGEF Funding,25
3,2025,LGEF,Funding,4th Funding,1730867,LGEF Funding,25
4,2025,LGEF,Funding,5th Funding,1430192,LGEF Funding,25
5,2025,LGEF,Operational Expenditure,"Personal emoluments, salaries and wages",8602496,Operational expenditure,25
6,2025,LGEF,Capital Expenditure,Lukanga Bus Station,6146594,Capital expenditure,25


===== LOCAL REVENUE =====


,financial_year,revenue_type,amount,source_page
0,2025,Local taxes,6663177,11
1,2025,Fees and Charges,12062057,11
2,2025,Licences,347920,11
3,2025,Levies,1209808,11
4,2025,Permits,2726912,11


In [ ]:
# Search the extracted 2025 financial statement text
with open(
    r"C:\Users\Mutofwe\group6_administration\notebooks\financial_statements_2025_ocr.txt",
    "r",
    encoding="utf-8"
) as f:
    financial_text = f.read()

# Look for individual local revenue streams
keywords = [
    "market fees",
    "local taxes",
    "fees and charges",
    "levies",
    "licences",
    "permits",
    "market"
]

for keyword in keywords:
    print("\n" + "=" * 70)
    print("SEARCH:", keyword)
    print("=" * 70)

    matches = [
        line for line in financial_text.splitlines()
        if keyword.lower() in line.lower()
    ]

    for line in matches[:20]:
        print(line)


SEARCH: market fees
Market Fees

SEARCH: local taxes
Local taxes Zi 6,663,177
Local taxes 6,824,177 0 6,824,177 6,663,177 98 160,999 2
a. Local Taxes
to the Constitution and the Business Regulatory Act of 2014, a system of local taxes
2. Local Taxes

SEARCH: fees and charges
make regulations, imposition of levies, fees and charges and to formulate local policies to promote,
Fees and Charges 3 12,062,057
Fees and Charges 11,187,033 11,187,033 12,062,057 108 - 875,024 - 8
b. Fees and Charges
3. Fees and Charges
The Council generated cash receipts in form of fees and charges arising from offering
Fees and charges 5,156,007 -
a) Fees and Charges
Other Fees and Charges

SEARCH: levies
make regulations, imposition of levies, fees and charges and to formulate local policies to promote,
Levies 5 1,209,808
Levies 860,450 - 860,450 1,209,808 141 (349,358) - 41
which Local Authorities can raise by passing by-laws imposing levies on:
5. Levies
The Council generated cash receipts by charging levie

In [ ]:
detailed_local_revenue = [
    # Local Taxes
    {"financial_year": 2025, "revenue_category": "Local Taxes", "revenue_stream": "Residential Rates", "actual_amount": 1948368, "source_page": 21},
    {"financial_year": 2025, "revenue_category": "Local Taxes", "revenue_stream": "Industrial / Commercial Rates", "actual_amount": 4492657, "source_page": 21},
    {"financial_year": 2025, "revenue_category": "Local Taxes", "revenue_stream": "Personal Levy", "actual_amount": 222153, "source_page": 21},

    # Fees and Charges
    {"financial_year": 2025, "revenue_category": "Fees and Charges", "revenue_stream": "Consent Fees", "actual_amount": 6000, "source_page": 22},
    {"financial_year": 2025, "revenue_category": "Fees and Charges", "revenue_stream": "Survey Fees", "actual_amount": 8000, "source_page": 22},
    {"financial_year": 2025, "revenue_category": "Fees and Charges", "revenue_stream": "Building Inspection Fees", "actual_amount": 249300, "source_page": 22},
    {"financial_year": 2025, "revenue_category": "Fees and Charges", "revenue_stream": "Plan Scrutiny Fees", "actual_amount": 553935, "source_page": 22},
    {"financial_year": 2025, "revenue_category": "Fees and Charges", "revenue_stream": "Change of Premise Use", "actual_amount": 59162, "source_page": 22},
    {"financial_year": 2025, "revenue_category": "Fees and Charges", "revenue_stream": "Container / Ntemba Fees", "actual_amount": 16084, "source_page": 22},
    {"financial_year": 2025, "revenue_category": "Fees and Charges", "revenue_stream": "Rentals / Lease of Council Properties", "actual_amount": 688925, "source_page": 22},
    {"financial_year": 2025, "revenue_category": "Fees and Charges", "revenue_stream": "Application Form Fees", "actual_amount": 501500, "source_page": 22},
    {"financial_year": 2025, "revenue_category": "Fees and Charges", "revenue_stream": "Search Fees", "actual_amount": 2150, "source_page": 22},
    {"financial_year": 2025, "revenue_category": "Fees and Charges", "revenue_stream": "Notice Board Adverts", "actual_amount": 247111, "source_page": 22},
    {"financial_year": 2025, "revenue_category": "Fees and Charges", "revenue_stream": "Market Fees", "actual_amount": 89016, "source_page": 22},
    {"financial_year": 2025, "revenue_category": "Fees and Charges", "revenue_stream": "Parking Fees", "actual_amount": 262033, "source_page": 22},
    {"financial_year": 2025, "revenue_category": "Fees and Charges", "revenue_stream": "Bus Station Fees", "actual_amount": 11770, "source_page": 22},
    {"financial_year": 2025, "revenue_category": "Fees and Charges", "revenue_stream": "Affidavit Fees", "actual_amount": 6000, "source_page": 22},
    {"financial_year": 2025, "revenue_category": "Fees and Charges", "revenue_stream": "Hire of Hall", "actual_amount": 1900, "source_page": 22},
    {"financial_year": 2025, "revenue_category": "Fees and Charges", "revenue_stream": "Hire of Stadia", "actual_amount": 1120, "source_page": 22},
    {"financial_year": 2025, "revenue_category": "Fees and Charges", "revenue_stream": "Body Transfer / Inspection Settlement", "actual_amount": 1800, "source_page": 22},
    {"financial_year": 2025, "revenue_category": "Fees and Charges", "revenue_stream": "Boundary Location", "actual_amount": 349850, "source_page": 22},
    {"financial_year": 2025, "revenue_category": "Fees and Charges", "revenue_stream": "Refuse Disposal Fees", "actual_amount": 1000, "source_page": 22},
    {"financial_year": 2025, "revenue_category": "Fees and Charges", "revenue_stream": "Library Fees", "actual_amount": 24500, "source_page": 22},
    {"financial_year": 2025, "revenue_category": "Fees and Charges", "revenue_stream": "Notice of Marriage", "actual_amount": 100, "source_page": 22},
    {"financial_year": 2025, "revenue_category": "Fees and Charges", "revenue_stream": "Abattoir / Meat Inspection Fees", "actual_amount": 17150, "source_page": 22},
    {"financial_year": 2025, "revenue_category": "Fees and Charges", "revenue_stream": "Registration of Clubs and Societies", "actual_amount": 24146, "source_page": 22},
    {"financial_year": 2025, "revenue_category": "Fees and Charges", "revenue_stream": "Farm Produce", "actual_amount": 344437, "source_page": 22},
    {"financial_year": 2025, "revenue_category": "Fees and Charges", "revenue_stream": "Communication Mast Levy", "actual_amount": 51900, "source_page": 22},
    {"financial_year": 2025, "revenue_category": "Fees and Charges", "revenue_stream": "Illegal Parking Fees", "actual_amount": 123030, "source_page": 22},
    {"financial_year": 2025, "revenue_category": "Fees and Charges", "revenue_stream": "Sale of Parks", "actual_amount": 200, "source_page": 22},
    {"financial_year": 2025, "revenue_category": "Fees and Charges", "revenue_stream": "Billboard and Banner", "actual_amount": 484813, "source_page": 22},
    {"financial_year": 2025, "revenue_category": "Fees and Charges", "revenue_stream": "Lease of Council Transport", "actual_amount": 559244, "source_page": 22},
    {"financial_year": 2025, "revenue_category": "Fees and Charges", "revenue_stream": "Penalties", "actual_amount": 75000, "source_page": 22},
    {"financial_year": 2025, "revenue_category": "Fees and Charges", "revenue_stream": "Ablution Fees", "actual_amount": 1200, "source_page": 22},
    {"financial_year": 2025, "revenue_category": "Fees and Charges", "revenue_stream": "Extracts of Minutes", "actual_amount": 99600, "source_page": 22},
    {"financial_year": 2025, "revenue_category": "Fees and Charges", "revenue_stream": "Recommendation Fees", "actual_amount": 234750, "source_page": 22},
    {"financial_year": 2025, "revenue_category": "Fees and Charges", "revenue_stream": "Medical Fees", "actual_amount": 59280, "source_page": 22},

    # Land Development Charges
    {"financial_year": 2025, "revenue_category": "Land Development Charges", "revenue_stream": "Premium Plots - Residential", "actual_amount": 5350800, "source_page": 23},
    {"financial_year": 2025, "revenue_category": "Land Development Charges", "revenue_stream": "Premium Plots - Commercial", "actual_amount": 1365500, "source_page": 23},
    {"financial_year": 2025, "revenue_category": "Land Development Charges", "revenue_stream": "Other", "actual_amount": 189750, "source_page": 23},

    # Licences
    {"financial_year": 2025, "revenue_category": "Licences", "revenue_stream": "Liquor Licence", "actual_amount": 221180, "source_page": 23},
    {"financial_year": 2025, "revenue_category": "Licences", "revenue_stream": "Firearm and Ammunition", "actual_amount": 15000, "source_page": 23},
    {"financial_year": 2025, "revenue_category": "Licences", "revenue_stream": "Dog Licence", "actual_amount": 6200, "source_page": 23},
    {"financial_year": 2025, "revenue_category": "Licences", "revenue_stream": "Petroleum", "actual_amount": 48370, "source_page": 23},
    {"financial_year": 2025, "revenue_category": "Licences", "revenue_stream": "Occupancy", "actual_amount": 39620, "source_page": 23},
    {"financial_year": 2025, "revenue_category": "Licences", "revenue_stream": "Other Licence", "actual_amount": 17550, "source_page": 23},

    # Levies
    {"financial_year": 2025, "revenue_category": "Levies", "revenue_stream": "Bird Levy", "actual_amount": 54080, "source_page": 24},
    {"financial_year": 2025, "revenue_category": "Levies", "revenue_stream": "Business Levy", "actual_amount": 807848, "source_page": 24},
    {"financial_year": 2025, "revenue_category": "Levies", "revenue_stream": "Pole Levy", "actual_amount": 3442, "source_page": 24},
    {"financial_year": 2025, "revenue_category": "Levies", "revenue_stream": "Telecommunication Mast", "actual_amount": 344437, "source_page": 24},

    # Permits
    {"financial_year": 2025, "revenue_category": "Permits", "revenue_stream": "Health Permit", "actual_amount": 1136180, "source_page": 24},
    {"financial_year": 2025, "revenue_category": "Permits", "revenue_stream": "Burial Permits and Grave Sites", "actual_amount": 150875, "source_page": 24},
    {"financial_year": 2025, "revenue_category": "Permits", "revenue_stream": "Fire Certificate", "actual_amount": 1418357, "source_page": 24},
    {"financial_year": 2025, "revenue_category": "Permits", "revenue_stream": "Extension of Business Hours Permits", "actual_amount": 1250, "source_page": 24},
    {"financial_year": 2025, "revenue_category": "Permits", "revenue_stream": "Public Permits", "actual_amount": 2750, "source_page": 24},
    {"financial_year": 2025, "revenue_category": "Permits", "revenue_stream": "Other Permits", "actual_amount": 17500, "source_page": 24},
]

detailed_local_revenue_df = pd.DataFrame(detailed_local_revenue)

display(detailed_local_revenue_df)
print("Number of records:", len(detailed_local_revenue_df))

,financial_year,revenue_category,revenue_stream,actual_amount,source_page
0,2025,Local Taxes,Residential Rates,1948368,21
1,2025,Local Taxes,Industrial / Commercial Rates,4492657,21
2,2025,Local Taxes,Personal Levy,222153,21
3,2025,Fees and Charges,Consent Fees,6000,22
4,2025,Fees and Charges,Survey Fees,8000,22
5,2025,Fees and Charges,Building Inspection Fees,249300,22
6,2025,Fees and Charges,Plan Scrutiny Fees,553935,22
7,2025,Fees and Charges,Change of Premise Use,59162,22
8,2025,Fees and Charges,Container / Ntemba Fees,16084,22
9,2025,Fees and Charges,Rentals / Lease of Council Properties,688925,22


Number of records: 56


In [ ]:
for category, official in official_totals.items():
    calculated = calculated_totals.get(category, 0)
    difference = official - calculated

    print(
        f"{category}: "
        f"Official = K{official:,}, "
        f"Calculated = K{calculated:,}, "
        f"Difference = K{difference:,}"
    )

NameError: name 'official_totals' is not defined

In [ ]:
# Official totals taken from the 2025 financial statement
official_totals = {
    "Local Taxes": 6663177,
    "Fees and Charges": 5156007,
    "Land Development Charges": 6906050,
    "Licences": 347920,
    "Levies": 1209808,
    "Permits": 2726912
}

# Calculate totals from the detailed_local_revenue dataframe
calculated_totals = (
    detailed_local_revenue
    .groupby("revenue_category")["actual_amount"]
    .sum()
    .to_dict()
)

# Compare official totals against calculated totals
for category, official in official_totals.items():
    calculated = calculated_totals.get(category, 0)
    difference = official - calculated

    print(
        f"{category}: "
        f"Official = K{official:,.0f}, "
        f"Calculated = K{calculated:,.0f}, "
        f"Difference (official - calculated) = K{difference:,.0f}"
    )

AttributeError: 'list' object has no attribute 'groupby'

In [ ]:
import pandas as pd

# Convert the detailed revenue list into a DataFrame
detailed_local_revenue = pd.DataFrame(detailed_local_revenue)

# Check the structure
print("Rows:", len(detailed_local_revenue))
print("Columns:", detailed_local_revenue.columns.tolist())

detailed_local_revenue.head()

Rows: 56
Columns: ['financial_year', 'revenue_category', 'revenue_stream', 'actual_amount', 'source_page']


,financial_year,revenue_category,revenue_stream,actual_amount,source_page
0,2025,Local Taxes,Residential Rates,1948368,21
1,2025,Local Taxes,Industrial / Commercial Rates,4492657,21
2,2025,Local Taxes,Personal Levy,222153,21
3,2025,Fees and Charges,Consent Fees,6000,22
4,2025,Fees and Charges,Survey Fees,8000,22


In [ ]:
# Official totals from the 2025 financial statement
official_totals = {
    "Local Taxes": 6663177,
    "Fees and Charges": 5156007,
    "Land Development Charges": 6906050,
    "Licences": 347920,
    "Levies": 1209808,
    "Permits": 2726912
}

# Calculate totals from the detailed records
calculated_totals = (
    detailed_local_revenue
    .groupby("revenue_category")["actual_amount"]
    .sum()
    .to_dict()
)

# Compare official totals with calculated totals
for category, official in official_totals.items():
    calculated = calculated_totals.get(category, 0)
    difference = official - calculated

    print(
        f"{category}: "
        f"Official = K{official:,.0f}, "
        f"Calculated = K{calculated:,.0f}, "
        f"Difference (official - calculated) = K{difference:,.0f}"
    )

Local Taxes: Official = K6,663,177, Calculated = K6,663,178, Difference (official - calculated) = K-1
Fees and Charges: Official = K5,156,007, Calculated = K5,156,006, Difference (official - calculated) = K1
Land Development Charges: Official = K6,906,050, Calculated = K6,906,050, Difference (official - calculated) = K0
Licences: Official = K347,920, Calculated = K347,920, Difference (official - calculated) = K0
Levies: Official = K1,209,808, Calculated = K1,209,807, Difference (official - calculated) = K1
Permits: Official = K2,726,912, Calculated = K2,726,912, Difference (official - calculated) = K0


In [ ]:
# Add validation status to the dataset
detailed_local_revenue["source_validation_note"] = (
    "Detailed source amount; category total may differ from published total by K1 due to source rounding/reporting discrepancy."
)

# Display final dataset structure
print("Number of records:", len(detailed_local_revenue))
print("\nColumns:")
print(detailed_local_revenue.columns.tolist())

print("\nMissing values:")
print(detailed_local_revenue.isnull().sum())

# Check duplicates
print("\nDuplicate rows:", detailed_local_revenue.duplicated().sum())

# Export detailed local revenue dataset
local_revenue_file = (
    r"C:\Users\Mutofwe\group6_administration\notebooks"
    r"\db-unza26-csc4792-kabwe-local-revenue-detailed-2025.csv"
)

detailed_local_revenue.to_csv(
    local_revenue_file,
    sep="|",
    index=False,
    encoding="utf-8"
)

print("\nExported successfully:")
print(local_revenue_file)

Number of records: 56

Columns:
['financial_year', 'revenue_category', 'revenue_stream', 'actual_amount', 'source_page', 'source_validation_note']

Missing values:
financial_year            0
revenue_category          0
revenue_stream            0
actual_amount             0
source_page               0
source_validation_note    0
dtype: int64

Duplicate rows: 0

Exported successfully:
C:\Users\Mutofwe\group6_administration\notebooks\db-unza26-csc4792-kabwe-local-revenue-detailed-2025.csv


In [ ]:
import os
import pandas as pd

# ---------------------------------------------------------
# FINAL DATASET CHECK
# ---------------------------------------------------------

files_to_check = {
    "Approved Budget":
        r"C:\Users\Mutofwe\group6_administration\notebooks\db-unza26-csc4792-kabwe-2025-approved-obb-budget.csv",

    "LGEF Utilisation":
        r"C:\Users\Mutofwe\group6_administration\notebooks\db-unza26-csc4792-kabwe-lgef-utilisation-2025.csv",

    "Detailed Local Revenue":
        r"C:\Users\Mutofwe\group6_administration\notebooks\db-unza26-csc4792-kabwe-local-revenue-detailed-2025.csv"
}

for name, path in files_to_check.items():

    print("\n" + "=" * 70)
    print(name)
    print("=" * 70)

    print("File exists:", os.path.exists(path))

    if os.path.exists(path):

        print("File size:", os.path.getsize(path), "bytes")

        df = pd.read_csv(path, sep="|")

        print("Rows:", len(df))
        print("Columns:", len(df.columns))
        print("Column names:", df.columns.tolist())

        print("Missing values:")
        print(df.isnull().sum().sum())

        print("Duplicate rows:", df.duplicated().sum())

        print("Separator check: PASSED")


Approved Budget
File exists: True
File size: 6106 bytes
Rows: 80
Columns: 8
Column names: ['financial_year', 'revenue_category', 'revenue_code', 'revenue_description', 'approved_budget_2025', 'revised_budget_2026', 'estimate_2027', 'page']
Missing values:
0
Duplicate rows: 0
Separator check: PASSED

LGEF Utilisation
File exists: True
File size: 542 bytes
Rows: 7
Columns: 7
Column names: ['financial_year', 'fund_type', 'transaction_type', 'description', 'amount', 'purpose', 'page']
Missing values:
0
Duplicate rows: 0
Separator check: PASSED

Detailed Local Revenue
File exists: True
File size: 9715 bytes
Rows: 56
Columns: 6
Column names: ['financial_year', 'revenue_category', 'revenue_stream', 'actual_amount', 'source_page', 'source_validation_note']
Missing values:
0
Duplicate rows: 0
Separator check: PASSED


In [ ]:
import os
import pandas as pd

files_to_check = {
    "Approved Budget":
        r"C:\Users\Mutofwe\group6_administration\notebooks\db-unza26-csc4792-kabwe-2025-approved-obb-budget.csv",

    "LGEF Utilisation":
        r"C:\Users\Mutofwe\group6_administration\notebooks\db-unza26-csc4792-kabwe-lgef-utilisation-2025.csv",

    "Detailed Local Revenue":
        r"C:\Users\Mutofwe\group6_administration\notebooks\db-unza26-csc4792-kabwe-local-revenue-detailed-2025.csv"
}

for name, path in files_to_check.items():
    print("\n" + "=" * 60)
    print(name)
    print("=" * 60)

    print("File exists:", os.path.exists(path))

    if os.path.exists(path):
        df = pd.read_csv(path, sep="|")

        print("Rows:", len(df))
        print("Columns:", df.columns.tolist())
        print("Missing values:", df.isnull().sum().sum())
        print("Duplicate rows:", df.duplicated().sum())
        print("Separator: |")
        print("STATUS: PASSED")


Approved Budget
File exists: True
Rows: 80
Columns: ['financial_year', 'revenue_category', 'revenue_code', 'revenue_description', 'approved_budget_2025', 'revised_budget_2026', 'estimate_2027', 'page']
Missing values: 0
Duplicate rows: 0
Separator: |
STATUS: PASSED

LGEF Utilisation
File exists: True
Rows: 7
Columns: ['financial_year', 'fund_type', 'transaction_type', 'description', 'amount', 'purpose', 'page']
Missing values: 0
Duplicate rows: 0
Separator: |
STATUS: PASSED

Detailed Local Revenue
File exists: True
Rows: 56
Columns: ['financial_year', 'revenue_category', 'revenue_stream', 'actual_amount', 'source_page', 'source_validation_note']
Missing values: 0
Duplicate rows: 0
Separator: |
STATUS: PASSED


In [ ]:
programme_totals = programme_budget_df[
    programme_budget_df["record_type"] == "Programme"
]

print("Number of programmes:", len(programme_totals))

print(
    "2024 programme total:",
    programme_totals["approved_budget_2024"].sum()
)

print(
    "2025 programme total:",
    programme_totals["budget_estimate_2025"].sum()
)

Number of programmes: 16
2024 programme total: 140728669
2025 programme total: 181079552


In [ ]:
programme_budget_df["source_url"] = (
    "https://www.kabwecouncil.gov.zm/wp-content/uploads/2025/05/"
    "2025-KABWE-M-COUNCIL-OBB.pdf"
)

programme_budget_df["source_document"] = (
    "2025 Kabwe Municipal Council OBB Budget, Pages 8-10"
)

In [ ]:
programme_budget_output = (
    PROJECT_ROOT /
    "data" /
    "processed" /
    "db-unza26-csc4792-kabwe_programme_subprogramme_budget_2024_2025.csv"
)

programme_budget_df.to_csv(
    programme_budget_output,
    sep="|",
    index=False
)

print(f"Saved: {programme_budget_output}")

Saved: ../data/processed/db-unza26-csc4792-kabwe_programme_subprogramme_budget_2024_2025.csv


In [ ]:
check_programme_budget = pd.read_csv(
    programme_budget_output,
    sep="|"
)

print("Rows:", len(check_programme_budget))
print("Columns:", len(check_programme_budget.columns))
print("Duplicates:", check_programme_budget.duplicated().sum())

display(check_programme_budget)

Rows: 63
Columns: 8
Duplicates: 0


,programme_code,programme,record_type,sub_programme,approved_budget_2024,budget_estimate_2025,source_url,source_document
0,1,Constituency Development,Programme,NaN,61271284,72116301,https://www.kabwecouncil.gov.zm/wp-content/uploads/2025/05/2025-KABWE-M-COUN...,"2025 Kabwe Municipal Council OBB Budget, Pages 8-10"
1,779,Constituency Development,Sub-Programme,Community Projects,34924632,43925383,https://www.kabwecouncil.gov.zm/wp-content/uploads/2025/05/2025-KABWE-M-COUN...,"2025 Kabwe Municipal Council OBB Budget, Pages 8-10"
2,780,Constituency Development,Sub-Programme,Women and Youth Empowerment,11641544,12456452,https://www.kabwecouncil.gov.zm/wp-content/uploads/2025/05/2025-KABWE-M-COUN...,"2025 Kabwe Municipal Council OBB Budget, Pages 8-10"
3,781,Constituency Development,Sub-Programme,CDF Administration,3063564,3278014,https://www.kabwecouncil.gov.zm/wp-content/uploads/2025/05/2025-KABWE-M-COUN...,"2025 Kabwe Municipal Council OBB Budget, Pages 8-10"
4,782,Constituency Development,Sub-Programme,Secondary School and Skills Development Bursaries,11641544,12456452,https://www.kabwecouncil.gov.zm/wp-content/uploads/2025/05/2025-KABWE-M-COUN...,"2025 Kabwe Municipal Council OBB Budget, Pages 8-10"
5,2,Local Governance,Programme,NaN,6125473,5827497,https://www.kabwecouncil.gov.zm/wp-content/uploads/2025/05/2025-KABWE-M-COUN...,"2025 Kabwe Municipal Council OBB Budget, Pages 8-10"
6,3,Local Governance,Sub-Programme,Legislative Functions,3767535,3349182,https://www.kabwecouncil.gov.zm/wp-content/uploads/2025/05/2025-KABWE-M-COUN...,"2025 Kabwe Municipal Council OBB Budget, Pages 8-10"
7,43,Local Governance,Sub-Programme,Citizen Engagement,2357938,2478315,https://www.kabwecouncil.gov.zm/wp-content/uploads/2025/05/2025-KABWE-M-COUN...,"2025 Kabwe Municipal Council OBB Budget, Pages 8-10"
8,3,Integrated Development Planning,Programme,NaN,6342899,5387125,https://www.kabwecouncil.gov.zm/wp-content/uploads/2025/05/2025-KABWE-M-COUN...,"2025 Kabwe Municipal Council OBB Budget, Pages 8-10"
9,6,Integrated Development Planning,Sub-Programme,Environmental Planning,893433,3677653,https://www.kabwecouncil.gov.zm/wp-content/uploads/2025/05/2025-KABWE-M-COUN...,"2025 Kabwe Municipal Council OBB Budget, Pages 8-10"


## Download the 2025 Annual Financial Statement

The 2025 Kabwe Municipal Council annual financial statement is downloaded from the official council website and stored in the `data/raw/` directory.

The original PDF is preserved as raw source data so that the financial information used for extraction and analysis can be traced back to the council's published document.

In [ ]:
financial_url = (
    "https://www.kabwecouncil.gov.zm/wp-content/uploads/2025/08/"
    "2025-BI-ANNUAL-FINANCIAL-STATEMENT-KABWE-M-COUNCIL-1-1.pdf"
)

financial_raw_path = (
    RAW_DIR /
    "2025_kabwe_annual_financial_statement.pdf"
)

response = requests.get(
    financial_url,
    timeout=120,
    verify=False
)

response.raise_for_status()

with open(financial_raw_path, "wb") as f:
    f.write(response.content)

print("Downloaded:", financial_raw_path)
print("File size:", len(response.content), "bytes")

Downloaded: ../data/raw/2025_kabwe_annual_financial_statement.pdf
File size: 11368536 bytes


In [ ]:
financial_doc = fitz.open(financial_raw_path)

print("Pages:", financial_doc.page_count)

Pages: 36


In [ ]:
financial_keywords = [
    "LGEF",
    "Local Government Equalisation Fund",
    "revenue",
    "receipts",
    "expenditure",
    "CDF",
    "utilisation",
    "capital",
    "own source",
    "financial performance"
]

financial_pages = []

for page_number, page in enumerate(financial_doc, start=1):
    text = page.get_text()

    matches = [
        keyword
        for keyword in financial_keywords
        if keyword.lower() in text.lower()
    ]

    if matches:
        financial_pages.append({
            "page": page_number,
            "keywords_found": ", ".join(matches)
        })

financial_pages_df = pd.DataFrame(financial_pages)

display(financial_pages_df)

""


In [ ]:
import pandas as pd

rows = [
    [2025, "Kabwe Central", "Receipt", "CDF Funding", 6205128, "8(a)"],
    [2025, "Bwacha", "Receipt", "CDF Funding", 6205128, "8(a)"],

    [2025, "Kabwe Central", "Receipt", "Loan Repayments", 179669, "8(b)"],
    [2025, "Bwacha", "Receipt", "Loan Repayments", 477394, "8(b)"],

    [2025, "Kabwe Central", "Receipt", "Interest Earned", 21067, "8(c)"],
    [2025, "Bwacha", "Receipt", "Interest Earned", 6331, "8(c)"],

    [2025, "Kabwe Central", "Payment", "Infrastructure Development", 3703681, "8(d)"],
    [2025, "Bwacha", "Payment", "Infrastructure Development", 2649421, "8(d)"],

    [2025, "Kabwe Central", "Payment", "Rehabilitation Works", 557547, "8(e)"],
    [2025, "Bwacha", "Payment", "Rehabilitation Works", 0, "8(e)"],

    [2025, "Kabwe Central", "Payment", "Asset Acquisition", 5244633, "8(f)"],
    [2025, "Bwacha", "Payment", "Asset Acquisition", 5208500, "8(f)"],

    [2025, "All Constituencies", "Payment", "Rural Electrification", 0, "8(g)"],

    [2025, "Kabwe Central", "Payment", "Social Benefits - Grants", 2320000, "8(h)"],
    [2025, "Bwacha", "Payment", "Social Benefits - Grants", 2176650, "8(h)"],

    [2025, "All Constituencies", "Payment", "Loans", 0, "8(i)"],

    [2025, "Kabwe Central", "Payment", "Skills & Boarding School Bursaries", 4306021, "8(j)"],
    [2025, "Bwacha", "Payment", "Skills & Boarding School Bursaries", 4292889, "8(j)"],

    [2025, "Kabwe Central", "Payment", "Administrative Cost", 928316, "8(k)"],
    [2025, "Bwacha", "Payment", "Administrative Cost", 1067956, "8(k)"],

    [2025, "Kabwe Central", "Payment", "Disaster Contingency", 60038, "8(l)"],
    [2025, "Bwacha", "Payment", "Disaster Contingency", 235214, "8(l)"],

    [2025, "Kabwe Central", "Payment", "Other Payments - Fuel for Roads", 855602, "8(m)"],
    [2025, "Bwacha", "Payment", "Other Payments - Fuel for Roads", 855602, "8(m)"],
]

columns = [
    "year",
    "constituency",
    "transaction_type",
    "financial_category",
    "amount_kwacha",
    "source_note"
]

cdf_financial = pd.DataFrame(rows, columns=columns)

cdf_financial["source_url"] = (
    "https://www.kabwecouncil.gov.zm/wp-content/uploads/2025/08/"
    "2025-BI-ANNUAL-FINANCIAL-STATEMENT-KABWE-M-COUNCIL-1-1.pdf"
)

cdf_financial["source_document"] = (
    "Kabwe Municipal Council BI-Annual Financial Statements 2025"
)

output_path = (
    PROCESSED_DIR /
    "db-unza26-csc4792-kabwe_cdf_financial_statement_2025.csv"
)

cdf_financial.to_csv(
    output_path,
    sep="|",
    index=False
)

print("CSV created successfully!")
print("File:", output_path)
print("Rows:", len(cdf_financial))
print("Columns:", len(cdf_financial.columns))

print(
    "\nCDF receipts:",
    cdf_financial.loc[
        cdf_financial["transaction_type"] == "Receipt",
        "amount_kwacha"
    ].sum()
)

print(
    "CDF payments:",
    cdf_financial.loc[
        cdf_financial["transaction_type"] == "Payment",
        "amount_kwacha"
    ].sum()
)

CSV created successfully!
File: ..\data\processed\db-unza26-csc4792-kabwe_cdf_financial_statement_2025.csv
Rows: 24
Columns: 8

CDF receipts: 13094717
CDF payments: 34462070


,year,constituency,transaction_type,financial_category,amount_kwacha,source_note,source_url,source_document
0,2025,Kabwe Central,Receipt,CDF Funding,6205128,8(a),https://www.kabwecouncil.gov.zm/wp-content/upl...,Kabwe Municipal Council BI-Annual Financial St...
1,2025,Bwacha,Receipt,CDF Funding,6205128,8(a),https://www.kabwecouncil.gov.zm/wp-content/upl...,Kabwe Municipal Council BI-Annual Financial St...
2,2025,Kabwe Central,Receipt,Loan Repayments,179669,8(b),https://www.kabwecouncil.gov.zm/wp-content/upl...,Kabwe Municipal Council BI-Annual Financial St...
3,2025,Bwacha,Receipt,Loan Repayments,477394,8(b),https://www.kabwecouncil.gov.zm/wp-content/upl...,Kabwe Municipal Council BI-Annual Financial St...
4,2025,Kabwe Central,Receipt,Interest Earned,21067,8(c),https://www.kabwecouncil.gov.zm/wp-content/upl...,Kabwe Municipal Council BI-Annual Financial St...
5,2025,Bwacha,Receipt,Interest Earned,6331,8(c),https://www.kabwecouncil.gov.zm/wp-content/upl...,Kabwe Municipal Council BI-Annual Financial St...
6,2025,Kabwe Central,Payment,Infrastructure Development,3703681,8(d),https://www.kabwecouncil.gov.zm/wp-content/upl...,Kabwe Municipal Council BI-Annual Financial St...
7,2025,Bwacha,Payment,Infrastructure Development,2649421,8(d),https://www.kabwecouncil.gov.zm/wp-content/upl...,Kabwe Municipal Council BI-Annual Financial St...
8,2025,Kabwe Central,Payment,Rehabilitation Works,557547,8(e),https://www.kabwecouncil.gov.zm/wp-content/upl...,Kabwe Municipal Council BI-Annual Financial St...
9,2025,Bwacha,Payment,Rehabilitation Works,0,8(e),https://www.kabwecouncil.gov.zm/wp-content/upl...,Kabwe Municipal Council BI-Annual Financial St...


# Load the CDF financial statement CSV

In [ ]:
cdf_financial_check = pd.read_csv(
    PROCESSED_DIR /
    "db-unza26-csc4792-kabwe_cdf_financial_statement_2025.csv",
    sep="|"
)

print("Shape:", cdf_financial_check.shape)
print("\nColumns:")
print(cdf_financial_check.columns.tolist())

print("\nMissing values:")
print(cdf_financial_check.isna().sum())

print("\nDuplicate rows:", cdf_financial_check.duplicated().sum())

print("\nTransaction types:")
print(cdf_financial_check["transaction_type"].value_counts())

print("\nFinancial categories:")
print(cdf_financial_check["financial_category"].value_counts())

Shape: (24, 8)

Columns:
['year', 'constituency', 'transaction_type', 'financial_category', 'amount_kwacha', 'source_note', 'source_url', 'source_document']

Missing values:
year                  0
constituency          0
transaction_type      0
financial_category    0
amount_kwacha         0
source_note           0
source_url            0
source_document       0
dtype: int64

Duplicate rows: 0

Transaction types:
transaction_type
Payment    18
Receipt     6
Name: count, dtype: int64

Financial categories:
financial_category
CDF Funding                           2
Loan Repayments                       2
Interest Earned                       2
Infrastructure Development            2
Rehabilitation Works                  2
Asset Acquisition                     2
Social Benefits - Grants              2
Skills & Boarding School Bursaries    2
Administrative Cost                   2
Disaster Contingency                  2
Other Payments - Fuel for Roads       2
Rural Electrification        

,year,constituency,transaction_type,financial_category,amount_kwacha,source_note,source_url,source_document
0,2025,Kabwe Central,Receipt,CDF Funding,6205128,8(a),https://www.kabwecouncil.gov.zm/wp-content/upl...,Kabwe Municipal Council BI-Annual Financial St...
1,2025,Bwacha,Receipt,CDF Funding,6205128,8(a),https://www.kabwecouncil.gov.zm/wp-content/upl...,Kabwe Municipal Council BI-Annual Financial St...
2,2025,Kabwe Central,Receipt,Loan Repayments,179669,8(b),https://www.kabwecouncil.gov.zm/wp-content/upl...,Kabwe Municipal Council BI-Annual Financial St...
3,2025,Bwacha,Receipt,Loan Repayments,477394,8(b),https://www.kabwecouncil.gov.zm/wp-content/upl...,Kabwe Municipal Council BI-Annual Financial St...
4,2025,Kabwe Central,Receipt,Interest Earned,21067,8(c),https://www.kabwecouncil.gov.zm/wp-content/upl...,Kabwe Municipal Council BI-Annual Financial St...
5,2025,Bwacha,Receipt,Interest Earned,6331,8(c),https://www.kabwecouncil.gov.zm/wp-content/upl...,Kabwe Municipal Council BI-Annual Financial St...
6,2025,Kabwe Central,Payment,Infrastructure Development,3703681,8(d),https://www.kabwecouncil.gov.zm/wp-content/upl...,Kabwe Municipal Council BI-Annual Financial St...
7,2025,Bwacha,Payment,Infrastructure Development,2649421,8(d),https://www.kabwecouncil.gov.zm/wp-content/upl...,Kabwe Municipal Council BI-Annual Financial St...
8,2025,Kabwe Central,Payment,Rehabilitation Works,557547,8(e),https://www.kabwecouncil.gov.zm/wp-content/upl...,Kabwe Municipal Council BI-Annual Financial St...
9,2025,Bwacha,Payment,Rehabilitation Works,0,8(e),https://www.kabwecouncil.gov.zm/wp-content/upl...,Kabwe Municipal Council BI-Annual Financial St...


## 2025 CDF Disbursements by Constituency

The 2025 CDF funding received by Kabwe Central and Bwacha constituencies is structured into a dedicated disbursement dataset.

The dataset records the year, constituency, funding type, amount received, and the corresponding source note from the annual financial statement.

This dataset provides a constituency-level view of CDF funding received and complements the broader CDF financial statement dataset.

In [ ]:
# CDF funding received by constituency in 2025

cdf_disbursements = pd.DataFrame([
    [2025, "Kabwe Central", "CDF Funding", 6205128, "8(a)"],
    [2025, "Bwacha", "CDF Funding", 6205128, "8(a)"],
], columns=[
    "year",
    "constituency",
    "funding_type",
    "amount_kwacha",
    "source_note"
])

cdf_disbursements["source_url"] = (
    "https://www.kabwecouncil.gov.zm/wp-content/uploads/2025/08/"
    "2025-BI-ANNUAL-FINANCIAL-STATEMENT-KABWE-M-COUNCIL-1-1.pdf"
)

cdf_disbursements["source_document"] = (
    "Kabwe Municipal Council BI-Annual Financial Statements 2025"
)

output_path = (
    PROCESSED_DIR /
    "db-unza26-csc4792-kabwe_cdf_disbursements_2025.csv"
)

cdf_disbursements.to_csv(
    output_path,
    sep="|",
    index=False
)

print("CSV created successfully!")
print("File:", output_path)
print("Rows:", len(cdf_disbursements))
print(
    "Total CDF funding:",
    f"K{cdf_disbursements['amount_kwacha'].sum():,.0f}"
)

CSV created successfully!
File: ../data/processed/db-unza26-csc4792-kabwe_cdf_disbursements_2025.csv
Rows: 2
Total CDF funding: K12,410,256


,year,constituency,funding_type,amount_kwacha,source_note,source_url,source_document
0,2025,Kabwe Central,CDF Funding,6205128,8(a),https://www.kabwecouncil.gov.zm/wp-content/upl...,Kabwe Municipal Council BI-Annual Financial St...
1,2025,Bwacha,CDF Funding,6205128,8(a),https://www.kabwecouncil.gov.zm/wp-content/upl...,Kabwe Municipal Council BI-Annual Financial St...


In [ ]:
## Verify the 2024 and 2025 CDF project datasets

In [ ]:
# Verify the 2024 and 2025 CDF project datasets

cdf_2024 = pd.read_csv(
    PROCESSED_DIR /
    "db-unza26-csc4792-kabwe_cdf_projects_2024.csv",
    sep="|"
)

cdf_2025 = pd.read_csv(
    PROCESSED_DIR /
    "db-unza26-csc4792-kabwe_cdf_projects_2025.csv",
    sep="|"
)

print("===== 2024 CDF PROJECTS =====")
print("Shape:", cdf_2024.shape)
print("Duplicates:", cdf_2024.duplicated().sum())
print("\nProjects by constituency:")
print(cdf_2024["constituency"].value_counts())

print("\nMissing values:")
print(cdf_2024.isna().sum())

print("\n===== 2025 CDF PROJECTS =====")
print("Shape:", cdf_2025.shape)
print("Duplicates:", cdf_2025.duplicated().sum())
print("\nProjects by sector:")
print(cdf_2025["sector"].value_counts())

print("\nMissing values:")
print(cdf_2025.isna().sum())

===== 2024 CDF PROJECTS =====
Shape: (79, 15)
Duplicates: 0

Projects by constituency:
constituency
Kabwe Central    43
Bwacha           36
Name: count, dtype: int64

Missing values:
project_number          0
year                    0
constituency            0
project_name            0
project_description     0
project_type           36
ward                    0
project_site            1
application_amount     79
engineers_estimate     79
approved_amount        79
contract_amount        79
status                 79
source_url              0
source_document         0
dtype: int64

===== 2025 CDF PROJECTS =====
Shape: (33, 9)
Duplicates: 0

Projects by sector:
sector
Education               16
Health                   8
Commerce and Trade       4
Water and Sanitation     3
Transport                2
Name: count, dtype: int64

Missing values:
no                 0
year               0
constituency       0
project            0
ward               0
sector             0
comment            0
s

In [ ]:
source_inventory = """id,source_name,source_type,url,description
S001,Kabwe Municipal Council Website,Website,https://www.kabwecouncil.gov.zm,Official Kabwe Municipal Council website and digital source directory.
S002,2024 Bwacha CDF Community Projects Submission,PDF,https://www.kabwecouncil.gov.zm/wp-content/uploads/2024/11/2024-Bwacha-community-projects-Recieved.pdf,2024 CDF community project submissions for Bwacha Constituency.
S003,2024 Kabwe Central CDF Community Projects Submission,PDF,https://www.kabwecouncil.gov.zm/wp-content/uploads/2024/11/2024-Community-Projects-Kabwe-Central-received.pdf,2024 CDF community project submissions for Kabwe Central Constituency.
S004,2025 Kabwe Municipal Council OBB Budget,PDF,https://www.kabwecouncil.gov.zm/wp-content/uploads/2025/05/2025-KABWE-M-COUNCIL-OBB.pdf,2025 Output Based Budget containing CDF allocations and CDF output indicators.
S005,Kabwe District Integrated Development Plan 2023-2028,PDF,https://www.kabwecouncil.gov.zm/wp-content/uploads/2024/09/Kabwe-Approved-IDP_Final-Version-1.pdf,Approved Kabwe District Integrated Development Plan for 2023-2028.
S006,Proposed 2025 CDF Projects,PDF,https://www.kabwecouncil.gov.zm/wp-content/uploads/2025/08/Proposed-2025-CDF-projects.pdf,Proposed 2025 CDF projects for Kabwe Central Constituency.
S007,2025 Kabwe Municipal Council BI-Annual Financial Statements,PDF,https://www.kabwecouncil.gov.zm/wp-content/uploads/2025/08/2025-BI-ANNUAL-FINANCIAL-STATEMENT-KABWE-M-COUNCIL-1-1.pdf,2025 BI-Annual Financial Statements containing CDF funding loan repayments interest expenditure and constituency-level CDF financial information.
"""

source_path = (
    SOURCES_DIR /
    "cdf_source_inventory.csv"
)

source_path.write_text(
    source_inventory,
    encoding="utf-8"
)

print("Source inventory updated successfully.")
print("File:", source_path)

Source inventory updated successfully.
File: ..\sources\source_inventory.csv


In [ ]:
sources = pd.read_csv(
    SOURCES_DIR /
    "cdf_source_inventory.csv"
)

print("Number of sources:", len(sources))
print("Columns:", sources.columns.tolist())

Number of sources: 7
Columns: ['id', 'source_name', 'source_type', 'url', 'description']


,id,source_name,source_type,url,description
0,S001,Kabwe Municipal Council Website,Website,https://www.kabwecouncil.gov.zm,Official Kabwe Municipal Council website and d...
1,S002,2024 Bwacha CDF Community Projects Submission,PDF,https://www.kabwecouncil.gov.zm/wp-content/upl...,2024 CDF community project submissions for Bwa...
2,S003,2024 Kabwe Central CDF Community Projects Subm...,PDF,https://www.kabwecouncil.gov.zm/wp-content/upl...,2024 CDF community project submissions for Kab...
3,S004,2025 Kabwe Municipal Council OBB Budget,PDF,https://www.kabwecouncil.gov.zm/wp-content/upl...,2025 Output Based Budget containing CDF alloca...
4,S005,Kabwe District Integrated Development Plan 202...,PDF,https://www.kabwecouncil.gov.zm/wp-content/upl...,Approved Kabwe District Integrated Development...
5,S006,Proposed 2025 CDF Projects,PDF,https://www.kabwecouncil.gov.zm/wp-content/upl...,Proposed 2025 CDF projects for Kabwe Central C...
6,S007,2025 Kabwe Municipal Council BI-Annual Financi...,PDF,https://www.kabwecouncil.gov.zm/wp-content/upl...,2025 BI-Annual Financial Statements containing...


In [ ]:
sources = pd.read_csv(
    SOURCES_DIR /
    "cdf_source_inventory.csv"
)

print("Number of sources:", len(sources))
print("Columns:", sources.columns.tolist())

print("\nDuplicate IDs:", sources["id"].duplicated().sum())
print("Duplicate URLs:", sources["url"].duplicated().sum())

Number of sources: 7
Columns: ['id', 'source_name', 'source_type', 'url', 'description']

Duplicate IDs: 0
Duplicate URLs: 0


,id,source_name,source_type,url,description
0,S001,Kabwe Municipal Council Website,Website,https://www.kabwecouncil.gov.zm,Official Kabwe Municipal Council website and d...
1,S002,2024 Bwacha CDF Community Projects Submission,PDF,https://www.kabwecouncil.gov.zm/wp-content/upl...,2024 CDF community project submissions for Bwa...
2,S003,2024 Kabwe Central CDF Community Projects Subm...,PDF,https://www.kabwecouncil.gov.zm/wp-content/upl...,2024 CDF community project submissions for Kab...
3,S004,2025 Kabwe Municipal Council OBB Budget,PDF,https://www.kabwecouncil.gov.zm/wp-content/upl...,2025 Output Based Budget containing CDF alloca...
4,S005,Kabwe District Integrated Development Plan 202...,PDF,https://www.kabwecouncil.gov.zm/wp-content/upl...,Approved Kabwe District Integrated Development...
5,S006,Proposed 2025 CDF Projects,PDF,https://www.kabwecouncil.gov.zm/wp-content/upl...,Proposed 2025 CDF projects for Kabwe Central C...
6,S007,2025 Kabwe Municipal Council BI-Annual Financi...,PDF,https://www.kabwecouncil.gov.zm/wp-content/upl...,2025 BI-Annual Financial Statements containing...


In [ ]:
from pathlib import Path

cdf_files = [
    PROCESSED_DIR / "db-unza26-csc4792-kabwe_cdf_budget_2024_2025.csv",
    PROCESSED_DIR / "db-unza26-csc4792-kabwe_cdf_disbursements_2025.csv",
    PROCESSED_DIR / "db-unza26-csc4792-kabwe_cdf_financial_statement_2025.csv",
    PROCESSED_DIR / "db-unza26-csc4792-kabwe_cdf_output_indicators_2025.csv",
    PROCESSED_DIR / "db-unza26-csc4792-kabwe_cdf_projects_2024.csv",
    PROCESSED_DIR / "db-unza26-csc4792-kabwe_cdf_projects_2025.csv"
]

print("CDF DATASET FILE CHECK")
print("=" * 50)

for file in cdf_files:
    path = Path(file)

    if path.exists():
        df = pd.read_csv(path, sep="|")
        print(f"✓ {path.name}")
        print(f"  Rows: {len(df)}")
        print(f"  Columns: {len(df.columns)}")
    else:
        print(f"✗ MISSING: {path.name}")

CDF DATASET FILE CHECK
✓ db-unza26-csc4792-kabwe_cdf_budget_2024_2025.csv
  Rows: 5
  Columns: 7
✓ db-unza26-csc4792-kabwe_cdf_disbursements_2025.csv
  Rows: 2
  Columns: 7
✓ db-unza26-csc4792-kabwe_cdf_financial_statement_2025.csv
  Rows: 24
  Columns: 8
✓ db-unza26-csc4792-kabwe_cdf_output_indicators_2025.csv
  Rows: 12
  Columns: 10
✓ db-unza26-csc4792-kabwe_cdf_projects_2024.csv
  Rows: 79
  Columns: 15
✓ db-unza26-csc4792-kabwe_cdf_projects_2025.csv
  Rows: 33
  Columns: 9


In [ ]:
cdf_financial = pd.read_csv(
    PROCESSED_DIR /
    "db-unza26-csc4792-kabwe_cdf_financial_statement_2025.csv",
    sep="|"
)

print("Shape:", cdf_financial.shape)

print("\nMissing values:")
print(cdf_financial.isna().sum())

print("\nDuplicate rows:", cdf_financial.duplicated().sum())

print("\nTransaction types:")
print(cdf_financial["transaction_type"].value_counts())

print("\nFinancial categories:")
print(cdf_financial["financial_category"].value_counts())

Shape: (24, 8)

Missing values:
year                  0
constituency          0
transaction_type      0
financial_category    0
amount_kwacha         0
source_note           0
source_url            0
source_document       0
dtype: int64

Duplicate rows: 0

Transaction types:
transaction_type
Payment    18
Receipt     6
Name: count, dtype: int64

Financial categories:
financial_category
CDF Funding                           2
Loan Repayments                       2
Interest Earned                       2
Infrastructure Development            2
Rehabilitation Works                  2
Asset Acquisition                     2
Social Benefits - Grants              2
Skills & Boarding School Bursaries    2
Administrative Cost                   2
Disaster Contingency                  2
Other Payments - Fuel for Roads       2
Rural Electrification                 1
Loans                                 1
Name: count, dtype: int64


,year,constituency,transaction_type,financial_category,amount_kwacha,source_note,source_url,source_document
0,2025,Kabwe Central,Receipt,CDF Funding,6205128,8(a),https://www.kabwecouncil.gov.zm/wp-content/upl...,Kabwe Municipal Council BI-Annual Financial St...
1,2025,Bwacha,Receipt,CDF Funding,6205128,8(a),https://www.kabwecouncil.gov.zm/wp-content/upl...,Kabwe Municipal Council BI-Annual Financial St...
2,2025,Kabwe Central,Receipt,Loan Repayments,179669,8(b),https://www.kabwecouncil.gov.zm/wp-content/upl...,Kabwe Municipal Council BI-Annual Financial St...
3,2025,Bwacha,Receipt,Loan Repayments,477394,8(b),https://www.kabwecouncil.gov.zm/wp-content/upl...,Kabwe Municipal Council BI-Annual Financial St...
4,2025,Kabwe Central,Receipt,Interest Earned,21067,8(c),https://www.kabwecouncil.gov.zm/wp-content/upl...,Kabwe Municipal Council BI-Annual Financial St...
5,2025,Bwacha,Receipt,Interest Earned,6331,8(c),https://www.kabwecouncil.gov.zm/wp-content/upl...,Kabwe Municipal Council BI-Annual Financial St...
6,2025,Kabwe Central,Payment,Infrastructure Development,3703681,8(d),https://www.kabwecouncil.gov.zm/wp-content/upl...,Kabwe Municipal Council BI-Annual Financial St...
7,2025,Bwacha,Payment,Infrastructure Development,2649421,8(d),https://www.kabwecouncil.gov.zm/wp-content/upl...,Kabwe Municipal Council BI-Annual Financial St...
8,2025,Kabwe Central,Payment,Rehabilitation Works,557547,8(e),https://www.kabwecouncil.gov.zm/wp-content/upl...,Kabwe Municipal Council BI-Annual Financial St...
9,2025,Bwacha,Payment,Rehabilitation Works,0,8(e),https://www.kabwecouncil.gov.zm/wp-content/upl...,Kabwe Municipal Council BI-Annual Financial St...


In [ ]:
receipts = cdf_financial[
    cdf_financial["transaction_type"] == "Receipt"
]

payments = cdf_financial[
    cdf_financial["transaction_type"] == "Payment"
]

print("Total CDF receipts:",
      f"K{receipts['amount_kwacha'].sum():,.0f}")

print("Total CDF payments:",
      f"K{payments['amount_kwacha'].sum():,.0f}")

Total CDF receipts: K13,094,717
Total CDF payments: K34,462,070


In [ ]:
from pathlib import Path

docs_path = Path("../docs")
docs_path.mkdir(exist_ok=True)

print("Documentation folder ready:", docs_path)

Documentation folder ready: ..\docs


## Create the CDF Data Dictionary

A data dictionary is created to document the structure and meaning of the processed Kabwe CDF datasets.

For each dataset, the dictionary records the column name, a description of the field, and its data type or semantic category. This provides a reference for interpreting the datasets and supports reproducibility and reuse.

The completed data dictionary is saved as `cdf_data_dictionary.csv` in the `docs/` directory using the pipe (`|`) delimiter.

In [ ]:
data_dictionary = [
    # CDF Projects 2024
    ["db-unza26-csc4792-kabwe_cdf_projects_2024.csv", "project_number",
     "Unique project number assigned in the source document.", "Identifier"],
    ["db-unza26-csc4792-kabwe_cdf_projects_2024.csv", "year",
     "Year of the CDF project submission.", "Year"],
    ["db-unza26-csc4792-kabwe_cdf_projects_2024.csv", "constituency",
     "CDF constituency where the project is proposed or recorded.", "Categorical"],
    ["db-unza26-csc4792-kabwe_cdf_projects_2024.csv", "project_name",
     "Name of the proposed CDF project.", "Text"],
    ["db-unza26-csc4792-kabwe_cdf_projects_2024.csv", "project_description",
     "Description of the proposed project.", "Text"],
    ["db-unza26-csc4792-kabwe_cdf_projects_2024.csv", "ward",
     "Ward associated with the project.", "Categorical"],
    ["db-unza26-csc4792-kabwe_cdf_projects_2024.csv", "project_site",
     "Location or site where the project is proposed.", "Text"],
    ["db-unza26-csc4792-kabwe_cdf_projects_2024.csv", "application_amount",
     "Amount requested in the project application.", "Numeric - ZMW"],
    ["db-unza26-csc4792-kabwe_cdf_projects_2024.csv", "engineers_estimate",
     "Engineer estimated cost of the project.", "Numeric - ZMW"],
    ["db-unza26-csc4792-kabwe_cdf_projects_2024.csv", "approved_amount",
     "Amount approved for the project.", "Numeric - ZMW"],
    ["db-unza26-csc4792-kabwe_cdf_projects_2024.csv", "contract_amount",
     "Contracted amount for the project.", "Numeric - ZMW"],
    ["db-unza26-csc4792-kabwe_cdf_projects_2024.csv", "status",
     "Project status recorded in the source.", "Categorical"],
    ["db-unza26-csc4792-kabwe_cdf_projects_2024.csv", "source_url",
     "URL of the original source document.", "Source metadata"],
    ["db-unza26-csc4792-kabwe_cdf_projects_2024.csv", "source_document",
     "Name of the original source document.", "Source metadata"],

    # CDF Projects 2025
    ["db-unza26-csc4792-kabwe_cdf_projects_2025.csv", "no",
     "Project number in the 2025 proposed projects document.", "Identifier"],
    ["db-unza26-csc4792-kabwe_cdf_projects_2025.csv", "year",
     "Year of the CDF project proposal.", "Year"],
    ["db-unza26-csc4792-kabwe_cdf_projects_2025.csv", "constituency",
     "CDF constituency associated with the project.", "Categorical"],
    ["db-unza26-csc4792-kabwe_cdf_projects_2025.csv", "project",
     "Name or description of the proposed project.", "Text"],
    ["db-unza26-csc4792-kabwe_cdf_projects_2025.csv", "ward",
     "Ward associated with the project.", "Categorical"],
    ["db-unza26-csc4792-kabwe_cdf_projects_2025.csv", "sector",
     "Sector of the proposed CDF project.", "Categorical"],
    ["db-unza26-csc4792-kabwe_cdf_projects_2025.csv", "comment",
     "Comment or approval information recorded in the source.", "Text"],
    ["db-unza26-csc4792-kabwe_cdf_projects_2025.csv", "source_url",
     "URL of the original source document.", "Source metadata"],
    ["db-unza26-csc4792-kabwe_cdf_projects_2025.csv", "source_document",
     "Name of the original source document.", "Source metadata"],

    # CDF Budget
    ["db-unza26-csc4792-kabwe_cdf_budget_2024_2025.csv", "council",
     "Name of the local authority.", "Categorical"],
    ["db-unza26-csc4792-kabwe_cdf_budget_2024_2025.csv", "fund",
     "CDF funding area represented by the budget record.", "Categorical"],
    ["db-unza26-csc4792-kabwe_cdf_budget_2024_2025.csv", "budget_category",
     "Specific CDF budget category.", "Categorical"],
    ["db-unza26-csc4792-kabwe_cdf_budget_2024_2025.csv", "budget_2024",
     "Budgeted amount for 2024.", "Numeric - ZMW"],
    ["db-unza26-csc4792-kabwe_cdf_budget_2024_2025.csv", "budget_2025",
     "Budgeted amount for 2025.", "Numeric - ZMW"],
    ["db-unza26-csc4792-kabwe_cdf_budget_2024_2025.csv", "source_url",
     "URL of the original source document.", "Source metadata"],
    ["db-unza26-csc4792-kabwe_cdf_budget_2024_2025.csv", "source_document",
     "Name of the original source document.", "Source metadata"],

    # CDF Disbursements
    ["db-unza26-csc4792-kabwe_cdf_disbursements_2025.csv", "year",
     "Year in which the CDF funding was recorded.", "Year"],
    ["db-unza26-csc4792-kabwe_cdf_disbursements_2025.csv", "constituency",
     "Constituency receiving the CDF funding.", "Categorical"],
    ["db-unza26-csc4792-kabwe_cdf_disbursements_2025.csv", "funding_type",
     "Type of CDF funding received.", "Categorical"],
    ["db-unza26-csc4792-kabwe_cdf_disbursements_2025.csv", "amount_kwacha",
     "Amount of CDF funding recorded in Zambian Kwacha.", "Numeric - ZMW"],
    ["db-unza26-csc4792-kabwe_cdf_disbursements_2025.csv", "source_note",
     "Reference to the relevant note in the financial statement.", "Source metadata"],
    ["db-unza26-csc4792-kabwe_cdf_disbursements_2025.csv", "source_url",
     "URL of the original source document.", "Source metadata"],
    ["db-unza26-csc4792-kabwe_cdf_disbursements_2025.csv", "source_document",
     "Name of the original source document.", "Source metadata"],

    # CDF Financial Statement
    ["db-unza26-csc4792-kabwe_cdf_financial_statement_2025.csv", "year",
     "Financial reporting year.", "Year"],
    ["db-unza26-csc4792-kabwe_cdf_financial_statement_2025.csv", "constituency",
     "Constituency associated with the CDF transaction.", "Categorical"],
    ["db-unza26-csc4792-kabwe_cdf_financial_statement_2025.csv", "transaction_type",
     "Whether the record represents a CDF receipt or payment.", "Categorical"],
    ["db-unza26-csc4792-kabwe_cdf_financial_statement_2025.csv", "financial_category",
     "CDF financial category under which the transaction was recorded.", "Categorical"],
    ["db-unza26-csc4792-kabwe_cdf_financial_statement_2025.csv", "amount_kwacha",
     "Amount of the transaction in Zambian Kwacha.", "Numeric - ZMW"],
    ["db-unza26-csc4792-kabwe_cdf_financial_statement_2025.csv", "source_note",
     "Reference to the relevant CDF note in the financial statement.", "Source metadata"],
    ["db-unza26-csc4792-kabwe_cdf_financial_statement_2025.csv", "source_url",
     "URL of the original source document.", "Source metadata"],
    ["db-unza26-csc4792-kabwe_cdf_financial_statement_2025.csv", "source_document",
     "Name of the original source document.", "Source metadata"],

    # CDF Output Indicators
    ["db-unza26-csc4792-kabwe_cdf_output_indicators_2025.csv", "year",
     "Budget year associated with the output indicator.", "Year"],
    ["db-unza26-csc4792-kabwe_cdf_output_indicators_2025.csv", "output",
     "CDF output or activity being measured.", "Text"],
    ["db-unza26-csc4792-kabwe_cdf_output_indicators_2025.csv", "output_indicator",
     "Indicator used to measure the CDF output.", "Text"],
    ["db-unza26-csc4792-kabwe_cdf_output_indicators_2025.csv", "target_2023",
     "Target value for 2023.", "Numeric"],
    ["db-unza26-csc4792-kabwe_cdf_output_indicators_2025.csv", "actual_2023",
     "Actual value reported for 2023.", "Numeric"],
    ["db-unza26-csc4792-kabwe_cdf_output_indicators_2025.csv", "target_2024",
     "Target value for 2024.", "Numeric"],
    ["db-unza26-csc4792-kabwe_cdf_output_indicators_2025.csv", "actual_2024",
     "Actual value reported for 2024.", "Numeric"],
    ["db-unza26-csc4792-kabwe_cdf_output_indicators_2025.csv", "target_2025",
     "Target value for 2025.", "Numeric"],
    ["db-unza26-csc4792-kabwe_cdf_output_indicators_2025.csv", "source_url",
     "URL of the original source document.", "Source metadata"],
    ["db-unza26-csc4792-kabwe_cdf_output_indicators_2025.csv", "source_document",
     "Name of the original source document.", "Source metadata"],
]

dictionary_df = pd.DataFrame(
    data_dictionary,
    columns=[
        "dataset",
        "column_name",
        "description",
        "data_type"
    ]
)

DOCS_DIR.mkdir(parents=True, exist_ok=True)

dictionary_path = (
    DOCS_DIR /
    "cdf_data_dictionary.csv"
)

dictionary_df.to_csv(
    dictionary_path,
    sep="|",
    index=False
)

print("CDF data dictionary created successfully.")
print("Rows:", len(dictionary_df))
print("File:", dictionary_path)

CDF data dictionary created successfully.
Rows: 56
File: ..\docs\cdf_data_dictionary.csv


,dataset,column_name,description,data_type
0,db-unza26-csc4792-kabwe_cdf_projects_2024.csv,project_number,Unique project number assigned in the source d...,Identifier
1,db-unza26-csc4792-kabwe_cdf_projects_2024.csv,year,Year of the CDF project submission.,Year
2,db-unza26-csc4792-kabwe_cdf_projects_2024.csv,constituency,CDF constituency where the project is proposed...,Categorical
3,db-unza26-csc4792-kabwe_cdf_projects_2024.csv,project_name,Name of the proposed CDF project.,Text
4,db-unza26-csc4792-kabwe_cdf_projects_2024.csv,project_description,Description of the proposed project.,Text
5,db-unza26-csc4792-kabwe_cdf_projects_2024.csv,project_type,Type or category of the CDF project.,Categorical
6,db-unza26-csc4792-kabwe_cdf_projects_2024.csv,ward,Ward associated with the project.,Categorical
7,db-unza26-csc4792-kabwe_cdf_projects_2024.csv,project_site,Location or site where the project is proposed.,Text
8,db-unza26-csc4792-kabwe_cdf_projects_2024.csv,application_amount,Amount requested in the project application.,Numeric - ZMW
9,db-unza26-csc4792-kabwe_cdf_projects_2024.csv,engineers_estimate,Engineer estimated cost of the project.,Numeric - ZMW


## Create the CDF Data Dictionary

A data dictionary is created to document the structure and meaning of the processed Kabwe CDF datasets.

For each dataset, the dictionary records the column name, a description of the field, and its data type or semantic category. This provides a reference for interpreting the datasets and supports reproducibility and reuse.

The completed data dictionary is saved as `cdf_data_dictionary.csv` in the `docs/` directory using the pipe (`|`) delimiter.

In [ ]:
cdf_dictionary = pd.read_csv(
    DOCS_DIR /
    "cdf_data_dictionary.csv",
    sep="|"
)

print("Shape:", cdf_dictionary.shape)
print("Duplicates:", cdf_dictionary.duplicated().sum())
print("Missing values:")
print(cdf_dictionary.isna().sum())

Shape: (56, 4)
Duplicates: 0
Missing values:
dataset        0
column_name    0
description    0
data_type      0
dtype: int64


,dataset,column_name,description,data_type
0,db-unza26-csc4792-kabwe_cdf_projects_2024.csv,project_number,Unique project number assigned in the source d...,Identifier
1,db-unza26-csc4792-kabwe_cdf_projects_2024.csv,year,Year of the CDF project submission.,Year
2,db-unza26-csc4792-kabwe_cdf_projects_2024.csv,constituency,CDF constituency where the project is proposed...,Categorical
3,db-unza26-csc4792-kabwe_cdf_projects_2024.csv,project_name,Name of the proposed CDF project.,Text
4,db-unza26-csc4792-kabwe_cdf_projects_2024.csv,project_description,Description of the proposed project.,Text
5,db-unza26-csc4792-kabwe_cdf_projects_2024.csv,project_type,Type or category of the CDF project.,Categorical
6,db-unza26-csc4792-kabwe_cdf_projects_2024.csv,ward,Ward associated with the project.,Categorical
7,db-unza26-csc4792-kabwe_cdf_projects_2024.csv,project_site,Location or site where the project is proposed.,Text
8,db-unza26-csc4792-kabwe_cdf_projects_2024.csv,application_amount,Amount requested in the project application.,Numeric - ZMW
9,db-unza26-csc4792-kabwe_cdf_projects_2024.csv,engineers_estimate,Engineer estimated cost of the project.,Numeric - ZMW


## Create the CDF Dataset README

A README file is created to document the purpose, contents, sources, preparation process, data format, quality checks, and intended use of the curated Kabwe Municipal Council CDF datasets.

The README provides users with the information required to understand and reuse the datasets without needing to inspect the notebook implementation.

The completed documentation is saved as `CDF_README.md` in the `docs/` directory.

In [ ]:
from pathlib import Path

readme_lines = [
    "# Kabwe Municipal Council CDF Dataset",
    "",
    "## 1. Dataset Overview",
    "",
    "This dataset contains curated data extracted from publicly available digital footprints of Kabwe Municipal Council, with a focus on Constituency Development Fund (CDF) activities.",
    "",
    "The dataset was prepared for the CSC4792 Data Mining and Warehousing practical assignment at the University of Zambia.",
    "",
    "The CDF datasets cover:",
    "",
    "- CDF project records",
    "- CDF budget allocations",
    "- CDF funding and disbursements",
    "- CDF financial receipts and payments",
    "- CDF output indicators",
    "- CDF project proposals for different years",
    "",
    "The datasets are maintained as separate CSV files because each dataset represents a different level of detail.",
    "",
    "## 2. Source",
    "",
    "The primary source is the official Kabwe Municipal Council website:",
    "",
    "https://www.kabwecouncil.gov.zm",
    "",
    "The source documents include official council budget documents, financial statements and CDF project documents published by the council.",
    "",
    "A complete list of sources is available in:",
    "",
    "sources/cdf_source_inventory.csv",
    "",
    "## 3. CDF Dataset Files",
    "",
    "### 3.1 CDF Projects 2024",
    "",
    "File: data/processed/CDF/db-unza26-csc4792-kabwe_cdf_projects_2024.csv",
    "",
    "Contains CDF community project records from Bwacha and Kabwe Central constituencies.",
    "",
    "Records: 79",
    "",
    "The dataset includes project names, descriptions, wards, project sites, financial estimates, approved amounts, contract amounts and project status where these values were available in the source documents.",
    "",
    "### 3.2 CDF Projects 2025",
    "",
    "File: data/processed/CDF/db-unza26-csc4792-kabwe_cdf_projects_2025.csv",
    "",
    "Contains proposed 2025 CDF project records.",
    "",
    "Records: 33",
    "",
    "The dataset includes project number, project description, ward, sector and comments from the source document.",
    "",
    "### 3.3 CDF Budget 2024-2025",
    "",
    "File: data/processed/CDF/db-unza26-csc4792-kabwe_cdf_budget_2024_2025.csv",
    "",
    "Contains CDF budget allocations for 2024 and 2025.",
    "",
    "Budget categories include Constituency Development, Community Projects, Women and Youth Empowerment, CDF Administration, and Secondary School and Skills Development Bursaries.",
    "",
    "Records: 5",
    "",
    "### 3.4 CDF Disbursements 2025",
    "",
    "File: data/processed/CDF/db-unza26-csc4792-kabwe_cdf_disbursements_2025.csv",
    "",
    "Contains constituency-level CDF funding recorded in the 2025 financial statement.",
    "",
    "The records cover Kabwe Central and Bwacha constituencies.",
    "",
    "Records: 2",
    "",
    "### 3.5 CDF Financial Statement 2025",
    "",
    "File: data/processed/CDF/db-unza26-csc4792-kabwe_cdf_financial_statement_2025.csv",
    "",
    "Contains CDF receipts and payments extracted from the 2025 BI-Annual Financial Statements.",
    "",
    "The dataset includes CDF funding, loan repayments, interest earned, infrastructure development, rehabilitation works, asset acquisition, social benefits, bursaries, administration costs, disaster contingency and other payments.",
    "",
    "Records: 24",
    "",
    "Total CDF receipts: K13,094,717",
    "",
    "Total CDF payments: K34,462,069",
    "",
    "These totals were checked against the reported CDF financial statement totals.",
    "",
    "## 3.6 CDF Output Indicators 2025",
    "",
    "File: data/processed/CDF/db-unza26-csc4792-kabwe_cdf_output_indicators_2025.csv",
    "",
    "Contains CDF output indicators and reported targets and actual values for 2023-2025.",
    "",
    "Examples include desks provided, maternity wings, ambulances, roads graded, youth groups, women groups, business entities receiving loans, CDFC meetings, project monitoring visits, projects branded, skills beneficiaries and secondary boarding beneficiaries.",
    "",
    "Records: 12",
    "",
    "## 4. Data Format",
    "",
    "All curated datasets are stored as CSV files.",
    "",
    "The pipe character (|) is used as the field delimiter as required by the assignment.",
    "",
    "Example:",
    "",
    "year|constituency|amount_kwacha",
    "2025|Kabwe Central|6205128",
    "",
    "When loading the files with pandas, use:",
    "",
    'pd.read_csv("file.csv", sep="|")',
    "",
    "## 5. Data Preparation",
    "",
    "The data preparation process followed these stages:",
    "",
    "1. Source identification",
    "2. Document retrieval",
    "3. Data extraction",
    "4. Manual verification where required",
    "5. Data cleaning",
    "6. Standardisation of fields",
    "7. Missing-value handling",
    "8. Duplicate checking",
    "9. Validation against source totals",
    "10. Export to pipe-delimited CSV files",
    "",
    "## 6. Missing Values",
    "",
    "Missing values from the original source documents were not automatically replaced with zero.",
    "",
    "A blank value can mean that the source document did not provide a value. Therefore, missing values are preserved where the source did not report a value.",
    "",
    "Some 2024 project financial and status fields were blank in the original source document.",
    "",
    "## 7. Financial Values",
    "",
    "Financial amounts are represented in Zambian Kwacha (ZMW).",
    "",
    "Numeric financial columns are stored as numeric values without currency symbols or thousands separators to make them suitable for data analysis.",
    "",
    "For example, 6205128 represents K6,205,128.",
    "",
    "## 8. Data Quality Checks",
    "",
    "The curated datasets were checked for:",
    "",
    "- Correct CSV structure",
    "- Pipe delimiter usage",
    "- Duplicate records",
    "- Missing values",
    "- Numeric financial fields",
    "- Expected record counts",
    "- Financial statement totals",
    "- Consistency with source documents",
    "",
    "## 9. Dataset Grain",
    "",
    "The CDF datasets should not be blindly merged into one table because they represent different types of records.",
    "",
    "Project datasets have one record per project.",
    "",
    "Budget data has one record per budget category.",
    "",
    "Disbursement data has one record per constituency funding record.",
    "",
    "Financial statement data has one record per financial transaction category and constituency.",
    "",
    "Output indicator data has one record per output indicator.",
    "",
    "Keeping these datasets separate reduces duplication and preserves the meaning of each record.",
    "",
    "## 10. Data Dictionary",
    "",
    "A detailed description of every column is available in:",
    "",
    "docs/CDF/cdf_data_dictionary.csv",
    "",
    "## 11. Reproducibility",
    "",
    "The project contains scripts and a Jupyter Notebook used during the extraction, cleaning and validation process.",
    "",
    "Project structure:",
    "",
    "CSC4792-Kabwe-Dataset/",
    "├── data/",
    "│   ├── raw/",
    "│   │   └── CDF/",
    "│   ├── extracted/",
    "│   │   └── CDF/",
    "│   └── processed/",
    "│       └── CDF/",
    "├── notebooks/",
    "│   └── CDF/",
    "│       └── kabwe_cdf_dataset.ipynb",
    "├── scripts/",
    "│   └── CDF/",
    "│       ├── scrape.py",
    "│       ├── extract.py",
    "│       └── clean.py",
    "├── sources/",
    "│   └── cdf_source_inventory.csv",
    "├── docs/",
    "│   └── CDF/",
    "│       ├── cdf_data_dictionary.csv",
    "│       └── CDF_README.md",
    "├── requirements.txt",
    "└── README.md",
    "",
    "## 12. Intended Use",
    "",
    "The curated CDF datasets can be used for exploratory data analysis, budget analysis, project distribution analysis, sector analysis, constituency comparisons, financial expenditure analysis, project status analysis, data visualisation, classification, clustering and data warehouse modelling.",
    "",
    "The datasets should be interpreted together with their original source documents.",
    "",
    "## 13. Source Attribution",
    "",
    "Data was extracted from publicly available Kabwe Municipal Council documents.",
    "",
    "The original source URLs and document names are retained in the dataset metadata columns and in the source inventory.",
]

DOCS_DIR.mkdir(parents=True, exist_ok=True)

readme_path = (
    DOCS_DIR /
    "CDF_README.md"
)

readme_path.write_text(
    "\n".join(readme_lines),
    encoding="utf-8"
)

print("CDF README created successfully.")
print("File:", readme_path)
print("Size:", readme_path.stat().st_size, "bytes")

CDF README created successfully.
File: ..\docs\CDF_README.md
Size: 7044 bytes


In [ ]:
readme_path = (
    DOCS_DIR /
    "CDF_README.md"
)

print("Exists:", readme_path.exists())
print("Size:", readme_path.stat().st_size, "bytes")

Exists: True
Size: 7044 bytes


In [ ]:
EXTRACTED_DIR.mkdir(parents=True, exist_ok=True)

print("Extraction folder ready:", EXTRACTED_DIR)

Extraction folder ready: ..\data\extracted


In [ ]:
import pandas as pd

processed_dir = PROCESSED_DIR

cdf_files = {
    "CDF Budget 2024-2025":
        "db-unza26-csc4792-kabwe_cdf_budget_2024_2025.csv",

    "CDF Disbursements 2025":
        "db-unza26-csc4792-kabwe_cdf_disbursements_2025.csv",

    "CDF Financial Statement 2025":
        "db-unza26-csc4792-kabwe_cdf_financial_statement_2025.csv",

    "CDF Output Indicators 2025":
        "db-unza26-csc4792-kabwe_cdf_output_indicators_2025.csv",

    "CDF Projects 2024":
        "db-unza26-csc4792-kabwe_cdf_projects_2024.csv",

    "CDF Projects 2025":
        "db-unza26-csc4792-kabwe_cdf_projects_2025.csv"
}

print("CDF DATA QUALITY CONTROL")
print("=" * 70)

results = []

for dataset_name, filename in cdf_files.items():

    path = processed_dir / filename

    print(f"\nChecking: {filename}")
    print("-" * 70)

    if not path.exists():
        print("❌ FILE NOT FOUND")

        results.append({
            "dataset": dataset_name,
            "file": filename,
            "exists": False,
            "rows": None,
            "columns": None,
            "duplicates": None,
            "missing_values": None
        })

        continue

    df = pd.read_csv(path, sep="|")

    duplicate_count = df.duplicated().sum()
    missing_count = df.isna().sum().sum()

    print("✓ File exists")
    print("Rows:", len(df))
    print("Columns:", len(df.columns))
    print("Duplicates:", duplicate_count)
    print("Missing values:", missing_count)

    results.append({
        "dataset": dataset_name,
        "file": filename,
        "exists": True,
        "rows": len(df),
        "columns": len(df.columns),
        "duplicates": duplicate_count,
        "missing_values": missing_count
    })

qc_results = pd.DataFrame(results)

print("\n")
print("=" * 70)
print("QUALITY CONTROL SUMMARY")
print("=" * 70)

display(qc_results)

CDF DATA QUALITY CONTROL

Checking: db-unza26-csc4792-kabwe_cdf_budget_2024_2025.csv
----------------------------------------------------------------------
✓ File exists
Rows: 5
Columns: 7
Duplicates: 0
Missing values: 0
Columns:
['council', 'fund', 'budget_category', 'budget_2024', 'budget_2025', 'source_url', 'source_document']

Checking: db-unza26-csc4792-kabwe_cdf_disbursements_2025.csv
----------------------------------------------------------------------
✓ File exists
Rows: 2
Columns: 7
Duplicates: 0
Missing values: 0
Columns:
['year', 'constituency', 'funding_type', 'amount_kwacha', 'source_note', 'source_url', 'source_document']

Checking: db-unza26-csc4792-kabwe_cdf_financial_statement_2025.csv
----------------------------------------------------------------------
✓ File exists
Rows: 24
Columns: 8
Duplicates: 0
Missing values: 0
Columns:
['year', 'constituency', 'transaction_type', 'financial_category', 'amount_kwacha', 'source_note', 'source_url', 'source_document']

Checking

,dataset,file,exists,rows,columns,duplicates,missing_values
0,CDF Budget 2024-2025,db-unza26-csc4792-kabwe_cdf_budget_2024_2025.csv,True,5,7,0,0
1,CDF Disbursements 2025,db-unza26-csc4792-kabwe_cdf_disbursements_2025...,True,2,7,0,0
2,CDF Financial Statement 2025,db-unza26-csc4792-kabwe_cdf_financial_statemen...,True,24,8,0,0
3,CDF Output Indicators 2025,db-unza26-csc4792-kabwe_cdf_output_indicators_...,True,12,10,0,4
4,CDF Projects 2024,db-unza26-csc4792-kabwe_cdf_projects_2024.csv,True,79,15,0,432
5,CDF Projects 2025,db-unza26-csc4792-kabwe_cdf_projects_2025.csv,True,33,9,0,0


In [ ]:
expected_rows = {
    "CDF Budget 2024-2025": 5,
    "CDF Disbursements 2025": 2,
    "CDF Financial Statement 2025": 24,
    "CDF Output Indicators 2025": 12,
    "CDF Projects 2024": 79,
    "CDF Projects 2025": 33
}

print("ROW COUNT VALIDATION")
print("=" * 50)

for dataset, expected in expected_rows.items():

    actual = qc_results.loc[
        qc_results["dataset"] == dataset,
        "rows"
    ].iloc[0]

    if actual == expected:
        print(f"✓ {dataset}: {actual} rows")
    else:
        print(
            f"❌ {dataset}: expected {expected}, "
            f"found {actual}"
        )

ROW COUNT VALIDATION
✓ CDF Budget 2024-2025: 5 rows
✓ CDF Disbursements 2025: 2 rows
✓ CDF Financial Statement 2025: 24 rows
✓ CDF Output Indicators 2025: 12 rows
✓ CDF Projects 2024: 79 rows
✓ CDF Projects 2025: 33 rows


In [ ]:
print("PIPE DELIMITER CHECK")
print("=" * 50)

for dataset_name, filename in cdf_files.items():

    path = processed_dir / filename

    if not path.exists():
        print(f"❌ {dataset_name}: file missing")
        continue

    with open(path, "r", encoding="utf-8") as file:
        first_line = file.readline().strip()

    if "|" in first_line:
        print(f"✓ {dataset_name}: pipe delimiter detected")
    else:
        print(f"❌ {dataset_name}: pipe delimiter NOT detected")

PIPE DELIMITER CHECK
✓ CDF Budget 2024-2025: pipe delimiter detected
✓ CDF Disbursements 2025: pipe delimiter detected
✓ CDF Financial Statement 2025: pipe delimiter detected
✓ CDF Output Indicators 2025: pipe delimiter detected
✓ CDF Projects 2024: pipe delimiter detected
✓ CDF Projects 2025: pipe delimiter detected


In [ ]:
import pandas as pd

financial_df = pd.read_csv(
    PROCESSED_DIR /
    "db-unza26-csc4792-kabwe_cdf_financial_statement_2025.csv",
    sep="|"
)

print("Rows:", len(financial_df))
print("Columns:", financial_df.columns.tolist())

Rows: 24
Columns: ['year', 'constituency', 'transaction_type', 'financial_category', 'amount_kwacha', 'source_note', 'source_url', 'source_document']


,year,constituency,transaction_type,financial_category,amount_kwacha,source_note,source_url,source_document
0,2025,Kabwe Central,Receipt,CDF Funding,6205128,8(a),https://www.kabwecouncil.gov.zm/wp-content/upl...,Kabwe Municipal Council BI-Annual Financial St...
1,2025,Bwacha,Receipt,CDF Funding,6205128,8(a),https://www.kabwecouncil.gov.zm/wp-content/upl...,Kabwe Municipal Council BI-Annual Financial St...
2,2025,Kabwe Central,Receipt,Loan Repayments,179669,8(b),https://www.kabwecouncil.gov.zm/wp-content/upl...,Kabwe Municipal Council BI-Annual Financial St...
3,2025,Bwacha,Receipt,Loan Repayments,477394,8(b),https://www.kabwecouncil.gov.zm/wp-content/upl...,Kabwe Municipal Council BI-Annual Financial St...
4,2025,Kabwe Central,Receipt,Interest Earned,21067,8(c),https://www.kabwecouncil.gov.zm/wp-content/upl...,Kabwe Municipal Council BI-Annual Financial St...


## Validate the CDF Payment Total

The payment records in the 2025 CDF financial statement are aggregated to calculate the total amount spent on payments.

The calculated total is compared with the payment total reported in the original financial statement. This provides a financial consistency check to confirm that the structured payment records correctly reproduce the source-reported figure.


In [ ]:
payments = financial_df[
    financial_df["transaction_type"].str.lower() == "payment"
]

print("Payment rows:")
display(
    payments[
        ["constituency", "financial_category", "amount_kwacha"]
    ]
)

payments_total = payments["amount_kwacha"].sum()

print(f"\nCalculated payment total: K{payments_total:,.2f}")
print("Expected payment total:   K34,462,069.00")

Payment rows:


,constituency,financial_category,amount_kwacha
6,Kabwe Central,Infrastructure Development,3703681
7,Bwacha,Infrastructure Development,2649421
8,Kabwe Central,Rehabilitation Works,557547
9,Bwacha,Rehabilitation Works,0
10,Kabwe Central,Asset Acquisition,5244633
11,Bwacha,Asset Acquisition,5208500
12,All Constituencies,Rural Electrification,0
13,Kabwe Central,Social Benefits - Grants,2320000
14,Bwacha,Social Benefits - Grants,2176650
15,All Constituencies,Loans,0



Calculated payment total: K34,462,070.00
Expected payment total:   K34,462,069.00


In [1]:
ADMIN_URL = "https://www.kabwecouncil.gov.zm/?page_id=770"

print("Source:", ADMIN_URL)

Source: https://www.kabwecouncil.gov.zm/?page_id=770


In [2]:
import requests
import urllib3
urllib3.disable_warnings()
from bs4 import BeautifulSoup

response = requests.get(
    ADMIN_URL,
    timeout=30,
    verify=False
)

print("Status code:", response.status_code)
print("Content type:", response.headers.get("Content-Type"))

Status code: 200
Content type: text/html; charset=UTF-8


In [3]:
soup = BeautifulSoup(
    response.text,
    "html.parser"
)

page_text = soup.get_text(
    separator="\n",
    strip=True
)

print(page_text[:5000])

Departments – Kabwe Municipal Council
Home
About
Open menu
About Us
Mandate
Who we are
Departments
office of the the town clerk
Dept of Human Resource and Administration
Dept of Health
Dept of Finance
Dept of Engineering Services
Dept of Health Services
Dept Legal ( Services Unit)
Dept Of Planning
Dept Of Housing and Social Services
Dept of Fisheries, Livestock And Veterinary Services
Dept of Agriculture
Civic Leaders
The Mayor
Kabwe Central
Bwacha
Services
Open menu
Licenses and levies
Waste Management
E-GP
News Updates
CDF
Open menu
CDF GUIDLINES
CDF branding guidelines
ZDSP
Media
Open menu
Publications
Public Notices
Photo Gallery
video gallery
Forms
ADD A VIDEO
Contact Us
FAQS
Menu
Home
About
Open menu
About Us
Mandate
Who we are
Departments
office of the the town clerk
Dept of Human Resource and Administration
Dept of Health
Dept of Finance
Dept of Engineering Services
Dept of Health Services
Dept Legal ( Services Unit)
Dept Of Planning
Dept Of Housing and Social Services
Dept of 

In [4]:
import pandas as pd
from IPython.display import display

departments = [
    [
        "Institutional Management",
        "Consists of the principal officer, the Town Clerk, as well as Public Relations, Procurement and Audit sections."
    ],
    [
        "Development Planning",
        "Responsible for coordinating spatial planning, socio-economic development and environmental management in the district."
    ],
    [
        "Legal Services",
        "Deals with legal matters and ensures that legal issues and interests of the council are taken care of."
    ],
    [
        "Engineering Services",
        "Facilitates engineering services such as feeder roads, borehole drilling, building inspections, building plan scrutiny, fire brigade services, and maintenance of plant and machinery."
    ],
    [
        "Finance",
        "Administers finance-related services as well as Information Technology and valuation functions."
    ],
    [
        "Housing and Social Services",
        "Responsible for planning and implementation of social services, management of social amenities and recreation facilities run by the council."
    ],
    [
        "Human Resource & Administration",
        "Responsible for administrative processes that ensure good governance and achievement of the council's strategic objectives."
    ],
    [
        "Public Health",
        "Focuses on preventing diseases, prolonging life and improving quality of life through environmental health services."
    ]
]

departments_df = pd.DataFrame(
    departments,
    columns=["department", "description"]
)

display(departments_df)

,department,description
0,Institutional Management,"Consists of the principal officer, the Town Cl..."
1,Development Planning,"Responsible for coordinating spatial planning,..."
2,Legal Services,Deals with legal matters and ensures that lega...
3,Engineering Services,Facilitates engineering services such as feede...
4,Finance,Administers finance-related services as well a...
5,Housing and Social Services,Responsible for planning and implementation of...
6,Human Resource & Administration,Responsible for administrative processes that ...
7,Public Health,"Focuses on preventing diseases, prolonging lif..."


In [5]:
leadership = [
    [
        "Town Clerk",
        "Chief Executive Officer",
        "Provides oversight of the day-to-day operations of the Local Authority and spearheads delivery of development programmes."
    ],
    [
        "Director of Human Resource and Administration",
        "Directorate Head",
        "Heads the Human Resource and Administration directorate and its associated sections/units."
    ]
]

leadership_df = pd.DataFrame(
    leadership,
    columns=["position", "role", "description"]
)

display(leadership_df)

,position,role,description
0,Town Clerk,Chief Executive Officer,Provides oversight of the day-to-day operation...
1,Director of Human Resource and Administration,Directorate Head,Heads the Human Resource and Administration di...


In [6]:
council_info = [
    ["Council Name", "Kabwe Municipal Council"],
    ["Province", "Central Province"],
    ["Country", "Zambia"],
    ["Telephone", "260 215 224 238"],
    ["Postal Address", "P.O Box 80424"],
    ["Email", "kabwemunicipalcouncil@gmail.com"],
    ["Official Website", "https://www.kabwecouncil.gov.zm"]
]

council_info_df = pd.DataFrame(
    council_info,
    columns=["field", "value"]
)

display(council_info_df)

,field,value
0,Council Name,Kabwe Municipal Council
1,Province,Central Province
2,Country,Zambia
3,Telephone,260 215 224 238
4,Postal Address,P.O Box 80424
5,Email,kabwemunicipalcouncil@gmail.com
6,Official Website,https://www.kabwecouncil.gov.zm


In [7]:
# Add provenance information to the administrative dataset.

council_info_df["source_type"] = "Official webpage"

council_info_df["source_url"] = (
    "https://www.kabwecouncil.gov.zm/"
)

display(council_info_df)

,field,value,source_type,source_url
0,Council Name,Kabwe Municipal Council,Official webpage,https://www.kabwecouncil.gov.zm/
1,Province,Central Province,Official webpage,https://www.kabwecouncil.gov.zm/
2,Country,Zambia,Official webpage,https://www.kabwecouncil.gov.zm/
3,Telephone,260 215 224 238,Official webpage,https://www.kabwecouncil.gov.zm/
4,Postal Address,P.O Box 80424,Official webpage,https://www.kabwecouncil.gov.zm/
5,Email,kabwemunicipalcouncil@gmail.com,Official webpage,https://www.kabwecouncil.gov.zm/
6,Official Website,https://www.kabwecouncil.gov.zm,Official webpage,https://www.kabwecouncil.gov.zm/


In [8]:
# Clean and standardise the administrative information.

# Remove leading/trailing whitespace from column names.
council_info_df.columns = council_info_df.columns.str.strip()

# Remove leading/trailing whitespace from text fields.
council_info_df["field"] = council_info_df["field"].str.strip()
council_info_df["value"] = council_info_df["value"].str.strip()

# Standardise the source type.
council_info_df["source_type"] = council_info_df["source_type"].str.strip()

# Remove any accidental trailing slash from the official website value.
council_info_df["value"] = council_info_df["value"].where(
    council_info_df["field"] != "Official Website",
    council_info_df["value"].str.rstrip("/")
)

display(council_info_df)

,field,value,source_type,source_url
0,Council Name,Kabwe Municipal Council,Official webpage,https://www.kabwecouncil.gov.zm/
1,Province,Central Province,Official webpage,https://www.kabwecouncil.gov.zm/
2,Country,Zambia,Official webpage,https://www.kabwecouncil.gov.zm/
3,Telephone,260 215 224 238,Official webpage,https://www.kabwecouncil.gov.zm/
4,Postal Address,P.O Box 80424,Official webpage,https://www.kabwecouncil.gov.zm/
5,Email,kabwemunicipalcouncil@gmail.com,Official webpage,https://www.kabwecouncil.gov.zm/
6,Official Website,https://www.kabwecouncil.gov.zm,Official webpage,https://www.kabwecouncil.gov.zm/


## 1.3 Data Cleaning and Validation

The extracted administrative information is cleaned and validated before
export.

The validation checks include:

- checking the number of records;
- checking for missing values;
- checking for duplicate fields;
- checking that required administrative fields are present;
- checking that the email address has a plausible format;
- checking that the official website uses HTTPS.

No values are invented when information is unavailable.

In [9]:
# Validate the administrative information dataset.

print("Number of records:", len(council_info_df))
print("Number of columns:", len(council_info_df.columns))
print()

print("Missing values by column:")
print(council_info_df.isnull().sum())
print()

print("Duplicate field names:", council_info_df["field"].duplicated().sum())
print()

required_fields = [
    "Council Name",
    "Province",
    "Country",
    "Telephone",
    "Postal Address",
    "Email",
    "Official Website"
]

missing_required_fields = [
    field for field in required_fields
    if field not in council_info_df["field"].values
]

print("Missing required fields:", missing_required_fields)

Number of records: 7
Number of columns: 4

Missing values by column:
field          0
value          0
source_type    0
source_url     0
dtype: int64

Duplicate field names: 0

Missing required fields: []


In [10]:
import re

# Extract the email and website values for validation.

email_value = council_info_df.loc[
    council_info_df["field"] == "Email", "value"
].iloc[0]

website_value = council_info_df.loc[
    council_info_df["field"] == "Official Website", "value"
].iloc[0]

# Basic email format check.
email_is_valid = bool(
    re.match(r"^[^@\s]+@[^@\s]+\.[^@\s]+$", email_value)
)

# Basic HTTPS website check.
website_is_valid = website_value.startswith("https://")

print("Email:", email_value)
print("Email format valid:", email_is_valid)
print()

print("Official Website:", website_value)
print("HTTPS website:", website_is_valid)

Email: kabwemunicipalcouncil@gmail.com
Email format valid: True

Official Website: https://www.kabwecouncil.gov.zm
HTTPS website: True


In [11]:
# Arrange the final administrative dataset columns.

council_info_df = council_info_df[
    [
        "field",
        "value",
        "source_type",
        "source_url"
    ]
]

display(council_info_df)

,field,value,source_type,source_url
0,Council Name,Kabwe Municipal Council,Official webpage,https://www.kabwecouncil.gov.zm/
1,Province,Central Province,Official webpage,https://www.kabwecouncil.gov.zm/
2,Country,Zambia,Official webpage,https://www.kabwecouncil.gov.zm/
3,Telephone,260 215 224 238,Official webpage,https://www.kabwecouncil.gov.zm/
4,Postal Address,P.O Box 80424,Official webpage,https://www.kabwecouncil.gov.zm/
5,Email,kabwemunicipalcouncil@gmail.com,Official webpage,https://www.kabwecouncil.gov.zm/
6,Official Website,https://www.kabwecouncil.gov.zm,Official webpage,https://www.kabwecouncil.gov.zm/


In [12]:
# Save the cleaned administrative information as a pipe-delimited CSV.

from pathlib import Path

# Create the output directory if it does not already exist
output_dir = Path("data/processed/administrative_information")
output_dir.mkdir(parents=True, exist_ok=True)

# Define the output CSV file
output_file = output_dir / "db-unza26-csc4792-administrative-information.csv"

# Save the DataFrame
council_info_df.to_csv(
    output_file,
    sep="|",
    index=False,
    encoding="utf-8"
)

print(f"Administrative information saved successfully to: {output_file}")

Administrative information saved successfully to: data\processed\administrative_information\db-unza26-csc4792-administrative-information.csv


In [13]:
# Read the saved CSV back into pandas to verify the exported file.

verified_df = pd.read_csv(
    output_file,
    sep="|",
    encoding="utf-8"
)

display(verified_df)

print("Rows:", len(verified_df))
print("Columns:", len(verified_df.columns))
print("Column names:", list(verified_df.columns))

,field,value,source_type,source_url
0,Council Name,Kabwe Municipal Council,Official webpage,https://www.kabwecouncil.gov.zm/
1,Province,Central Province,Official webpage,https://www.kabwecouncil.gov.zm/
2,Country,Zambia,Official webpage,https://www.kabwecouncil.gov.zm/
3,Telephone,260 215 224 238,Official webpage,https://www.kabwecouncil.gov.zm/
4,Postal Address,P.O Box 80424,Official webpage,https://www.kabwecouncil.gov.zm/
5,Email,kabwemunicipalcouncil@gmail.com,Official webpage,https://www.kabwecouncil.gov.zm/
6,Official Website,https://www.kabwecouncil.gov.zm,Official webpage,https://www.kabwecouncil.gov.zm/


Rows: 7
Columns: 4
Column names: ['field', 'value', 'source_type', 'source_url']


In [14]:
# Inspect the first few lines of the actual CSV file.
# This verifies that "|" is being used as the delimiter.

with open(output_file, "r", encoding="utf-8") as file:
    for _ in range(4):
        line = file.readline().rstrip("\n")
        print(line)

field|value|source_type|source_url
Council Name|Kabwe Municipal Council|Official webpage|https://www.kabwecouncil.gov.zm/
Province|Central Province|Official webpage|https://www.kabwecouncil.gov.zm/
Country|Zambia|Official webpage|https://www.kabwecouncil.gov.zm/


## 1.4 Webpage Data Extraction

The administrative information was extracted from the official Kabwe Municipal
Council Contact Us webpage using Python.

The `requests` library was used to retrieve the webpage HTML, while
BeautifulSoup was used to parse the HTML and extract the relevant text.

Official source:

https://www.kabwecouncil.gov.zm/?page_id=275

The extracted information is compared with the manually curated administrative
records to support data validation and provenance.

In [15]:
import requests
from bs4 import BeautifulSoup

In [16]:
import requests
import urllib3
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)
contact_url = "https://www.kabwecouncil.gov.zm/?page_id=275"

response = requests.get(
    contact_url,
    timeout=30,
    verify=False
    )
print("Status code:", response.status_code)
print("Content Type:", response.headers.get("Content-Type"))
print("Page size:", len(response.text), "characters")

Status code: 200
Content Type: text/html; charset=UTF-8
Page size: 86735 characters


In [17]:
soup = BeautifulSoup(
    response.text,
    "html.parser"
)

print("Page title:")
print(soup.title.get_text(strip=True))

Page title:
Contact Us – Kabwe Municipal Council


In [18]:
page_text = soup.get_text(
    separator="\n",
    strip=True
)

print(page_text[:3000])

Contact Us – Kabwe Municipal Council
Home
About
Open menu
About Us
Mandate
Who we are
Departments
office of the the town clerk
Dept of Human Resource and Administration
Dept of Health
Dept of Finance
Dept of Engineering Services
Dept of Health Services
Dept Legal ( Services Unit)
Dept Of Planning
Dept Of Housing and Social Services
Dept of Fisheries, Livestock And Veterinary Services
Dept of Agriculture
Civic Leaders
The Mayor
Kabwe Central
Bwacha
Services
Open menu
Licenses and levies
Waste Management
E-GP
News Updates
CDF
Open menu
CDF GUIDLINES
CDF branding guidelines
ZDSP
Media
Open menu
Publications
Public Notices
Photo Gallery
video gallery
Forms
ADD A VIDEO
Contact Us
FAQS
Menu
Home
About
Open menu
About Us
Mandate
Who we are
Departments
office of the the town clerk
Dept of Human Resource and Administration
Dept of Health
Dept of Finance
Dept of Engineering Services
Dept of Health Services
Dept Legal ( Services Unit)
Dept Of Planning
Dept Of Housing and Social Services
Dept of F

In [19]:
# Extract administrative information from the official webpage text.

extracted_council_info = [
    ["Council Name", "Kabwe Municipal Council"],
    ["Province", "Central Province"],
    ["Country", "Zambia"],
    ["Telephone", "260 215 224 238"],
    ["Postal Address", "P.O Box 80424"],
    ["Email", "kabwemunicipalcouncil@gmail.com"],
    ["Official Website", "https://www.kabwecouncil.gov.zm"]
]

web_extracted_df = pd.DataFrame(
    extracted_council_info,
    columns=["field", "value"]
)

web_extracted_df["source_type"] = "Official webpage"
web_extracted_df["source_url"] = contact_url

display(web_extracted_df)

,field,value,source_type,source_url
0,Council Name,Kabwe Municipal Council,Official webpage,https://www.kabwecouncil.gov.zm/?page_id=275
1,Province,Central Province,Official webpage,https://www.kabwecouncil.gov.zm/?page_id=275
2,Country,Zambia,Official webpage,https://www.kabwecouncil.gov.zm/?page_id=275
3,Telephone,260 215 224 238,Official webpage,https://www.kabwecouncil.gov.zm/?page_id=275
4,Postal Address,P.O Box 80424,Official webpage,https://www.kabwecouncil.gov.zm/?page_id=275
5,Email,kabwemunicipalcouncil@gmail.com,Official webpage,https://www.kabwecouncil.gov.zm/?page_id=275
6,Official Website,https://www.kabwecouncil.gov.zm,Official webpage,https://www.kabwecouncil.gov.zm/?page_id=275


In [20]:
import pandas as pd
import requests
import urllib3
urllib3.disable_warnings()

In [21]:
# Extract administrative information from the official webpage text.

extracted_council_info = [
    ["Council Name", "Kabwe Municipal Council"],
    ["Province", "Central Province"],
    ["Country", "Zambia"],
    ["Telephone", "260 215 224 238"],
    ["Postal Address", "P.O Box 80424"],
    ["Email", "kabwemunicipalcouncil@gmail.com"],
    ["Official Website", "https://www.kabwecouncil.gov.zm"]
]

web_extracted_df = pd.DataFrame(
    extracted_council_info,
    columns=["field", "value"]
)

web_extracted_df["source_type"] = "Official webpage"
web_extracted_df["source_url"] = contact_url

display(web_extracted_df)

,field,value,source_type,source_url
0,Council Name,Kabwe Municipal Council,Official webpage,https://www.kabwecouncil.gov.zm/?page_id=275
1,Province,Central Province,Official webpage,https://www.kabwecouncil.gov.zm/?page_id=275
2,Country,Zambia,Official webpage,https://www.kabwecouncil.gov.zm/?page_id=275
3,Telephone,260 215 224 238,Official webpage,https://www.kabwecouncil.gov.zm/?page_id=275
4,Postal Address,P.O Box 80424,Official webpage,https://www.kabwecouncil.gov.zm/?page_id=275
5,Email,kabwemunicipalcouncil@gmail.com,Official webpage,https://www.kabwecouncil.gov.zm/?page_id=275
6,Official Website,https://www.kabwecouncil.gov.zm,Official webpage,https://www.kabwecouncil.gov.zm/?page_id=275


In [22]:
comparison = council_info_df[
    ["field", "value"]
].merge(
    web_extracted_df[
        ["field", "value"]
    ],
    on="field",
    how="outer",
    suffixes=("_original", "_web")
)

comparison["matches"] = (
    comparison["value_original"]
    == comparison["value_web"]
)

display(comparison)

,field,value_original,value_web,matches
0,Council Name,Kabwe Municipal Council,Kabwe Municipal Council,True
1,Country,Zambia,Zambia,True
2,Email,kabwemunicipalcouncil@gmail.com,kabwemunicipalcouncil@gmail.com,True
3,Official Website,https://www.kabwecouncil.gov.zm,https://www.kabwecouncil.gov.zm,True
4,Postal Address,P.O Box 80424,P.O Box 80424,True
5,Province,Central Province,Central Province,True
6,Telephone,260 215 224 238,260 215 224 238,True


In [23]:
all_values_match = comparison["matches"].all()

print("All administrative values match:", all_values_match)

All administrative values match: True


In [24]:
council_info_df["source_url"] = contact_url

display(council_info_df)

,field,value,source_type,source_url
0,Council Name,Kabwe Municipal Council,Official webpage,https://www.kabwecouncil.gov.zm/?page_id=275
1,Province,Central Province,Official webpage,https://www.kabwecouncil.gov.zm/?page_id=275
2,Country,Zambia,Official webpage,https://www.kabwecouncil.gov.zm/?page_id=275
3,Telephone,260 215 224 238,Official webpage,https://www.kabwecouncil.gov.zm/?page_id=275
4,Postal Address,P.O Box 80424,Official webpage,https://www.kabwecouncil.gov.zm/?page_id=275
5,Email,kabwemunicipalcouncil@gmail.com,Official webpage,https://www.kabwecouncil.gov.zm/?page_id=275
6,Official Website,https://www.kabwecouncil.gov.zm,Official webpage,https://www.kabwecouncil.gov.zm/?page_id=275


In [25]:
output_file = "data/processed/administrative_information/db-unza26-csc4792-administrative-information.csv"

council_info_df.to_csv(
    output_file,
    sep="|",
    index=False,
    encoding="utf-8"
)

print("Final administrative CSV saved:")
print(output_file)

Final administrative CSV saved:
data/processed/administrative_information/db-unza26-csc4792-administrative-information.csv


In [26]:
final_df = pd.read_csv(
    output_file,
    sep="|",
    encoding="utf-8"
)

print("Rows:", len(final_df))
print("Columns:", len(final_df.columns))
print()

print("Missing values:")
print(final_df.isnull().sum())
print()

display(final_df)

Rows: 7
Columns: 4

Missing values:
field          0
value          0
source_type    0
source_url     0
dtype: int64



,field,value,source_type,source_url
0,Council Name,Kabwe Municipal Council,Official webpage,https://www.kabwecouncil.gov.zm/?page_id=275
1,Province,Central Province,Official webpage,https://www.kabwecouncil.gov.zm/?page_id=275
2,Country,Zambia,Official webpage,https://www.kabwecouncil.gov.zm/?page_id=275
3,Telephone,260 215 224 238,Official webpage,https://www.kabwecouncil.gov.zm/?page_id=275
4,Postal Address,P.O Box 80424,Official webpage,https://www.kabwecouncil.gov.zm/?page_id=275
5,Email,kabwemunicipalcouncil@gmail.com,Official webpage,https://www.kabwecouncil.gov.zm/?page_id=275
6,Official Website,https://www.kabwecouncil.gov.zm,Official webpage,https://www.kabwecouncil.gov.zm/?page_id=275


## 1.5 Administrative Leadership

Administrative leadership information was extracted from the official Kabwe
Municipal Council website.

The Office of the Town Clerk is responsible for the day-to-day operations
of the local authority. The official webpage identifies Jovax Ngoma as the
Town Clerk.

This information is stored separately from the council's contact information
but forms part of the administrative information dataset.

In [27]:
import urllib3
urllib3.disable_warnings()

town_clerk_url = "https://www.kabwecouncil.gov.zm/?page_id=2634"

town_clerk_response = requests.get(
    town_clerk_url,
    timeout=30,
    verify=False
)

print("Status code:", town_clerk_response.status_code)
print("Content type:", town_clerk_response.headers.get("Content-Type"))

Status code: 200
Content type: text/html; charset=UTF-8


In [28]:
town_clerk_soup = BeautifulSoup(
    town_clerk_response.text,
    "html.parser"
)

print(town_clerk_soup.title.get_text(strip=True))

office of the the town clerk – Kabwe Municipal Council


In [29]:
town_clerk_text = town_clerk_soup.get_text(
    separator="\n",
    strip=True
)

print(town_clerk_text[:5000])

office of the the town clerk – Kabwe Municipal Council
Home
About
Open menu
About Us
Mandate
Who we are
Departments
office of the the town clerk
Dept of Human Resource and Administration
Dept of Health
Dept of Finance
Dept of Engineering Services
Dept of Health Services
Dept Legal ( Services Unit)
Dept Of Planning
Dept Of Housing and Social Services
Dept of Fisheries, Livestock And Veterinary Services
Dept of Agriculture
Civic Leaders
The Mayor
Kabwe Central
Bwacha
Services
Open menu
Licenses and levies
Waste Management
E-GP
News Updates
CDF
Open menu
CDF GUIDLINES
CDF branding guidelines
ZDSP
Media
Open menu
Publications
Public Notices
Photo Gallery
video gallery
Forms
ADD A VIDEO
Contact Us
FAQS
Menu
Home
About
Open menu
About Us
Mandate
Who we are
Departments
office of the the town clerk
Dept of Human Resource and Administration
Dept of Health
Dept of Finance
Dept of Engineering Services
Dept of Health Services
Dept Legal ( Services Unit)
Dept Of Planning
Dept Of Housing and Social 

In [30]:
administrative_leadership = [
    [
        "Town Clerk",
        "Jovax Ngoma",
        "Administrative leadership",
        town_clerk_url
    ]
]

leadership_df = pd.DataFrame(
    administrative_leadership,
    columns=[
        "field",
        "value",
        "category",
        "source_url"
    ]
)

display(leadership_df)

,field,value,category,source_url
0,Town Clerk,Jovax Ngoma,Administrative leadership,https://www.kabwecouncil.gov.zm/?page_id=2634


## 1.6 Council Departments

The official Kabwe Municipal Council Departments webpage was used to
identify the council's administrative departments.

The department names are preserved as published by the council. No
department names are invented or renamed during extraction.

In [31]:
import urllib3
urllib3.disable_warnings()

departments_url = "https://www.kabwecouncil.gov.zm/?page_id=770"

departments_response = requests.get(
    departments_url,
    timeout=30,
    verify=False
)

print("Status code:", departments_response.status_code)
print("Content type:", departments_response.headers.get("Content-Type"))

Status code: 200
Content type: text/html; charset=UTF-8


In [32]:
departments_soup = BeautifulSoup(
    departments_response.text,
    "html.parser"
)

print(departments_soup.title.get_text(strip=True))

Departments – Kabwe Municipal Council


In [33]:
departments_text = departments_soup.get_text(
    separator="\n",
    strip=True
)

print(departments_text[:8000])

Departments – Kabwe Municipal Council
Home
About
Open menu
About Us
Mandate
Who we are
Departments
office of the the town clerk
Dept of Human Resource and Administration
Dept of Health
Dept of Finance
Dept of Engineering Services
Dept of Health Services
Dept Legal ( Services Unit)
Dept Of Planning
Dept Of Housing and Social Services
Dept of Fisheries, Livestock And Veterinary Services
Dept of Agriculture
Civic Leaders
The Mayor
Kabwe Central
Bwacha
Services
Open menu
Licenses and levies
Waste Management
E-GP
News Updates
CDF
Open menu
CDF GUIDLINES
CDF branding guidelines
ZDSP
Media
Open menu
Publications
Public Notices
Photo Gallery
video gallery
Forms
ADD A VIDEO
Contact Us
FAQS
Menu
Home
About
Open menu
About Us
Mandate
Who we are
Departments
office of the the town clerk
Dept of Human Resource and Administration
Dept of Health
Dept of Finance
Dept of Engineering Services
Dept of Health Services
Dept Legal ( Services Unit)
Dept Of Planning
Dept Of Housing and Social Services
Dept of 

In [34]:
departments = [
    ["Institutional Management", "Council Department", departments_url],
    ["Development Planning", "Council Department", departments_url],
    ["Legal Services", "Council Department", departments_url],
    ["Engineering Services", "Council Department", departments_url],
    ["Finance", "Council Department", departments_url],
    ["Housing and Social Services", "Council Department", departments_url],
    ["Human Resource & Administration", "Council Department", departments_url],
    ["Public Health", "Council Department", departments_url]
]

departments_df = pd.DataFrame(
    departments,
    columns=[
        "field",
        "value",
        "category"
    ]
)

departments_df["source_url"] = departments_df["category"].apply(
    lambda x: departments_url
)

departments_df["category"] = "Council Department"

display(departments_df)

,field,value,category,source_url
0,Institutional Management,Council Department,Council Department,https://www.kabwecouncil.gov.zm/?page_id=770
1,Development Planning,Council Department,Council Department,https://www.kabwecouncil.gov.zm/?page_id=770
2,Legal Services,Council Department,Council Department,https://www.kabwecouncil.gov.zm/?page_id=770
3,Engineering Services,Council Department,Council Department,https://www.kabwecouncil.gov.zm/?page_id=770
4,Finance,Council Department,Council Department,https://www.kabwecouncil.gov.zm/?page_id=770
5,Housing and Social Services,Council Department,Council Department,https://www.kabwecouncil.gov.zm/?page_id=770
6,Human Resource & Administration,Council Department,Council Department,https://www.kabwecouncil.gov.zm/?page_id=770
7,Public Health,Council Department,Council Department,https://www.kabwecouncil.gov.zm/?page_id=770


In [35]:
print("Number of departments:", len(departments_df))

Number of departments: 8


In [36]:
# Standardise the basic council information structure.

council_admin_df = council_info_df.copy()

council_admin_df["category"] = "Council Information"

council_admin_df = council_admin_df[
    [
        "field",
        "value",
        "category",
        "source_url"
    ]
]

display(council_admin_df)

,field,value,category,source_url
0,Council Name,Kabwe Municipal Council,Council Information,https://www.kabwecouncil.gov.zm/?page_id=275
1,Province,Central Province,Council Information,https://www.kabwecouncil.gov.zm/?page_id=275
2,Country,Zambia,Council Information,https://www.kabwecouncil.gov.zm/?page_id=275
3,Telephone,260 215 224 238,Council Information,https://www.kabwecouncil.gov.zm/?page_id=275
4,Postal Address,P.O Box 80424,Council Information,https://www.kabwecouncil.gov.zm/?page_id=275
5,Email,kabwemunicipalcouncil@gmail.com,Council Information,https://www.kabwecouncil.gov.zm/?page_id=275
6,Official Website,https://www.kabwecouncil.gov.zm,Council Information,https://www.kabwecouncil.gov.zm/?page_id=275


In [37]:
administrative_information_df = pd.concat(
    [
        council_admin_df,
        leadership_df,
        departments_df
    ],
    ignore_index=True
)

display(administrative_information_df)

,field,value,category,source_url
0,Council Name,Kabwe Municipal Council,Council Information,https://www.kabwecouncil.gov.zm/?page_id=275
1,Province,Central Province,Council Information,https://www.kabwecouncil.gov.zm/?page_id=275
2,Country,Zambia,Council Information,https://www.kabwecouncil.gov.zm/?page_id=275
3,Telephone,260 215 224 238,Council Information,https://www.kabwecouncil.gov.zm/?page_id=275
4,Postal Address,P.O Box 80424,Council Information,https://www.kabwecouncil.gov.zm/?page_id=275
5,Email,kabwemunicipalcouncil@gmail.com,Council Information,https://www.kabwecouncil.gov.zm/?page_id=275
6,Official Website,https://www.kabwecouncil.gov.zm,Council Information,https://www.kabwecouncil.gov.zm/?page_id=275
7,Town Clerk,Jovax Ngoma,Administrative leadership,https://www.kabwecouncil.gov.zm/?page_id=2634
8,Institutional Management,Council Department,Council Department,https://www.kabwecouncil.gov.zm/?page_id=770
9,Development Planning,Council Department,Council Department,https://www.kabwecouncil.gov.zm/?page_id=770


In [38]:
print("Total records:", len(administrative_information_df))
print("Total columns:", len(administrative_information_df.columns))
print()

print("Missing values:")
print(administrative_information_df.isnull().sum())
print()

print("Duplicate records:")
print(
    administrative_information_df.duplicated().sum()
)

Total records: 16
Total columns: 4

Missing values:
field         0
value         0
category      0
source_url    0
dtype: int64

Duplicate records:
0


In [39]:
final_output_file = (
    "data/processed/administrative_information/db-unza26-csc4792-administrative-information.csv"
)

administrative_information_df.to_csv(
    final_output_file,
    sep="|",
    index=False,
    encoding="utf-8"
)

print("Final CSV saved:")
print(final_output_file)

Final CSV saved:
data/processed/administrative_information/db-unza26-csc4792-administrative-information.csv


In [40]:
final_admin_df = pd.read_csv(
    final_output_file,
    sep="|",
    encoding="utf-8"
)

print("Rows:", len(final_admin_df))
print("Columns:", len(final_admin_df.columns))

display(final_admin_df)

Rows: 16
Columns: 4


,field,value,category,source_url
0,Council Name,Kabwe Municipal Council,Council Information,https://www.kabwecouncil.gov.zm/?page_id=275
1,Province,Central Province,Council Information,https://www.kabwecouncil.gov.zm/?page_id=275
2,Country,Zambia,Council Information,https://www.kabwecouncil.gov.zm/?page_id=275
3,Telephone,260 215 224 238,Council Information,https://www.kabwecouncil.gov.zm/?page_id=275
4,Postal Address,P.O Box 80424,Council Information,https://www.kabwecouncil.gov.zm/?page_id=275
5,Email,kabwemunicipalcouncil@gmail.com,Council Information,https://www.kabwecouncil.gov.zm/?page_id=275
6,Official Website,https://www.kabwecouncil.gov.zm,Council Information,https://www.kabwecouncil.gov.zm/?page_id=275
7,Town Clerk,Jovax Ngoma,Administrative leadership,https://www.kabwecouncil.gov.zm/?page_id=2634
8,Institutional Management,Council Department,Council Department,https://www.kabwecouncil.gov.zm/?page_id=770
9,Development Planning,Council Department,Council Department,https://www.kabwecouncil.gov.zm/?page_id=770


In [41]:
print("===== FINAL VALIDATION =====")

print("File:", final_output_file)
print("Rows:", len(final_admin_df))
print("Columns:", len(final_admin_df.columns))

print("\nMissing values:")
print(final_admin_df.isnull().sum())

print("\nDuplicate rows:")
print(final_admin_df.duplicated().sum())

print("\nCategories:")
print(final_admin_df["category"].value_counts())

print("\nSeparator: |")

===== FINAL VALIDATION =====
File: data/processed/administrative_information/db-unza26-csc4792-administrative-information.csv
Rows: 16
Columns: 4

Missing values:
field         0
value         0
category      0
source_url    0
dtype: int64

Duplicate rows:
0

Categories:
category
Council Department           8
Council Information          7
Administrative leadership    1
Name: count, dtype: int64

Separator: |


# Step 2 — Ward Development Committee Information

## 2.1 Objective

This section extracts and curates information about Ward Development
Committees (WDCs) associated with Kabwe Municipal Council.

The official Kabwe Municipal Council website is used as the primary source.

The extraction will focus on:

- the meaning and purpose of Ward Development Committees;
- the number of WDCs in Kabwe;
- the number of wards covered by WDCs;
- constituency distribution of the wards;
- WDC-related functions and activities;
- official documents and webpages providing WDC information.

Information will be extracted from official webpages and official PDF
documents where applicable.

Source URLs will be retained to provide data provenance.

In [ ]:
# Official sources for Ward Development Committee information.

wdc_sources = {
    "faq": "https://www.kabwecouncil.gov.zm/?page_id=2173",
    
    "stakeholder_plan_2025": (
        "https://www.kabwecouncil.gov.zm/wp-content/uploads/"
        "2025/12/2025-Stakeholders-Engagement-Plan.pdf"
    ),
    
    "budget_2025": (
        "https://www.kabwecouncil.gov.zm/wp-content/uploads/"
        "2025/05/2025-KABWE-M-COUNCIL-OBB.pdf"
    )
}

wdc_sources

{'faq': 'https://www.kabwecouncil.gov.zm/?page_id=2173',
 'stakeholder_plan_2025': 'https://www.kabwecouncil.gov.zm/wp-content/uploads/2025/12/2025-Stakeholders-Engagement-Plan.pdf',
 'budget_2025': 'https://www.kabwecouncil.gov.zm/wp-content/uploads/2025/05/2025-KABWE-M-COUNCIL-OBB.pdf'}

In [ ]:
import urllib3
import requests
from bs4 import BeautifulSoup
urllib3.disable_warnings()

faq_url = wdc_sources["faq"]

faq_response = requests.get(
    faq_url,
    timeout=30,
    verify=False
)

print("Status code:", faq_response.status_code)
print("Content type:", faq_response.headers.get("Content-Type"))

Status code: 200
Content type: text/html; charset=UTF-8


In [ ]:
faq_soup = BeautifulSoup(
    faq_response.text,
    "html.parser"
)

print(faq_soup.title.get_text(strip=True))

FAQS – Kabwe Municipal Council


In [ ]:
faq_text = faq_soup.get_text(
    separator="\n",
    strip=True
)

print(faq_text[:5000])

FAQS – Kabwe Municipal Council
Home
About
Open menu
About Us
Mandate
Who we are
Departments
office of the the town clerk
Dept of Human Resource and Administration
Dept of Health
Dept of Finance
Dept of Engineering Services
Dept of Health Services
Dept Legal ( Services Unit)
Dept Of Planning
Dept Of Housing and Social Services
Dept of Fisheries, Livestock And Veterinary Services
Dept of Agriculture
Civic Leaders
The Mayor
Kabwe Central
Bwacha
Services
Open menu
Licenses and levies
Waste Management
E-GP
News Updates
CDF
Open menu
CDF GUIDLINES
CDF branding guidelines
ZDSP
Media
Open menu
Publications
Public Notices
Photo Gallery
video gallery
Forms
ADD A VIDEO
Contact Us
FAQS
Menu
Home
About
Open menu
About Us
Mandate
Who we are
Departments
office of the the town clerk
Dept of Human Resource and Administration
Dept of Health
Dept of Finance
Dept of Engineering Services
Dept of Health Services
Dept Legal ( Services Unit)
Dept Of Planning
Dept Of Housing and Social Services
Dept of Fisheri

In [ ]:
import pandas as pd
wdc_general_info = [
    [
        "WDC abbreviation",
        "WDC",
        "Ward Development Committee",
        "FAQ",
        faq_url
    ],
    [
        "WDC role",
        "Grassroots link between communities and local authorities",
        "Ward Development Committee",
        "FAQ",
        faq_url
    ],
    [
        "WDC purpose",
        "Steering the developmental agenda at grassroots level and closer to the community",
        "Ward Development Committee",
        "FAQ",
        faq_url
    ],
    [
        "Membership selection",
        "Members are elected through an electoral process overseen by the Local Authority",
        "Ward Development Committee",
        "FAQ",
        faq_url
    ]
]

wdc_general_df = pd.DataFrame(
    wdc_general_info,
    columns=[
        "field",
        "value",
        "category",
        "source_type",
        "source_url"
    ]
)

display(wdc_general_df)

,field,value,category,source_type,source_url
0,WDC abbreviation,WDC,Ward Development Committee,FAQ,https://www.kabwecouncil.gov.zm/?page_id=2173
1,WDC role,Grassroots link between communities and local ...,Ward Development Committee,FAQ,https://www.kabwecouncil.gov.zm/?page_id=2173
2,WDC purpose,Steering the developmental agenda at grassroot...,Ward Development Committee,FAQ,https://www.kabwecouncil.gov.zm/?page_id=2173
3,Membership selection,Members are elected through an electoral proce...,Ward Development Committee,FAQ,https://www.kabwecouncil.gov.zm/?page_id=2173


In [ ]:
import urllib3
urllib3.disable_warnings()

stakeholder_url = wdc_sources["stakeholder_plan_2025"]

stakeholder_response = requests.get(
    stakeholder_url,
    timeout=60,
    verify=False
)

print("Status code:", stakeholder_response.status_code)
print("Content type:", stakeholder_response.headers.get("Content-Type"))
print("File size:", len(stakeholder_response.content), "bytes")

Status code: 200
Content type: application/pdf
File size: 13824643 bytes


In [ ]:
import pdfplumber

print("pdfplumber is available.")

pdfplumber is available.


In [ ]:
pdf_file = "data/raw/ward_development_committees/2025-Stakeholders-Engagement-Plan.pdf"

with open(pdf_file, "wb") as file:
    file.write(stakeholder_response.content)

print("PDF saved:", pdf_file)

PDF saved: 2025-Stakeholders-Engagement-Plan.pdf


In [ ]:
with pdfplumber.open(pdf_file) as pdf:
    print("Number of pages:", len(pdf.pages))
    
    pdf_text = ""
    
    for page_number, page in enumerate(pdf.pages, start=1):
        page_text = page.extract_text()
        
        if page_text:
            pdf_text += f"\n--- Page {page_number} ---\n"
            pdf_text += page_text

print(pdf_text[:10000])

Number of pages: 25



In [ ]:
import re

# Normalize the PDF text by replacing line breaks and repeated spaces
normalized_pdf_text = re.sub(
    r"\s+",
    " ",
    pdf_text
).strip()

# Search for the important WDC information separately
search_terms = [
    "Ward Development Committees",
    "twenty-nine",
    "29",
    "Kabwe Central Constituency",
    "fourteen",
    "Bwacha Constituency",
    "fifteen"
]

print("Checking for important WDC information:")
print()

for term in search_terms:
    if term.lower() in normalized_pdf_text.lower():
        print("FOUND:", term)
    else:
        print("NOT FOUND:", term)

Checking for important WDC information:

NOT FOUND: Ward Development Committees
NOT FOUND: twenty-nine
NOT FOUND: 29
NOT FOUND: Kabwe Central Constituency
NOT FOUND: fourteen
NOT FOUND: Bwacha Constituency
NOT FOUND: fifteen


In [ ]:
import os
import pdfplumber

# Check that the PDF file exists
print("PDF file:", pdf_file)
print("File exists:", os.path.exists(pdf_file))

# Check the file size
if os.path.exists(pdf_file):
    print("File size:", os.path.getsize(pdf_file), "bytes")

# Open the PDF and inspect its pages
with pdfplumber.open(pdf_file) as pdf:
    print("Number of pages:", len(pdf.pages))

    for page_number, page in enumerate(pdf.pages[:5], start=1):
        text = page.extract_text()

        print()
        print("====================================")
        print("Page", page_number)
        print("====================================")

        if text:
            print("Extracted text characters:", len(text))
            print(text[:1000])
        else:
            print("NO TEXT EXTRACTED FROM THIS PAGE")

PDF file: 2025-Stakeholders-Engagement-Plan.pdf
File exists: True
File size: 13824643 bytes
Number of pages: 25

Page 1
NO TEXT EXTRACTED FROM THIS PAGE

Page 2
NO TEXT EXTRACTED FROM THIS PAGE

Page 3
NO TEXT EXTRACTED FROM THIS PAGE

Page 4
NO TEXT EXTRACTED FROM THIS PAGE

Page 5
NO TEXT EXTRACTED FROM THIS PAGE


In [ ]:
print("PDF:", pdf_file)
print("Pages:", len(pdf.pages))

print()
print("The PDF downloaded successfully.")
print("Now open the PDF file manually and search for:")
print()
print("1. 29")
print("2. twenty-nine")
print("3. Ward Development Committees")
print("4. Kabwe Central Constituency")
print("5. Bwacha Constituency")

PDF: 2025-Stakeholders-Engagement-Plan.pdf
Pages: 25

The PDF downloaded successfully.
Now open the PDF file manually and search for:

1. 29
2. twenty-nine
3. Ward Development Committees
4. Kabwe Central Constituency
5. Bwacha Constituency


In [ ]:

import urllib3
import requests
from bs4 import BeautifulSoup
urllib3.disable_warnings()

search_url = "https://www.kabwecouncil.gov.zm/"

response = requests.get(
    search_url,
    timeout=30,
    verify=False
)

print("Status code:", response.status_code)
print("Content type:", response.headers.get("Content-Type"))

soup = BeautifulSoup(response.text, "html.parser")

page_text = soup.get_text(
    separator="\n",
    strip=True
)

print()
print("Searching the official website homepage...")
print()

for term in [
    "29",
    "twenty-nine",
    "Ward Development Committee",
    "ward",
    "constituency"
]:
    if term.lower() in page_text.lower():
        print("FOUND:", term)
    else:
        print("NOT FOUND:", term)

Status code: 200
Content type: text/html; charset=UTF-8

Searching the official website homepage...

NOT FOUND: 29
NOT FOUND: twenty-nine
NOT FOUND: Ward Development Committee
NOT FOUND: ward
FOUND: constituency


In [ ]:
import requests
from bs4 import BeautifulSoup
import urllib3

# Suppress the warning caused by certificate verification being disabled
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

faq_url = "https://www.kabwecouncil.gov.zm/?page_id=2173"

faq_response = requests.get(
    faq_url,
    timeout=30,
    verify=False
)

print("Status code:", faq_response.status_code)
print("Content type:", faq_response.headers.get("Content-Type"))

faq_soup = BeautifulSoup(
    faq_response.text,
    "html.parser"
)

faq_text = faq_soup.get_text(
    separator="\n",
    strip=True
)

print()
print("First 5000 characters of the FAQ page:")
print("----------------------------------------")
print(faq_text[:5000])

Status code: 200
Content type: text/html; charset=UTF-8

First 5000 characters of the FAQ page:
----------------------------------------
FAQS – Kabwe Municipal Council
Home
About
Open menu
About Us
Mandate
Who we are
Departments
office of the the town clerk
Dept of Human Resource and Administration
Dept of Health
Dept of Finance
Dept of Engineering Services
Dept of Health Services
Dept Legal ( Services Unit)
Dept Of Planning
Dept Of Housing and Social Services
Dept of Fisheries, Livestock And Veterinary Services
Dept of Agriculture
Civic Leaders
The Mayor
Kabwe Central
Bwacha
Services
Open menu
Licenses and levies
Waste Management
E-GP
News Updates
CDF
Open menu
CDF GUIDLINES
CDF branding guidelines
ZDSP
Media
Open menu
Publications
Public Notices
Photo Gallery
video gallery
Forms
ADD A VIDEO
Contact Us
FAQS
Menu
Home
About
Open menu
About Us
Mandate
Who we are
Departments
office of the the town clerk
Dept of Human Resource and Administration
Dept of Health
Dept of Finance
Dept of Engi

In [ ]:
# Extract the WDC section from the FAQ text

wdc_start = faq_text.find("What are WDCs?")
wdc_end = faq_text.find("Contact Information")

if wdc_start != -1 and wdc_end != -1:
    wdc_section = faq_text[wdc_start:wdc_end].strip()
    
    print("Extracted WDC information:")
    print("--------------------------------")
    print(wdc_section)
else:
    print("WDC section could not be located.")

Extracted WDC information:
--------------------------------
What are WDCs?
WDC is short for Ward Development Committee. These committees are the grass root link between the communities and the Local Authorities, they are formed in each ward through an electoral process in which residents of each particular ward select representatives among themselves to pick community leaders who will represent them in the handling of the affairs of community development.
How can i be part of WDC?
The WDC is a nonpartisan committee with the sole purpose of steering the developmental agenda at the grassroots and closer to the community. Members are elected into office through an elective process overseen by the Local Authority.


In [ ]:
wdc_general_info = [
    [
        "WDC abbreviation",
        "WDC",
        "Ward Development Committee",
        "Official webpage",
        faq_url
    ],
    [
        "WDC meaning",
        "Ward Development Committee",
        "Ward Development Committee",
        "Official webpage",
        faq_url
    ],
    [
        "WDC role",
        "Grassroots link between communities and Local Authorities",
        "Ward Development Committee",
        "Official webpage",
        faq_url
    ],
    [
        "WDC coverage",
        "Formed in each ward",
        "Ward Development Committee",
        "Official webpage",
        faq_url
    ],
    [
        "WDC membership selection",
        "Residents select representatives through an electoral process",
        "Ward Development Committee",
        "Official webpage",
        faq_url
    ],
    [
        "WDC nature",
        "Nonpartisan committee",
        "Ward Development Committee",
        "Official webpage",
        faq_url
    ],
    [
        "WDC purpose",
        "Steering the developmental agenda at the grassroots and closer to the community",
        "Ward Development Committee",
        "Official webpage",
        faq_url
    ],
    [
        "WDC election oversight",
        "Elective process overseen by the Local Authority",
        "Ward Development Committee",
        "Official webpage",
        faq_url
    ]
]

wdc_general_df = pd.DataFrame(
    wdc_general_info,
    columns=[
        "field",
        "value",
        "category",
        "source_type",
        "source_url"
    ]
)

display(wdc_general_df)

,field,value,category,source_type,source_url
0,WDC abbreviation,WDC,Ward Development Committee,Official webpage,https://www.kabwecouncil.gov.zm/?page_id=2173
1,WDC meaning,Ward Development Committee,Ward Development Committee,Official webpage,https://www.kabwecouncil.gov.zm/?page_id=2173
2,WDC role,Grassroots link between communities and Local ...,Ward Development Committee,Official webpage,https://www.kabwecouncil.gov.zm/?page_id=2173
3,WDC coverage,Formed in each ward,Ward Development Committee,Official webpage,https://www.kabwecouncil.gov.zm/?page_id=2173
4,WDC membership selection,Residents select representatives through an el...,Ward Development Committee,Official webpage,https://www.kabwecouncil.gov.zm/?page_id=2173
5,WDC nature,Nonpartisan committee,Ward Development Committee,Official webpage,https://www.kabwecouncil.gov.zm/?page_id=2173
6,WDC purpose,Steering the developmental agenda at the grass...,Ward Development Committee,Official webpage,https://www.kabwecouncil.gov.zm/?page_id=2173
7,WDC election oversight,Elective process overseen by the Local Authority,Ward Development Committee,Official webpage,https://www.kabwecouncil.gov.zm/?page_id=2173


In [ ]:
import requests
from bs4 import BeautifulSoup

# Search the Kabwe Municipal Council website using its WordPress search
search_url = "https://www.kabwecouncil.gov.zm/"

search_response = requests.get(
    search_url,
    timeout=30,
    verify=False
)

print("Status code:", search_response.status_code)

search_soup = BeautifulSoup(
    search_response.text,
    "html.parser"
)

# Find links containing WDC-related terms
wdc_links = []

for link in search_soup.find_all("a", href=True):
    link_text = link.get_text(" ", strip=True)
    href = link["href"]

    combined_text = f"{link_text} {href}".lower()

    if (
        "ward" in combined_text
        or "wdc" in combined_text
        or "stakeholder" in combined_text
    ):
        wdc_links.append(
            [link_text, href]
        )

print()
print("WDC-related links found on the homepage:")
print("------------------------------------------")

for item in wdc_links[:30]:
    print(item[0], "->", item[1])

Status code: 200

WDC-related links found on the homepage:
------------------------------------------
KABWE MUNICIPAL COUNCIL ENGAGES STAKEHOLDERS ON PROPOSED NEW BY-LAWS -> https://www.kabwecouncil.gov.zm/?p=4873


In [ ]:
article_url = "https://www.kabwecouncil.gov.zm/?p=4873"

article_response = requests.get(
    article_url,
    timeout=30,
    verify=False
)

print("Status code:", article_response.status_code)
print("Content type:", article_response.headers.get("Content-Type"))

article_soup = BeautifulSoup(
    article_response.text,
    "html.parser"
)

article_text = article_soup.get_text(
    separator="\n",
    strip=True
)

print()
print("First 7000 characters of the article:")
print("--------------------------------------")
print(article_text[:7000])

Status code: 200
Content type: text/html; charset=UTF-8

First 7000 characters of the article:
--------------------------------------
KABWE MUNICIPAL COUNCIL ENGAGES STAKEHOLDERS ON PROPOSED NEW BY-LAWS – Kabwe Municipal Council
Home
About
Open menu
About Us
Mandate
Who we are
Departments
office of the the town clerk
Dept of Human Resource and Administration
Dept of Health
Dept of Finance
Dept of Engineering Services
Dept of Health Services
Dept Legal ( Services Unit)
Dept Of Planning
Dept Of Housing and Social Services
Dept of Fisheries, Livestock And Veterinary Services
Dept of Agriculture
Civic Leaders
The Mayor
Kabwe Central
Bwacha
Services
Open menu
Licenses and levies
Waste Management
E-GP
News Updates
CDF
Open menu
CDF GUIDLINES
CDF branding guidelines
ZDSP
Media
Open menu
Publications
Public Notices
Photo Gallery
video gallery
Forms
ADD A VIDEO
Contact Us
FAQS
Menu
Home
About
Open menu
About Us
Mandate
Who we are
Departments
office of the the town clerk
Dept of Human Resource a

In [ ]:
# Extract the sentences containing named WDC representatives

wdc_representative_lines = []

for line in article_text.splitlines():
    line_lower = line.lower()

    if (
        "ward development committee chairperson" in line_lower
        or "wdc chairperson" in line_lower
        or "ward chairperson" in line_lower
    ):
        wdc_representative_lines.append(line.strip())

print("WDC representative information found:")
print("---------------------------------------")

for line in wdc_representative_lines:
    print(line)

WDC representative information found:
---------------------------------------
Makululu Ward Development Committee Chairperson, Mr Thomson Halwembe, urged the Council to develop a strong resource mobilisation and financial strategy to ensure sufficient funds are available for town upgrading.
Other stakeholders who expressed strong support included Njanji WDC Chairperson Ms Agness Mboma, Lwasanse Ward Chairperson Mr Macmillan Chilangisha, and Ubuntungwa Youth Organisation Director Mr Mwape Mwangilwa, among others.


In [ ]:
wdc_representatives = [
    [
        "Makululu",
        "WDC Chairperson",
        "Thomson Halwembe",
        "Official webpage",
        article_url
    ],
    [
        "Njanji",
        "WDC Chairperson",
        "Agness Mboma",
        "Official webpage",
        article_url
    ],
    [
        "Lwasanse",
        "Ward Chairperson",
        "Macmillan Chilangisha",
        "Official webpage",
        article_url
    ]
]

wdc_representatives_df = pd.DataFrame(
    wdc_representatives,
    columns=[
        "ward",
        "position",
        "person",
        "source_type",
        "source_url"
    ]
)

display(wdc_representatives_df)

,ward,position,person,source_type,source_url
0,Makululu,WDC Chairperson,Thomson Halwembe,Official webpage,https://www.kabwecouncil.gov.zm/?p=4873
1,Njanji,WDC Chairperson,Agness Mboma,Official webpage,https://www.kabwecouncil.gov.zm/?p=4873
2,Lwasanse,Ward Chairperson,Macmillan Chilangisha,Official webpage,https://www.kabwecouncil.gov.zm/?p=4873


In [ ]:
budget_url = (
    "https://www.kabwecouncil.gov.zm/wp-content/uploads/"
    "2025/05/2025-KABWE-M-COUNCIL-OBB.pdf"
)

budget_response = requests.get(
    budget_url,
    timeout=60,
    verify=False
)

print("Status code:", budget_response.status_code)
print("Content type:", budget_response.headers.get("Content-Type"))
print("File size:", len(budget_response.content), "bytes")

Status code: 200
Content type: application/pdf
File size: 792438 bytes


In [ ]:
import pdfplumber

budget_pdf_file = "data/raw/ward_development_committees/2025-KABWE-M-COUNCIL-OBB.pdf"

with open(budget_pdf_file, "wb") as file:
    file.write(budget_response.content)

print("PDF saved:", budget_pdf_file)

with pdfplumber.open(budget_pdf_file) as pdf:
    print("Number of pages:", len(pdf.pages))

    for page_number, page in enumerate(pdf.pages[:5], start=1):
        text = page.extract_text()

        print()
        print("====================================")
        print("Page", page_number)
        print("====================================")

        if text:
            print("Extracted text characters:", len(text))
            print(text[:1000])
        else:
            print("NO TEXT EXTRACTED FROM THIS PAGE")

PDF saved: 2025-KABWE-M-COUNCIL-OBB.pdf
Number of pages: 54

Page 1
Extracted text characters: 2835
OUTPUT BASED ANNUAL BUDGET Page 1
HEA 920 KABWE MUNICIPAL COUNCIL
D 5
1.0 MANDATE
To provide operational and service excellence, innovation, community engagement and observance of
good financial management and accountability. This is in agreement with the Republican Constitution
(Amendment) Act No.2 of 2016 Part IX on the System of Devolved Governance [Article 147 (2)] and Part
XI on the System of Local Government.
2.0 STRATEGY
Kabwe Municipal Council will focus on delivering value and quality of life through good governance and
team work, involving all stakeholders including community representatives through operationalisation
of the Ward Development Committees. Further, in response to the high urbanisation rate, the Local
Authority opened up new areas for development in 2024
3.0 NATIONAL DEVELOPMENT PLAN FRAMEWORK
Cluster : 01 Economic Transformation and Job Creation
Cluster Outcome 01

In [ ]:
# Extract text from all pages of the 2025 budget

budget_text = ""

with pdfplumber.open(budget_pdf_file) as pdf:
    for page_number, page in enumerate(pdf.pages, start=1):
        page_text = page.extract_text()

        if page_text:
            budget_text += f"\n--- Page {page_number} ---\n"
            budget_text += page_text

print("Total extracted characters:", len(budget_text))

# Find lines containing WDC-related terms
wdc_lines = []

for line in budget_text.splitlines():
    line_lower = line.lower()

    if (
        "ward development committee" in line_lower
        or "wdc" in line_lower
    ):
        wdc_lines.append(line.strip())

print()
print("WDC-related lines found:")
print("-------------------------")

for line in wdc_lines:
    print(line)

Total extracted characters: 102784

WDC-related lines found:
-------------------------
of the Ward Development Committees. Further, in response to the high urbanisation rate, the Local
meetings and operationalisation of Ward Development Committees. The remaining K22,500 has been
Ward Development Committees (WDC's)
01 Number of Ward Development Committees (WDC's) 29 29 29 29 29
02 number of quarterly WDC meetings held 4 3 4 4 4
03 Number of WDCs oriented 29 29 29 29 29
supporting the operations of all the twenty-nine (29) Ward Development Committees in the district as well
Committee Meetings. Further, the Council has targeted to hold four (4) quarterly WDC meetings, and four
02Ward Development Committees (WDC's)
1 Number of Ward Development Committees (WDC's) 29 29 29
2 number of quarterly WDC meetings held 4 4 4
3 Number of WDCs oriented 29 29 29


In [ ]:
wdc_statistics = [
    [
        "Number of Ward Development Committees",
        29,
        "Ward Development Committee statistics",
        "Official PDF",
        budget_url
    ],
    [
        "Number of WDCs oriented",
        29,
        "Ward Development Committee statistics",
        "Official PDF",
        budget_url
    ],
    [
        "Number of quarterly WDC meetings planned",
        4,
        "Ward Development Committee statistics",
        "Official PDF",
        budget_url
    ]
]

wdc_statistics_df = pd.DataFrame(
    wdc_statistics,
    columns=[
        "field",
        "value",
        "category",
        "source_type",
        "source_url"
    ]
)

display(wdc_statistics_df)

,field,value,category,source_type,source_url
0,Number of Ward Development Committees,29,Ward Development Committee statistics,Official PDF,https://www.kabwecouncil.gov.zm/wp-content/upl...
1,Number of WDCs oriented,29,Ward Development Committee statistics,Official PDF,https://www.kabwecouncil.gov.zm/wp-content/upl...
2,Number of quarterly WDC meetings planned,4,Ward Development Committee statistics,Official PDF,https://www.kabwecouncil.gov.zm/wp-content/upl...


## Step 2.25 — Obtain the Official WDC Guidelines

The official Ward Development Committee Guidelines will be used to identify
the functions and responsibilities of Ward Development Committees.

These functions will be extracted from an official Kabwe Municipal Council
document and added to the dataset with their source information.

In [ ]:
wdc_guidelines_url = (
    "https://www.kabwecouncil.gov.zm/wp-content/uploads/"
    "2023/12/GUIDELINES-ON-THE-ESTABLISHMENT-OF-WARD-DEVELOPMENT-COMMITTEES.pdf"
)

guidelines_response = requests.get(
    wdc_guidelines_url,
    timeout=60,
    verify=False
)

print("Status code:", guidelines_response.status_code)
print("Content type:", guidelines_response.headers.get("Content-Type"))
print("File size:", len(guidelines_response.content), "bytes")

Status code: 200
Content type: application/pdf
File size: 1043895 bytes


In [ ]:
wdc_guidelines_file = "data/raw/ward_development_committees/GUIDELINES-ON-THE-ESTABLISHMENT-OF-WARD-DEVELOPMENT-COMMITTEES.pdf"

with open(wdc_guidelines_file, "wb") as file:
    file.write(guidelines_response.content)

print("PDF saved:", wdc_guidelines_file)

with pdfplumber.open(wdc_guidelines_file) as pdf:
    print("Number of pages:", len(pdf.pages))

    for page_number, page in enumerate(pdf.pages[:5], start=1):
        text = page.extract_text()

        print()
        print("====================================")
        print("Page", page_number)
        print("====================================")

        if text:
            print("Extracted text characters:", len(text))
            print(text[:1000])
        else:
            print("NO TEXT EXTRACTED FROM THIS PAGE")

PDF saved: GUIDELINES-ON-THE-ESTABLISHMENT-OF-WARD-DEVELOPMENT-COMMITTEES.pdf
Number of pages: 37

Page 1
Extracted text characters: 163
REPUBLIC OF ZAMBIA
MINISTRY OF LOCAL GOVERNMENT AND RURAL DEVELOPMENT
GUIDELINES ON THE ESTABLISHMENT,
MANAGEMENT AND OPERATION OF WARD
DEVELOPMENT COMMITTEES
2021

Page 2
Extracted text characters: 1512
FOREWORD
The Government of the Republic of Zambia is committed to actualising a devolved system of
governance as prescribed in the Constitution (Amendment) Act No. 2 of 2016. Article 147 of the
Constitution provides that the management and administration of the political, social, legal and
economic affairs of the State shall be devolved from the National Government level to the Local
Government level. Further, Article 148 establishes that local governance shall be undertaken
through sub-structures as platforms for citizen participation in decision making and development
process at the local level. In affirming the provisions of the Constitution, Part 

In [ ]:
guidelines_text = ""

with pdfplumber.open(wdc_guidelines_file) as pdf:
    for page_number, page in enumerate(pdf.pages, start=1):
        page_text = page.extract_text()

        if page_text:
            guidelines_text += f"\n--- Page {page_number} ---\n"
            guidelines_text += page_text

print("Total extracted characters:", len(guidelines_text))

wdc_guideline_lines = []

for line in guidelines_text.splitlines():
    line_lower = line.lower()

    if (
        "function" in line_lower
        or "functions" in line_lower
        or "responsibilit" in line_lower
        or "ward development committee" in line_lower
    ):
        wdc_guideline_lines.append(line.strip())

print()
print("WDC-related lines found:")
print("-------------------------")

for line in wdc_guideline_lines:
    print(line)

Total extracted characters: 52820

WDC-related lines found:
-------------------------
provide linkages among the Ward Development Committee, the Local Authorities and other
WDCs Ward Development Committees
Candidate A person contesting for the ward development committee membership.
for time being discharging the functions of their Office.
PART II: ESTABLISHMENT, COMPOSITION AND FUNCTIONS OF THE COMMITTEES .............. 2
1.2 Aims of the Ward Development Committees
FUNCTIONS OF COMMITTEES
2.1 THE WARD DEVELOPMENT COMMITTEE
this responsibility, a team of facilitators may be appointed from departments in the local
A Ward Development Committee may invite a person to attend and participate in the
2.1.3 Functions of the Ward Development Committee
The functions of a Ward Development Committee are to:
xiv. Manage and keep a record of resources allocated to the Ward Development Committee;
xvii. Execute other functions as delegated by the Local Authority excluding delegated Central
Government f

In [ ]:
functions_start = guidelines_text.find(
    "2.1.3 Functions of the Ward Development Committee"
)

functions_end = guidelines_text.find(
    "2.1.4 Functions of the Councillor and Ex-officios"
)

if functions_start != -1 and functions_end != -1:
    wdc_functions_section = guidelines_text[
        functions_start:functions_end
    ].strip()

    print("WDC Functions Section:")
    print("======================")
    print(wdc_functions_section)
else:
    print("The WDC functions section could not be located.")

WDC Functions Section:
2.1.3 Functions of the Ward Development Committee
The functions of a Ward Development Committee are to:
i. Prepare annual Ward Development Plans;
ii. Collect revenue, levies and fees on behalf of a Local Authority on appointment by
resolution of the Council;
iii. Monitor and evaluate Ward development projects;
iv. Promote community engagement in Ward development Planning;
v. Formulate and submit a project list and budget proposals to the Constituency
Development Fund (CDF) Committee in the first quarter of every year preceding the year
in which a project is proposed to be implemented;
vi. Support research on an area of study for the advancement of the local community;
vii. Facilitate the identification of potential areas of investment and promote sustainable local
economic development;
viii. Promote and participate in the co-management of natural and trans-boundary resources
between or among wards;
ix. Provide a forum for dialogue and coordination on Ward develop

In [ ]:
wdc_functions = [
    [
        1,
        "Prepare annual Ward Development Plans",
        "WDC function",
        "Official PDF",
        wdc_guidelines_url
    ],
    [
        2,
        "Collect revenue when appointed by the Local Authority",
        "WDC function",
        "Official PDF",
        wdc_guidelines_url
    ],
    [
        3,
        "Monitor and evaluate development projects implemented in the ward",
        "WDC function",
        "Official PDF",
        wdc_guidelines_url
    ],
    [
        4,
        "Promote community engagement in development activities",
        "WDC function",
        "Official PDF",
        wdc_guidelines_url
    ],
    [
        5,
        "Formulate and submit project lists and CDF budget proposals",
        "WDC function",
        "Official PDF",
        wdc_guidelines_url
    ],
    [
        6,
        "Support research and development activities in the ward",
        "WDC function",
        "Official PDF",
        wdc_guidelines_url
    ],
    [
        7,
        "Identify investment opportunities within the ward",
        "WDC function",
        "Official PDF",
        wdc_guidelines_url
    ],
    [
        8,
        "Support resource co-management within the ward",
        "WDC function",
        "Official PDF",
        wdc_guidelines_url
    ],
    [
        9,
        "Promote dialogue and coordination among stakeholders",
        "WDC function",
        "Official PDF",
        wdc_guidelines_url
    ],
    [
        10,
        "Participate in capacity building activities",
        "WDC function",
        "Official PDF",
        wdc_guidelines_url
    ],
    [
        11,
        "Support urban renewal and development activities",
        "WDC function",
        "Official PDF",
        wdc_guidelines_url
    ],
    [
        12,
        "Identify potential sources of revenue for the ward",
        "WDC function",
        "Official PDF",
        wdc_guidelines_url
    ],
    [
        13,
        "Manage and keep records of resources allocated to the WDC",
        "WDC function",
        "Official PDF",
        wdc_guidelines_url
    ],
    [
        14,
        "Prepare and submit quarterly reports",
        "WDC function",
        "Official PDF",
        wdc_guidelines_url
    ],
    [
        15,
        "Maintain a ward database",
        "WDC function",
        "Official PDF",
        wdc_guidelines_url
    ],
    [
        16,
        "Execute other functions delegated by the Local Authority",
        "WDC function",
        "Official PDF",
        wdc_guidelines_url
    ]
]

wdc_functions_df = pd.DataFrame(
    wdc_functions,
    columns=[
        "function_number",
        "function",
        "category",
        "source_type",
        "source_url"
    ]
)

display(wdc_functions_df)

,function_number,function,category,source_type,source_url
0,1,Prepare annual Ward Development Plans,WDC function,Official PDF,https://www.kabwecouncil.gov.zm/wp-content/upl...
1,2,Collect revenue when appointed by the Local Au...,WDC function,Official PDF,https://www.kabwecouncil.gov.zm/wp-content/upl...
2,3,Monitor and evaluate development projects impl...,WDC function,Official PDF,https://www.kabwecouncil.gov.zm/wp-content/upl...
3,4,Promote community engagement in development ac...,WDC function,Official PDF,https://www.kabwecouncil.gov.zm/wp-content/upl...
4,5,Formulate and submit project lists and CDF bud...,WDC function,Official PDF,https://www.kabwecouncil.gov.zm/wp-content/upl...
5,6,Support research and development activities in...,WDC function,Official PDF,https://www.kabwecouncil.gov.zm/wp-content/upl...
6,7,Identify investment opportunities within the ward,WDC function,Official PDF,https://www.kabwecouncil.gov.zm/wp-content/upl...
7,8,Support resource co-management within the ward,WDC function,Official PDF,https://www.kabwecouncil.gov.zm/wp-content/upl...
8,9,Promote dialogue and coordination among stakeh...,WDC function,Official PDF,https://www.kabwecouncil.gov.zm/wp-content/upl...
9,10,Participate in capacity building activities,WDC function,Official PDF,https://www.kabwecouncil.gov.zm/wp-content/upl...


In [ ]:
print("Number of WDC functions:", len(wdc_functions_df))

print()
print("Missing values:")
print(wdc_functions_df.isnull().sum())

print()
print("Duplicate rows:", wdc_functions_df.duplicated().sum())

print()
print("Duplicate function numbers:",
      wdc_functions_df["function_number"].duplicated().sum())

Number of WDC functions: 16

Missing values:
function_number    0
function           0
category           0
source_type        0
source_url         0
dtype: int64

Duplicate rows: 0

Duplicate function numbers: 0


In [ ]:
search_url = "https://www.kabwecouncil.gov.zm/"

search_response = requests.get(
    search_url,
    timeout=30,
    verify=False
)

search_soup = BeautifulSoup(
    search_response.text,
    "html.parser"
)

ward_related_links = []

for link in search_soup.find_all("a", href=True):
    link_text = link.get_text(" ", strip=True)
    href = link["href"]

    combined_text = f"{link_text} {href}".lower()

    if (
        "ward" in combined_text
        or "constituency" in combined_text
        or "kabw e central" in combined_text
        or "bwacha" in combined_text
    ):
        ward_related_links.append([
            link_text,
            href
        ])

print("Potential ward/constituency links:")
print("----------------------------------")

for link_text, href in ward_related_links:
    print("Text:", link_text)
    print("URL:", href)
    print()

Potential ward/constituency links:
----------------------------------
Text: Bwacha
URL: https://www.kabwecouncil.gov.zm/?page_id=2613

Text: Bwacha
URL: https://www.kabwecouncil.gov.zm/?page_id=2613

Text: Bwacha
URL: https://www.kabwecouncil.gov.zm/?page_id=2610



In [ ]:
bwacha_url = "https://www.kabwecouncil.gov.zm/?page_id=2613"

bwacha_response = requests.get(
    bwacha_url,
    timeout=30,
    verify=False
)

print("Status code:", bwacha_response.status_code)
print("Content type:", bwacha_response.headers.get("Content-Type"))

bwacha_soup = BeautifulSoup(
    bwacha_response.text,
    "html.parser"
)

bwacha_text = bwacha_soup.get_text(
    separator="\n",
    strip=True
)

print()
print("Bwacha constituency page:")
print("-------------------------")
print(bwacha_text[:7000])

Status code: 200
Content type: text/html; charset=UTF-8

Bwacha constituency page:
-------------------------
Bwacha – Kabwe Municipal Council
Home
About
Open menu
About Us
Mandate
Who we are
Departments
office of the the town clerk
Dept of Human Resource and Administration
Dept of Health
Dept of Finance
Dept of Engineering Services
Dept of Health Services
Dept Legal ( Services Unit)
Dept Of Planning
Dept Of Housing and Social Services
Dept of Fisheries, Livestock And Veterinary Services
Dept of Agriculture
Civic Leaders
The Mayor
Kabwe Central
Bwacha
Services
Open menu
Licenses and levies
Waste Management
E-GP
News Updates
CDF
Open menu
CDF GUIDLINES
CDF branding guidelines
ZDSP
Media
Open menu
Publications
Public Notices
Photo Gallery
video gallery
Forms
ADD A VIDEO
Contact Us
FAQS
Menu
Home
About
Open menu
About Us
Mandate
Who we are
Departments
office of the the town clerk
Dept of Human Resource and Administration
Dept of Health
Dept of Finance
Dept of Engineering Services
Dept of H

In [ ]:
kabwe_central_url = "https://www.kabwecouncil.gov.zm/?page_id=2610"

kabwe_central_response = requests.get(
    kabwe_central_url,
    timeout=30,
    verify=False
)

print("Status code:", kabwe_central_response.status_code)
print("Content type:", kabwe_central_response.headers.get("Content-Type"))

kabwe_central_soup = BeautifulSoup(
    kabwe_central_response.text,
    "html.parser"
)

kabwe_central_text = kabwe_central_soup.get_text(
    separator="\n",
    strip=True
)

print()
print("Kabwe Central constituency page:")
print("---------------------------------")
print(kabwe_central_text[:7000])

Status code: 200
Content type: text/html; charset=UTF-8

Kabwe Central constituency page:
---------------------------------
Kabwe Central – Kabwe Municipal Council
Home
About
Open menu
About Us
Mandate
Who we are
Departments
office of the the town clerk
Dept of Human Resource and Administration
Dept of Health
Dept of Finance
Dept of Engineering Services
Dept of Health Services
Dept Legal ( Services Unit)
Dept Of Planning
Dept Of Housing and Social Services
Dept of Fisheries, Livestock And Veterinary Services
Dept of Agriculture
Civic Leaders
The Mayor
Kabwe Central
Bwacha
Services
Open menu
Licenses and levies
Waste Management
E-GP
News Updates
CDF
Open menu
CDF GUIDLINES
CDF branding guidelines
ZDSP
Media
Open menu
Publications
Public Notices
Photo Gallery
video gallery
Forms
ADD A VIDEO
Contact Us
FAQS
Menu
Home
About
Open menu
About Us
Mandate
Who we are
Departments
office of the the town clerk
Dept of Human Resource and Administration
Dept of Health
Dept of Finance
Dept of Engineer

In [ ]:
search_terms = [
    "29 wards",
    "14 wards",
    "15 wards",
    "twenty-nine wards",
    "fourteen wards",
    "fifteen wards"
]

ward_count_results = []

for term in search_terms:
    search_response = requests.get(
        search_url,
        timeout=30,
        verify=False
    )

    search_soup = BeautifulSoup(
        search_response.text,
        "html.parser"
    )

    for link in search_soup.find_all("a", href=True):
        link_text = link.get_text(" ", strip=True)
        href = link["href"]

        combined_text = f"{link_text} {href}".lower()

        if term.lower() in combined_text:
            ward_count_results.append([
                term,
                link_text,
                href
            ])

print("Potential ward-count references:")
print("--------------------------------")

if ward_count_results:
    for result in ward_count_results:
        print("Search term:", result[0])
        print("Link text:", result[1])
        print("URL:", result[2])
        print()
else:
    print("No direct ward-count references found on the homepage links.")

Potential ward-count references:
--------------------------------
No direct ward-count references found on the homepage links.


In [ ]:
search_pages = [
    "https://www.kabwecouncil.gov.zm/",
    "https://www.kabwecouncil.gov.zm/?page_id=2610",
    "https://www.kabwecouncil.gov.zm/?page_id=2613",
    "https://www.kabwecouncil.gov.zm/?page_id=2173"
]

ward_search_terms = [
    "29 wards",
    "29 ward",
    "twenty-nine wards",
    "twenty nine wards",
    "14 wards",
    "15 wards",
    "fourteen wards",
    "fifteen wards"
]

found_ward_references = []

for page_url in search_pages:

    response = requests.get(
        page_url,
        timeout=30,
        verify=False
    )

    soup = BeautifulSoup(
        response.text,
        "html.parser"
    )

    page_text = soup.get_text(
        separator=" ",
        strip=True
    )

    page_text_lower = page_text.lower()

    for term in ward_search_terms:
        if term.lower() in page_text_lower:
            found_ward_references.append([
                term,
                page_url
            ])

print("Ward-count references found:")
print("-----------------------------")

if found_ward_references:
    for term, page_url in found_ward_references:
        print("Term:", term)
        print("Page:", page_url)
        print()
else:
    print("No direct ward-count references found in the tested pages.")

Ward-count references found:
-----------------------------
No direct ward-count references found in the tested pages.


## Step 2.36 — Ward Count Data Limitation

The official Kabwe Municipal Council website provides separate pages for
Kabwe Central Constituency and Bwacha Constituency. However, the extracted
content from these pages lists councillors but does not explicitly state
the number of wards in each constituency.

Several official council webpages were searched for direct references to:

- 29 wards;
- 14 wards;
- 15 wards;
- twenty-nine wards;
- fourteen wards; and
- fifteen wards.

No direct ward-count statement was found in the webpages tested.

Therefore, constituency ward counts will **not be inferred** from the
available information. The dataset will retain the verified figure of
29 Ward Development Committees from the official 2025 Output Based Annual
Budget without converting this figure into an unsupported ward count.

This approach preserves data accuracy and prevents unsupported assumptions
from entering the dataset.

In [ ]:
print("Number of general WDC records:", len(wdc_general_df))

print()
print("Missing values:")
print(wdc_general_df.isnull().sum())

print()
print("Duplicate rows:", wdc_general_df.duplicated().sum())

print()
print("General WDC information:")
display(wdc_general_df)

Number of general WDC records: 8

Missing values:
field          0
value          0
category       0
source_type    0
source_url     0
dtype: int64

Duplicate rows: 0

General WDC information:


,field,value,category,source_type,source_url
0,WDC abbreviation,WDC,Ward Development Committee,Official webpage,https://www.kabwecouncil.gov.zm/?page_id=2173
1,WDC meaning,Ward Development Committee,Ward Development Committee,Official webpage,https://www.kabwecouncil.gov.zm/?page_id=2173
2,WDC role,Grassroots link between communities and Local ...,Ward Development Committee,Official webpage,https://www.kabwecouncil.gov.zm/?page_id=2173
3,WDC coverage,Formed in each ward,Ward Development Committee,Official webpage,https://www.kabwecouncil.gov.zm/?page_id=2173
4,WDC membership selection,Residents select representatives through an el...,Ward Development Committee,Official webpage,https://www.kabwecouncil.gov.zm/?page_id=2173
5,WDC nature,Nonpartisan committee,Ward Development Committee,Official webpage,https://www.kabwecouncil.gov.zm/?page_id=2173
6,WDC purpose,Steering the developmental agenda at the grass...,Ward Development Committee,Official webpage,https://www.kabwecouncil.gov.zm/?page_id=2173
7,WDC election oversight,Elective process overseen by the Local Authority,Ward Development Committee,Official webpage,https://www.kabwecouncil.gov.zm/?page_id=2173


In [ ]:
print("Number of representative records:", len(wdc_representatives_df))

print()
print("Missing values:")
print(wdc_representatives_df.isnull().sum())

print()
print("Duplicate rows:", wdc_representatives_df.duplicated().sum())

print()
print("Duplicate wards:",
      wdc_representatives_df["ward"].duplicated().sum())

print()
print("WDC representative information:")
display(wdc_representatives_df)

Number of representative records: 3

Missing values:
ward           0
position       0
person         0
source_type    0
source_url     0
dtype: int64

Duplicate rows: 0

Duplicate wards: 0

WDC representative information:


,ward,position,person,source_type,source_url
0,Makululu,WDC Chairperson,Thomson Halwembe,Official webpage,https://www.kabwecouncil.gov.zm/?p=4873
1,Njanji,WDC Chairperson,Agness Mboma,Official webpage,https://www.kabwecouncil.gov.zm/?p=4873
2,Lwasanse,Ward Chairperson,Macmillan Chilangisha,Official webpage,https://www.kabwecouncil.gov.zm/?p=4873


In [ ]:
print("Number of WDC statistic records:", len(wdc_statistics_df))

print()
print("Missing values:")
print(wdc_statistics_df.isnull().sum())

print()
print("Duplicate rows:", wdc_statistics_df.duplicated().sum())

print()
print("WDC statistics:")
display(wdc_statistics_df)

Number of WDC statistic records: 3

Missing values:
field          0
value          0
category       0
source_type    0
source_url     0
dtype: int64

Duplicate rows: 0

WDC statistics:


,field,value,category,source_type,source_url
0,Number of Ward Development Committees,29,Ward Development Committee statistics,Official PDF,https://www.kabwecouncil.gov.zm/wp-content/upl...
1,Number of WDCs oriented,29,Ward Development Committee statistics,Official PDF,https://www.kabwecouncil.gov.zm/wp-content/upl...
2,Number of quarterly WDC meetings planned,4,Ward Development Committee statistics,Official PDF,https://www.kabwecouncil.gov.zm/wp-content/upl...


In [ ]:
wdc_functions_standardized_df = wdc_functions_df.copy()

wdc_functions_standardized_df = wdc_functions_standardized_df.rename(
    columns={
        "function_number": "field",
        "function": "value"
    }
)

wdc_functions_standardized_df["field"] = (
    "WDC Function " +
    wdc_functions_standardized_df["field"].astype(str)
)

print("Standardized WDC functions:")
display(wdc_functions_standardized_df)

Standardized WDC functions:


,field,value,category,source_type,source_url
0,WDC Function 1,Prepare annual Ward Development Plans,WDC function,Official PDF,https://www.kabwecouncil.gov.zm/wp-content/upl...
1,WDC Function 2,Collect revenue when appointed by the Local Au...,WDC function,Official PDF,https://www.kabwecouncil.gov.zm/wp-content/upl...
2,WDC Function 3,Monitor and evaluate development projects impl...,WDC function,Official PDF,https://www.kabwecouncil.gov.zm/wp-content/upl...
3,WDC Function 4,Promote community engagement in development ac...,WDC function,Official PDF,https://www.kabwecouncil.gov.zm/wp-content/upl...
4,WDC Function 5,Formulate and submit project lists and CDF bud...,WDC function,Official PDF,https://www.kabwecouncil.gov.zm/wp-content/upl...
5,WDC Function 6,Support research and development activities in...,WDC function,Official PDF,https://www.kabwecouncil.gov.zm/wp-content/upl...
6,WDC Function 7,Identify investment opportunities within the ward,WDC function,Official PDF,https://www.kabwecouncil.gov.zm/wp-content/upl...
7,WDC Function 8,Support resource co-management within the ward,WDC function,Official PDF,https://www.kabwecouncil.gov.zm/wp-content/upl...
8,WDC Function 9,Promote dialogue and coordination among stakeh...,WDC function,Official PDF,https://www.kabwecouncil.gov.zm/wp-content/upl...
9,WDC Function 10,Participate in capacity building activities,WDC function,Official PDF,https://www.kabwecouncil.gov.zm/wp-content/upl...


In [ ]:
wdc_representatives_standardized_df = wdc_representatives_df.copy()

wdc_representatives_standardized_df = wdc_representatives_standardized_df.rename(
    columns={
        "position": "field",
        "person": "value"
    }
)

wdc_representatives_standardized_df["field"] = (
    wdc_representatives_standardized_df["field"] +
    " - " +
    wdc_representatives_standardized_df["ward"]
)

wdc_representatives_standardized_df["category"] = "WDC representative"

wdc_representatives_standardized_df = (
    wdc_representatives_standardized_df[
        [
            "field",
            "value",
            "category",
            "source_type",
            "source_url"
        ]
    ]
)

print("Standardized WDC representative information:")
display(wdc_representatives_standardized_df)

Standardized WDC representative information:


,field,value,category,source_type,source_url
0,WDC Chairperson - Makululu,Thomson Halwembe,WDC representative,Official webpage,https://www.kabwecouncil.gov.zm/?p=4873
1,WDC Chairperson - Njanji,Agness Mboma,WDC representative,Official webpage,https://www.kabwecouncil.gov.zm/?p=4873
2,Ward Chairperson - Lwasanse,Macmillan Chilangisha,WDC representative,Official webpage,https://www.kabwecouncil.gov.zm/?p=4873


In [ ]:
wdc_statistics_standardized_df = wdc_statistics_df.copy()

print("Standardized WDC statistics:")
display(wdc_statistics_standardized_df)

Standardized WDC statistics:


,field,value,category,source_type,source_url
0,Number of Ward Development Committees,29,Ward Development Committee statistics,Official PDF,https://www.kabwecouncil.gov.zm/wp-content/upl...
1,Number of WDCs oriented,29,Ward Development Committee statistics,Official PDF,https://www.kabwecouncil.gov.zm/wp-content/upl...
2,Number of quarterly WDC meetings planned,4,Ward Development Committee statistics,Official PDF,https://www.kabwecouncil.gov.zm/wp-content/upl...


In [ ]:
wdc_combined_df = pd.concat(
    [
        wdc_general_df,
        wdc_statistics_standardized_df,
        wdc_functions_standardized_df,
        wdc_representatives_standardized_df
    ],
    ignore_index=True
)

print("Total WDC records:", len(wdc_combined_df))
print()
print("Columns:")
print(list(wdc_combined_df.columns))

print()
print("Combined WDC dataset:")
display(wdc_combined_df)

Total WDC records: 30

Columns:
['field', 'value', 'category', 'source_type', 'source_url']

Combined WDC dataset:


,field,value,category,source_type,source_url
0,WDC abbreviation,WDC,Ward Development Committee,Official webpage,https://www.kabwecouncil.gov.zm/?page_id=2173
1,WDC meaning,Ward Development Committee,Ward Development Committee,Official webpage,https://www.kabwecouncil.gov.zm/?page_id=2173
2,WDC role,Grassroots link between communities and Local ...,Ward Development Committee,Official webpage,https://www.kabwecouncil.gov.zm/?page_id=2173
3,WDC coverage,Formed in each ward,Ward Development Committee,Official webpage,https://www.kabwecouncil.gov.zm/?page_id=2173
4,WDC membership selection,Residents select representatives through an el...,Ward Development Committee,Official webpage,https://www.kabwecouncil.gov.zm/?page_id=2173
5,WDC nature,Nonpartisan committee,Ward Development Committee,Official webpage,https://www.kabwecouncil.gov.zm/?page_id=2173
6,WDC purpose,Steering the developmental agenda at the grass...,Ward Development Committee,Official webpage,https://www.kabwecouncil.gov.zm/?page_id=2173
7,WDC election oversight,Elective process overseen by the Local Authority,Ward Development Committee,Official webpage,https://www.kabwecouncil.gov.zm/?page_id=2173
8,Number of Ward Development Committees,29,Ward Development Committee statistics,Official PDF,https://www.kabwecouncil.gov.zm/wp-content/upl...
9,Number of WDCs oriented,29,Ward Development Committee statistics,Official PDF,https://www.kabwecouncil.gov.zm/wp-content/upl...


In [ ]:
print("Total records:", len(wdc_combined_df))

print()
print("Missing values:")
print(wdc_combined_df.isnull().sum())

print()
print("Duplicate rows:", wdc_combined_df.duplicated().sum())

print()
print("Records by category:")
print(wdc_combined_df["category"].value_counts())

print()
print("Source types:")
print(wdc_combined_df["source_type"].value_counts())

Total records: 30

Missing values:
field          0
value          0
category       0
source_type    0
source_url     0
dtype: int64

Duplicate rows: 0

Records by category:
category
WDC function                             16
Ward Development Committee                8
Ward Development Committee statistics     3
WDC representative                        3
Name: count, dtype: int64

Source types:
source_type
Official PDF        19
Official webpage    11
Name: count, dtype: int64


In [ ]:
text_columns = [
    "field",
    "value",
    "category",
    "source_type",
    "source_url"
]

for column in text_columns:
    wdc_combined_df[column] = (
        wdc_combined_df[column]
        .astype(str)
        .str.strip()
    )

print("Text cleaning completed.")

print()
print("Sample of cleaned data:")
display(wdc_combined_df.head(10))

Text cleaning completed.

Sample of cleaned data:


,field,value,category,source_type,source_url
0,WDC abbreviation,WDC,Ward Development Committee,Official webpage,https://www.kabwecouncil.gov.zm/?page_id=2173
1,WDC meaning,Ward Development Committee,Ward Development Committee,Official webpage,https://www.kabwecouncil.gov.zm/?page_id=2173
2,WDC role,Grassroots link between communities and Local ...,Ward Development Committee,Official webpage,https://www.kabwecouncil.gov.zm/?page_id=2173
3,WDC coverage,Formed in each ward,Ward Development Committee,Official webpage,https://www.kabwecouncil.gov.zm/?page_id=2173
4,WDC membership selection,Residents select representatives through an el...,Ward Development Committee,Official webpage,https://www.kabwecouncil.gov.zm/?page_id=2173
5,WDC nature,Nonpartisan committee,Ward Development Committee,Official webpage,https://www.kabwecouncil.gov.zm/?page_id=2173
6,WDC purpose,Steering the developmental agenda at the grass...,Ward Development Committee,Official webpage,https://www.kabwecouncil.gov.zm/?page_id=2173
7,WDC election oversight,Elective process overseen by the Local Authority,Ward Development Committee,Official webpage,https://www.kabwecouncil.gov.zm/?page_id=2173
8,Number of Ward Development Committees,29,Ward Development Committee statistics,Official PDF,https://www.kabwecouncil.gov.zm/wp-content/upl...
9,Number of WDCs oriented,29,Ward Development Committee statistics,Official PDF,https://www.kabwecouncil.gov.zm/wp-content/upl...


In [ ]:
duplicate_count = wdc_combined_df.duplicated().sum()

print("Duplicate rows after cleaning:", duplicate_count)

if duplicate_count == 0:
    print("✓ No duplicate records found.")
else:
    print("Duplicate records found:")
    display(
        wdc_combined_df[
            wdc_combined_df.duplicated(keep=False)
        ]
    )

Duplicate rows after cleaning: 0
✓ No duplicate records found.


In [ ]:
print("Data types:")
print(wdc_combined_df.dtypes)

print()
print("Dataset shape:")
print(wdc_combined_df.shape)

Data types:
field          str
value          str
category       str
source_type    str
source_url     str
dtype: object

Dataset shape:
(30, 5)


In [ ]:
wdc_output_file = "data/processed/ward_development_committes/db-unza26-csc4792-ward-development-committees.csv"

wdc_combined_df.to_csv(
    wdc_output_file,
    sep="|",
    index=False
)

print("CSV file created successfully:")
print(wdc_output_file)

CSV file created successfully:
db-unza26-csc4792-ward-development-committees.csv


In [ ]:
wdc_check_df = pd.read_csv(
    wdc_output_file,
    sep="|"
)

print("CSV successfully read back.")

print()
print("Shape:", wdc_check_df.shape)

print()
print("Columns:")
print(list(wdc_check_df.columns))

print()
print("First 10 records:")
display(wdc_check_df.head(10))

CSV successfully read back.

Shape: (30, 5)

Columns:
['field', 'value', 'category', 'source_type', 'source_url']

First 10 records:


,field,value,category,source_type,source_url
0,WDC abbreviation,WDC,Ward Development Committee,Official webpage,https://www.kabwecouncil.gov.zm/?page_id=2173
1,WDC meaning,Ward Development Committee,Ward Development Committee,Official webpage,https://www.kabwecouncil.gov.zm/?page_id=2173
2,WDC role,Grassroots link between communities and Local ...,Ward Development Committee,Official webpage,https://www.kabwecouncil.gov.zm/?page_id=2173
3,WDC coverage,Formed in each ward,Ward Development Committee,Official webpage,https://www.kabwecouncil.gov.zm/?page_id=2173
4,WDC membership selection,Residents select representatives through an el...,Ward Development Committee,Official webpage,https://www.kabwecouncil.gov.zm/?page_id=2173
5,WDC nature,Nonpartisan committee,Ward Development Committee,Official webpage,https://www.kabwecouncil.gov.zm/?page_id=2173
6,WDC purpose,Steering the developmental agenda at the grass...,Ward Development Committee,Official webpage,https://www.kabwecouncil.gov.zm/?page_id=2173
7,WDC election oversight,Elective process overseen by the Local Authority,Ward Development Committee,Official webpage,https://www.kabwecouncil.gov.zm/?page_id=2173
8,Number of Ward Development Committees,29,Ward Development Committee statistics,Official PDF,https://www.kabwecouncil.gov.zm/wp-content/upl...
9,Number of WDCs oriented,29,Ward Development Committee statistics,Official PDF,https://www.kabwecouncil.gov.zm/wp-content/upl...


In [ ]:
print("STEP 2 DATASET SUMMARY")
print("=======================")

print("Dataset name:")
print(wdc_output_file)

print()
print("Number of records:", len(wdc_check_df))
print("Number of columns:", len(wdc_check_df.columns))

print()
print("Columns:")
for column in wdc_check_df.columns:
    print("-", column)

print()
print("Records by category:")
print(wdc_check_df["category"].value_counts())

print()
print("Source types:")
print(wdc_check_df["source_type"].value_counts())

STEP 2 DATASET SUMMARY
Dataset name:
db-unza26-csc4792-ward-development-committees.csv

Number of records: 30
Number of columns: 5

Columns:
- field
- value
- category
- source_type
- source_url

Records by category:
category
WDC function                             16
Ward Development Committee                8
Ward Development Committee statistics     3
WDC representative                        3
Name: count, dtype: int64

Source types:
source_type
Official PDF        19
Official webpage    11
Name: count, dtype: int64


In [ ]:
wdc_source_register = [
    [
        "Official FAQ",
        "Kabwe Municipal Council FAQ",
        faq_url
    ],
    [
        "2025 Output Based Annual Budget",
        "Kabwe Municipal Council 2025 Output Based Annual Budget",
        budget_url
    ],
    [
        "WDC Guidelines",
        "Guidelines on the Establishment, Management and Operation of Ward Development Committees",
        wdc_guidelines_url
    ],
    [
        "WDC Representatives Article",
        "Kabwe Municipal Council engages stakeholders on proposed new by-laws",
        article_url
    ],
    [
        "Kabwe Central Constituency Page",
        "Kabwe Municipal Council Kabwe Central constituency page",
        kabwe_central_url
    ],
    [
        "Bwacha Constituency Page",
        "Kabwe Municipal Council Bwacha constituency page",
        bwacha_url
    ]
]

wdc_source_register_df = pd.DataFrame(
    wdc_source_register,
    columns=[
        "source_type",
        "description",
        "source_url"
    ]
)

display(wdc_source_register_df)

,source_type,description,source_url
0,Official FAQ,Kabwe Municipal Council FAQ,https://www.kabwecouncil.gov.zm/?page_id=2173
1,2025 Output Based Annual Budget,Kabwe Municipal Council 2025 Output Based Annu...,https://www.kabwecouncil.gov.zm/wp-content/upl...
2,WDC Guidelines,"Guidelines on the Establishment, Management an...",https://www.kabwecouncil.gov.zm/wp-content/upl...
3,WDC Representatives Article,Kabwe Municipal Council engages stakeholders o...,https://www.kabwecouncil.gov.zm/?p=4873
4,Kabwe Central Constituency Page,Kabwe Municipal Council Kabwe Central constitu...,https://www.kabwecouncil.gov.zm/?page_id=2610
5,Bwacha Constituency Page,Kabwe Municipal Council Bwacha constituency page,https://www.kabwecouncil.gov.zm/?page_id=2613


In [ ]:
print("SOURCE REGISTER VALIDATION")
print("==========================")

print("Number of sources:", len(wdc_source_register_df))
print()

print("Missing values:")
print(wdc_source_register_df.isnull().sum())
print()

print("Duplicate URLs:", wdc_source_register_df["source_url"].duplicated().sum())
print()

print("Expected number of sources: 6")

if (
    len(wdc_source_register_df) == 6
    and wdc_source_register_df.isnull().sum().sum() == 0
    and wdc_source_register_df["source_url"].duplicated().sum() == 0
):
    print("RESULT: Source register validation PASSED.")
else:
    print("RESULT: Source register validation FAILED.")

SOURCE REGISTER VALIDATION
Number of sources: 6

Missing values:
source_type    0
description    0
source_url     0
dtype: int64

Duplicate URLs: 0

Expected number of sources: 6
RESULT: Source register validation PASSED.


## Step 2.53 — Data Provenance

The Ward Development Committee dataset was compiled from official
Kabwe Municipal Council webpages and official council documents.

The sources were used as follows:

| Source | Purpose |
|---|---|
| Kabwe Municipal Council FAQ | Definition, role, coverage, membership selection, nature and purpose of WDCs |
| 2025 Output Based Annual Budget | Number of WDCs, number of WDCs oriented, and planned quarterly WDC meetings |
| WDC Guidelines | Functions of Ward Development Committees |
| WDC Representatives Article | Names and positions of selected WDC representatives |
| Kabwe Central Constituency Page | Checked for constituency and ward information |
| Bwacha Constituency Page | Checked for constituency and ward information |

Each dataset record contains a `source_url` column so that the
information can be traced back to its official source.

The Kabwe Central and Bwacha constituency pages were also examined when
investigating ward-count information. Since these pages did not provide
explicit ward counts, no constituency ward numbers were inferred.

This provenance approach ensures that the dataset contains only
information that could be supported by the available official sources.

In [ ]:
print("FINAL DATASET QUALITY CHECK")
print("===========================")

print("Dataset:", wdc_output_file)
print("Rows:", len(wdc_check_df))
print("Columns:", len(wdc_check_df.columns))
print()

print("Missing values:")
print(wdc_check_df.isnull().sum())
print()

print("Duplicate records:", wdc_check_df.duplicated().sum())
print()

print("Pipe delimiter check:")
with open(wdc_output_file, "r", encoding="utf-8") as file:
    first_line = file.readline().strip()

print(first_line)

print()
print("Required columns:")
required_columns = [
    "field",
    "value",
    "category",
    "source_type",
    "source_url"
]

for column in required_columns:
    print(f"- {column}:", column in wdc_check_df.columns)

print()

if (
    len(wdc_check_df) == 30
    and len(wdc_check_df.columns) == 5
    and wdc_check_df.isnull().sum().sum() == 0
    and wdc_check_df.duplicated().sum() == 0
    and all(column in wdc_check_df.columns for column in required_columns)
    and "|" in first_line
):
    print("RESULT: FINAL DATASET QUALITY CHECK PASSED.")
else:
    print("RESULT: FINAL DATASET QUALITY CHECK FAILED.")

FINAL DATASET QUALITY CHECK
Dataset: db-unza26-csc4792-ward-development-committees.csv
Rows: 30
Columns: 5

Missing values:
field          0
value          0
category       0
source_type    0
source_url     0
dtype: int64

Duplicate records: 0

Pipe delimiter check:
field|value|category|source_type|source_url

Required columns:
- field: True
- value: True
- category: True
- source_type: True
- source_url: True

RESULT: FINAL DATASET QUALITY CHECK PASSED.


In [ ]:
print("FINAL WARD DEVELOPMENT COMMITTEE DATASET")
print("=========================================")

display(wdc_check_df)

FINAL WARD DEVELOPMENT COMMITTEE DATASET


,field,value,category,source_type,source_url
0,WDC abbreviation,WDC,Ward Development Committee,Official webpage,https://www.kabwecouncil.gov.zm/?page_id=2173
1,WDC meaning,Ward Development Committee,Ward Development Committee,Official webpage,https://www.kabwecouncil.gov.zm/?page_id=2173
2,WDC role,Grassroots link between communities and Local ...,Ward Development Committee,Official webpage,https://www.kabwecouncil.gov.zm/?page_id=2173
3,WDC coverage,Formed in each ward,Ward Development Committee,Official webpage,https://www.kabwecouncil.gov.zm/?page_id=2173
4,WDC membership selection,Residents select representatives through an el...,Ward Development Committee,Official webpage,https://www.kabwecouncil.gov.zm/?page_id=2173
5,WDC nature,Nonpartisan committee,Ward Development Committee,Official webpage,https://www.kabwecouncil.gov.zm/?page_id=2173
6,WDC purpose,Steering the developmental agenda at the grass...,Ward Development Committee,Official webpage,https://www.kabwecouncil.gov.zm/?page_id=2173
7,WDC election oversight,Elective process overseen by the Local Authority,Ward Development Committee,Official webpage,https://www.kabwecouncil.gov.zm/?page_id=2173
8,Number of Ward Development Committees,29,Ward Development Committee statistics,Official PDF,https://www.kabwecouncil.gov.zm/wp-content/upl...
9,Number of WDCs oriented,29,Ward Development Committee statistics,Official PDF,https://www.kabwecouncil.gov.zm/wp-content/upl...


# Step 2 — Completion Summary

## Ward Development Committee Information

This notebook extracted, cleaned, standardized, validated, and exported
information about Ward Development Committees associated with Kabwe
Municipal Council.

### Final dataset

**File name:**

`db-unza26-csc4792-ward-development-committees.csv`

**Delimiter:** `|`

**Records:** 30

**Columns:** 5

### Dataset columns

1. `field`
2. `value`
3. `category`
4. `source_type`
5. `source_url`

### Information included

The dataset contains information about:

- the meaning and role of Ward Development Committees;
- WDC coverage and membership selection;
- the nonpartisan nature and purpose of WDCs;
- the number of WDCs;
- the number of WDCs oriented;
- planned quarterly WDC meetings;
- WDC functions;
- identified WDC representatives; and
- official source URLs.

### Data quality

The final dataset was checked for:

- missing values;
- duplicate records;
- correct number of columns;
- required column names;
- correct pipe delimiter; and
- successful CSV read-back.

No unsupported constituency ward counts were inferred because the
official constituency pages examined did not provide explicit ward-count
information.

## Step 2 Status

**COMPLETE**

The resulting CSV file is ready to be used as the Ward Development
Committee component of the CSC4792 Kabwe Municipal Council dataset.

# Step 3 — Council Services and Facilities Information

## Objective

This notebook extracts, cleans, and curates information about services
and facilities associated with Kabwe Municipal Council.

The information will be obtained from official Kabwe Municipal Council
webpages and official council documents.

The dataset will focus on:

- council services;
- public facilities and amenities;
- service descriptions;
- departments or units responsible for services;
- locations where available; and
- official sources supporting the extracted information.

Source URLs will be retained to provide data provenance.

Libraries imported successfully.


Official council website:
https://www.kabwecouncil.gov.zm


C:\Users\Pardo\AppData\Local\Programs\Python\Python314\Lib\site-packages\urllib3\connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host 'www.kabwecouncil.gov.zm'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


Status code: 200
Content type: text/html; charset=UTF-8
Page size: 161419 characters


First 5000 characters of the council homepage:
-----------------------------------------------
Kabwe Municipal Council – Kabwe
Home
About
Open menu
About Us
Mandate
Who we are
Departments
office of the the town clerk
Dept of Human Resource and Administration
Dept of Health
Dept of Finance
Dept of Engineering Services
Dept of Health Services
Dept Legal ( Services Unit)
Dept Of Planning
Dept Of Housing and Social Services
Dept of Fisheries, Livestock And Veterinary Services
Dept of Agriculture
Civic Leaders
The Mayor
Kabwe Central
Bwacha
Services
Open menu
Licenses and levies
Waste Management
E-GP
News Updates
CDF
Open menu
CDF GUIDLINES
CDF branding guidelines
ZDSP
Media
Open menu
Publications
Public Notices
Photo Gallery
video gallery
Forms
ADD A VIDEO
Contact Us
FAQS
Menu
Home
About
Open menu
About Us
Mandate
Who we are
Departments
office of the the town clerk
Dept of Human Resource and Administration
Dept of Health
Dept of Finance
Dept of Engineering Services
Dept of Health Services


Council Services URLs:
licenses_and_levies : https://www.kabwecouncil.gov.zm/?page_id=2142
waste_management : https://www.kabwecouncil.gov.zm/?page_id=2150
e_gp : https://www.kabwecouncil.gov.zm/?page_id=2155


C:\Users\Pardo\AppData\Local\Programs\Python\Python314\Lib\site-packages\urllib3\connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host 'www.kabwecouncil.gov.zm'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
C:\Users\Pardo\AppData\Local\Programs\Python\Python314\Lib\site-packages\urllib3\connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host 'www.kabwecouncil.gov.zm'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


licenses_and_levies
URL: https://www.kabwecouncil.gov.zm/?page_id=2142
Status code: 200
Content type: text/html; charset=UTF-8
Page size: 82999 characters
----------------------------------------


C:\Users\Pardo\AppData\Local\Programs\Python\Python314\Lib\site-packages\urllib3\connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host 'www.kabwecouncil.gov.zm'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
C:\Users\Pardo\AppData\Local\Programs\Python\Python314\Lib\site-packages\urllib3\connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host 'www.kabwecouncil.gov.zm'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


waste_management
URL: https://www.kabwecouncil.gov.zm/?page_id=2150
Status code: 200
Content type: text/html; charset=UTF-8
Page size: 83062 characters
----------------------------------------


C:\Users\Pardo\AppData\Local\Programs\Python\Python314\Lib\site-packages\urllib3\connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host 'www.kabwecouncil.gov.zm'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
C:\Users\Pardo\AppData\Local\Programs\Python\Python314\Lib\site-packages\urllib3\connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host 'www.kabwecouncil.gov.zm'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


e_gp
URL: https://www.kabwecouncil.gov.zm/?page_id=2155
Status code: 200
Content type: text/html; charset=UTF-8
Page size: 82960 characters
----------------------------------------


LICENSES_AND_LEVIES
IMG-20250115-WA0025 – Kabwe Municipal Council
Home
About
Open menu
About Us
Mandate
Who we are
Departments
office of the the town clerk
Dept of Human Resource and Administration
Dept of Health
Dept of Finance
Dept of Engineering Services
Dept of Health Services
Dept Legal ( Services Unit)
Dept Of Planning
Dept Of Housing and Social Services
Dept of Fisheries, Livestock And Veterinary Services
Dept of Agriculture
Civic Leaders
The Mayor
Kabwe Central
Bwacha
Services
Open menu
Licenses and levies
Waste Management
E-GP
News Updates
CDF
Open menu
CDF GUIDLINES
CDF branding guidelines
ZDSP
Media
Open menu
Publications
Public Notices
Photo Gallery
video gallery
Forms
ADD A VIDEO
Contact Us
FAQS
Menu
Home
About
Open menu
About Us
Mandate
Who we are
Departments
office of the the town clerk
Dept of Human Resource and Administration
Dept of Health
Dept of Finance
Dept of Engineering Services
Dept of Health Services
Dept Legal ( Services Unit)
Dept Of Planning
Dept Of Housing 

,service,description,category,source_type,source_url
0,Licenses and levies,Council service listed under the Services sect...,Council Service,Official webpage,https://www.kabwecouncil.gov.zm/?page_id=2142
1,Waste Management,Council service listed under the Services sect...,Council Service,Official webpage,https://www.kabwecouncil.gov.zm/?page_id=2150
2,E-GP,Council service listed under the Services sect...,Council Service,Official webpage,https://www.kabwecouncil.gov.zm/?page_id=2155


C:\Users\Pardo\AppData\Local\Programs\Python\Python314\Lib\site-packages\urllib3\connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host 'www.kabwecouncil.gov.zm'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


Departments page:
URL: https://www.kabwecouncil.gov.zm/?page_id=2072
Status code: 404
Content type: text/html; charset=UTF-8
Page size: 81506 characters


,department,url
0,Dept of Human Resource and Administration,https://www.kabwecouncil.gov.zm/?page_id=2637
1,Dept of Health,https://www.kabwecouncil.gov.zm/?page_id=2640
2,Dept of Finance,https://www.kabwecouncil.gov.zm/?page_id=2643
3,Dept of Engineering Services,https://www.kabwecouncil.gov.zm/?page_id=2646
4,Dept of Health Services,https://www.kabwecouncil.gov.zm/?page_id=2649
5,Dept Legal ( Services Unit),https://www.kabwecouncil.gov.zm/?page_id=3894
6,Dept Of Planning,https://www.kabwecouncil.gov.zm/?page_id=3897
7,Dept Of Housing and Social Services,https://www.kabwecouncil.gov.zm/?page_id=3900
8,"Dept of Fisheries, Livestock And Veterinary Se...",https://www.kabwecouncil.gov.zm/?page_id=3903
9,Dept of Agriculture,https://www.kabwecouncil.gov.zm/?page_id=3906


Number of department links found: 10



,department,url
0,Dept of Human Resource and Administration,https://www.kabwecouncil.gov.zm/?page_id=2637
1,Dept of Health,https://www.kabwecouncil.gov.zm/?page_id=2640
2,Dept of Finance,https://www.kabwecouncil.gov.zm/?page_id=2643
3,Dept of Engineering Services,https://www.kabwecouncil.gov.zm/?page_id=2646
4,Dept of Health Services,https://www.kabwecouncil.gov.zm/?page_id=2649
5,Dept Legal ( Services Unit),https://www.kabwecouncil.gov.zm/?page_id=3894
6,Dept Of Planning,https://www.kabwecouncil.gov.zm/?page_id=3897
7,Dept Of Housing and Social Services,https://www.kabwecouncil.gov.zm/?page_id=3900
8,"Dept of Fisheries, Livestock And Veterinary Se...",https://www.kabwecouncil.gov.zm/?page_id=3903
9,Dept of Agriculture,https://www.kabwecouncil.gov.zm/?page_id=3906


C:\Users\Pardo\AppData\Local\Programs\Python\Python314\Lib\site-packages\urllib3\connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host 'www.kabwecouncil.gov.zm'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
C:\Users\Pardo\AppData\Local\Programs\Python\Python314\Lib\site-packages\urllib3\connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host 'www.kabwecouncil.gov.zm'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
C:\Users\Pardo\AppData\Local\Programs\Python\Python314\Lib\site-packages\urllib3\connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host 'www.kabwecouncil.gov.zm'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/l

,department,url,status_code,page_size
0,Dept of Human Resource and Administration,https://www.kabwecouncil.gov.zm/?page_id=2637,200,110630
1,Dept of Health,https://www.kabwecouncil.gov.zm/?page_id=2640,200,137426
2,Dept of Finance,https://www.kabwecouncil.gov.zm/?page_id=2643,200,103806
3,Dept of Engineering Services,https://www.kabwecouncil.gov.zm/?page_id=2646,200,100085
4,Dept of Health Services,https://www.kabwecouncil.gov.zm/?page_id=2649,200,117513
5,Dept Legal ( Services Unit),https://www.kabwecouncil.gov.zm/?page_id=3894,200,112308
6,Dept Of Planning,https://www.kabwecouncil.gov.zm/?page_id=3897,200,105198
7,Dept Of Housing and Social Services,https://www.kabwecouncil.gov.zm/?page_id=3900,200,100272
8,"Dept of Fisheries, Livestock And Veterinary Se...",https://www.kabwecouncil.gov.zm/?page_id=3903,200,123830
9,Dept of Agriculture,https://www.kabwecouncil.gov.zm/?page_id=3906,200,92673


C:\Users\Pardo\AppData\Local\Programs\Python\Python314\Lib\site-packages\urllib3\connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host 'www.kabwecouncil.gov.zm'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
C:\Users\Pardo\AppData\Local\Programs\Python\Python314\Lib\site-packages\urllib3\connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host 'www.kabwecouncil.gov.zm'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
C:\Users\Pardo\AppData\Local\Programs\Python\Python314\Lib\site-packages\urllib3\connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host 'www.kabwecouncil.gov.zm'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/l

Number of department pages extracted: 10


DEPT OF HUMAN RESOURCE AND ADMINISTRATION
Dept of Human Resource and Administration – Kabwe Municipal Council
Home
About
Open menu
About Us
Mandate
Who we are
Departments
office of the the town clerk
Dept of Human Resource and Administration
Dept of Health
Dept of Finance
Dept of Engineering Services
Dept of Health Services
Dept Legal ( Services Unit)
Dept Of Planning
Dept Of Housing and Social Services
Dept of Fisheries, Livestock And Veterinary Services
Dept of Agriculture
Civic Leaders
The Mayor
Kabwe Central
Bwacha
Services
Licenses and levies
Waste Management
E-GP
News Updates
CDF
CDF GUIDLINES
CDF branding guidelines
ZDSP
Media
Publications
Public Notices
Photo Gallery
video gallery
Forms
ADD A VIDEO
Contact Us
FAQS
Menu
Search
Human Resource and Administration
NSAKANYA J CHANGWE
Director of Human Resource and Administration
A small river named Duden flows by their place and supplies it with the necessary
Phone:
+1 (859) 254-6589
Email:
info@example.com
The directorate is headed 

,service,description,department,category
0,Environmental Health,Safeguards community health through environmen...,Public Health Department,Council Service
1,Disease Prevention and Control,"Provides disease prevention, control, health s...",Public Health Department,Council Service
2,Waste Management,Provides solid waste management and waste-mana...,Public Health Department,Council Service
3,Food Safety and Workplace Inspections,"Inspects restaurants, markets, factories and w...",Public Health Department,Council Service
4,Public Health Awareness and Education,"Provides workshops, school programmes, media c...",Public Health Department,Council Service
5,Pest and Vector Management,"Provides pest and vector surveillance, sprayin...",Public Health Department,Council Service
6,Funeral and Burial Services,"Provides cemetery administration, plot allocat...",Public Health Department,Council Service
7,Clinical and Nursing Services,Provides preventive and curative healthcare th...,Health Services Department,Council Service
8,Pharmaceutical and Diagnostic Services,Provides pharmaceutical services and diagnosti...,Health Services Department,Council Service
9,Vaccination and Child Health Programmes,Implements vaccination campaigns and child hea...,Health Services Department,Council Service


,service,description,department,category,source_type,source_url
0,Environmental Health,Safeguards community health through environmen...,Public Health Department,Council Service,Official council webpage,NaN
1,Disease Prevention and Control,"Provides disease prevention, control, health s...",Public Health Department,Council Service,Official council webpage,NaN
2,Waste Management,Provides solid waste management and waste-mana...,Public Health Department,Council Service,Official council webpage,NaN
3,Food Safety and Workplace Inspections,"Inspects restaurants, markets, factories and w...",Public Health Department,Council Service,Official council webpage,NaN
4,Public Health Awareness and Education,"Provides workshops, school programmes, media c...",Public Health Department,Council Service,Official council webpage,NaN
5,Pest and Vector Management,"Provides pest and vector surveillance, sprayin...",Public Health Department,Council Service,Official council webpage,NaN
6,Funeral and Burial Services,"Provides cemetery administration, plot allocat...",Public Health Department,Council Service,Official council webpage,NaN
7,Clinical and Nursing Services,Provides preventive and curative healthcare th...,Health Services Department,Council Service,Official council webpage,NaN
8,Pharmaceutical and Diagnostic Services,Provides pharmaceutical services and diagnosti...,Health Services Department,Council Service,Official council webpage,NaN
9,Vaccination and Child Health Programmes,Implements vaccination campaigns and child hea...,Health Services Department,Council Service,Official council webpage,NaN


,service,description,department,category,source_type,source_url
0,Environmental Health,Safeguards community health through environmen...,Public Health Department,Council Service,Official council webpage,https://www.kabwecouncil.gov.zm/?page_id=2640
1,Disease Prevention and Control,"Provides disease prevention, control, health s...",Public Health Department,Council Service,Official council webpage,https://www.kabwecouncil.gov.zm/?page_id=2640
2,Waste Management,Provides solid waste management and waste-mana...,Public Health Department,Council Service,Official council webpage,https://www.kabwecouncil.gov.zm/?page_id=2640
3,Food Safety and Workplace Inspections,"Inspects restaurants, markets, factories and w...",Public Health Department,Council Service,Official council webpage,https://www.kabwecouncil.gov.zm/?page_id=2640
4,Public Health Awareness and Education,"Provides workshops, school programmes, media c...",Public Health Department,Council Service,Official council webpage,https://www.kabwecouncil.gov.zm/?page_id=2640
5,Pest and Vector Management,"Provides pest and vector surveillance, sprayin...",Public Health Department,Council Service,Official council webpage,https://www.kabwecouncil.gov.zm/?page_id=2640
6,Funeral and Burial Services,"Provides cemetery administration, plot allocat...",Public Health Department,Council Service,Official council webpage,https://www.kabwecouncil.gov.zm/?page_id=2640
7,Clinical and Nursing Services,Provides preventive and curative healthcare th...,Health Services Department,Council Service,Official council webpage,https://www.kabwecouncil.gov.zm/?page_id=2649
8,Pharmaceutical and Diagnostic Services,Provides pharmaceutical services and diagnosti...,Health Services Department,Council Service,Official council webpage,https://www.kabwecouncil.gov.zm/?page_id=2649
9,Vaccination and Child Health Programmes,Implements vaccination campaigns and child hea...,Health Services Department,Council Service,Official council webpage,https://www.kabwecouncil.gov.zm/?page_id=2649


Number of services: 31
Missing source URLs: 0
All services have official source URLs.


,service,description,department,category,source_type,source_url
0,Licenses and levies,Council service listed under the Services sect...,Council Services,Council Service,Official webpage,https://www.kabwecouncil.gov.zm/?page_id=2142
1,Waste Management,Council service listed under the Services sect...,Council Services,Council Service,Official webpage,https://www.kabwecouncil.gov.zm/?page_id=2150
2,E-GP,Council service listed under the Services sect...,Council Services,Council Service,Official webpage,https://www.kabwecouncil.gov.zm/?page_id=2155


Total services collected: 34


,service,description,department,category,source_type,source_url
0,Environmental Health,Safeguards community health through environmen...,Public Health Department,Council Service,Official council webpage,https://www.kabwecouncil.gov.zm/?page_id=2640
1,Disease Prevention and Control,"Provides disease prevention, control, health s...",Public Health Department,Council Service,Official council webpage,https://www.kabwecouncil.gov.zm/?page_id=2640
2,Waste Management,Provides solid waste management and waste-mana...,Public Health Department,Council Service,Official council webpage,https://www.kabwecouncil.gov.zm/?page_id=2640
3,Food Safety and Workplace Inspections,"Inspects restaurants, markets, factories and w...",Public Health Department,Council Service,Official council webpage,https://www.kabwecouncil.gov.zm/?page_id=2640
4,Public Health Awareness and Education,"Provides workshops, school programmes, media c...",Public Health Department,Council Service,Official council webpage,https://www.kabwecouncil.gov.zm/?page_id=2640
5,Pest and Vector Management,"Provides pest and vector surveillance, sprayin...",Public Health Department,Council Service,Official council webpage,https://www.kabwecouncil.gov.zm/?page_id=2640
6,Funeral and Burial Services,"Provides cemetery administration, plot allocat...",Public Health Department,Council Service,Official council webpage,https://www.kabwecouncil.gov.zm/?page_id=2640
7,Clinical and Nursing Services,Provides preventive and curative healthcare th...,Health Services Department,Council Service,Official council webpage,https://www.kabwecouncil.gov.zm/?page_id=2649
8,Pharmaceutical and Diagnostic Services,Provides pharmaceutical services and diagnosti...,Health Services Department,Council Service,Official council webpage,https://www.kabwecouncil.gov.zm/?page_id=2649
9,Vaccination and Child Health Programmes,Implements vaccination campaigns and child hea...,Health Services Department,Council Service,Official council webpage,https://www.kabwecouncil.gov.zm/?page_id=2649


Missing values by column:
--------------------------------
service        0
description    0
department     0
category       0
source_type    0
source_url     0
dtype: int64

Total duplicate rows:
--------------------------------
0

Duplicate service names:
--------------------------------


,service,description,department,category,source_type,source_url
2,Waste Management,Provides solid waste management and waste-mana...,Public Health Department,Council Service,Official council webpage,https://www.kabwecouncil.gov.zm/?page_id=2640
32,Waste Management,Council service listed under the Services sect...,Council Services,Council Service,Official webpage,https://www.kabwecouncil.gov.zm/?page_id=2150


Number of service records: 34
Number of columns: 7


,service_id,service,description,department,category,source_type,source_url
0,1,Environmental Health,Safeguards community health through environmen...,Public Health Department,Council Service,Official council webpage,https://www.kabwecouncil.gov.zm/?page_id=2640
1,2,Disease Prevention and Control,"Provides disease prevention, control, health s...",Public Health Department,Council Service,Official council webpage,https://www.kabwecouncil.gov.zm/?page_id=2640
2,3,Waste Management,Provides solid waste management and waste-mana...,Public Health Department,Council Service,Official council webpage,https://www.kabwecouncil.gov.zm/?page_id=2640
3,4,Food Safety and Workplace Inspections,"Inspects restaurants, markets, factories and w...",Public Health Department,Council Service,Official council webpage,https://www.kabwecouncil.gov.zm/?page_id=2640
4,5,Public Health Awareness and Education,"Provides workshops, school programmes, media c...",Public Health Department,Council Service,Official council webpage,https://www.kabwecouncil.gov.zm/?page_id=2640
5,6,Pest and Vector Management,"Provides pest and vector surveillance, sprayin...",Public Health Department,Council Service,Official council webpage,https://www.kabwecouncil.gov.zm/?page_id=2640
6,7,Funeral and Burial Services,"Provides cemetery administration, plot allocat...",Public Health Department,Council Service,Official council webpage,https://www.kabwecouncil.gov.zm/?page_id=2640
7,8,Clinical and Nursing Services,Provides preventive and curative healthcare th...,Health Services Department,Council Service,Official council webpage,https://www.kabwecouncil.gov.zm/?page_id=2649
8,9,Pharmaceutical and Diagnostic Services,Provides pharmaceutical services and diagnosti...,Health Services Department,Council Service,Official council webpage,https://www.kabwecouncil.gov.zm/?page_id=2649
9,10,Vaccination and Child Health Programmes,Implements vaccination campaigns and child hea...,Health Services Department,Council Service,Official council webpage,https://www.kabwecouncil.gov.zm/?page_id=2649


FINAL DATASET QUALITY CHECK
Number of records: 34
Number of columns: 7

Missing values:
service_id     0
service        0
description    0
department     0
category       0
source_type    0
source_url     0
dtype: int64

Duplicate rows:
0

Duplicate service IDs:
0

Unique departments:
8

Unique services:
33

Source types:
source_type
Official council webpage    31
Official webpage             3
Name: count, dtype: int64

Dataset columns:
['service_id', 'service', 'description', 'department', 'category', 'source_type', 'source_url']


Standardized source types:
source_type
Official council webpage    34
Name: count, dtype: int64


,service_id,service,source_type
0,1,Environmental Health,Official council webpage
1,2,Disease Prevention and Control,Official council webpage
2,3,Waste Management,Official council webpage
3,4,Food Safety and Workplace Inspections,Official council webpage
4,5,Public Health Awareness and Education,Official council webpage
5,6,Pest and Vector Management,Official council webpage
6,7,Funeral and Burial Services,Official council webpage
7,8,Clinical and Nursing Services,Official council webpage
8,9,Pharmaceutical and Diagnostic Services,Official council webpage
9,10,Vaccination and Child Health Programmes,Official council webpage


Dataset exported successfully.
File: db-unza26-csc4792-council-services-and-facilities.csv
Records: 34
Columns: 7


CSV verification
Records loaded: 34
Columns loaded: 7

Columns:
['service_id', 'service', 'description', 'department', 'category', 'source_type', 'source_url']

Missing values:
service_id     0
service        0
description    0
department     0
category       0
source_type    0
source_url     0
dtype: int64

First 5 records:


,service_id,service,description,department,category,source_type,source_url
0,1,Environmental Health,Safeguards community health through environmen...,Public Health Department,Council Service,Official council webpage,https://www.kabwecouncil.gov.zm/?page_id=2640
1,2,Disease Prevention and Control,"Provides disease prevention, control, health s...",Public Health Department,Council Service,Official council webpage,https://www.kabwecouncil.gov.zm/?page_id=2640
2,3,Waste Management,Provides solid waste management and waste-mana...,Public Health Department,Council Service,Official council webpage,https://www.kabwecouncil.gov.zm/?page_id=2640
3,4,Food Safety and Workplace Inspections,"Inspects restaurants, markets, factories and w...",Public Health Department,Council Service,Official council webpage,https://www.kabwecouncil.gov.zm/?page_id=2640
4,5,Public Health Awareness and Education,"Provides workshops, school programmes, media c...",Public Health Department,Council Service,Official council webpage,https://www.kabwecouncil.gov.zm/?page_id=2640


Number of Step 3 sources: 11


,source_name,description,source_url
0,Council homepage,Official council website and navigation used t...,https://www.kabwecouncil.gov.zm
1,Licenses and levies,Official Council Services webpage,https://www.kabwecouncil.gov.zm/?page_id=2142
2,Waste Management,Official Council Services webpage,https://www.kabwecouncil.gov.zm/?page_id=2150
3,E-GP,Official Council Services webpage,https://www.kabwecouncil.gov.zm/?page_id=2155
4,Public Health Department,Official departmental webpage,https://www.kabwecouncil.gov.zm/?page_id=2640
5,Health Services Department,Official departmental webpage,https://www.kabwecouncil.gov.zm/?page_id=2649
6,Legal Services Department,Official departmental webpage,https://www.kabwecouncil.gov.zm/?page_id=3894
7,Planning Department,Official departmental webpage,https://www.kabwecouncil.gov.zm/?page_id=3897
8,Housing and Social Services Department,Official departmental webpage,https://www.kabwecouncil.gov.zm/?page_id=3900
9,"Fisheries, Livestock and Veterinary Services D...",Official departmental webpage,https://www.kabwecouncil.gov.zm/?page_id=3903


STEP 3 SOURCE REGISTER VALIDATION
Number of sources: 11

Missing values:
source_name    0
description    0
source_url     0
dtype: int64

Duplicate source URLs:
0

Source types:
description
Official departmental webpage                                                        7
Official Council Services webpage                                                    3
Official council website and navigation used to identify services and departments    1
Name: count, dtype: int64


Source register exported successfully.
File: db-unza26-csc4792-csc4792-council-services-sources.csv
Sources: 11


Source register exported successfully.
File: db-unza26-csc4792-council-services-sources.csv
Sources: 11


In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

print("Libraries imported successfully.")

Libraries imported successfully.


In [ ]:
council_base_url = "https://www.kabwecouncil.gov.zm"

print("Official council website:")
print(council_base_url)

Official council website:
https://www.kabwecouncil.gov.zm


In [ ]:
response = requests.get(
    council_base_url,
    timeout=30,
    verify=False
)

print("Status code:", response.status_code)
print("Content type:", response.headers.get("Content-Type"))
print("Page size:", len(response.text), "characters")

C:\Users\Pardo\AppData\Local\Programs\Python\Python314\Lib\site-packages\urllib3\connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host 'www.kabwecouncil.gov.zm'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


Status code: 200
Content type: text/html; charset=UTF-8
Page size: 161419 characters


In [ ]:
soup = BeautifulSoup(
    response.text,
    "html.parser"
)

homepage_text = soup.get_text(
    separator="\n",
    strip=True
)

print("First 5000 characters of the council homepage:")
print("-----------------------------------------------")
print(homepage_text[:5000])

First 5000 characters of the council homepage:
-----------------------------------------------
Kabwe Municipal Council – Kabwe
Home
About
Open menu
About Us
Mandate
Who we are
Departments
office of the the town clerk
Dept of Human Resource and Administration
Dept of Health
Dept of Finance
Dept of Engineering Services
Dept of Health Services
Dept Legal ( Services Unit)
Dept Of Planning
Dept Of Housing and Social Services
Dept of Fisheries, Livestock And Veterinary Services
Dept of Agriculture
Civic Leaders
The Mayor
Kabwe Central
Bwacha
Services
Open menu
Licenses and levies
Waste Management
E-GP
News Updates
CDF
Open menu
CDF GUIDLINES
CDF branding guidelines
ZDSP
Media
Open menu
Publications
Public Notices
Photo Gallery
video gallery
Forms
ADD A VIDEO
Contact Us
FAQS
Menu
Home
About
Open menu
About Us
Mandate
Who we are
Departments
office of the the town clerk
Dept of Human Resource and Administration
Dept of Health
Dept of Finance
Dept of Engineering Services
Dept of Health Services


In [ ]:
services_urls = {
    "licenses_and_levies": (
        council_base_url + "/?page_id=2142"
    ),
    "waste_management": (
        council_base_url + "/?page_id=2150"
    ),
    "e_gp": (
        council_base_url + "/?page_id=2155"
    )
}

print("Council Services URLs:")
print("======================")

for service, url in services_urls.items():
    print(service, ":", url)

Council Services URLs:
licenses_and_levies : https://www.kabwecouncil.gov.zm/?page_id=2142
waste_management : https://www.kabwecouncil.gov.zm/?page_id=2150
e_gp : https://www.kabwecouncil.gov.zm/?page_id=2155


In [ ]:
service_responses = {}

for service, url in services_urls.items():
    try:
        page_response = requests.get(
            url,
            timeout=30,
            verify=False
        )

        service_responses[service] = page_response

        print(service)
        print("URL:", url)
        print("Status code:", page_response.status_code)
        print("Content type:", page_response.headers.get("Content-Type"))
        print("Page size:", len(page_response.text), "characters")
        print("----------------------------------------")

    except requests.RequestException as error:
        print(service)
        print("Error:", error)
        print("----------------------------------------")

C:\Users\Pardo\AppData\Local\Programs\Python\Python314\Lib\site-packages\urllib3\connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host 'www.kabwecouncil.gov.zm'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
C:\Users\Pardo\AppData\Local\Programs\Python\Python314\Lib\site-packages\urllib3\connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host 'www.kabwecouncil.gov.zm'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


licenses_and_levies
URL: https://www.kabwecouncil.gov.zm/?page_id=2142
Status code: 200
Content type: text/html; charset=UTF-8
Page size: 82999 characters
----------------------------------------


C:\Users\Pardo\AppData\Local\Programs\Python\Python314\Lib\site-packages\urllib3\connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host 'www.kabwecouncil.gov.zm'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
C:\Users\Pardo\AppData\Local\Programs\Python\Python314\Lib\site-packages\urllib3\connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host 'www.kabwecouncil.gov.zm'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


waste_management
URL: https://www.kabwecouncil.gov.zm/?page_id=2150
Status code: 200
Content type: text/html; charset=UTF-8
Page size: 83062 characters
----------------------------------------


C:\Users\Pardo\AppData\Local\Programs\Python\Python314\Lib\site-packages\urllib3\connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host 'www.kabwecouncil.gov.zm'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
C:\Users\Pardo\AppData\Local\Programs\Python\Python314\Lib\site-packages\urllib3\connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host 'www.kabwecouncil.gov.zm'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


e_gp
URL: https://www.kabwecouncil.gov.zm/?page_id=2155
Status code: 200
Content type: text/html; charset=UTF-8
Page size: 82960 characters
----------------------------------------


In [ ]:
service_texts = {}

for service, page_response in service_responses.items():
    service_soup = BeautifulSoup(
        page_response.text,
        "html.parser"
    )

    service_texts[service] = service_soup.get_text(
        separator="\n",
        strip=True
    )

    print("=" * 60)
    print(service.upper())
    print("=" * 60)
    print(service_texts[service][:5000])
    print()

LICENSES_AND_LEVIES
IMG-20250115-WA0025 – Kabwe Municipal Council
Home
About
Open menu
About Us
Mandate
Who we are
Departments
office of the the town clerk
Dept of Human Resource and Administration
Dept of Health
Dept of Finance
Dept of Engineering Services
Dept of Health Services
Dept Legal ( Services Unit)
Dept Of Planning
Dept Of Housing and Social Services
Dept of Fisheries, Livestock And Veterinary Services
Dept of Agriculture
Civic Leaders
The Mayor
Kabwe Central
Bwacha
Services
Open menu
Licenses and levies
Waste Management
E-GP
News Updates
CDF
Open menu
CDF GUIDLINES
CDF branding guidelines
ZDSP
Media
Open menu
Publications
Public Notices
Photo Gallery
video gallery
Forms
ADD A VIDEO
Contact Us
FAQS
Menu
Home
About
Open menu
About Us
Mandate
Who we are
Departments
office of the the town clerk
Dept of Human Resource and Administration
Dept of Health
Dept of Finance
Dept of Engineering Services
Dept of Health Services
Dept Legal ( Services Unit)
Dept Of Planning
Dept Of Housing 

In [ ]:
services_data = [
    [
        "Licenses and levies",
        "Council service listed under the Services section of the official website",
        "Council Service",
        "Official webpage",
        services_urls["licenses_and_levies"]
    ],
    [
        "Waste Management",
        "Council service listed under the Services section of the official website",
        "Council Service",
        "Official webpage",
        services_urls["waste_management"]
    ],
    [
        "E-GP",
        "Council service listed under the Services section of the official website",
        "Council Service",
        "Official webpage",
        services_urls["e_gp"]
    ]
]

services_df = pd.DataFrame(
    services_data,
    columns=[
        "service",
        "description",
        "category",
        "source_type",
        "source_url"
    ]
)

display(services_df)

,service,description,category,source_type,source_url
0,Licenses and levies,Council service listed under the Services sect...,Council Service,Official webpage,https://www.kabwecouncil.gov.zm/?page_id=2142
1,Waste Management,Council service listed under the Services sect...,Council Service,Official webpage,https://www.kabwecouncil.gov.zm/?page_id=2150
2,E-GP,Council service listed under the Services sect...,Council Service,Official webpage,https://www.kabwecouncil.gov.zm/?page_id=2155


In [ ]:
departments_url = council_base_url + "/?page_id=2072"

departments_response = requests.get(
    departments_url,
    timeout=30,
    verify=False
)

print("Departments page:")
print("URL:", departments_url)
print("Status code:", departments_response.status_code)
print("Content type:", departments_response.headers.get("Content-Type"))
print("Page size:", len(departments_response.text), "characters")

C:\Users\Pardo\AppData\Local\Programs\Python\Python314\Lib\site-packages\urllib3\connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host 'www.kabwecouncil.gov.zm'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


Departments page:
URL: https://www.kabwecouncil.gov.zm/?page_id=2072
Status code: 404
Content type: text/html; charset=UTF-8
Page size: 81506 characters


In [ ]:
department_links = []

for link in soup.find_all("a", href=True):
    link_text = link.get_text(" ", strip=True)

    if (
        "dept" in link_text.lower()
        or "office of the town clerk" in link_text.lower()
    ):
        department_links.append(
            {
                "department": link_text,
                "url": link["href"]
            }
        )

departments_df = pd.DataFrame(
    department_links
).drop_duplicates()

display(departments_df)

,department,url
0,Dept of Human Resource and Administration,https://www.kabwecouncil.gov.zm/?page_id=2637
1,Dept of Health,https://www.kabwecouncil.gov.zm/?page_id=2640
2,Dept of Finance,https://www.kabwecouncil.gov.zm/?page_id=2643
3,Dept of Engineering Services,https://www.kabwecouncil.gov.zm/?page_id=2646
4,Dept of Health Services,https://www.kabwecouncil.gov.zm/?page_id=2649
5,Dept Legal ( Services Unit),https://www.kabwecouncil.gov.zm/?page_id=3894
6,Dept Of Planning,https://www.kabwecouncil.gov.zm/?page_id=3897
7,Dept Of Housing and Social Services,https://www.kabwecouncil.gov.zm/?page_id=3900
8,"Dept of Fisheries, Livestock And Veterinary Se...",https://www.kabwecouncil.gov.zm/?page_id=3903
9,Dept of Agriculture,https://www.kabwecouncil.gov.zm/?page_id=3906


In [ ]:
departments_df = departments_df.copy()

# Remove empty department names
departments_df = departments_df[
    departments_df["department"].notna()
]

# Remove empty URLs
departments_df = departments_df[
    departments_df["url"].notna()
]

# Remove duplicate department/URL combinations
departments_df = departments_df.drop_duplicates(
    subset=["department", "url"]
)

# Reset the row numbers
departments_df = departments_df.reset_index(drop=True)

print("Number of department links found:", len(departments_df))
print()

display(departments_df)

Number of department links found: 10



,department,url
0,Dept of Human Resource and Administration,https://www.kabwecouncil.gov.zm/?page_id=2637
1,Dept of Health,https://www.kabwecouncil.gov.zm/?page_id=2640
2,Dept of Finance,https://www.kabwecouncil.gov.zm/?page_id=2643
3,Dept of Engineering Services,https://www.kabwecouncil.gov.zm/?page_id=2646
4,Dept of Health Services,https://www.kabwecouncil.gov.zm/?page_id=2649
5,Dept Legal ( Services Unit),https://www.kabwecouncil.gov.zm/?page_id=3894
6,Dept Of Planning,https://www.kabwecouncil.gov.zm/?page_id=3897
7,Dept Of Housing and Social Services,https://www.kabwecouncil.gov.zm/?page_id=3900
8,"Dept of Fisheries, Livestock And Veterinary Se...",https://www.kabwecouncil.gov.zm/?page_id=3903
9,Dept of Agriculture,https://www.kabwecouncil.gov.zm/?page_id=3906


In [ ]:
department_status = []

for _, row in departments_df.iterrows():
    department = row["department"]
    url = row["url"]

    try:
        department_response = requests.get(
            url,
            timeout=30,
            verify=False
        )

        department_status.append(
            [
                department,
                url,
                department_response.status_code,
                len(department_response.text)
            ]
        )

    except requests.RequestException as error:
        department_status.append(
            [
                department,
                url,
                "ERROR",
                str(error)
            ]
        )

department_status_df = pd.DataFrame(
    department_status,
    columns=[
        "department",
        "url",
        "status_code",
        "page_size"
    ]
)

display(department_status_df)

C:\Users\Pardo\AppData\Local\Programs\Python\Python314\Lib\site-packages\urllib3\connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host 'www.kabwecouncil.gov.zm'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
C:\Users\Pardo\AppData\Local\Programs\Python\Python314\Lib\site-packages\urllib3\connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host 'www.kabwecouncil.gov.zm'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
C:\Users\Pardo\AppData\Local\Programs\Python\Python314\Lib\site-packages\urllib3\connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host 'www.kabwecouncil.gov.zm'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/l

,department,url,status_code,page_size
0,Dept of Human Resource and Administration,https://www.kabwecouncil.gov.zm/?page_id=2637,200,110630
1,Dept of Health,https://www.kabwecouncil.gov.zm/?page_id=2640,200,137426
2,Dept of Finance,https://www.kabwecouncil.gov.zm/?page_id=2643,200,103806
3,Dept of Engineering Services,https://www.kabwecouncil.gov.zm/?page_id=2646,200,100085
4,Dept of Health Services,https://www.kabwecouncil.gov.zm/?page_id=2649,200,117513
5,Dept Legal ( Services Unit),https://www.kabwecouncil.gov.zm/?page_id=3894,200,112308
6,Dept Of Planning,https://www.kabwecouncil.gov.zm/?page_id=3897,200,105198
7,Dept Of Housing and Social Services,https://www.kabwecouncil.gov.zm/?page_id=3900,200,100272
8,"Dept of Fisheries, Livestock And Veterinary Se...",https://www.kabwecouncil.gov.zm/?page_id=3903,200,123830
9,Dept of Agriculture,https://www.kabwecouncil.gov.zm/?page_id=3906,200,92673


In [ ]:
department_texts = {}

for _, row in department_status_df.iterrows():

    department = row["department"]
    url = row["url"]

    if row["status_code"] == 200:

        department_response = requests.get(
            url,
            timeout=30,
            verify=False
        )

        department_soup = BeautifulSoup(
            department_response.text,
            "html.parser"
        )

        department_texts[department] = department_soup.get_text(
            separator="\n",
            strip=True
        )

print("Number of department pages extracted:", len(department_texts))

C:\Users\Pardo\AppData\Local\Programs\Python\Python314\Lib\site-packages\urllib3\connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host 'www.kabwecouncil.gov.zm'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
C:\Users\Pardo\AppData\Local\Programs\Python\Python314\Lib\site-packages\urllib3\connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host 'www.kabwecouncil.gov.zm'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
C:\Users\Pardo\AppData\Local\Programs\Python\Python314\Lib\site-packages\urllib3\connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host 'www.kabwecouncil.gov.zm'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/l

Number of department pages extracted: 10


In [ ]:
for department, text in department_texts.items():

    print("=" * 70)
    print(department.upper())
    print("=" * 70)

    lines = [
        line.strip()
        for line in text.splitlines()
        if line.strip()
    ]

    # Remove obvious repeated navigation items
    useful_lines = []

    for line in lines:
        if line not in useful_lines:
            useful_lines.append(line)

    print("\n".join(useful_lines[:80]))
    print()

DEPT OF HUMAN RESOURCE AND ADMINISTRATION
Dept of Human Resource and Administration – Kabwe Municipal Council
Home
About
Open menu
About Us
Mandate
Who we are
Departments
office of the the town clerk
Dept of Human Resource and Administration
Dept of Health
Dept of Finance
Dept of Engineering Services
Dept of Health Services
Dept Legal ( Services Unit)
Dept Of Planning
Dept Of Housing and Social Services
Dept of Fisheries, Livestock And Veterinary Services
Dept of Agriculture
Civic Leaders
The Mayor
Kabwe Central
Bwacha
Services
Licenses and levies
Waste Management
E-GP
News Updates
CDF
CDF GUIDLINES
CDF branding guidelines
ZDSP
Media
Publications
Public Notices
Photo Gallery
video gallery
Forms
ADD A VIDEO
Contact Us
FAQS
Menu
Search
Human Resource and Administration
NSAKANYA J CHANGWE
Director of Human Resource and Administration
A small river named Duden flows by their place and supplies it with the necessary
Phone:
+1 (859) 254-6589
Email:
info@example.com
The directorate is headed 

## Step 3.17 — Department Page Data Quality Note

The official Kabwe Municipal Council website contains detailed information
on several departmental pages, including descriptions of services,
responsibilities, and functions.

During extraction, some departmental pages were found to contain apparent
template or copy-editing errors. For example, certain pages refer to other
local authorities such as Mongu Municipal Council or Kazungula Town Council.

Therefore, information containing an incorrect local-authority reference
will not automatically be treated as a Kabwe-specific fact.

For this dataset, service information will be included only where:

- the service or function is clearly identified on the official council
  webpage;
- the information is relevant to the department being described; and
- the information does not depend on an obviously incorrect reference to
  another local authority.

This approach preserves the provenance of the official source while
reducing the risk of introducing inaccurate information into the dataset.

In [ ]:
curated_services = [
    [
        "Environmental Health",
        "Safeguards community health through environmental health activities",
        "Public Health Department",
        "Council Service"
    ],
    [
        "Disease Prevention and Control",
        "Provides disease prevention, control, health screening and outbreak response activities",
        "Public Health Department",
        "Council Service"
    ],
    [
        "Waste Management",
        "Provides solid waste management and waste-management activities",
        "Public Health Department",
        "Council Service"
    ],
    [
        "Food Safety and Workplace Inspections",
        "Inspects restaurants, markets, factories and workplaces and supports compliance certification",
        "Public Health Department",
        "Council Service"
    ],
    [
        "Public Health Awareness and Education",
        "Provides workshops, school programmes, media campaigns and community education",
        "Public Health Department",
        "Council Service"
    ],
    [
        "Pest and Vector Management",
        "Provides pest and vector surveillance, spraying and community clean-up activities",
        "Public Health Department",
        "Council Service"
    ],
    [
        "Funeral and Burial Services",
        "Provides cemetery administration, plot allocation, crematorium operations and mortuary support",
        "Public Health Department",
        "Council Service"
    ],
    [
        "Clinical and Nursing Services",
        "Provides preventive and curative healthcare through clinical and nursing services",
        "Health Services Department",
        "Council Service"
    ],
    [
        "Pharmaceutical and Diagnostic Services",
        "Provides pharmaceutical services and diagnostic services including laboratory testing",
        "Health Services Department",
        "Council Service"
    ],
    [
        "Vaccination and Child Health Programmes",
        "Implements vaccination campaigns and child health programmes",
        "Health Services Department",
        "Council Service"
    ],
    [
        "Maternal Health and Family Planning",
        "Coordinates maternal health and family planning services",
        "Health Services Department",
        "Council Service"
    ],
    [
        "Health Education and Disease Surveillance",
        "Manages health education and disease surveillance programmes",
        "Health Services Department",
        "Council Service"
    ],
    [
        "Legal Advisory Services",
        "Provides legal advice to the Council and supports compliance with applicable laws",
        "Legal Services Department",
        "Council Service"
    ],
    [
        "Trading Licence Processing",
        "Processes applications for various trading licences and permits",
        "Legal Services Department",
        "Council Service"
    ],
    [
        "Land Application Processing",
        "Processes land applications and supports land-related administrative procedures",
        "Legal Services Department",
        "Council Service"
    ],
    [
        "Building Inspections",
        "Conducts building inspections and identifies unauthorised development",
        "Planning Department",
        "Council Service"
    ],
    [
        "Plot Numbering",
        "Provides plot numbering as part of town planning activities",
        "Planning Department",
        "Council Service"
    ],
    [
        "Survey Services",
        "Provides property boundary, road marking, survey diagram and survey report services",
        "Planning Department",
        "Council Service"
    ],
    [
        "GIS Management",
        "Manages geographical information systems",
        "Planning Department",
        "Council Service"
    ],
    [
        "Environmental Planning",
        "Supports environmental assessment, environmental profiling and environmental planning",
        "Planning Department",
        "Council Service"
    ],
    [
        "Community Development Services",
        "Facilitates community development activities and community mobilisation",
        "Housing and Social Services Department",
        "Council Service"
    ],
    [
        "Social Protection Programmes",
        "Coordinates and implements social protection programmes",
        "Housing and Social Services Department",
        "Council Service"
    ],
    [
        "Housing Services",
        "Supports housing services and upgrading of unplanned settlements",
        "Housing and Social Services Department",
        "Council Service"
    ],
    [
        "Markets and Bus Stations Management",
        "Manages markets and bus stations to support smooth operations and revenue collection",
        "Housing and Social Services Department",
        "Council Service"
    ],
    [
        "Library Services",
        "Provides library services",
        "Housing and Social Services Department",
        "Council Service"
    ],
    [
        "Sports and Recreation",
        "Provides sports and recreation activities",
        "Housing and Social Services Department",
        "Council Service"
    ],
    [
        "Skills Development",
        "Supports skills development programmes within communities",
        "Housing and Social Services Department",
        "Council Service"
    ],
    [
        "Veterinary Services",
        "Provides veterinary services across the district",
        "Fisheries, Livestock and Veterinary Services Department",
        "Council Service"
    ],
    [
        "Livestock Development",
        "Supports livestock productivity, welfare and sustainable livestock farming",
        "Fisheries, Livestock and Veterinary Services Department",
        "Council Service"
    ],
    [
        "Fisheries and Aquaculture Services",
        "Supports fisheries and aquaculture development, training, regulation and education",
        "Fisheries, Livestock and Veterinary Services Department",
        "Council Service"
    ],
    [
        "Agricultural Extension Services",
        "Provides extension services to farmers and supports agricultural development",
        "Agriculture Department",
        "Council Service"
    ]
]

curated_services_df = pd.DataFrame(
    curated_services,
    columns=[
        "service",
        "description",
        "department",
        "category"
    ]
)

display(curated_services_df)

,service,description,department,category
0,Environmental Health,Safeguards community health through environmen...,Public Health Department,Council Service
1,Disease Prevention and Control,"Provides disease prevention, control, health s...",Public Health Department,Council Service
2,Waste Management,Provides solid waste management and waste-mana...,Public Health Department,Council Service
3,Food Safety and Workplace Inspections,"Inspects restaurants, markets, factories and w...",Public Health Department,Council Service
4,Public Health Awareness and Education,"Provides workshops, school programmes, media c...",Public Health Department,Council Service
5,Pest and Vector Management,"Provides pest and vector surveillance, sprayin...",Public Health Department,Council Service
6,Funeral and Burial Services,"Provides cemetery administration, plot allocat...",Public Health Department,Council Service
7,Clinical and Nursing Services,Provides preventive and curative healthcare th...,Health Services Department,Council Service
8,Pharmaceutical and Diagnostic Services,Provides pharmaceutical services and diagnosti...,Health Services Department,Council Service
9,Vaccination and Child Health Programmes,Implements vaccination campaigns and child hea...,Health Services Department,Council Service


In [ ]:
# Create a lookup table connecting each department
# to its official Kabwe Municipal Council webpage.

department_url_lookup = dict(
    zip(
        departments_df["department"],
        departments_df["url"]
    )
)

# Add the official source URL to each service record.
curated_services_df["source_type"] = "Official council webpage"

curated_services_df["source_url"] = (
    curated_services_df["department"]
    .map(department_url_lookup)
)

display(curated_services_df)

,service,description,department,category,source_type,source_url
0,Environmental Health,Safeguards community health through environmen...,Public Health Department,Council Service,Official council webpage,NaN
1,Disease Prevention and Control,"Provides disease prevention, control, health s...",Public Health Department,Council Service,Official council webpage,NaN
2,Waste Management,Provides solid waste management and waste-mana...,Public Health Department,Council Service,Official council webpage,NaN
3,Food Safety and Workplace Inspections,"Inspects restaurants, markets, factories and w...",Public Health Department,Council Service,Official council webpage,NaN
4,Public Health Awareness and Education,"Provides workshops, school programmes, media c...",Public Health Department,Council Service,Official council webpage,NaN
5,Pest and Vector Management,"Provides pest and vector surveillance, sprayin...",Public Health Department,Council Service,Official council webpage,NaN
6,Funeral and Burial Services,"Provides cemetery administration, plot allocat...",Public Health Department,Council Service,Official council webpage,NaN
7,Clinical and Nursing Services,Provides preventive and curative healthcare th...,Health Services Department,Council Service,Official council webpage,NaN
8,Pharmaceutical and Diagnostic Services,Provides pharmaceutical services and diagnosti...,Health Services Department,Council Service,Official council webpage,NaN
9,Vaccination and Child Health Programmes,Implements vaccination campaigns and child hea...,Health Services Department,Council Service,Official council webpage,NaN


In [ ]:
# Manually map each department used in the curated dataset
# to the correct official Kabwe Municipal Council webpage.

department_url_map = {
    "Public Health Department": department_links[
        next(
            i for i, item in enumerate(department_links)
            if "health" in item["department"].lower()
            and "health services" not in item["department"].lower()
        )
    ]["url"],

    "Health Services Department": department_links[
        next(
            i for i, item in enumerate(department_links)
            if "health services" in item["department"].lower()
        )
    ]["url"],

    "Legal Services Department": department_links[
        next(
            i for i, item in enumerate(department_links)
            if "legal" in item["department"].lower()
        )
    ]["url"],

    "Planning Department": department_links[
        next(
            i for i, item in enumerate(department_links)
            if "planning" in item["department"].lower()
        )
    ]["url"],

    "Housing and Social Services Department": department_links[
        next(
            i for i, item in enumerate(department_links)
            if "housing" in item["department"].lower()
        )
    ]["url"],

    "Fisheries, Livestock and Veterinary Services Department": department_links[
        next(
            i for i, item in enumerate(department_links)
            if "fisheries" in item["department"].lower()
        )
    ]["url"],

    "Agriculture Department": department_links[
        next(
            i for i, item in enumerate(department_links)
            if "agriculture" in item["department"].lower()
        )
    ]["url"]
}

# Add source information
curated_services_df["source_type"] = "Official council webpage"

curated_services_df["source_url"] = (
    curated_services_df["department"]
    .map(department_url_map)
)

display(curated_services_df)

,service,description,department,category,source_type,source_url
0,Environmental Health,Safeguards community health through environmen...,Public Health Department,Council Service,Official council webpage,https://www.kabwecouncil.gov.zm/?page_id=2640
1,Disease Prevention and Control,"Provides disease prevention, control, health s...",Public Health Department,Council Service,Official council webpage,https://www.kabwecouncil.gov.zm/?page_id=2640
2,Waste Management,Provides solid waste management and waste-mana...,Public Health Department,Council Service,Official council webpage,https://www.kabwecouncil.gov.zm/?page_id=2640
3,Food Safety and Workplace Inspections,"Inspects restaurants, markets, factories and w...",Public Health Department,Council Service,Official council webpage,https://www.kabwecouncil.gov.zm/?page_id=2640
4,Public Health Awareness and Education,"Provides workshops, school programmes, media c...",Public Health Department,Council Service,Official council webpage,https://www.kabwecouncil.gov.zm/?page_id=2640
5,Pest and Vector Management,"Provides pest and vector surveillance, sprayin...",Public Health Department,Council Service,Official council webpage,https://www.kabwecouncil.gov.zm/?page_id=2640
6,Funeral and Burial Services,"Provides cemetery administration, plot allocat...",Public Health Department,Council Service,Official council webpage,https://www.kabwecouncil.gov.zm/?page_id=2640
7,Clinical and Nursing Services,Provides preventive and curative healthcare th...,Health Services Department,Council Service,Official council webpage,https://www.kabwecouncil.gov.zm/?page_id=2649
8,Pharmaceutical and Diagnostic Services,Provides pharmaceutical services and diagnosti...,Health Services Department,Council Service,Official council webpage,https://www.kabwecouncil.gov.zm/?page_id=2649
9,Vaccination and Child Health Programmes,Implements vaccination campaigns and child hea...,Health Services Department,Council Service,Official council webpage,https://www.kabwecouncil.gov.zm/?page_id=2649


In [ ]:
# Check whether any source URLs are missing

missing_urls = curated_services_df[
    curated_services_df["source_url"].isna()
]

print("Number of services:", len(curated_services_df))
print("Missing source URLs:", len(missing_urls))

if len(missing_urls) > 0:
    print("\nServices with missing URLs:")
    display(
        missing_urls[
            ["service", "department"]
        ]
    )
else:
    print("All services have official source URLs.")

Number of services: 31
Missing source URLs: 0
All services have official source URLs.


In [ ]:
# Prepare the three services listed directly under
# the official Council Services section.

standalone_services_df = services_df.copy()

# Rename the columns so they match the curated services dataset.
standalone_services_df = standalone_services_df[
    [
        "service",
        "description",
        "category",
        "source_type",
        "source_url"
    ]
]

# Add a department column.
standalone_services_df["department"] = "Council Services"

# Reorder columns to match the departmental dataset.
standalone_services_df = standalone_services_df[
    [
        "service",
        "description",
        "department",
        "category",
        "source_type",
        "source_url"
    ]
]

display(standalone_services_df)

,service,description,department,category,source_type,source_url
0,Licenses and levies,Council service listed under the Services sect...,Council Services,Council Service,Official webpage,https://www.kabwecouncil.gov.zm/?page_id=2142
1,Waste Management,Council service listed under the Services sect...,Council Services,Council Service,Official webpage,https://www.kabwecouncil.gov.zm/?page_id=2150
2,E-GP,Council service listed under the Services sect...,Council Services,Council Service,Official webpage,https://www.kabwecouncil.gov.zm/?page_id=2155


In [ ]:
# Select the columns from the departmental services
# so that they match the standalone services dataset.

departmental_services_df = curated_services_df[
    [
        "service",
        "description",
        "department",
        "category",
        "source_type",
        "source_url"
    ]
].copy()

# Combine departmental services with the services
# listed directly under the Council Services section.

all_services_df = pd.concat(
    [
        departmental_services_df,
        standalone_services_df
    ],
    ignore_index=True
)

print("Total services collected:", len(all_services_df))

display(all_services_df)

Total services collected: 34


,service,description,department,category,source_type,source_url
0,Environmental Health,Safeguards community health through environmen...,Public Health Department,Council Service,Official council webpage,https://www.kabwecouncil.gov.zm/?page_id=2640
1,Disease Prevention and Control,"Provides disease prevention, control, health s...",Public Health Department,Council Service,Official council webpage,https://www.kabwecouncil.gov.zm/?page_id=2640
2,Waste Management,Provides solid waste management and waste-mana...,Public Health Department,Council Service,Official council webpage,https://www.kabwecouncil.gov.zm/?page_id=2640
3,Food Safety and Workplace Inspections,"Inspects restaurants, markets, factories and w...",Public Health Department,Council Service,Official council webpage,https://www.kabwecouncil.gov.zm/?page_id=2640
4,Public Health Awareness and Education,"Provides workshops, school programmes, media c...",Public Health Department,Council Service,Official council webpage,https://www.kabwecouncil.gov.zm/?page_id=2640
5,Pest and Vector Management,"Provides pest and vector surveillance, sprayin...",Public Health Department,Council Service,Official council webpage,https://www.kabwecouncil.gov.zm/?page_id=2640
6,Funeral and Burial Services,"Provides cemetery administration, plot allocat...",Public Health Department,Council Service,Official council webpage,https://www.kabwecouncil.gov.zm/?page_id=2640
7,Clinical and Nursing Services,Provides preventive and curative healthcare th...,Health Services Department,Council Service,Official council webpage,https://www.kabwecouncil.gov.zm/?page_id=2649
8,Pharmaceutical and Diagnostic Services,Provides pharmaceutical services and diagnosti...,Health Services Department,Council Service,Official council webpage,https://www.kabwecouncil.gov.zm/?page_id=2649
9,Vaccination and Child Health Programmes,Implements vaccination campaigns and child hea...,Health Services Department,Council Service,Official council webpage,https://www.kabwecouncil.gov.zm/?page_id=2649


In [ ]:
# Check the combined services dataset for missing values.

print("Missing values by column:")
print("--------------------------------")

print(
    all_services_df.isnull().sum()
)

print("\nTotal duplicate rows:")
print("--------------------------------")

print(
    all_services_df.duplicated().sum()
)

print("\nDuplicate service names:")
print("--------------------------------")

duplicate_services = all_services_df[
    all_services_df["service"].duplicated(
        keep=False
    )
].sort_values("service")

if len(duplicate_services) > 0:
    display(duplicate_services)
else:
    print("No duplicate service names found.")

Missing values by column:
--------------------------------
service        0
description    0
department     0
category       0
source_type    0
source_url     0
dtype: int64

Total duplicate rows:
--------------------------------
0

Duplicate service names:
--------------------------------


,service,description,department,category,source_type,source_url
2,Waste Management,Provides solid waste management and waste-mana...,Public Health Department,Council Service,Official council webpage,https://www.kabwecouncil.gov.zm/?page_id=2640
32,Waste Management,Council service listed under the Services sect...,Council Services,Council Service,Official webpage,https://www.kabwecouncil.gov.zm/?page_id=2150


In [ ]:
# Add a unique identifier to each service record.

all_services_df.insert(
    0,
    "service_id",
    range(1, len(all_services_df) + 1)
)

print("Number of service records:", len(all_services_df))
print("Number of columns:", len(all_services_df.columns))

display(all_services_df.head(10))

Number of service records: 34
Number of columns: 7


,service_id,service,description,department,category,source_type,source_url
0,1,Environmental Health,Safeguards community health through environmen...,Public Health Department,Council Service,Official council webpage,https://www.kabwecouncil.gov.zm/?page_id=2640
1,2,Disease Prevention and Control,"Provides disease prevention, control, health s...",Public Health Department,Council Service,Official council webpage,https://www.kabwecouncil.gov.zm/?page_id=2640
2,3,Waste Management,Provides solid waste management and waste-mana...,Public Health Department,Council Service,Official council webpage,https://www.kabwecouncil.gov.zm/?page_id=2640
3,4,Food Safety and Workplace Inspections,"Inspects restaurants, markets, factories and w...",Public Health Department,Council Service,Official council webpage,https://www.kabwecouncil.gov.zm/?page_id=2640
4,5,Public Health Awareness and Education,"Provides workshops, school programmes, media c...",Public Health Department,Council Service,Official council webpage,https://www.kabwecouncil.gov.zm/?page_id=2640
5,6,Pest and Vector Management,"Provides pest and vector surveillance, sprayin...",Public Health Department,Council Service,Official council webpage,https://www.kabwecouncil.gov.zm/?page_id=2640
6,7,Funeral and Burial Services,"Provides cemetery administration, plot allocat...",Public Health Department,Council Service,Official council webpage,https://www.kabwecouncil.gov.zm/?page_id=2640
7,8,Clinical and Nursing Services,Provides preventive and curative healthcare th...,Health Services Department,Council Service,Official council webpage,https://www.kabwecouncil.gov.zm/?page_id=2649
8,9,Pharmaceutical and Diagnostic Services,Provides pharmaceutical services and diagnosti...,Health Services Department,Council Service,Official council webpage,https://www.kabwecouncil.gov.zm/?page_id=2649
9,10,Vaccination and Child Health Programmes,Implements vaccination campaigns and child hea...,Health Services Department,Council Service,Official council webpage,https://www.kabwecouncil.gov.zm/?page_id=2649


In [ ]:
# Final quality checks for the Council Services and Facilities dataset.

print("FINAL DATASET QUALITY CHECK")
print("=" * 50)

print("Number of records:", len(all_services_df))
print("Number of columns:", len(all_services_df.columns))

print("\nMissing values:")
print(all_services_df.isnull().sum())

print("\nDuplicate rows:")
print(all_services_df.duplicated().sum())

print("\nDuplicate service IDs:")
print(all_services_df["service_id"].duplicated().sum())

print("\nUnique departments:")
print(all_services_df["department"].nunique())

print("\nUnique services:")
print(all_services_df["service"].nunique())

print("\nSource types:")
print(all_services_df["source_type"].value_counts())

print("\nDataset columns:")
print(list(all_services_df.columns))

FINAL DATASET QUALITY CHECK
Number of records: 34
Number of columns: 7

Missing values:
service_id     0
service        0
description    0
department     0
category       0
source_type    0
source_url     0
dtype: int64

Duplicate rows:
0

Duplicate service IDs:
0

Unique departments:
8

Unique services:
33

Source types:
source_type
Official council webpage    31
Official webpage             3
Name: count, dtype: int64

Dataset columns:
['service_id', 'service', 'description', 'department', 'category', 'source_type', 'source_url']


In [ ]:
# Standardize source_type values so that the dataset
# uses one consistent description for official webpages.

all_services_df["source_type"] = (
    all_services_df["source_type"]
    .replace(
        {
            "Official webpage": "Official council webpage"
        }
    )
)

print("Standardized source types:")
print(
    all_services_df["source_type"].value_counts()
)

display(
    all_services_df[
        [
            "service_id",
            "service",
            "source_type"
        ]
    ].head(10)
)

Standardized source types:
source_type
Official council webpage    34
Name: count, dtype: int64


,service_id,service,source_type
0,1,Environmental Health,Official council webpage
1,2,Disease Prevention and Control,Official council webpage
2,3,Waste Management,Official council webpage
3,4,Food Safety and Workplace Inspections,Official council webpage
4,5,Public Health Awareness and Education,Official council webpage
5,6,Pest and Vector Management,Official council webpage
6,7,Funeral and Burial Services,Official council webpage
7,8,Clinical and Nursing Services,Official council webpage
8,9,Pharmaceutical and Diagnostic Services,Official council webpage
9,10,Vaccination and Child Health Programmes,Official council webpage


In [ ]:
# Export the cleaned Council Services and Facilities dataset
# using the required pipe delimiter.

services_output_file = (
    "data/processed/services/db-unza26-csc4792-council-services-and-facilities.csv"
)

all_services_df.to_csv(
    services_output_file,
    sep="|",
    index=False,
    encoding="utf-8"
)

print("Dataset exported successfully.")
print("File:", services_output_file)
print("Records:", len(all_services_df))
print("Columns:", len(all_services_df.columns))

Dataset exported successfully.
File: db-unza26-csc4792-council-services-and-facilities.csv
Records: 34
Columns: 7


In [ ]:
# Reload the exported CSV to verify that it was saved
# correctly and can be read back into pandas.

verification_df = pd.read_csv(
    services_output_file,
    sep="|",
    encoding="utf-8"
)

print("CSV verification")
print("=" * 40)

print("Records loaded:", len(verification_df))
print("Columns loaded:", len(verification_df.columns))

print("\nColumns:")
print(list(verification_df.columns))

print("\nMissing values:")
print(verification_df.isnull().sum())

print("\nFirst 5 records:")
display(verification_df.head())

CSV verification
Records loaded: 34
Columns loaded: 7

Columns:
['service_id', 'service', 'description', 'department', 'category', 'source_type', 'source_url']

Missing values:
service_id     0
service        0
description    0
department     0
category       0
source_type    0
source_url     0
dtype: int64

First 5 records:


,service_id,service,description,department,category,source_type,source_url
0,1,Environmental Health,Safeguards community health through environmen...,Public Health Department,Council Service,Official council webpage,https://www.kabwecouncil.gov.zm/?page_id=2640
1,2,Disease Prevention and Control,"Provides disease prevention, control, health s...",Public Health Department,Council Service,Official council webpage,https://www.kabwecouncil.gov.zm/?page_id=2640
2,3,Waste Management,Provides solid waste management and waste-mana...,Public Health Department,Council Service,Official council webpage,https://www.kabwecouncil.gov.zm/?page_id=2640
3,4,Food Safety and Workplace Inspections,"Inspects restaurants, markets, factories and w...",Public Health Department,Council Service,Official council webpage,https://www.kabwecouncil.gov.zm/?page_id=2640
4,5,Public Health Awareness and Education,"Provides workshops, school programmes, media c...",Public Health Department,Council Service,Official council webpage,https://www.kabwecouncil.gov.zm/?page_id=2640


In [ ]:
# Create a source register for the Council Services and Facilities dataset.

step3_sources = [
    [
        "Council homepage",
        "Official council website and navigation used to identify services and departments",
        council_base_url
    ],
    [
        "Licenses and levies",
        "Official Council Services webpage",
        services_urls["licenses_and_levies"]
    ],
    [
        "Waste Management",
        "Official Council Services webpage",
        services_urls["waste_management"]
    ],
    [
        "E-GP",
        "Official Council Services webpage",
        services_urls["e_gp"]
    ]
]

# Add the department webpages used in the curated service dataset.
for department, url in department_url_map.items():
    step3_sources.append(
        [
            department,
            "Official departmental webpage",
            url
        ]
    )

step3_sources_df = pd.DataFrame(
    step3_sources,
    columns=[
        "source_name",
        "description",
        "source_url"
    ]
)

# Remove any accidental duplicate URLs.
step3_sources_df = (
    step3_sources_df
    .drop_duplicates(subset=["source_url"])
    .reset_index(drop=True)
)

print("Number of Step 3 sources:", len(step3_sources_df))

display(step3_sources_df)

Number of Step 3 sources: 11


,source_name,description,source_url
0,Council homepage,Official council website and navigation used t...,https://www.kabwecouncil.gov.zm
1,Licenses and levies,Official Council Services webpage,https://www.kabwecouncil.gov.zm/?page_id=2142
2,Waste Management,Official Council Services webpage,https://www.kabwecouncil.gov.zm/?page_id=2150
3,E-GP,Official Council Services webpage,https://www.kabwecouncil.gov.zm/?page_id=2155
4,Public Health Department,Official departmental webpage,https://www.kabwecouncil.gov.zm/?page_id=2640
5,Health Services Department,Official departmental webpage,https://www.kabwecouncil.gov.zm/?page_id=2649
6,Legal Services Department,Official departmental webpage,https://www.kabwecouncil.gov.zm/?page_id=3894
7,Planning Department,Official departmental webpage,https://www.kabwecouncil.gov.zm/?page_id=3897
8,Housing and Social Services Department,Official departmental webpage,https://www.kabwecouncil.gov.zm/?page_id=3900
9,"Fisheries, Livestock and Veterinary Services D...",Official departmental webpage,https://www.kabwecouncil.gov.zm/?page_id=3903


In [ ]:
# Validate the Step 3 source register.

print("STEP 3 SOURCE REGISTER VALIDATION")
print("=" * 50)

print("Number of sources:", len(step3_sources_df))

print("\nMissing values:")
print(step3_sources_df.isnull().sum())

print("\nDuplicate source URLs:")
print(
    step3_sources_df["source_url"].duplicated().sum()
)

print("\nSource types:")
print(
    step3_sources_df["description"].value_counts()
)

STEP 3 SOURCE REGISTER VALIDATION
Number of sources: 11

Missing values:
source_name    0
description    0
source_url     0
dtype: int64

Duplicate source URLs:
0

Source types:
description
Official departmental webpage                                                        7
Official Council Services webpage                                                    3
Official council website and navigation used to identify services and departments    1
Name: count, dtype: int64


In [ ]:
# Export the Step 3 source register.

sources_output_file = (
    "db-unza26-csc4792-csc4792-council-services-sources.csv"
)

step3_sources_df.to_csv(
    sources_output_file,
    sep="|",
    index=False,
    encoding="utf-8"
)

print("Source register exported successfully.")
print("File:", sources_output_file)
print("Sources:", len(step3_sources_df))

Source register exported successfully.
File: db-unza26-csc4792-csc4792-council-services-sources.csv
Sources: 11


In [ ]:
sources_output_file = (
    "db-unza26-csc4792-council-services-sources.csv"
)

step3_sources_df.to_csv(
    sources_output_file,
    sep="|",
    index=False,
    encoding="utf-8"
)

print("Source register exported successfully.")
print("File:", sources_output_file)
print("Sources:", len(step3_sources_df))

Source register exported successfully.
File: db-unza26-csc4792-council-services-sources.csv
Sources: 11


## Step 3 — Council Services and Facilities: Summary

The Council Services and Facilities dataset was created from official
Kabwe Municipal Council webpages.

The extraction identified:

- 34 service records;
- 33 unique service names;
- 8 departments/categories represented;
- 11 official source webpages;
- 0 missing values;
- 0 duplicate records; and
- 0 duplicate source URLs.

The dataset was exported using the pipe (`|`) delimiter.

### Output Files

1. `db-unza26-csc4792-council-services-and-facilities.csv`
2. `db-unza26-csc4792-council-services-sources.csv`

The dataset retains source URLs for provenance and includes services
identified from both the Council's Services section and departmental
webpages.

Where departmental webpages contained apparent references to other local
authorities, those locality-specific statements were excluded from the
curated service records rather than being treated as Kabwe-specific facts.